In [ ]:
import os
import warnings
import random
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
import optuna
import joblib
from sklearn.inspection import permutation_importance
from catboost import CatBoostRegressor
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import json
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor


In [ ]:
ALLOW_FUTURE_VALIDATION = True  

warnings.filterwarnings("ignore")

# PreProcess

### DATA LOADING

In [ ]:
Data = pd.read_excel("/Users/dhanujiamanda/Documents/Projects/Agentic AI /Pipeline/Agentic-AI-for-Pharma-Stockout-Problem/data/fact_monthly_closed.xlsx")
Data.to_csv("/Users/dhanujiamanda/Documents/Projects/Agentic AI /Pipeline/Agentic-AI-for-Pharma-Stockout-Problem/data/fact_monthly_closed.csv", index=False)

In [ ]:
def force_itemcode_str(df):
    df = df.copy()
    if "ItemCode" in df.columns:
        df["ItemCode"] = df["ItemCode"].astype(str)
    if "ItemCode_Original" in df.columns:
        df["ItemCode_Original"] = df["ItemCode_Original"].astype(str)
    return df

Data = force_itemcode_str(Data)

In [ ]:
# To delete incomplete month
Data = Data[~((Data["Year"] == 2026) & (Data["Month_Number"] == 2))].copy()

Data = Data.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)


In [ ]:
# Load focus item codes
focus_file = "/Users/dhanujiamanda/Documents/Projects/Agentic AI /Pipeline/Agentic-AI-for-Pharma-Stockout-Problem/data/FocusItemCodes.xlsx"   

focus_df = pd.read_excel(focus_file)

# clean and convert code column
focus_df["Code"] = pd.to_numeric(focus_df["Code"], errors="coerce")
focus_df = focus_df.dropna(subset=["Code"])
focus_df["Code"] = focus_df["Code"].astype(int).astype(str)

PHARMA_SKUS = sorted(focus_df["Code"].unique().tolist())

print("Focus SKU count from file:", len(PHARMA_SKUS))
print("First 10 SKUs:", PHARMA_SKUS[:10])


# Filter main dataset
Data["ItemCode"] = pd.to_numeric(Data["ItemCode"], errors="coerce")
Data = Data.dropna(subset=["ItemCode"]).copy()
Data["ItemCode"] = Data["ItemCode"].astype(int).astype(str)

Data = Data[Data["ItemCode"].isin(PHARMA_SKUS)].copy()

Data = force_itemcode_str(Data)

print("Filtered pharma rows:", len(Data))
print("Filtered pharma SKUs:", Data["ItemCode"].nunique())


# Missing SKU check
input_skus = set(PHARMA_SKUS)
found_skus = set(Data["ItemCode"].dropna().astype(str).unique())
missing_skus = sorted(list(input_skus - found_skus))

print("Requested pharma SKU count:", len(input_skus))
print("Found in dataset:", len(found_skus))
print("Missing from dataset:", len(missing_skus))

if missing_skus:
    print("First 50 missing SKUs:", missing_skus[:50])

In [ ]:
# Clean raw negatives
for c in [
    "Secondary_Sales_Qty",
    "Primary_Sales_Qty",
    "Free_Qty",
    "Available_Primary_Inventory_Qty",
    "Distributor_Inventory_Qty",
    "Blocked_Stock_Qty",
    "Inspection_Stock_Qty",
    "Total_Primary_Inventory_Qty"
]:
    if c in Data.columns:
        Data[c] = Data[c].clip(lower=0)

# Base observed movement
Data["Observed_Demand"] = Data["Secondary_Sales_Qty"].clip(lower=0)

In [ ]:
print(Data.info())
print(Data.head(5))
print(Data.count())

### DATA QUALITY CHECKS

In [ ]:
# Check Duplicates
dup_count = Data.duplicated().sum()
print(dup_count)

# Check Nulls
null_count = Data.isnull().sum()
print("Nulls:\n", null_count)

### DEMAND SIGNAL CONSTRUCTION

In [ ]:
# =========================
# PAST-ONLY HELPER FEATURES
# =========================
grp = Data.groupby("ItemCode")

# Lag demand
Data["Lag1_Obs"] = grp["Observed_Demand"].shift(1)
Data["Lag2_Obs"] = grp["Observed_Demand"].shift(2)
Data["Lag3_Obs"] = grp["Observed_Demand"].shift(3)
Data["Lag6_Obs"] = grp["Observed_Demand"].shift(6)
Data["Lag12_Obs"] = grp["Observed_Demand"].shift(12)

# Rolling stats from observed demand
Data["Rolling3M_Obs_Mean"] = grp["Observed_Demand"].transform(lambda x: x.rolling(3, min_periods=1).mean().shift(1))
Data["Rolling6M_Obs_Mean"] = grp["Observed_Demand"].transform(lambda x: x.rolling(6, min_periods=1).mean().shift(1))
Data["Rolling3M_Obs_Std"] = grp["Observed_Demand"].transform(lambda x: x.rolling(3, min_periods=1).std().shift(1)).fillna(0)

# Safe baseline
Data["Baseline_Demand"] = Data["Rolling3M_Obs_Mean"].fillna(Data["Lag1_Obs"]).fillna(0)

# Uplift ratio
# safer denominator
safe_baseline = np.maximum(Data["Baseline_Demand"], 1)

Data["Uplift_vs_Baseline"] = Data["Observed_Demand"] / safe_baseline

# clip extreme values (VERY IMPORTANT)
Data["Uplift_vs_Baseline"] = Data["Uplift_vs_Baseline"].clip(0, 5)

# Z-score past-only
Data["Z_Score_Obs"] = (
    (Data["Observed_Demand"] - Data["Rolling3M_Obs_Mean"]) /
    (Data["Rolling3M_Obs_Std"] + 1)
).fillna(0)



In [ ]:
# =========================
# RECURRING BONUS SKU DETECTION
# =========================
def detect_recurring_bonus_skus(df,
                                min_bonus_months=3,
                                gap_tolerance=1,
                                uplift_threshold=1.4):
    """
    Detect SKUs where bonus months happen in a stable repeated interval
    and those months consistently create strong uplift.
    """
    df = force_itemcode_str(df)
    out = []

    for item, g in df.groupby("ItemCode"):
        g = g.sort_values(["Year", "Month_Number"]).copy()
        g["Time_Index"] = g["Year"].astype(int) * 12 + g["Month_Number"].astype(int)

        bonus_rows = g[g["Bonus_Flag"] == 1].copy()

        recurring_flag = 0
        cycle_len = 0
        avg_gap = np.nan
        bonus_freq_12m = 0.0
        avg_bonus_uplift = 1.0

        if len(g) > 0:
            bonus_freq_12m = bonus_rows.shape[0] / len(g)

        if len(bonus_rows) >= min_bonus_months:
            gaps = bonus_rows["Time_Index"].diff().dropna()

            if len(gaps) > 0:
                avg_gap = gaps.mean()

                rounded_gap = int(round(avg_gap))
                stable_gap = ((gaps - rounded_gap).abs() <= gap_tolerance).mean()

                avg_bonus_uplift = bonus_rows["Uplift_vs_Baseline"].replace([np.inf, -np.inf], np.nan).clip(0,5).fillna(1.0).median()

                # recurring if gaps are stable and uplift is meaningful
                if stable_gap >= 0.6 and avg_bonus_uplift >= uplift_threshold:
                    recurring_flag = 1
                    cycle_len = rounded_gap

        out.append({
            "ItemCode": item,
            "Recurring_Bonus_SKU": recurring_flag,
            "Bonus_Cycle_Length": cycle_len,
            "Avg_Bonus_Gap": avg_gap if pd.notna(avg_gap) else 0,
            "Bonus_Frequency_All": bonus_freq_12m,
            "Avg_Bonus_Uplift": avg_bonus_uplift
        })

    return pd.DataFrame(out)

In [ ]:
# =========================
# BONUS CYCLE FEATURES
# =========================
def add_bonus_cycle_features(df):
    df = force_itemcode_str(df)
    df = df.sort_values(["ItemCode", "Year", "Month_Number"]).copy()

    pieces = []

    for item_code, g in df.groupby("ItemCode", sort=False):
        g = g.sort_values(["Year", "Month_Number"]).copy()
        g["ItemCode"] = item_code
        g["Time_Index"] = g["Year"].astype(int) * 12 + g["Month_Number"].astype(int)

        bonus_time_idx = g.loc[g["Bonus_Flag"] == 1, "Time_Index"].tolist()

        months_since_last_bonus = []
        expected_bonus_this_month = []

        for i in range(len(g)):
            current_t = g["Time_Index"].iloc[i]
            past_bonus = [t for t in bonus_time_idx if t < current_t]

            if len(past_bonus) == 0:
                months_since_last_bonus.append(999)
            else:
                months_since_last_bonus.append(current_t - past_bonus[-1])

            cyc = g["Bonus_Cycle_Length"].iloc[i]
            recurring = g["Recurring_Bonus_SKU"].iloc[i]

            if recurring == 1 and cyc > 0 and len(past_bonus) > 0:
                expected_bonus_this_month.append(
                    1 if abs((current_t - past_bonus[-1]) - cyc) <= 1 else 0
                )
            else:
                expected_bonus_this_month.append(0)

        g["Months_Since_Last_Bonus"] = months_since_last_bonus
        g["Expected_Bonus_Month"] = expected_bonus_this_month

        g["Bonus_Flag_Lag1"] = g["Bonus_Flag"].shift(1).fillna(0)
        g["Bonus_Flag_Lag2"] = g["Bonus_Flag"].shift(2).fillna(0)
        g["Bonus_Flag_Lag3"] = g["Bonus_Flag"].shift(3).fillna(0)

        g["Bonus_Frequency_12M"] = (
            g["Bonus_Flag"]
            .rolling(12, min_periods=1)
            .mean()
            .shift(1)
            .fillna(0)
        )

        g = g.drop(columns=["Time_Index"], errors="ignore")
        pieces.append(g)

    out = pd.concat(pieces, axis=0, ignore_index=True)
    return out

def add_bonus_cycle_features_foldsafe(train_df, valid_df):
    """
    Build train bonus-cycle features on train only.
    Build valid bonus-cycle features sequentially using train history + prior valid rows only.
    """
    train_df = force_itemcode_str(train_df)
    valid_df = force_itemcode_str(valid_df)
    
    train_df = train_df.copy().sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)
    valid_df = valid_df.copy().sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    # train rows can be built normally
    train_out = add_bonus_cycle_features(train_df)

    valid_rows = []

    for item_code, g_valid in valid_df.groupby("ItemCode", sort=False):
        history = train_df[train_df["ItemCode"] == item_code].copy()
        g_valid = g_valid.sort_values(["Year", "Month_Number"]).copy()

        for _, row in g_valid.iterrows():
            row_df = pd.DataFrame([row])

            temp = pd.concat([history, row_df], ignore_index=True)
            temp = temp.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

            temp = add_bonus_cycle_features(temp)
            valid_rows.append(temp.iloc[-1].copy())

            history = pd.concat([history, row_df], ignore_index=True)

    valid_out = pd.DataFrame(valid_rows) if valid_rows else valid_df.head(0).copy()

    train_out = force_itemcode_str(train_out)
    valid_out = force_itemcode_str(valid_out)

    return train_out, valid_out

In [ ]:
# Build recurring bonus pattern features on full historical data - only for EDA purposed
bonus_pattern_df = detect_recurring_bonus_skus(Data)[[
    "ItemCode",
    "Recurring_Bonus_SKU",
    "Bonus_Cycle_Length",
    "Avg_Bonus_Gap",
    "Bonus_Frequency_All",
    "Avg_Bonus_Uplift"
]].copy() 
Data = force_itemcode_str(Data)
bonus_pattern_df = force_itemcode_str(bonus_pattern_df)

Data = Data.drop(columns=[
    "Recurring_Bonus_SKU",
    "Bonus_Cycle_Length",
    "Avg_Bonus_Gap",
    "Bonus_Frequency_All",
    "Avg_Bonus_Uplift",
    "Months_Since_Last_Bonus",
    "Expected_Bonus_Month",
    "Bonus_Flag_Lag1",
    "Bonus_Flag_Lag2",
    "Bonus_Flag_Lag3",
    "Bonus_Frequency_12M"
], errors="ignore")
Data = Data.merge(bonus_pattern_df, on="ItemCode", how="left")
Data = force_itemcode_str(Data)

for c in ["Recurring_Bonus_SKU", "Bonus_Cycle_Length", "Avg_Bonus_Gap", "Bonus_Frequency_All"]:
    Data[c] = Data[c].fillna(0)

Data["Avg_Bonus_Uplift"] = Data["Avg_Bonus_Uplift"].fillna(1.0)

# actually compute cycle-based historical features
Data = add_bonus_cycle_features(Data)

In [ ]:
# =========================
# CHANNEL / STOCK FLOW FEATURES
# =========================
Data["Net_Available_Stock"] = (
    Data["Total_Primary_Inventory_Qty"]
    - Data["Blocked_Stock_Qty"]
    - Data["Inspection_Stock_Qty"]
).clip(lower=0)

Data["Primary_Stock_Cover"] = np.where(
    Data["Baseline_Demand"] <= 0,
    0,
    Data["Net_Available_Stock"] / (Data["Baseline_Demand"] + 1)
)

Data["Distributor_Stock_Cover"] = np.where(
    Data["Baseline_Demand"] <= 0,
    0,
    Data["Distributor_Inventory_Qty"] / (Data["Baseline_Demand"] + 1)
)

Data["Primary_to_Distributor_Ratio"] = np.where(
    Data["Distributor_Inventory_Qty"] <= 0,
    0,
    Data["Net_Available_Stock"] / (Data["Distributor_Inventory_Qty"] + 1)
)

Data["Blocked_Stock_Ratio"] = np.where(
    Data["Total_Primary_Inventory_Qty"] <= 0,
    0,
    Data["Blocked_Stock_Qty"] / (Data["Total_Primary_Inventory_Qty"] + 1)
)

Data["Inspection_Stock_Ratio"] = np.where(
    Data["Total_Primary_Inventory_Qty"] <= 0,
    0,
    Data["Inspection_Stock_Qty"] / (Data["Total_Primary_Inventory_Qty"] + 1)
)

Data["Primary_Inv_Change"] = Data.groupby("ItemCode")["Net_Available_Stock"].diff().fillna(0)
Data["Distributor_Inv_Change"] = Data.groupby("ItemCode")["Distributor_Inventory_Qty"].diff().fillna(0)

Data["Primary_to_Distributor_Ratio"] = Data["Primary_to_Distributor_Ratio"].clip(upper=Data["Primary_to_Distributor_Ratio"].quantile(0.99))

Data["Distributor_Inv_Change"] = Data["Distributor_Inv_Change"].clip(upper=Data["Distributor_Inv_Change"].quantile(0.99))
Data["Primary_Inv_Change"] = Data["Primary_Inv_Change"].clip(upper=Data["Primary_Inv_Change"].quantile(0.99))


### BUSINESS RULE ADJUSTMENTS

In [ ]:
Data["Effective_Demand"] = Data["Observed_Demand"].copy()

In [ ]:
# ─── RULE: Supply-constraint correction ────────────────────────────────────────────────── 

Data["Supply_Baseline"] = Data["Rolling3M_Obs_Mean"].fillna(Data["Lag1_Obs"]).fillna(Data["Observed_Demand"])

supply_constrained = (Data["Supply_Constraint_Flag"] == 1)

Data["Effective_Demand"] = np.where(
    supply_constrained,
    np.maximum(Data["Observed_Demand"], 0.85 * Data["Supply_Baseline"]),
    Data["Effective_Demand"]
)

'''
If supply was constrained (stock not available), observed sales may be artificially low.
So we cap demand to a safer value: last 3-month average secondary sales (shifted to avoid leakage).

If Supply_Constraint_Flag == 1:
   Effective_Demand = min(current sales, rolling average)
Else:
   keep current demand

if constrained, observed sales may be lower than true pull.
Instead of min(current, rolling), use max(current, a safe baseline fraction)
'''

In [ ]:
# ─── RULE: Irregular bonus spike detection via Z-score ────────────────────────────────────────────────── 

irregular_bonus_spike = (
    (Data["Bonus_Flag"] == 1) &
    (Data["Recurring_Bonus_SKU"] == 0) &
    (Data["Z_Score_Obs"] > 2.0) &
    (Data["Uplift_vs_Baseline"] > 1.6)
)
'''
High Z-score means current demand is unusually higher than its recent baseline.
+1 in denominator prevents division exploding for stable/low-variance SKUs.
'''

In [ ]:
# ─── RULE: Stockout-like demand suppression (past-only) ────────────────────────────────────────────────── 

grp = Data.groupby("ItemCode")

prev_obs = grp["Observed_Demand"].shift(1)
prev_primary_cover = grp["Primary_Stock_Cover"].shift(1)
prev_dist_cover = grp["Distributor_Stock_Cover"].shift(1)

stockout_drop_condition = (
    (Data["Observed_Demand"] < 0.65 * prev_obs.fillna(Data["Observed_Demand"])) &
    (Data["Supply_Constraint_Flag"] == 1) &
    (
        (prev_primary_cover.fillna(99) < 1.0) |
        (prev_dist_cover.fillna(99) < 1.0)
    )
)

In [ ]:
# Start clean demand
Data["Clean_Demand"] = Data["Effective_Demand"].copy()

# For irregular bonus spikes: smooth partially
Data.loc[irregular_bonus_spike, "Clean_Demand"] = (
    0.60 * Data.loc[irregular_bonus_spike, "Observed_Demand"] +
    0.40 * Data.loc[irregular_bonus_spike, "Baseline_Demand"]
)

# For stockout-like drop: normalize upward toward baseline
Data.loc[stockout_drop_condition, "Clean_Demand"] = np.maximum(
    Data.loc[stockout_drop_condition, "Observed_Demand"],
    0.90 * Data.loc[stockout_drop_condition, "Baseline_Demand"]
)

Data["Clean_Demand"] = Data["Clean_Demand"].clip(lower=0)

# Flags for model
Data["Bonus_Shock"] = irregular_bonus_spike.astype(int)
# Data["Recurring_Bonus_Month"] = 0
Data["Supply_Shock"] = stockout_drop_condition.astype(int)

# PROMO INTENSITY FEATURES
Data["Free_Ratio"] = np.where(
    Data["Primary_Sales_Qty"] <= 0,
    0,
    Data["Free_Qty"] / (Data["Primary_Sales_Qty"] + 1)
)

grp = Data.groupby("ItemCode")

In [ ]:
# DROP these BEFORE MODELING
Data = Data.drop(columns=[
    "Months_Since_Last_Bonus",
    "Expected_Bonus_Month",
    "Bonus_Flag_Lag1",
    "Bonus_Flag_Lag2",
    "Bonus_Flag_Lag3",
    "Bonus_Frequency_12M"
], errors="ignore")

# Model 

## MODEL FEATURE ENGINEERING

#### Core Config

In [ ]:
ACTUAL_TARGET_COL = "Target"
MODEL_TARGET_COL = "Residual_Target"
BASELINE_COL = "Residual_Baseline"

#### Helper Functions

##### Other

In [ ]:




def apply_sku_history_profile(train_df, valid_df):
    train_df = train_df.copy()
    valid_df = valid_df.copy()

    train_df = force_itemcode_str(train_df)
    valid_df = force_itemcode_str(valid_df)

    profile = (
        train_df.groupby("ItemCode")
        .agg(
            SKU_Mean_Demand=("Clean_Demand", "mean"),
            SKU_Std_Demand=("Clean_Demand", "std"),
            SKU_ZeroRate=("Clean_Demand", lambda x: (x == 0).mean())
        )
        .reset_index()
    )
    profile["SKU_Std_Demand"] = profile["SKU_Std_Demand"].fillna(0)
    profile["SKU_CV"] = np.where(
        profile["SKU_Mean_Demand"] <= 0,
        0,
        profile["SKU_Std_Demand"] / (profile["SKU_Mean_Demand"] + 1)
    )
    profile = force_itemcode_str(profile)

    keep_cols = ["ItemCode", "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]

    train_df = train_df.drop(columns=["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"], errors="ignore")
    valid_df = valid_df.drop(columns=["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"], errors="ignore")

    train_df = train_df.merge(profile[keep_cols], on="ItemCode", how="left")
    valid_df = valid_df.merge(profile[keep_cols], on="ItemCode", how="left")

    train_df = force_itemcode_str(train_df)
    valid_df = force_itemcode_str(valid_df)

    for c in ["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]:
        train_df[c] = train_df[c].fillna(0)
        valid_df[c] = valid_df[c].fillna(0)

    return train_df, valid_df, profile[keep_cols]


##### Main Feature building

In [ ]:
def rebuild_time_features(df):
    df = force_itemcode_str(df)
    df = df.sort_values(["ItemCode", "Year", "Month_Number"]).copy()
    grp = df.groupby("ItemCode")

    # demand history
    for lag in [1, 2, 3, 6, 12]:
        df[f"Lag{lag}"] = grp["Clean_Demand"].shift(lag)

    df["Rolling3M_Mean"] = grp["Clean_Demand"].transform(lambda x: x.rolling(3, min_periods=1).mean().shift(1))
    df["Rolling6M_Mean"] = grp["Clean_Demand"].transform(lambda x: x.rolling(6, min_periods=1).mean().shift(1))
    df["Rolling3M_Std"] = grp["Clean_Demand"].transform(lambda x: x.rolling(3, min_periods=1).std().shift(1)).fillna(0)

    # calendar
    df["Month_Sin"] = np.sin(2 * np.pi * df["Month_Number"] / 12)
    df["Month_Cos"] = np.cos(2 * np.pi * df["Month_Number"] / 12)
    
    quarter = ((df["Month_Number"] - 1) // 3) + 1
    df["Quarter_Sin"] = np.sin(2 * np.pi * quarter / 4)
    df["Quarter_Cos"] = np.cos(2 * np.pi * quarter / 4)

    # demand dynamics
    df["Momentum"] = df["Lag1"] - df["Lag3"]
    
    # DEMAND TRANSITION / REGIME CHANGE SIGNALS
    df["Demand_Trend_3M"] = df["Lag1"] - df["Rolling3M_Mean"]
    df["Demand_Acceleration"] = ((df["Lag1"] - df["Lag2"]) - (df["Lag2"] - df["Lag3"]))
    df["CV_3M"] = df["Rolling3M_Std"] / (df["Rolling3M_Mean"] + 1)
    df["Lag1_vs_Rolling3"] = df["Lag1"] / (df["Rolling3M_Mean"] + 1)
    df["Lag2_vs_Rolling3"] = df["Lag2"] / (df["Rolling3M_Mean"] + 1)

    df["Recent_Drop_Flag"] = (
        (df["Lag1_vs_Rolling3"] < 0.45) & (df["Rolling3M_Mean"] > 20)
    ).astype(int)

    df["Demand_Rebound_Risk"] = (
        (df["Recent_Drop_Flag"] == 1) &
        (df["Rolling6M_Mean"] > df["Rolling3M_Mean"] * 1.4)
    ).astype(int)
    
    df["Is_Zero"] = (df["Clean_Demand"] == 0).astype(int)
    df["ZeroRate_6M"] = grp["Is_Zero"].transform(lambda x: x.rolling(6, min_periods=1).mean().shift(1)).fillna(0)

    # stock features
    df["Net_Available_Stock"] = (
        df["Total_Primary_Inventory_Qty"]
        - df["Blocked_Stock_Qty"]
        - df["Inspection_Stock_Qty"]
    ).clip(lower=0)

    df["Inventory_Pressure"] = np.where(
        df["Lag1"].fillna(0) <= 0,
        0,
        df["Available_Primary_Inventory_Qty"] / (df["Lag1"] + 1)
    )

    df["Stock_Cover_Months"] = np.where(
        df["Rolling3M_Mean"].fillna(0) <= 0,
        0,
        df["Net_Available_Stock"] / (df["Rolling3M_Mean"] + 1)
    )

    df["Primary_Stock_Cover"] = np.where(
        df["Rolling3M_Mean"].fillna(0) <= 0,
        0,
        df["Net_Available_Stock"] / (df["Rolling3M_Mean"] + 1)
    )

    df["Distributor_Stock_Cover"] = np.where(
        df["Rolling3M_Mean"].fillna(0) <= 0,
        0,
        df["Distributor_Inventory_Qty"] / (df["Rolling3M_Mean"] + 1)
    )

    df["Demand_to_Stock_Ratio"] = np.where(
        df["Net_Available_Stock"] <= 0,
        0,
        df["Rolling3M_Mean"] / (df["Net_Available_Stock"] + 1)
    )

    df["Primary_to_Distributor_Ratio"] = np.where(
        df["Distributor_Inventory_Qty"] <= 0,
        0,
        df["Net_Available_Stock"] / (df["Distributor_Inventory_Qty"] + 1)
    )

    df["Blocked_Stock_Ratio"] = np.where(
        df["Total_Primary_Inventory_Qty"] <= 0,
        0,
        df["Blocked_Stock_Qty"] / (df["Total_Primary_Inventory_Qty"] + 1)
    )

    df["Inspection_Stock_Ratio"] = np.where(
        df["Total_Primary_Inventory_Qty"] <= 0,
        0,
        df["Inspection_Stock_Qty"] / (df["Total_Primary_Inventory_Qty"] + 1)
    )

    df["Primary_Inv_Change"] = grp["Net_Available_Stock"].diff().fillna(0)
    df["Distributor_Inv_Change"] = grp["Distributor_Inventory_Qty"].diff().fillna(0)

    # INVENTORY TRANSITION SIGNALS
    df["Primary_Stock_Cover_Change"] = (
        df["Primary_Stock_Cover"] - grp["Primary_Stock_Cover"].shift(1).fillna(0))

    df["Distributor_Stock_Cover_Change"] = (
        df["Distributor_Stock_Cover"] - grp["Distributor_Stock_Cover"].shift(1).fillna(0))

    df["Primary_Stockout_Risk"] = (
        (df["Primary_Stock_Cover"] < 0.5) & (df["Distributor_Stock_Cover"] < 1.0)).astype(int)

    df["Distributor_Buffer_Available"] = (df["Distributor_Stock_Cover"] >= 1.0).astype(int)

    df["Stock_Recovery_Flag"] = (
        (df["Primary_Stock_Cover_Change"] > 0.8) | (df["Primary_Inv_Change"] > df["Rolling3M_Mean"])
    ).astype(int)

    df["Suppressed_Demand_Flag"] = (
        (df["Supply_Constraint_Flag"] == 1) &
        (df["Clean_Demand"] < 0.75 * df["Rolling3M_Mean"]) &
        (df["Distributor_Stock_Cover"] < 1.0)
    ).astype(int)

    # supply history
    df["Supply_Constraint_Lag1"] = grp["Supply_Constraint_Flag"].shift(1).fillna(0)
    df["Supply_Constraint_Lag2"] = grp["Supply_Constraint_Flag"].shift(2).fillna(0)

    df["Primary_Stock_Cover_Lag1"] = grp["Primary_Stock_Cover"].shift(1).fillna(0)
    df["Distributor_Stock_Cover_Lag1"] = grp["Distributor_Stock_Cover"].shift(1).fillna(0)

    # promo raw history
    df["Free_Qty_Lag1"] = grp["Free_Qty"].shift(1).fillna(0)
    df["Free_Ratio_Lag1"] = grp["Free_Ratio"].shift(1).fillna(0)
    df["Free_Qty_Rolling3"] = grp["Free_Qty"].transform(lambda x: x.rolling(3, min_periods=1).mean().shift(1)).fillna(0)

    df["Promo_Intensity_History"] = np.where(
        df["Rolling3M_Mean"].fillna(0) <= 0,
        0,
        df["Free_Qty_Rolling3"] / (df["Rolling3M_Mean"] + 1)
    )

    # promo timing history
    df["Bonus_Flag_Lag1"] = grp["Bonus_Flag"].shift(1).fillna(0)
    df["Bonus_Flag_Lag2"] = grp["Bonus_Flag"].shift(2).fillna(0)
    df["Bonus_Flag_Lag3"] = grp["Bonus_Flag"].shift(3).fillna(0)

    df["Bonus_Frequency_12M"] = grp["Bonus_Flag"].transform(lambda x: x.rolling(12, min_periods=1).mean().shift(1)).fillna(0)

    # months since last bonus
    months_since = []

    for _, g in df.groupby("ItemCode", sort=False):
        g = g.sort_values(["Year", "Month_Number"]).copy()
        g["Time_Index"] = g["Year"].astype(int) * 12 + g["Month_Number"].astype(int)

        bonus_time_idx = g.loc[g["Bonus_Flag"] == 1, "Time_Index"].tolist()

        out = []
        for current_t in g["Time_Index"]:
            past_bonus = [t for t in bonus_time_idx if t < current_t]
            if len(past_bonus) == 0:
                out.append(999)
            else:
                out.append(current_t - past_bonus[-1])

        months_since.extend(out)

    df["Months_Since_Last_Bonus"] = months_since

    df["Expected_Bonus_Month"] = np.where(
        (df["Recurring_Bonus_SKU"] == 1) &
        (df["Bonus_Cycle_Length"] > 0) &
        (np.abs(df["Months_Since_Last_Bonus"] - df["Bonus_Cycle_Length"]) <= 1),
        1, 0
    )

    df["Expected_Bonus_NextMonth"] = np.where(
        (df["Recurring_Bonus_SKU"] == 1) &
        (df["Bonus_Cycle_Length"] > 0) &
        (np.abs((df["Months_Since_Last_Bonus"] + 1) - df["Bonus_Cycle_Length"]) <= 1),
        1, 0
    )

    # PROMO CYCLE / SPECIAL BONUS SIGNALS

    df["Bonus_Cycle_Position"] = np.where(
        df["Bonus_Cycle_Length"] <= 0,
        0,
        df["Months_Since_Last_Bonus"] / (df["Bonus_Cycle_Length"] + 1)
    )

    df["Near_Bonus_Cycle"] = (
        (df["Recurring_Bonus_SKU"] == 1) &
        (df["Bonus_Cycle_Length"] > 0) &
        (np.abs((df["Months_Since_Last_Bonus"] + 1) - df["Bonus_Cycle_Length"]) <= 1)
    ).astype(int)

    grp = df.groupby("ItemCode")
    df["Post_Bonus_Month_Flag"] = grp["Bonus_Flag"].shift(1).fillna(0)

    df["Recurring_Bonus_Month"] = np.where(
        (df["Recurring_Bonus_SKU"] == 1) &
        (
            (df["Bonus_Flag"] == 1) |
            (
                (df["Bonus_Cycle_Length"] > 0) &
                (np.abs(df["Months_Since_Last_Bonus"] - df["Bonus_Cycle_Length"]) <= 1)
            )
        ),
        1, 0
    )

    # promo uplift history
    safe_mean = np.maximum(df["Rolling3M_Mean"].fillna(0), 1.0)
    df["Realized_Uplift"] = (df["Clean_Demand"] / safe_mean).clip(0, 6)

    df["Bonus_Demand_Only"] = np.where(
        df["Bonus_Flag"] == 1,
        df["Clean_Demand"],
        np.nan
    )

    grp2 = df.groupby("ItemCode")
    df["Promo_Uplift_Lag1"] = grp2["Realized_Uplift"].shift(1).fillna(1.0)
    df["Promo_Uplift_Lag2"] = grp2["Realized_Uplift"].shift(2).fillna(1.0)
    df["Promo_Uplift_6M"] = grp2["Realized_Uplift"].transform(
        lambda x: x.rolling(6, min_periods=1).mean().shift(1)
    ).fillna(1.0)
    df["Last_Bonus_Demand"] = grp2["Bonus_Demand_Only"].transform(
        lambda x: x.shift(1).ffill()
    ).fillna(0)

    df["Promo_Uplift_Lag1"] = df["Promo_Uplift_Lag1"].clip(0.5, 3.0)
    df["Promo_Uplift_Lag2"] = df["Promo_Uplift_Lag2"].clip(0.5, 3.0)
    df["Promo_Uplift_6M"] = df["Promo_Uplift_6M"].clip(0.5, 2.5)

    df["Promo_Strong_Uplift_Flag"] = (
        (df["Avg_Bonus_Uplift"] >= 1.35) |
        (df["Promo_Uplift_6M"] >= 1.35)
    ).astype(int)

    df["Promo_Frequent_Weak_Flag"] = (
        (df["Bonus_Frequency_12M"] >= 0.5) &
        (df["Avg_Bonus_Uplift"] < 1.25) &
        (df["Promo_Uplift_6M"] < 1.25)
    ).astype(int)

    df["Recurring_Special_Bonus_Risk"] = (
        (df["Near_Bonus_Cycle"] == 1) &
        (df["Promo_Strong_Uplift_Flag"] == 1)
    ).astype(int)

    # use these for model training
    df["Expected_Bonus_Sin"] = df["Expected_Bonus_NextMonth"] * np.sin(2 * np.pi * df["Month_Number"] / 12)
    df["Expected_Bonus_Cos"] = df["Expected_Bonus_NextMonth"] * np.cos(2 * np.pi * df["Month_Number"] / 12)

    # ============================================================
    # BEHAVIOR-LEVEL FEATURES
    # past-only: detects spiky / cyclic / volatile / promo behavior
    # ============================================================

    grp3 = df.groupby("ItemCode")

    df["Behavior_CV_6M"] = grp3["Clean_Demand"].transform(
        lambda x: x.rolling(6, min_periods=3).std().shift(1) /
                  (x.rolling(6, min_periods=3).mean().shift(1) + 1)
    ).fillna(0)

    df["Behavior_Peak_Ratio_6M"] = grp3["Clean_Demand"].transform(
        lambda x: x.rolling(6, min_periods=3).max().shift(1) /
                  (x.rolling(6, min_periods=3).mean().shift(1) + 1)
    ).fillna(0)

    def safe_autocorr_lag3(s):
        if len(s) < 8 or s.std() == 0:
            return 0
        return s.autocorr(lag=3)

    def safe_autocorr_lag6(s):
        if len(s) < 10 or s.std() == 0:
            return 0
        return s.autocorr(lag=6)

    df["Behavior_Autocorr_Lag3_12M"] = grp3["Clean_Demand"].transform(
        lambda x: x.shift(1).rolling(12, min_periods=8).apply(safe_autocorr_lag3, raw=False)
    ).fillna(0)

    df["Behavior_Autocorr_Lag6_18M"] = grp3["Clean_Demand"].transform(
        lambda x: x.shift(1).rolling(18, min_periods=10).apply(safe_autocorr_lag6, raw=False)
    ).fillna(0)

    df["Behavior_ZeroRate_12M"] = grp3["Clean_Demand"].transform(
        lambda x: (x.shift(1) == 0).rolling(12, min_periods=3).mean()
    ).fillna(0)

    df["Behavior_PromoRate_12M"] = grp3["Bonus_Flag"].transform(
        lambda x: x.shift(1).rolling(12, min_periods=3).mean()
    ).fillna(0)

    df["Lag1_to_Rolling3_Ratio"] = df["Lag1"] / (df["Rolling3M_Mean"] + 1)
    df["Lag1_to_Rolling6_Ratio"] = df["Lag1"] / (df["Rolling6M_Mean"] + 1)

    df["Recent_Spike_Flag"] = (
        (df["Lag1_to_Rolling3_Ratio"] > 1.8) |
        (df["Lag2"] > 1.8 * (df["Rolling3M_Mean"] + 1))
    ).astype(int)

    df["Post_Spike_Drop_Risk"] = (
        (df["Lag1_to_Rolling3_Ratio"] > 1.8) &
        (df["Behavior_Peak_Ratio_6M"] > 2.0)
    ).astype(int)

    df["Cycle_Pos_3"] = grp3.cumcount() % 3
    df["Cycle_Pos_6"] = grp3.cumcount() % 6

    # keep only row-safe features here
    df = df.drop(columns=["Bonus_Demand_Only"], errors="ignore")

    return df


##### Classification

In [ ]:
def classify_sku_behavior(row):
    zero_rate = float(row.get("Behavior_ZeroRate_12M", 0) or 0)
    cv = float(row.get("Behavior_CV_6M", 0) or 0)

    if zero_rate >= 0.40:
        return "INTERMITTENT"

    if int(row.get("Suppressed_Demand_Flag", 0) or 0) == 1:
        return "SUPPRESSED_DEMAND"

    if int(row.get("Recurring_Special_Bonus_Risk", 0) or 0) == 1:
        return "PROMO_RECURRING_SPECIAL_SPIKE"

    if int(row.get("Promo_Frequent_Weak_Flag", 0) or 0) == 1:
        return "PROMO_FREQUENT_WEAK"

    if int(row.get("Demand_Rebound_Risk", 0) or 0) == 1:
        return "RECENT_DROP_REBOUND_RISK"

    if int(row.get("Recent_Spike_Flag", 0) or 0) == 1:
        return "RECENT_SPIKE"

    if cv >= 1.0:
        return "VOLATILE"

    return "STABLE"


##### Bonus 

Full-data bonus features below are used only for base demand cleaning / signal engineering.
Fold-safe bonus features for model training are recomputed later inside train/valid functions.

In [ ]:
# Bonus Detection
def apply_recurring_bonus_features_foldsafe(train_df, valid_df):
    train_df = force_itemcode_str(train_df)
    valid_df = force_itemcode_str(valid_df)

    train_df = train_df.copy()
    valid_df = valid_df.copy()

    keep_cols = [
        "ItemCode",
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ]

    bonus_pattern_df = detect_recurring_bonus_skus(train_df)[keep_cols].copy()
    bonus_pattern_df = force_itemcode_str(bonus_pattern_df)

    train_df = train_df.drop(columns=keep_cols[1:], errors="ignore")
    valid_df = valid_df.drop(columns=keep_cols[1:], errors="ignore")

    train_df = train_df.merge(bonus_pattern_df, on="ItemCode", how="left")
    valid_df = valid_df.merge(bonus_pattern_df, on="ItemCode", how="left")
    train_df = force_itemcode_str(train_df)
    valid_df = force_itemcode_str(valid_df)

    for c in [
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All"
    ]:
        train_df[c] = train_df[c].fillna(0)
        valid_df[c] = valid_df[c].fillna(0)

    train_df["Avg_Bonus_Uplift"] = train_df["Avg_Bonus_Uplift"].fillna(1.0)
    valid_df["Avg_Bonus_Uplift"] = valid_df["Avg_Bonus_Uplift"].fillna(1.0)

    # KEY FIX: no combined train+valid bonus-cycle rebuild
    train_df, valid_df = add_bonus_cycle_features_foldsafe(train_df, valid_df)

    if "ItemCode" not in train_df.columns:
        raise KeyError(f"ItemCode missing in train_df after apply_recurring_bonus_features_foldsafe. Columns: {train_df.columns.tolist()}")

    if "ItemCode" not in valid_df.columns:
        raise KeyError(f"ItemCode missing in valid_df after apply_recurring_bonus_features_foldsafe. Columns: {valid_df.columns.tolist()}")

    train_df = train_df.reset_index(drop=True)
    valid_df = valid_df.reset_index(drop=True)

    return train_df, valid_df, bonus_pattern_df


def build_promo_profile(df, min_bonus_months=3, corr_threshold_promo=0.45, corr_threshold_pure=0.70):
    df = force_itemcode_str(df)
    df = df.copy().sort_values(["ItemCode", "Year", "Month_Number"])

    out = []

    for item, g in df.groupby("ItemCode"):
        g = g.copy()

        if "Clean_Demand" not in g.columns:
            continue

        bonus_months = g[g["Bonus_Flag"] == 1]
        non_bonus_months = g[g["Bonus_Flag"] == 0]

        bonus_count = len(bonus_months)
        total_count = len(g)
        bonus_freq = bonus_count / total_count if total_count > 0 else 0.0

        bonus_avg = float(bonus_months["Clean_Demand"].mean()) if bonus_count > 0 else 0.0
        non_bonus_avg = float(non_bonus_months["Clean_Demand"].mean()) if len(non_bonus_months) > 0 else 0.0

        bonus_std = float(bonus_months["Clean_Demand"].std()) if bonus_count > 1 else 0.0
        non_bonus_std = float(non_bonus_months["Clean_Demand"].std()) if len(non_bonus_months) > 1 else 0.0

        if non_bonus_avg < 5:
            uplift_ratio = 1.0
        else:
            uplift_ratio = bonus_avg / max(non_bonus_avg, 1.0)

        uplift_ratio = np.clip(uplift_ratio, 0, 5)

        # safer correlation
        if g["Bonus_Flag"].nunique() > 1 and g["Clean_Demand"].nunique() > 1:
            corr = g["Bonus_Flag"].corr(g["Clean_Demand"])
            corr = 0.0 if pd.isna(corr) else float(corr)
        else:
            corr = 0.0

        # share of demand happening in bonus months
        total_demand = float(g["Clean_Demand"].sum())
        bonus_demand_share = float(bonus_months["Clean_Demand"].sum() / total_demand) if total_demand > 0 else 0.0

        # profile rules
        if (
            bonus_count >= min_bonus_months
            and corr >= corr_threshold_pure
            and uplift_ratio >= 2.0
            and bonus_demand_share >= 0.65
        ):
            promo_profile = "PURE_PROMO"
        elif (
            bonus_count >= min_bonus_months
            and corr >= corr_threshold_promo
            and uplift_ratio >= 1.25
        ):
            promo_profile = "PROMO_INFLUENCED"
        else:
            promo_profile = "NORMAL"

        out.append({
            "ItemCode": item,
            "Promo_Profile": promo_profile,
            "Bonus_Corr": corr,
            "Bonus_Frequency_Profile": bonus_freq,
            "Bonus_Avg_Demand": bonus_avg,
            "NonBonus_Avg_Demand": non_bonus_avg,
            "Bonus_Uplift_Ratio_Profile": uplift_ratio,
            "Bonus_Std_Demand": bonus_std,
            "NonBonus_Std_Demand": non_bonus_std,
            "Bonus_Demand_Share": bonus_demand_share,
            "Bonus_Month_Count": bonus_count
        })

    out_df = pd.DataFrame(out)
    out_df = force_itemcode_str(out_df)
    return out_df


def merge_promo_profile(df, promo_profile_df):
    df = force_itemcode_str(df)
    promo_profile_df = force_itemcode_str(promo_profile_df)
    df = df.copy()

    keep_cols = [
        "ItemCode",
        "Promo_Profile",
        "Bonus_Corr",
        "Bonus_Frequency_Profile",
        "Bonus_Avg_Demand",
        "NonBonus_Avg_Demand",
        "Bonus_Uplift_Ratio_Profile",
        "Bonus_Std_Demand",
        "NonBonus_Std_Demand",
        "Bonus_Demand_Share",
        "Bonus_Month_Count"
    ]

    df = df.drop(columns=[c for c in keep_cols if c != "ItemCode"], errors="ignore")
    df = df.merge(promo_profile_df[keep_cols], on="ItemCode", how="left")
    df = force_itemcode_str(df)

    df["Promo_Profile"] = df["Promo_Profile"].fillna("NORMAL")
    for c in [
        "Bonus_Corr",
        "Bonus_Frequency_Profile",
        "Bonus_Avg_Demand",
        "NonBonus_Avg_Demand",
        "Bonus_Uplift_Ratio_Profile",
        "Bonus_Std_Demand",
        "NonBonus_Std_Demand",
        "Bonus_Demand_Share",
        "Bonus_Month_Count"
    ]:
        df[c] = df[c].fillna(0)

    return df


## MODEL PIPELINE 

In [ ]:
# SHARED EXTRA SIGNALS
EXTRA_SIGNAL_FEATURES = [
    "Demand_Trend_3M",
    "Demand_Acceleration",
    "CV_3M",
    "Lag1_vs_Rolling3",
    "Lag2_vs_Rolling3",
    "Recent_Drop_Flag",
    "Demand_Rebound_Risk",

    "Primary_Stock_Cover_Change",
    "Distributor_Stock_Cover_Change",
    "Primary_Stockout_Risk",
    "Distributor_Buffer_Available",
    "Stock_Recovery_Flag",
    "Suppressed_Demand_Flag",

    "Bonus_Cycle_Position",
    "Near_Bonus_Cycle",
    "Promo_Strong_Uplift_Flag",
    "Promo_Frequent_Weak_Flag",
    "Recurring_Special_Bonus_Risk",

    "Expected_Bonus_Sin",
    "Expected_Bonus_Cos",
]

SINGLE_FEATURE_COLS = [
    "ItemCode", "ABC_Class",

    "Lag1", "Lag2", "Lag3", "Lag6", "Lag12",
    "Rolling3M_Mean", "Rolling6M_Mean", "Rolling3M_Std",
    "Momentum",
    "Month_Sin", "Month_Cos",
    "Quarter_Sin", "Quarter_Cos",

    "Bonus_Flag",
    "Free_Qty",
    "Free_Ratio",
    "Bonus_Flag_Lag1",
    "Free_Qty_Lag1",
    "Free_Ratio_Lag1",
    "Free_Qty_Rolling3",
    "Promo_Intensity_History",
    "Bonus_Frequency_12M",
    "Expected_Bonus_NextMonth",
    "Post_Bonus_Month_Flag",

    "Supply_Constraint_Flag",
    "Supply_Constraint_Lag1",
    "Supply_Constraint_Lag2",

    "Available_Primary_Inventory_Qty",
    "Distributor_Inventory_Qty",
    "Net_Available_Stock",
    "Stock_Cover_Months",
    "Demand_to_Stock_Ratio",
    "Primary_Stock_Cover",
    "Distributor_Stock_Cover",
    "Distributor_Stock_Cover_Lag1",

    "Inventory_Pressure",
    "Supply_Shock",

    "Recurring_Bonus_SKU",
    "Recurring_Bonus_Month",
    "Bonus_Cycle_Length",
    "Months_Since_Last_Bonus",
    "Avg_Bonus_Uplift",

    "Promo_Uplift_Lag1",
    "Promo_Uplift_Lag2",
    "Promo_Uplift_6M",
    "Last_Bonus_Demand",

    "Bonus_Corr",
    "Bonus_Frequency_Profile",
    "Bonus_Uplift_Ratio_Profile",
    "Bonus_Demand_Share",
    "Bonus_Month_Count",

    "ZeroRate_6M",
    "SKU_Mean_Demand",
    "SKU_ZeroRate",
    "SKU_CV",

    "Behavior_CV_6M",
    "Behavior_Peak_Ratio_6M",
    "Behavior_Autocorr_Lag3_12M",
    "Behavior_Autocorr_Lag6_18M",
    "Behavior_ZeroRate_12M",
    "Behavior_PromoRate_12M",
    "Lag1_to_Rolling3_Ratio",
    "Lag1_to_Rolling6_Ratio",
    "Recent_Spike_Flag",
    "Post_Spike_Drop_Risk",
    "Cycle_Pos_3",
    "Cycle_Pos_6",
] + EXTRA_SIGNAL_FEATURES


In [ ]:
# =========================
# FEATURE SETS BY SEGMENT
# =========================

LONG_FEATURE_COLS = SINGLE_FEATURE_COLS.copy()

MEDIUM_FEATURE_COLS = [
    f for f in SINGLE_FEATURE_COLS
    if f not in [
        "Lag12",
        "Rolling6M_Mean",
        "Quarter_Sin",
        "Quarter_Cos",
    ]
]

SHORT_FEATURE_COLS = [
    "Lag1",
    "Lag2",
    "Rolling3M_Mean",
    "SKU_Mean_Demand",
    "Last_Bonus_Demand",
    "Avg_Bonus_Uplift",
    "Bonus_Flag",
    "Expected_Bonus_NextMonth",

    "Supply_Constraint_Flag",
    "Available_Primary_Inventory_Qty",
    "Distributor_Inventory_Qty",

    "Recent_Drop_Flag",
    "Demand_Rebound_Risk",
    "Primary_Stockout_Risk",
    "Distributor_Buffer_Available",
    "Suppressed_Demand_Flag",
    "Near_Bonus_Cycle",
    "Promo_Frequent_Weak_Flag",
    "Recurring_Special_Bonus_Risk",

    "Short_History_Length",
    "Short_Mean_Demand",
    "Short_ZeroRate",
    "Short_Bonus_Frequency",
    "Short_Bonus_Demand_Share",
    "Short_Supply_Rate",
]

# =========================
# FINAL FEATURE BUILD BEFORE SEGMENT COPIES
# =========================
Data = rebuild_time_features(Data)

Data["Behavior_Type"] = Data.apply(classify_sku_behavior, axis=1)

Data = add_history_length_from_subset(Data)

Data_long = Data[Data["History_Segment"] == "LONG"].copy()
Data_medium = Data[Data["History_Segment"] == "MEDIUM"].copy()
Data_short = Data[Data["History_Segment"] == "SHORT"].copy()

print("LONG SKUs:", Data_long["ItemCode"].nunique())
print("MEDIUM SKUs:", Data_medium["ItemCode"].nunique())
print("SHORT SKUs:", Data_short["ItemCode"].nunique())

print("Behavior_Type in Data:", "Behavior_Type" in Data.columns)
print("Behavior_Type in Data_long:", "Behavior_Type" in Data_long.columns)
print("Behavior_Type in Data_medium:", "Behavior_Type" in Data_medium.columns)
print("Behavior_Type in Data_short:", "Behavior_Type" in Data_short.columns)

### Long History Segment

In [ ]:
def permutation_rank(model, eval_df, feature_cols, target_col, n_repeats=5):
    """
    Permutation importance on an evaluation dataframe.
    Use validation windows for pruning, not the final holdout test window.
    """
    X = sanitize(eval_df[feature_cols])
    y = eval_df[target_col].values

    r = permutation_importance(
        model,
        X,
        y,
        scoring="neg_mean_absolute_error",
        n_repeats=n_repeats,
        random_state=42
    )

    imp = pd.DataFrame({
        "feature": feature_cols,
        "perm_importance": r.importances_mean
    }).sort_values("perm_importance", ascending=False)

    return imp


def prepare_long_deploy_frame(full_data):
    deploy_df = full_data.copy().sort_values(["ItemCode", "Year", "Month_Number"])
    deploy_df = force_itemcode_str(deploy_df)

    deploy_df = add_history_length_from_subset(deploy_df, deploy_df)
    deploy_df = deploy_df[deploy_df["History_Segment"] == "LONG"].copy()

    if deploy_df.empty:
        raise ValueError("No LONG rows available for deployment.")

    bonus_pattern_df = detect_recurring_bonus_skus(deploy_df)[[
        "ItemCode",
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ]].copy()
    bonus_pattern_df = force_itemcode_str(bonus_pattern_df)

    deploy_df = deploy_df.drop(columns=[
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ], errors="ignore")

    deploy_df = force_itemcode_str(deploy_df)
    deploy_df = deploy_df.merge(bonus_pattern_df, on="ItemCode", how="left")
    deploy_df = force_itemcode_str(deploy_df)

    deploy_df, _ = apply_sku_cap(deploy_df.copy(), deploy_df.copy())

    for c in ["Recurring_Bonus_SKU", "Bonus_Cycle_Length", "Avg_Bonus_Gap", "Bonus_Frequency_All"]:
        deploy_df[c] = deploy_df[c].fillna(0)

    deploy_df["Avg_Bonus_Uplift"] = deploy_df["Avg_Bonus_Uplift"].fillna(1.0)

    sku_total = deploy_df.groupby("ItemCode")["Clean_Demand"].sum().sort_values(ascending=False)
    total_sum = sku_total.sum()

    if total_sum > 0:
        cum_pct = sku_total.cumsum() / total_sum
        abc_series = pd.cut(cum_pct, bins=[0, 0.7, 0.9, 1.0], labels=[0, 1, 2])
        abc_map = abc_series.to_dict()
        deploy_df["ABC_Class"] = deploy_df["ItemCode"].map(abc_map).fillna(2)
    else:
        abc_map = {}
        deploy_df["ABC_Class"] = 2

    promo_profile_df = build_promo_profile(deploy_df)
    deploy_df = merge_promo_profile(deploy_df, promo_profile_df)

    sku_profile_df = (
        deploy_df.groupby("ItemCode")
        .agg(
            SKU_Mean_Demand=("Clean_Demand", "mean"),
            SKU_Std_Demand=("Clean_Demand", "std"),
            SKU_ZeroRate=("Clean_Demand", lambda x: (x == 0).mean())
        )
        .reset_index()
    )

    sku_profile_df["SKU_Std_Demand"] = sku_profile_df["SKU_Std_Demand"].fillna(0)
    sku_profile_df["SKU_CV"] = np.where(
        sku_profile_df["SKU_Mean_Demand"] <= 0,
        0,
        sku_profile_df["SKU_Std_Demand"] / (sku_profile_df["SKU_Mean_Demand"] + 1)
    )

    sku_profile_df = sku_profile_df[["ItemCode", "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]]
    sku_profile_df = force_itemcode_str(sku_profile_df)
    
    deploy_df = deploy_df.drop(columns=["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"], errors="ignore")
    deploy_df = deploy_df.merge(sku_profile_df, on="ItemCode", how="left")
    deploy_df = force_itemcode_str(deploy_df)

    for c in ["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]:
        deploy_df[c] = deploy_df[c].fillna(0)

    deploy_df = add_bonus_cycle_features(deploy_df)
    deploy_df = rebuild_time_features(deploy_df)

    caps = compute_clip_caps(deploy_df, cols=["Inventory_Pressure", "Stock_Cover_Months"], q=0.99)
    deploy_df = apply_clip_caps(deploy_df, caps)

    deploy_df = recompute_target(deploy_df)
    deploy_df = add_residual_target(deploy_df)
    deploy_df = deploy_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()

    deploy_df["ItemCode_Original"] = deploy_df["ItemCode"]
    deploy_df, _, itemcode_categories = encode_itemcode(deploy_df, deploy_df)

    return deploy_df, {
        "abc_map": abc_map,
        "promo_profile_df": promo_profile_df,
        "clip_caps": caps,
        "sku_profile_df": sku_profile_df,
        "itemcode_categories": itemcode_categories
    }



def rebuild_time_features_foldsafe(train_df, valid_df):
    """
    Build train features on train only.
    Build valid features sequentially using train history + prior valid rows only.
    """
    train_df = force_itemcode_str(train_df)
    valid_df = force_itemcode_str(valid_df)
    
    train_df = train_df.copy().sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)
    valid_df = valid_df.copy().sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    # Train can be rebuilt normally
    train_out = rebuild_time_features(train_df)

    valid_rows = []

    for item_code, g_valid in valid_df.groupby("ItemCode", sort=False):
        g_train = train_df[train_df["ItemCode"] == item_code].copy()
        history = g_train.copy()

        g_valid = g_valid.sort_values(["Year", "Month_Number"]).copy()

        for _, row in g_valid.iterrows():
            row_df = pd.DataFrame([row])

            # Build current row using only history available up to that row
            temp = pd.concat([history, row_df], ignore_index=True)
            temp = temp.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)
            temp = rebuild_time_features(temp)

            valid_rows.append(temp.iloc[-1].copy())

            # After creating row-safe features, append raw row to history for next step
            history = pd.concat([history, row_df], ignore_index=True)

    valid_out = pd.DataFrame(valid_rows) if valid_rows else valid_df.head(0).copy()

    train_out = force_itemcode_str(train_out)
    valid_out = force_itemcode_str(valid_out)
    return train_out, valid_out

#### XGB_LONG

##### Tune

In [ ]:
# TUNING FUNCTION BY SEGMENT
LONG_TUNE_WINDOWS = get_long_tune_windows(Data, TIME_WINDOWS, n_folds=2, fold_size_months=12)
print("LONG_TUNE_WINDOWS:", LONG_TUNE_WINDOWS)

def tune_residual_xgb_long(full_data, feature_cols, n_trials=40, study_name=None):
    full_data = full_data.copy()

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 700, 1400),
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.06),
            "max_depth": trial.suggest_int("max_depth", 4, 7),
            "max_leaves": 64,
            "grow_policy": "lossguide",
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 6),
            "subsample": trial.suggest_float("subsample", 0.75, 0.9),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.75, 0.9),
            "gamma": trial.suggest_float("gamma", 0, 0.3),
            "reg_lambda": trial.suggest_float("reg_lambda", 2, 12),
            "reg_alpha": trial.suggest_float("reg_alpha", 0, 3),
        }

        scores = []
        temp = add_period_index(full_data)

        for tune_window in LONG_TUNE_WINDOWS:

            train_df = temp[temp["Period_Index"] < tune_window["valid_start_idx"]].copy()
            valid_df = temp[temp["Period_Index"].between(tune_window["valid_start_idx"], tune_window["valid_end_idx"])].copy()

            train_df, valid_df, _ = prepare_long_frame_foldsafe(train_df, valid_df)
            if train_df is None or valid_df is None:
                continue

            train_df, valid_df, _ = encode_itemcode(train_df, valid_df)

            assert_features_exist(train_df, feature_cols, where="XGB_LONG_TUNE_TRAIN")
            assert_features_exist(valid_df, feature_cols, where="XGB_LONG_TUNE_VALID")

            model = xgb.XGBRegressor(
                objective="reg:squarederror",
                eval_metric="rmse",
                random_state=42,
                tree_method="hist",
                n_jobs=-1,
                **params
            )

            Xtr = sanitize(train_df[feature_cols])
            ytr = train_df[MODEL_TARGET_COL]
            Xva = sanitize(valid_df[feature_cols])

            model.fit(
                Xtr,
                ytr,
                sample_weight=build_train_weights(train_df),
                verbose=False
            )

            pred_residual = model.predict(Xva)
            pred_final = np.clip(valid_df[BASELINE_COL].values + pred_residual, 0, None)

            scores.append(wmape(valid_df[ACTUAL_TARGET_COL].values, pred_final))

        return 999999.0 if len(scores) == 0 else np.mean(scores)

    study = optuna.create_study(direction="minimize", study_name=study_name)
    study.optimize(objective, n_trials=n_trials)

    return study.best_params, study


In [ ]:
long_best_params, long_study = tune_residual_xgb_long(
    full_data=Data,
    feature_cols=LONG_FEATURE_COLS,
    n_trials=40,
    study_name="residual_xgb_long"
)
print("LONG best params:", long_best_params)

##### Feature Pruning

In [ ]:
LONG_VALID_WINDOW = get_long_validation_window(TIME_WINDOWS, valid_months=12)

print("LONG validation window:",
      LONG_VALID_WINDOW["valid_start_idx"],
      LONG_VALID_WINDOW["valid_end_idx"])

In [ ]:
def train_eval_validation_xgb_long(
    full_data,
    feature_cols,
    best_params,
    valid_window,
    yearly_boost=0.25
):
    full_data = add_period_index(full_data)

    train_df = full_data[full_data["Period_Index"] < valid_window["valid_start_idx"]].copy()
    valid_df = full_data[full_data["Period_Index"].between(valid_window["valid_start_idx"], valid_window["valid_end_idx"])].copy()

    if train_df.empty or valid_df.empty:
        raise ValueError("Need both train and dynamic validation data.")

    train_df, valid_df, prep_artifacts = prepare_long_frame_foldsafe(train_df, valid_df)

    if train_df is None or valid_df is None:
        raise ValueError("No usable LONG rows after fold-safe validation preparation.")

    train_df["ItemCode_Original"] = train_df["ItemCode"]
    valid_df["ItemCode_Original"] = valid_df["ItemCode"]

    train_df, valid_df, itemcode_categories = encode_itemcode(train_df, valid_df)
    prep_artifacts["itemcode_categories"] = itemcode_categories

    assert_features_exist(train_df, feature_cols, where="XGB_LONG_VALID_TRAIN")
    assert_features_exist(valid_df, feature_cols, where="XGB_LONG_VALID_EVAL")

    model = xgb.XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=42,
        tree_method="hist",
        n_jobs=-1,
        **best_params
    )

    model.fit(
        sanitize(train_df[feature_cols]),
        train_df[MODEL_TARGET_COL],
        sample_weight=build_train_weights(train_df, yearly_boost=yearly_boost),
        verbose=False
    )

    valid_df["Pred_Residual"] = model.predict(sanitize(valid_df[feature_cols]))
    valid_df["Pred"] = np.clip(valid_df[BASELINE_COL] + valid_df["Pred_Residual"], 0, None)

    metrics = evaluate_all_metrics(valid_df[ACTUAL_TARGET_COL].values, valid_df["Pred"].values)

    return model, valid_df, metrics, prep_artifacts

def iterative_feature_prune_xgb_long_holdout(
    full_data,
    start_features,
    best_params,
    valid_window,
    drop_k=1,
    min_features=18,
    max_rounds=12,
    tolerance=0.10,
    n_repeats=5
):
    """
    Feature pruning using the validation holdout window only.
    The final holdout test window is not used here.
    """
    history = []
    features = start_features.copy()

    model, eval_df, m, _ = train_eval_validation_xgb_long(
        full_data=full_data,
        feature_cols=features,
        best_params=best_params,
        valid_window=valid_window
    )

    best_wmape = m["WMAPE"]
    last_accepted_state = (features.copy(), model, eval_df.copy(), m.copy())

    for r in range(1, max_rounds + 1):
        if len(features) <= min_features:
            break

        imp = permutation_rank(
            model=model,
            eval_df=eval_df,
            feature_cols=features,
            target_col=MODEL_TARGET_COL,
            n_repeats=n_repeats
        )

        protected = {
            "ItemCode",
            "ABC_Class",
            "Lag1",
            "Rolling3M_Mean",
            "Month_Sin",
            "Month_Cos",
            "Recurring_Bonus_SKU",
            "Expected_Bonus_NextMonth"
        }

        drop_candidates = [
            f for f in imp.sort_values("perm_importance").feature.tolist()
            if f not in protected
        ]
        to_drop = drop_candidates[:drop_k]

        if not to_drop:
            break

        new_features = [f for f in features if f not in to_drop]

        new_model, new_eval_df, new_m, _ = train_eval_validation_xgb_long(
            full_data=full_data,
            feature_cols=new_features,
            best_params=best_params,
            valid_window=valid_window
        )

        history.append({
            "round": r,
            "model": "XGBoost",
            "segment": "LONG",
            "valid_start_idx": valid_window["valid_start_idx"],
            "valid_end_idx": valid_window["valid_end_idx"],
            "dropped": to_drop,
            "n_features": len(new_features),
            **new_m
        })

        if new_m["WMAPE"] <= best_wmape + tolerance:
            features = new_features
            model, eval_df = new_model, new_eval_df
            best_wmape = min(best_wmape, new_m["WMAPE"])
            last_accepted_state = (features.copy(), model, eval_df.copy(), new_m.copy())
        else:
            break

    results_df = pd.DataFrame(history)

    return (
        last_accepted_state[0],
        last_accepted_state[1],
        last_accepted_state[2],
        last_accepted_state[3],
        results_df
    )

In [ ]:
# LONG
long_best_feats, _, _, long_best_metrics, long_prune_log = iterative_feature_prune_xgb_long_holdout(
    full_data=Data,
    start_features=LONG_FEATURE_COLS,
    best_params=long_best_params,
    valid_window=LONG_VALID_WINDOW,
    drop_k=1,
    min_features=18,
    max_rounds=12,
    tolerance=0.10,
    n_repeats=5
)

print("LONG best metrics:", long_best_metrics)
print("LONG best features:", long_best_feats)
print(long_prune_log)

#### CAT_LONG

In [ ]:
# ============================================================
# CATBOOST LONG
# ============================================================

# 1) TUNING
def tune_residual_catboost_long(full_data, feature_cols, n_trials=40, study_name=None):
    full_data = full_data.copy()

    def objective(trial):
        params = {
            "loss_function": "RMSE",
            "eval_metric": "RMSE",
            "random_seed": 42,
            "verbose": 0,

            "iterations": trial.suggest_int("iterations", 500, 1400),
            "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.06),
            "depth": trial.suggest_int("depth", 4, 8),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 2.0, 15.0),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 40),
            "subsample": trial.suggest_float("subsample", 0.70, 0.95),
            "rsm": trial.suggest_float("rsm", 0.70, 1.00),
            "random_strength": trial.suggest_float("random_strength", 0.0, 3.0),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 3.0)
        }

        scores = []

        full_data_pi = add_period_index(full_data)

        for window in LONG_TUNE_WINDOWS:
            train_df = full_data_pi[full_data_pi["Period_Index"] < window["valid_start_idx"]].copy()
            valid_df = full_data_pi[full_data_pi["Period_Index"].between(window["valid_start_idx"], window["valid_end_idx"])].copy()

            train_df, valid_df, _ = prepare_long_frame_foldsafe(train_df, valid_df)
            
            if train_df is None or valid_df is None:
                continue

            train_df, valid_df, _ = encode_itemcode(train_df, valid_df)

            assert_features_exist(train_df, feature_cols, where="CATBOOST_LONG_TUNE_TRAIN")
            assert_features_exist(valid_df, feature_cols, where="CATBOOST_LONG_TUNE_VALID")

            Xtr = sanitize(train_df[feature_cols])
            ytr = train_df[MODEL_TARGET_COL]
            Xva = sanitize(valid_df[feature_cols])

            model = CatBoostRegressor(**params)

            model.fit(
                Xtr,
                ytr,
                sample_weight=build_train_weights(train_df),
                verbose=False
            )

            pred_residual = model.predict(Xva)
            pred_final = np.clip(valid_df[BASELINE_COL].values + pred_residual, 0, None)

            scores.append(wmape(valid_df[ACTUAL_TARGET_COL].values, pred_final))

        return 999999.0 if len(scores) == 0 else np.mean(scores)

    study = optuna.create_study(direction="minimize", study_name=study_name)
    study.optimize(objective, n_trials=n_trials)

    return study.best_params, study

In [ ]:
# 3) FEATURE PRUNING

def train_eval_validation_catboost_long(
    full_data,
    feature_cols,
    best_params,
    valid_window,
    yearly_boost=0.25
):
    full_data = add_period_index(full_data)

    train_df = full_data[full_data["Period_Index"] < valid_window["valid_start_idx"]].copy()
    valid_df = full_data[full_data["Period_Index"].between(valid_window["valid_start_idx"], valid_window["valid_end_idx"])].copy()

    if train_df.empty or valid_df.empty:
        raise ValueError("Need both train and dynamic validation data.")

    train_df, valid_df, prep_artifacts = prepare_long_frame_foldsafe(train_df, valid_df)

    if train_df is None or valid_df is None:
        raise ValueError("No usable LONG rows after fold-safe validation preparation.")

    train_df["ItemCode_Original"] = train_df["ItemCode"]
    valid_df["ItemCode_Original"] = valid_df["ItemCode"]

    train_df, valid_df, itemcode_categories = encode_itemcode(train_df, valid_df)
    prep_artifacts["itemcode_categories"] = itemcode_categories

    assert_features_exist(train_df, feature_cols, where="CATBOOST_LONG_VALID_TRAIN")
    assert_features_exist(valid_df, feature_cols, where="CATBOOST_LONG_VALID_EVAL")

    model = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42,
        verbose=0,
        **best_params
    )

    model.fit(
        sanitize(train_df[feature_cols]),
        train_df[MODEL_TARGET_COL],
        sample_weight=build_train_weights(train_df, yearly_boost=yearly_boost),
        verbose=False
    )

    valid_df["Pred_Residual"] = model.predict(sanitize(valid_df[feature_cols]))
    valid_df["Pred"] = np.clip(valid_df[BASELINE_COL] + valid_df["Pred_Residual"], 0, None)

    metrics = evaluate_all_metrics(valid_df[ACTUAL_TARGET_COL].values, valid_df["Pred"].values)

    return model, valid_df, metrics, prep_artifacts


def iterative_feature_prune_catboost_long_holdout(
    full_data,
    start_features,
    best_params,
    valid_window,
    drop_k=1,
    min_features=18,
    max_rounds=12,
    tolerance=0.10,
    n_repeats=5
):
    history = []
    features = start_features.copy()

    model, eval_df, m, _ = train_eval_validation_catboost_long(
        full_data=full_data,
        feature_cols=features,
        best_params=best_params,
        valid_window=valid_window
    )

    best_wmape = m["WMAPE"]
    last_accepted_state = (features.copy(), model, eval_df.copy(), m.copy())

    for r in range(1, max_rounds + 1):
        if len(features) <= min_features:
            break

        imp = permutation_rank(
            model=model,
            eval_df=eval_df,
            feature_cols=features,
            target_col=MODEL_TARGET_COL,
            n_repeats=n_repeats
        )

        protected = {
            "ItemCode",
            "ABC_Class",
            "Lag1",
            "Rolling3M_Mean",
            "Month_Sin",
            "Month_Cos",
            "Recurring_Bonus_SKU",
            "Expected_Bonus_NextMonth"
        }

        drop_candidates = [
            f for f in imp.sort_values("perm_importance").feature.tolist()
            if f not in protected
        ]
        to_drop = drop_candidates[:drop_k]

        if not to_drop:
            break

        new_features = [f for f in features if f not in to_drop]

        new_model, new_eval_df, new_m, _ = train_eval_validation_catboost_long(
            full_data=full_data,
            feature_cols=new_features,
            best_params=best_params,
            valid_window=valid_window
        )

        history.append({
            "round": r,
            "model": "CATBOOST",
            "segment": "LONG",
            "valid_start_idx": valid_window["valid_start_idx"],
            "valid_end_idx": valid_window["valid_end_idx"],
            "dropped": to_drop,
            "n_features": len(new_features),
            **new_m
        })

        if new_m["WMAPE"] <= best_wmape + tolerance:
            features = new_features
            model, eval_df = new_model, new_eval_df
            best_wmape = min(best_wmape, new_m["WMAPE"])
            last_accepted_state = (features.copy(), model, eval_df.copy(), new_m.copy())
        else:
            break

    results_df = pd.DataFrame(history)

    return (
        last_accepted_state[0],
        last_accepted_state[1],
        last_accepted_state[2],
        last_accepted_state[3],
        results_df
    )

In [ ]:
# ============================================================
# CATBOOST LONG RUN
# ============================================================

catboost_long_best_params, catboost_long_study = tune_residual_catboost_long(
    full_data=Data,
    feature_cols=LONG_FEATURE_COLS,
    n_trials=40,
    study_name="residual_catboost_long"
)
print("CATBOOST LONG best params:", catboost_long_best_params)

catboost_long_best_feats, _, _, catboost_long_best_metrics, catboost_long_prune_log = iterative_feature_prune_catboost_long_holdout(
    full_data=Data,
    start_features=LONG_FEATURE_COLS,
    best_params=catboost_long_best_params,
    valid_window=LONG_VALID_WINDOW,
    drop_k=1,
    min_features=18,
    max_rounds=12,
    tolerance=0.10,
    n_repeats=5
)

print("CATBOOST LONG best metrics:", catboost_long_best_metrics)
print("CATBOOST LONG best features:", catboost_long_best_feats)
print(catboost_long_prune_log)

#### LGBM_LONG

In [ ]:
# ============================================================
# LIGHTGBM LONG
# ============================================================

# 1) TUNING
def tune_residual_lgbm_long(full_data, feature_cols, n_trials=40, study_name=None):
    full_data = full_data.copy()

    def objective(trial):
        params = {
            "objective": "regression",
            "metric": "rmse",
            "random_state": 42,
            "n_jobs": -1,
            "verbosity": -1,

            "n_estimators": trial.suggest_int("n_estimators", 500, 1400),
            "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.06),
            "num_leaves": trial.suggest_int("num_leaves", 31, 127),
            "max_depth": trial.suggest_int("max_depth", 4, 8),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 60),
            "subsample": trial.suggest_float("subsample", 0.70, 0.95),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.70, 0.95),
            "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 8.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 12.0),
            "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 0.3)
        }

        scores = []

        full_data_pi = add_period_index(full_data)

        for window in LONG_TUNE_WINDOWS:
            train_df = full_data_pi[full_data_pi["Period_Index"] < window["valid_start_idx"]].copy()
            valid_df = full_data_pi[full_data_pi["Period_Index"].between(window["valid_start_idx"], window["valid_end_idx"])].copy()

            train_df, valid_df, _ = prepare_long_frame_foldsafe(train_df, valid_df)
            if train_df is None or valid_df is None:
                continue

            train_df, valid_df, _ = encode_itemcode(train_df, valid_df)

            assert_features_exist(train_df, feature_cols, where="LGBM_LONG_TUNE_TRAIN")
            assert_features_exist(valid_df, feature_cols, where="LGBM_LONG_TUNE_VALID")

            Xtr = sanitize(train_df[feature_cols])
            ytr = train_df[MODEL_TARGET_COL]
            Xva = sanitize(valid_df[feature_cols])

            model = LGBMRegressor(**params)

            model.fit(
                Xtr,
                ytr,
                sample_weight=build_train_weights(train_df)
            )

            pred_residual = model.predict(Xva)
            pred_final = np.clip(valid_df[BASELINE_COL].values + pred_residual, 0, None)

            scores.append(wmape(valid_df[ACTUAL_TARGET_COL].values, pred_final))

        return 999999.0 if len(scores) == 0 else np.mean(scores)

    study = optuna.create_study(direction="minimize", study_name=study_name)
    study.optimize(objective, n_trials=n_trials)

    return study.best_params, study

In [ ]:
# 3) FEATURE PRUNING

def train_eval_validation_lgbm_long(
    full_data,
    feature_cols,
    best_params,
    valid_window,
    yearly_boost=0.25
):
    full_data = add_period_index(full_data)

    train_df = full_data[full_data["Period_Index"] < valid_window["valid_start_idx"]].copy()
    valid_df = full_data[full_data["Period_Index"].between(valid_window["valid_start_idx"], valid_window["valid_end_idx"])].copy()

    if train_df.empty or valid_df.empty:
        raise ValueError("Need both train and dynamic validation data.")

    train_df, valid_df, prep_artifacts = prepare_long_frame_foldsafe(train_df, valid_df)

    if train_df is None or valid_df is None:
        raise ValueError("No usable LONG rows after fold-safe validation preparation.")

    train_df["ItemCode_Original"] = train_df["ItemCode"]
    valid_df["ItemCode_Original"] = valid_df["ItemCode"]

    train_df, valid_df, itemcode_categories = encode_itemcode(train_df, valid_df)
    prep_artifacts["itemcode_categories"] = itemcode_categories

    assert_features_exist(train_df, feature_cols, where="LGBM_LONG_VALID_TRAIN")
    assert_features_exist(valid_df, feature_cols, where="LGBM_LONG_VALID_EVAL")

    model = LGBMRegressor(
        objective="regression",
        metric="rmse",
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
        **best_params
    )

    model.fit(
        sanitize(train_df[feature_cols]),
        train_df[MODEL_TARGET_COL],
        sample_weight=build_train_weights(train_df, yearly_boost=yearly_boost)
    )

    valid_df["Pred_Residual"] = model.predict(sanitize(valid_df[feature_cols]))
    valid_df["Pred"] = np.clip(valid_df[BASELINE_COL] + valid_df["Pred_Residual"], 0, None)

    metrics = evaluate_all_metrics(valid_df[ACTUAL_TARGET_COL].values, valid_df["Pred"].values)

    return model, valid_df, metrics, prep_artifacts


def iterative_feature_prune_lgbm_long_holdout(
    full_data,
    start_features,
    best_params,
    valid_window,
    drop_k=1,
    min_features=18,
    max_rounds=12,
    tolerance=0.10,
    n_repeats=5
):
    history = []
    features = start_features.copy()

    model, eval_df, m, _ = train_eval_validation_lgbm_long(
        full_data=full_data,
        feature_cols=features,
        best_params=best_params,
        valid_window=valid_window
    )

    best_wmape = m["WMAPE"]
    last_accepted_state = (features.copy(), model, eval_df.copy(), m.copy())

    for r in range(1, max_rounds + 1):
        if len(features) <= min_features:
            break

        imp = permutation_rank(
            model=model,
            eval_df=eval_df,
            feature_cols=features,
            target_col=MODEL_TARGET_COL,
            n_repeats=n_repeats
        )

        protected = {
            "ItemCode",
            "ABC_Class",
            "Lag1",
            "Rolling3M_Mean",
            "Month_Sin",
            "Month_Cos",
            "Recurring_Bonus_SKU",
            "Expected_Bonus_NextMonth"
        }

        drop_candidates = [
            f for f in imp.sort_values("perm_importance").feature.tolist()
            if f not in protected
        ]
        to_drop = drop_candidates[:drop_k]

        if not to_drop:
            break

        new_features = [f for f in features if f not in to_drop]

        new_model, new_eval_df, new_m, _ = train_eval_validation_lgbm_long(
            full_data=full_data,
            feature_cols=new_features,
            best_params=best_params,
            valid_window=valid_window
        )

        history.append({
            "round": r,
            "model": "LIGHTGBM",
            "segment": "LONG",
            "valid_start_idx": valid_window["valid_start_idx"],
            "valid_end_idx": valid_window["valid_end_idx"],
            "dropped": to_drop,
            "n_features": len(new_features),
            **new_m
        })

        if new_m["WMAPE"] <= best_wmape + tolerance:
            features = new_features
            model, eval_df = new_model, new_eval_df
            best_wmape = min(best_wmape, new_m["WMAPE"])
            last_accepted_state = (features.copy(), model, eval_df.copy(), new_m.copy())
        else:
            break

    results_df = pd.DataFrame(history)

    return (
        last_accepted_state[0],
        last_accepted_state[1],
        last_accepted_state[2],
        last_accepted_state[3],
        results_df
    )

In [ ]:
# ============================================================
# LIGHTGBM LONG RUN
# ============================================================

lgbm_long_best_params, lgbm_long_study = tune_residual_lgbm_long(
    full_data=Data,
    feature_cols=LONG_FEATURE_COLS,
    n_trials=40,
    study_name="residual_lgbm_long"
)
print("LIGHTGBM LONG best params:", lgbm_long_best_params)

lgbm_long_best_feats, _, _, lgbm_long_best_metrics, lgbm_long_prune_log = iterative_feature_prune_lgbm_long_holdout(
    full_data=Data,
    start_features=LONG_FEATURE_COLS,
    best_params=lgbm_long_best_params,
    valid_window=LONG_VALID_WINDOW,
    drop_k=1,
    min_features=18,
    max_rounds=12,
    tolerance=0.10,
    n_repeats=5
)

print("LIGHTGBM LONG best metrics:", lgbm_long_best_metrics)
print("LIGHTGBM LONG best features:", lgbm_long_best_feats)
print(lgbm_long_prune_log)

#### DL_LONG

In [ ]:
# 1) CONFIG
GRU_SEED = 42

GRU_SEQ_LEN = 18

GRU_LR = 8e-4
GRU_WEIGHT_DECAY = 1e-5
GRU_DROPOUT = 0.25
GRU_EMBED_DIM = 32

GRU_DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

GRU_BATCH_SIZE = 256
GRU_EPOCHS = 10
GRU_HIDDEN_SIZE = 32
GRU_NUM_LAYERS = 1

GRU_HOLDOUT_MONTHS = 12
GRU_RECENT_MONTHS = 4

GRU_ABC_WEIGHT_MAP = {0: 2.5, 1: 1.2, 2: 1.0}
GRU_PROMO_WEIGHT = 1.35
GRU_SUPPLY_WEIGHT = 1.20
GRU_RECURRING_PROMO_WEIGHT = 1.25
GRU_EXPECTED_PROMO_WEIGHT = 1.20

GRU_UNDER_PENALTY = 1.10

GRU_EVAL_DIR = "gru_long_eval_artifacts"
GRU_DEPLOY_DIR = "gru_long_deploy_artifacts"

In [ ]:
# =========================
# GRU DEBUG CONFIG
# =========================
GRU_DEBUG = True
GRU_DEBUG_SKUS = {"600308", "600311", "600315", "600319"}   # add/remove as needed
GRU_DEBUG_MAX_SEQ_ROWS = 8

In [ ]:
# 2) REPRODUCIBILITY
def gru_seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# 3) GRU HELPERS
def gru_signed_log_transform(x):
    x = np.asarray(x, dtype=float)
    return np.sign(x) * np.log1p(np.abs(x))

def gru_signed_log_inverse(x):
    x = np.asarray(x, dtype=float)
    return np.sign(x) * np.expm1(np.abs(x))

def gru_infer_next_year_month(year, month_number):
    year = int(year)
    month_number = int(month_number)
    if month_number == 12:
        return year + 1, 1
    return year, month_number + 1

def make_item_mapping_from_train(train_df):
    train_df = force_itemcode_str(train_df)

    key_col = "ItemCode_Original" if "ItemCode_Original" in train_df.columns else "ItemCode"
    item_codes = sorted(train_df[key_col].astype(str).unique().tolist())

    return {item: i for i, item in enumerate(item_codes)}

# 4.1) LONG DATA ADAPTER FOR GRU
def prepare_long_gru_from_prepared_df(prepared_df):
    df = force_itemcode_str(prepared_df)
    df = df.copy().sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    if df.empty:
        return df

    df["Residual_Target_Log"] = gru_signed_log_transform(df[MODEL_TARGET_COL].fillna(0))
    df["Residual_Target_Log"] = df["Residual_Target_Log"].clip(-7.0, 7.0)

    return df

def prepare_long_gru_eval_data_for_window(full_data, valid_window):
    full_data = add_period_index(full_data)

    train_df = full_data[full_data["Period_Index"] < valid_window["valid_start_idx"]].copy()
    valid_df = full_data[
        full_data["Period_Index"].between(valid_window["valid_start_idx"], valid_window["valid_end_idx"])
    ].copy()

    if train_df.empty or valid_df.empty:
        raise ValueError("No usable LONG rows for GRU window evaluation.")

    train_prep, valid_prep, prep_artifacts = prepare_long_frame_foldsafe(train_df, valid_df)

    if train_prep is None or valid_prep is None:
        raise ValueError("No usable LONG rows after GRU fold-safe preparation.")

    train_prep["ItemCode_Original"] = train_prep["ItemCode"].astype(str)
    valid_prep["ItemCode_Original"] = valid_prep["ItemCode"].astype(str)

    train_prep, valid_prep, itemcode_categories = encode_itemcode(train_prep, valid_prep)
    prep_artifacts["itemcode_categories"] = itemcode_categories

    train_prep = prepare_long_gru_from_prepared_df(train_prep)
    valid_prep = prepare_long_gru_from_prepared_df(valid_prep)

    artifacts = {
        "abc_map": prep_artifacts["abc_map"],
        "promo_profile_df": prep_artifacts["promo_profile_df"],
        "sku_profile_df": prep_artifacts["sku_profile_df"],
        "clip_caps": prep_artifacts["clip_caps"],
        "itemcode_categories": prep_artifacts["itemcode_categories"]
    }

    return train_prep, valid_prep, artifacts

def prepare_long_gru_deploy_data(full_data):
    deploy_prep, deploy_artifacts = prepare_long_deploy_frame(full_data)
    deploy_prep = prepare_long_gru_from_prepared_df(deploy_prep)
    return deploy_prep, deploy_artifacts


# 5) FEATURE SETS FOR GRU
GRU_LONG_SEQ_FEATURES = [
    "Clean_Demand",
    "Secondary_Sales_Qty",
    "Primary_Sales_Qty",
    "Free_Qty",
    "Free_Ratio",

    "Bonus_Flag",
    "Bonus_Flag_Lag1",
    "Bonus_Flag_Lag2",
    "Bonus_Frequency_12M",
    "Recurring_Bonus_SKU",
    "Bonus_Cycle_Length",
    "Months_Since_Last_Bonus",
    "Expected_Bonus_NextMonth",
    "Post_Bonus_Month_Flag",
    "Avg_Bonus_Uplift",
    "Promo_Uplift_Lag1",
    "Promo_Uplift_Lag2",
    "Promo_Uplift_6M",
    "Last_Bonus_Demand",

    "Supply_Constraint_Flag",
    "Supply_Constraint_Lag1",
    "Supply_Constraint_Lag2",
    "Supply_Shock",

    "Available_Primary_Inventory_Qty",
    "Distributor_Inventory_Qty",
    "Net_Available_Stock",
    "Inventory_Pressure",
    "Stock_Cover_Months",
    "Primary_Stock_Cover",
    "Distributor_Stock_Cover",
    "Demand_to_Stock_Ratio",

    "Lag1", "Lag2", "Lag3", "Lag6", "Lag12",
    "Rolling3M_Mean",
    "Rolling6M_Mean",
    "Rolling3M_Std",
    "Momentum",

    "Month_Sin", "Month_Cos",
    "Quarter_Sin", "Quarter_Cos",
    "ZeroRate_6M",
] + EXTRA_SIGNAL_FEATURES

GRU_LONG_SEQ_FEATURES = list(dict.fromkeys(GRU_LONG_SEQ_FEATURES))

GRU_LONG_STATIC_FEATURES = [
    "ABC_Class",
    "SKU_Mean_Demand",
    "SKU_ZeroRate",
    "SKU_CV",
    "Recurring_Bonus_SKU",
    "Bonus_Cycle_Length",
    "Avg_Bonus_Uplift",
    "Bonus_Corr",
    "Bonus_Frequency_Profile",
    "Bonus_Uplift_Ratio_Profile",
    "Bonus_Demand_Share",
    "Bonus_Month_Count",
]

GRU_LONG_TARGET_COL = ACTUAL_TARGET_COL
GRU_LONG_BASELINE_COL = BASELINE_COL
GRU_LONG_RESIDUAL_COL = MODEL_TARGET_COL
GRU_LONG_RESIDUAL_LOG_COL = "Residual_Target_Log"

In [ ]:
def gru_should_debug(sku_code):
    sku_code = str(sku_code)
    return bool(GRU_DEBUG) and (len(GRU_DEBUG_SKUS) == 0 or sku_code in GRU_DEBUG_SKUS)


def gru_debug_print(*args):
    if GRU_DEBUG:
        print(*args)


def gru_debug_df(title, df, max_rows=5):
    if not GRU_DEBUG:
        return
    print(f"\n[GRU DEBUG DF] {title}")
    if df is None:
        print("None")
        return
    if len(df) == 0:
        print("EMPTY")
        return
    print(df.head(max_rows))
    print("shape:", df.shape)

In [ ]:
# 6) DATASET
class LongGRUSequenceDataset(Dataset):
    def __init__(self, X_seq, X_static, X_item, y_res_log, sample_w):
        self.X_seq = X_seq
        self.X_static = X_static
        self.X_item = X_item
        self.y_res_log = y_res_log
        self.sample_w = sample_w

    def __len__(self):
        return len(self.y_res_log)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.X_seq[idx], dtype=torch.float32),
            torch.tensor(self.X_static[idx], dtype=torch.float32),
            torch.tensor(self.X_item[idx], dtype=torch.long),
            torch.tensor(self.y_res_log[idx], dtype=torch.float32),
            torch.tensor(self.sample_w[idx], dtype=torch.float32),
        )

# 7) MODEL
class LongGRUResidualForecaster(nn.Module):
    def __init__(
        self,
        num_items,
        seq_input_dim,
        static_input_dim,
        embed_dim=32,
        hidden_size=64,
        num_layers=2,
        dropout=0.25,
    ):
        super().__init__()

        self.item_embedding = nn.Embedding(num_items, embed_dim)

        self.gru = nn.GRU(
            input_size=seq_input_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.seq_fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.static_fc = nn.Sequential(
            nn.Linear(static_input_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.head = nn.Sequential(
            nn.Linear(64 + 32 + embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x_seq, x_static, x_item):
        out, _ = self.gru(x_seq)
        last_hidden = out[:, -1, :]

        seq_repr = self.seq_fc(last_hidden)
        static_repr = self.static_fc(x_static)
        item_repr = self.item_embedding(x_item)

        x = torch.cat([seq_repr, static_repr, item_repr], dim=1)
        pred_res_log = self.head(x).squeeze(1)
        return pred_res_log

# 8) LOSS
class WeightedAsymmetricMAELoss(nn.Module):
    def __init__(self, under_penalty=1.75):
        super().__init__()
        self.under_penalty = under_penalty

    def forward(self, preds, targets, sample_weights):
        err = preds - targets
        abs_err = torch.abs(err)
        penalty = torch.where(err < 0, self.under_penalty, 1.0)
        loss = abs_err * penalty * sample_weights
        return loss.mean()

# 9) SCALERS
@dataclass
class LongGRUScalerBundle:
    seq_scaler: StandardScaler
    static_scaler: StandardScaler

def fit_long_gru_scalers(train_df):
    seq_scaler = StandardScaler()
    static_scaler = StandardScaler()

    seq_scaler.fit(train_df[GRU_LONG_SEQ_FEATURES].fillna(0))
    static_scaler.fit(train_df[GRU_LONG_STATIC_FEATURES].fillna(0))

    return LongGRUScalerBundle(
        seq_scaler=seq_scaler,
        static_scaler=static_scaler
    )

# ============================================================
# 10) BUILD SEQUENCES
# ============================================================
def build_long_gru_sequences(df, item_to_idx, scalers, seq_len=18):
    X_seq, X_static, X_item = [], [], []
    y_res_log, sample_w = [], []
    meta = []

    work = df.copy()
    work = force_itemcode_str(work)

    key_col = "ItemCode_Original" if "ItemCode_Original" in work.columns else "ItemCode"
    work[key_col] = work[key_col].astype(str)

    debug_summary = []

    for item, g in work.groupby(key_col):
        item = str(item)

        if item not in item_to_idx:
            if gru_should_debug(item):
                print(f"[GRU BUILD DEBUG] SKU={item} skipped: not in item_to_idx")
            continue

        g = g.sort_values(["Year", "Month_Number"]).copy().reset_index(drop=True)
        usable_idx = g.index[g[GRU_LONG_RESIDUAL_LOG_COL].notna()].tolist()

        seq_count_for_sku = 0

        if gru_should_debug(item):
            debug_summary.append({
                "ItemCode": item,
                "Rows": len(g),
                "Usable_Target_Rows": len(usable_idx),
                "Min_YearMonth": f"{int(g['Year'].min())}-{int(g.loc[g['Year'].idxmin(), 'Month_Number']):02d}" if len(g) else None,
                "Max_YearMonth": f"{int(g['Year'].max())}-{int(g.loc[g['Year'].idxmax(), 'Month_Number']):02d}" if len(g) else None,
            })

        for idx in usable_idx:
            start = idx - seq_len + 1
            if start < 0:
                continue

            seq_slice = g.iloc[start:idx + 1].copy()
            if len(seq_slice) != seq_len:
                continue

            seq_vals = seq_slice[GRU_LONG_SEQ_FEATURES].fillna(0).values
            seq_vals = scalers.seq_scaler.transform(seq_vals)

            static_vals = seq_slice.iloc[-1][GRU_LONG_STATIC_FEATURES].fillna(0).values.reshape(1, -1)
            static_vals = scalers.static_scaler.transform(static_vals)[0]

            y_val = float(g.iloc[idx][GRU_LONG_RESIDUAL_LOG_COL])

            abc_class = int(seq_slice.iloc[-1]["ABC_Class"])
            bonus_flag = int(seq_slice.iloc[-1]["Bonus_Flag"])
            supply_flag = int(seq_slice.iloc[-1]["Supply_Constraint_Flag"])
            recurring_flag = int(seq_slice.iloc[-1]["Recurring_Bonus_SKU"])
            expected_bonus_flag = int(seq_slice.iloc[-1]["Expected_Bonus_NextMonth"])

            w = GRU_ABC_WEIGHT_MAP.get(abc_class, 1.0)
            if bonus_flag == 1:
                w *= GRU_PROMO_WEIGHT
            if supply_flag == 1:
                w *= GRU_SUPPLY_WEIGHT
            if recurring_flag == 1:
                w *= GRU_RECURRING_PROMO_WEIGHT
            if expected_bonus_flag == 1:
                w *= GRU_EXPECTED_PROMO_WEIGHT

            X_seq.append(seq_vals.astype(np.float32))
            X_static.append(static_vals.astype(np.float32))
            X_item.append(item_to_idx[item])
            y_res_log.append(np.float32(y_val))
            sample_w.append(np.float32(w))

            seq_count_for_sku += 1

            meta.append({
                "ItemCode": item,
                "ItemCode_Original": item,
                "Year": int(g.iloc[idx]["Year"]),
                "Month_Number": int(g.iloc[idx]["Month_Number"]),
                "ABC_Class": abc_class,
                "Bonus_Flag": bonus_flag,
                "Supply_Constraint_Flag": supply_flag,
                "Recurring_Bonus_SKU": recurring_flag,
                "Expected_Bonus_NextMonth": expected_bonus_flag,
                "Actual": float(g.iloc[idx][GRU_LONG_TARGET_COL]),
                "Residual_Baseline": float(g.iloc[idx][GRU_LONG_BASELINE_COL]),
                "Residual_Target": float(g.iloc[idx][GRU_LONG_RESIDUAL_COL]),
            })

        if gru_should_debug(item):
            print(
                f"[GRU BUILD DEBUG] SKU={item} | rows={len(g)} | usable_idx={len(usable_idx)} | built_sequences={seq_count_for_sku}"
            )

    if GRU_DEBUG and len(debug_summary) > 0:
        debug_df = pd.DataFrame(debug_summary)
        gru_debug_df("GRU build summary sample", debug_df, max_rows=20)

    meta_columns = [
        "ItemCode",
        "ItemCode_Original",
        "Year",
        "Month_Number",
        "ABC_Class",
        "Bonus_Flag",
        "Supply_Constraint_Flag",
        "Recurring_Bonus_SKU",
        "Expected_Bonus_NextMonth",
        "Actual",
        "Residual_Baseline",
        "Residual_Target",
    ]

    meta_df = pd.DataFrame(meta, columns=meta_columns)

    return (
        np.array(X_seq, dtype=np.float32),
        np.array(X_static, dtype=np.float32),
        np.array(X_item, dtype=np.int64),
        np.array(y_res_log, dtype=np.float32),
        np.array(sample_w, dtype=np.float32),
        meta_df,
    )

def filter_sequence_pack_by_period_index(X_seq, X_static, X_item, y, w, meta_df, start_idx, end_idx):
    if meta_df.empty:
        return X_seq[:0], X_static[:0], X_item[:0], y[:0], w[:0], meta_df.copy()

    meta = meta_df.copy()
    meta["Period_Index"] = meta["Year"].astype(int) * 12 + meta["Month_Number"].astype(int)

    mask = meta["Period_Index"].between(start_idx, end_idx)
    idx = np.where(mask.values)[0]

    return (
        X_seq[idx],
        X_static[idx],
        X_item[idx],
        y[idx],
        w[idx],
        meta.iloc[idx].reset_index(drop=True)
    )

# ============================================================
# 11) ARRAY SPLIT
# ============================================================
'''
def split_long_gru_sequence_arrays(X_seq, X_static, X_item, y, w, meta_df):
    train_mask = meta_df["Year"] <= GRU_TRAIN_END_YEAR
    valid_mask = meta_df["Year"] == GRU_VALID_YEAR
    test_mask = meta_df["Year"] == GRU_TEST_YEAR

    def take(mask):
        idx = np.where(mask.values)[0]
        return (
            X_seq[idx],
            X_static[idx],
            X_item[idx],
            y[idx],
            w[idx],
            meta_df.iloc[idx].reset_index(drop=True)
        )

    return take(train_mask), take(valid_mask), take(test_mask)
'''

# ============================================================
# 12) TRAIN / PREDICT
# ============================================================
def train_long_gru_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for x_seq, x_static, x_item, y_res_log, sample_w in loader:
        x_seq = x_seq.to(GRU_DEVICE)
        x_static = x_static.to(GRU_DEVICE)
        x_item = x_item.to(GRU_DEVICE)
        y_res_log = y_res_log.to(GRU_DEVICE)
        sample_w = sample_w.to(GRU_DEVICE)

        optimizer.zero_grad()
        preds = model(x_seq, x_static, x_item)
        loss = criterion(preds, y_res_log, sample_w)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * len(y_res_log)

    return total_loss / len(loader.dataset)

def split_gru_train_valid_before_window(X_seq, X_static, X_item, y, w, meta_df, valid_window):
    if meta_df.empty:
        return None, None

    meta = meta_df.copy()
    meta["Period_Index"] = meta["Year"].astype(int) * 12 + meta["Month_Number"].astype(int)

    trainable_meta = meta[meta["Period_Index"] < valid_window["valid_start_idx"]].copy()

    if trainable_meta.empty:
        return None, None

    unique_periods = (
        trainable_meta[["Period_Index"]]
        .drop_duplicates()
        .sort_values("Period_Index")
        .reset_index(drop=True)
    )

    if len(unique_periods) >= 4:
        valid_periods = unique_periods.tail(4)["Period_Index"].tolist()
        train_mask = meta["Period_Index"].isin(unique_periods.iloc[:-4]["Period_Index"])
        valid_mask = meta["Period_Index"].isin(valid_periods)
    else:
        n = len(meta_df)
        split_idx = int(n * 0.85)
        train_mask = np.zeros(n, dtype=bool)
        valid_mask = np.zeros(n, dtype=bool)
        train_mask[:split_idx] = True
        valid_mask[split_idx:] = True

    def take(mask):
        idx = np.where(mask)[0]
        return (
            X_seq[idx],
            X_static[idx],
            X_item[idx],
            y[idx],
            w[idx],
            meta_df.iloc[idx].reset_index(drop=True)
        )

    return take(train_mask), take(valid_mask)

@torch.no_grad()
def predict_long_gru_residual_log(model, loader):
    model.eval()

    preds_all, y_all = [], []

    for x_seq, x_static, x_item, y_res_log, sample_w in loader:
        x_seq = x_seq.to(GRU_DEVICE)
        x_static = x_static.to(GRU_DEVICE)
        x_item = x_item.to(GRU_DEVICE)

        preds = model(x_seq, x_static, x_item).cpu().numpy()
        preds_all.append(preds)
        y_all.append(y_res_log.numpy())

    preds_all = np.concatenate(preds_all)
    y_all = np.concatenate(y_all)
    return preds_all, y_all

def evaluate_long_gru_on_loader(model, loader, meta_df):
    pred_res_log, true_res_log = predict_long_gru_residual_log(model, loader)

    pred_res_log = np.clip(pred_res_log, -7.0, 7.0)
    true_res_log = np.clip(true_res_log, -7.0, 7.0)

    pred_residual = gru_signed_log_inverse(pred_res_log)
    true_residual = gru_signed_log_inverse(true_res_log)

    out = meta_df.copy()
    out["Pred_Residual_Log"] = pred_res_log
    out["Pred_Residual"] = pred_residual
    out["True_Residual"] = true_residual

    out["Pred"] = out["Residual_Baseline"] + out["Pred_Residual"]
    out["Pred"] = out["Pred"].clip(lower=0)

    out["Error"] = out["Actual"] - out["Pred"]
    out["Abs_Error"] = np.abs(out["Error"])

    metrics = evaluate_all_metrics(out["Actual"].values, out["Pred"].values)
    return metrics, out

def fit_long_gru_model(train_dataset, valid_dataset, valid_meta, num_items, seq_input_dim, static_input_dim):
    train_loader = DataLoader(train_dataset, batch_size=GRU_BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=GRU_BATCH_SIZE, shuffle=False)

    print("GRU_DEVICE:", GRU_DEVICE)
    print("Train batches:", len(train_loader))
    print("Valid batches:", len(valid_loader))

    model = LongGRUResidualForecaster(
        num_items=num_items,
        seq_input_dim=seq_input_dim,
        static_input_dim=static_input_dim,
        embed_dim=GRU_EMBED_DIM,
        hidden_size=GRU_HIDDEN_SIZE,
        num_layers=GRU_NUM_LAYERS,
        dropout=GRU_DROPOUT,
    ).to(GRU_DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=GRU_LR, weight_decay=GRU_WEIGHT_DECAY)
    criterion = WeightedAsymmetricMAELoss(under_penalty=GRU_UNDER_PENALTY)

    best_valid_wmape = float("inf")
    best_state = None
    patience = 8
    wait = 0

    for epoch in range(1, GRU_EPOCHS + 1):
        train_loss = train_long_gru_one_epoch(model, train_loader, optimizer, criterion)
        valid_metrics, _ = evaluate_long_gru_on_loader(model, valid_loader, valid_meta)

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.5f} | "
            f"Valid WMAPE: {valid_metrics['WMAPE']:.4f} | "
            f"Valid Bias: {valid_metrics['Bias']:.4f}"
        )

        if valid_metrics["WMAPE"] < best_valid_wmape:
            best_valid_wmape = valid_metrics["WMAPE"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping triggered.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model


# 13) STANDARDIZE OUTPUT FOR MODEL COMPARISON
def convert_gru_long_output_for_comparison(test_result_df):
    out = force_itemcode_str(test_result_df.copy())

    if "ItemCode_Original" in out.columns:
        out["ItemCode_Original"] = out["ItemCode_Original"].astype(str)
        out["ItemCode"] = out["ItemCode_Original"].astype(str)
    else:
        out["ItemCode"] = out["ItemCode"].astype(str)
        out["ItemCode_Original"] = out["ItemCode"]

    out["Segment"] = "LONG"
    out["Model_Name"] = "GRU"
    out["Actual"] = pd.to_numeric(out["Actual"], errors="coerce")
    out["Pred"] = pd.to_numeric(out["Pred"], errors="coerce").clip(lower=0)
    out["Error"] = out["Actual"] - out["Pred"]
    out["Abs_Error"] = np.abs(out["Error"])

    keep_cols = [
        "ItemCode",
        "ItemCode_Original",
        "Year",
        "Month_Number",
        "Actual",
        "Pred",
        "Error",
        "Abs_Error",
        "Segment",
        "Model_Name"
    ]

    extra_cols = [c for c in out.columns if c not in keep_cols]
    return out[keep_cols + extra_cols].copy()


# 14) LONG GRU HOLDOUT EVALUATION PIPELINE
# 14) LONG GRU HOLDOUT EVALUATION PIPELINE
def run_long_gru_window_evaluation(full_data, valid_window, run_label="GRU_WINDOW"):
    print(f"\n========== LONG GRU EVALUATION PIPELINE → {run_label} ==========")

    train_prep, valid_prep, artifacts_meta = prepare_long_gru_eval_data_for_window(
        full_data=full_data,
        valid_window=valid_window
    )

    if train_prep.empty or valid_prep.empty:
        raise ValueError(f"Prepared LONG GRU data is empty for {run_label}.")

    item_to_idx = make_item_mapping_from_train(train_prep)

    key_col_train = "ItemCode_Original" if "ItemCode_Original" in train_prep.columns else "ItemCode"
    key_col_valid = "ItemCode_Original" if "ItemCode_Original" in valid_prep.columns else "ItemCode"

    train_prep[key_col_train] = train_prep[key_col_train].astype(str)
    valid_prep[key_col_valid] = valid_prep[key_col_valid].astype(str)

    train_prep = train_prep[train_prep[key_col_train].isin(item_to_idx.keys())].copy()
    valid_prep = valid_prep[valid_prep[key_col_valid].isin(item_to_idx.keys())].copy()

    if train_prep.empty:
        raise ValueError(f"GRU train_prep became empty after key filtering for {run_label}.")
    if valid_prep.empty:
        raise ValueError(f"GRU valid_prep became empty after key filtering for {run_label}.")

    scalers = fit_long_gru_scalers(train_prep)

    print("GRU key_col_train:", key_col_train)
    print("GRU key_col_valid:", key_col_valid)
    print("GRU train_prep rows after key filter:", len(train_prep))
    print("GRU valid_prep rows after key filter:", len(valid_prep))
    print("GRU unique train SKUs:", train_prep[key_col_train].nunique())
    print("GRU unique valid SKUs:", valid_prep[key_col_valid].nunique())

    train_seq_source = train_prep.copy()
    valid_seq_source = pd.concat([train_prep, valid_prep], ignore_index=True).sort_values(
        ["ItemCode", "Year", "Month_Number"]
    )

    train_all = build_long_gru_sequences(
        train_seq_source, item_to_idx, scalers, seq_len=GRU_SEQ_LEN
    )
    valid_all = build_long_gru_sequences(
        valid_seq_source, item_to_idx, scalers, seq_len=GRU_SEQ_LEN
    )

    X_seq_train_all, X_static_train_all, X_item_train_all, y_train_all, w_train_all, meta_train_all = train_all

    X_seq_valid_eval, X_static_valid_eval, X_item_valid_eval, y_valid_eval, w_valid_eval, meta_valid_eval = filter_sequence_pack_by_period_index(
        *valid_all,
        start_idx=valid_window["valid_start_idx"],
        end_idx=valid_window["valid_end_idx"]
    )

    print("GRU total train sequences:", len(y_train_all))
    print("GRU total valid eval sequences:", len(y_valid_eval))

    if len(y_train_all) == 0:
        raise ValueError(f"No LONG GRU train sequences were created for {run_label}.")
    if len(y_valid_eval) == 0:
        raise ValueError(f"No LONG GRU eval sequences were created for {run_label}.")

    train_pack, early_valid_pack = split_gru_train_valid_before_window(
        X_seq_train_all,
        X_static_train_all,
        X_item_train_all,
        y_train_all,
        w_train_all,
        meta_train_all,
        valid_window=valid_window
    )

    if train_pack is None or early_valid_pack is None:
        raise ValueError(f"GRU split_gru_train_valid_before_window returned None for {run_label}.")

    X_seq_train, X_static_train, X_item_train, y_train, w_train, meta_train = train_pack
    X_seq_early_valid, X_static_early_valid, X_item_early_valid, y_early_valid, w_early_valid, meta_early_valid = early_valid_pack

    print("GRU final train sequences:", len(y_train))
    print("GRU early valid sequences:", len(y_early_valid))

    if len(y_train) == 0 or len(y_early_valid) == 0:
        raise ValueError(f"No LONG GRU train/early-valid split could be created for {run_label}.")

    train_dataset = LongGRUSequenceDataset(
        X_seq_train, X_static_train, X_item_train, y_train, w_train
    )
    early_valid_dataset = LongGRUSequenceDataset(
        X_seq_early_valid, X_static_early_valid, X_item_early_valid, y_early_valid, w_early_valid
    )
    eval_dataset = LongGRUSequenceDataset(
        X_seq_valid_eval, X_static_valid_eval, X_item_valid_eval, y_valid_eval, w_valid_eval
    )

    model = fit_long_gru_model(
        train_dataset=train_dataset,
        valid_dataset=early_valid_dataset,
        valid_meta=meta_early_valid,
        num_items=len(item_to_idx),
        seq_input_dim=X_seq_train.shape[2],
        static_input_dim=X_static_train.shape[1],
    )

    eval_loader = DataLoader(eval_dataset, batch_size=GRU_BATCH_SIZE, shuffle=False)
    eval_metrics, eval_result_df = evaluate_long_gru_on_loader(model, eval_loader, meta_valid_eval)

    eval_std_df = convert_gru_long_output_for_comparison(eval_result_df)

    artifacts = {
        "model": model,
        "model_type": "GRU",
        "model_name": "GRU",
        "segment": "LONG",
        "seq_features": GRU_LONG_SEQ_FEATURES,
        "static_features": GRU_LONG_STATIC_FEATURES,
        "seq_len": GRU_SEQ_LEN,
        "embed_dim": GRU_EMBED_DIM,
        "hidden_size": GRU_HIDDEN_SIZE,
        "num_layers": GRU_NUM_LAYERS,
        "dropout": GRU_DROPOUT,
        "item_to_idx": item_to_idx,
        "abc_map": artifacts_meta["abc_map"],
        "promo_profile_df": artifacts_meta["promo_profile_df"],
        "sku_profile_df": artifacts_meta["sku_profile_df"],
        "clip_caps": artifacts_meta["clip_caps"],
        "itemcode_categories": artifacts_meta["itemcode_categories"],
        "eval_metrics": eval_metrics,
        "run_label": run_label
    }

    return artifacts, eval_result_df, eval_std_df, eval_metrics, scalers

#### COMPARISON

	•	per-SKU comparison table
	•	champion map: ItemCode -> Best_Model
	•	deployment artifacts per model
	•	inference routing:
	•	lookup SKU in champion map
	•	load chosen model
	•	forecast with that model
	•	save Used_Model

` ============================================================ `
  * LONG MODEL COMPARISON + CHAMPION MAP
  * DUAL-VIEW SELECTION
  * Model A = stable model trained before Holdout12 start evaluated on Holdout12 + Recent4
  * Model B = recent-aware model trained before Recent4 start evaluated on Recent4 only

 * FINAL SCORE:
   0.4 * Holdout12_WMAPE_A +
   0.3 * Recent4_WMAPE_A +
   0.3 * Recent4_WMAPE_B

 NOTE:
 For now this patch uses TREE MODELS only:
   XGBOOST / CATBOOST / LIGHTGBM
 GRU will be patched separately after inference changes.
` ============================================================ `


In [ ]:
RECENT_MONTHS = TIME_WINDOWS.get("recent_months", 4)

LONG_RECENT4_WINDOW = get_long_recent_window(
    time_windows=TIME_WINDOWS,
    recent_months=RECENT_MONTHS
)

LONG_HOLDOUT12_WINDOW = get_long_pre_recent_holdout_window(
    time_windows=TIME_WINDOWS,
    holdout_months=12,
    recent_months=RECENT_MONTHS
)

LONG_MODEL_A_WINDOW = combine_long_model_a_window(
    time_windows=TIME_WINDOWS,
    holdout_months=12,
    recent_months=RECENT_MONTHS
)

print("\nLONG WINDOWS")
print("LONG_HOLDOUT12_WINDOW:", LONG_HOLDOUT12_WINDOW)
print("LONG_RECENT4_WINDOW:", LONG_RECENT4_WINDOW)
print("LONG_MODEL_A_WINDOW:", LONG_MODEL_A_WINDOW)

In [ ]:
def build_sku_behavior_profile(df):
    df = force_itemcode_str(df)
    df = df.copy().sort_values(["ItemCode", "Year", "Month_Number"])

    def safe_autocorr(x, lag):
        x = x.dropna()
        if len(x) <= lag or x.std() == 0:
            return 0
        return x.autocorr(lag=lag)

    profile = (
        df.groupby("ItemCode")
        .agg(
            SKU_Mean_Demand=("Clean_Demand", "mean"),
            SKU_Std_Demand=("Clean_Demand", "std"),
            SKU_Zero_Rate=("Clean_Demand", lambda x: (x == 0).mean()),
            SKU_NonZero_Months=("Clean_Demand", lambda x: (x > 0).sum()),
            SKU_Total_Months=("Clean_Demand", "count"),
            SKU_Max_Demand=("Clean_Demand", "max"),
            Autocorr_Lag3=("Clean_Demand", lambda x: safe_autocorr(x, 3)),
            Autocorr_Lag6=("Clean_Demand", lambda x: safe_autocorr(x, 6)),
            Promo_Rate=("Bonus_Flag", "mean")
        )
        .reset_index()
    )

    profile["SKU_Std_Demand"] = profile["SKU_Std_Demand"].fillna(0)

    profile["SKU_CV"] = np.where(
        profile["SKU_Mean_Demand"] <= 0,
        0,
        profile["SKU_Std_Demand"] / (profile["SKU_Mean_Demand"] + 1)
    )

    profile["Peak_Ratio"] = np.where(
        profile["SKU_Mean_Demand"] <= 0,
        0,
        profile["SKU_Max_Demand"] / (profile["SKU_Mean_Demand"] + 1)
    )

    profile["Intermittent_Flag"] = np.where(
        (profile["SKU_Zero_Rate"] >= 0.40) |
        (profile["SKU_NonZero_Months"] < 8),
        1, 0
    )

    profile["Highly_Volatile_Flag"] = np.where(
        profile["SKU_CV"] >= 1.5,
        1, 0
    )

    profile["Spiky_Flag"] = np.where(
        profile["Peak_Ratio"] >= 2.5,
        1, 0
    )

    profile["Cyclic_Flag"] = np.where(
        (profile["Autocorr_Lag3"] >= 0.50) |
        (profile["Autocorr_Lag6"] >= 0.50),
        1, 0
    )

    return profile


def assign_final_routing(df):
    df = force_itemcode_str(df).copy()

    metric_cols = [
        "Best_Model_Score",
        "Best_Model_Holdout12_WMAPE",
        "Best_Model_Recent4A_WMAPE",
        "Best_Model_Recent4B_WMAPE",
        "SKU_CV",
        "SKU_Zero_Rate",
        "Intermittent_Flag",
        "Highly_Volatile_Flag"
    ]

    for c in metric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # stricter unreliability
    df["Unreliable_Flag"] = np.where(
        (
            (df["Best_Model_Holdout12_WMAPE"] > 90) &
            (df["Best_Model_Recent4B_WMAPE"] > 90)
        ) |
        (
            (df["Best_Model_Score"] > 100) &
            (df["Best_Model_Recent4B_WMAPE"] > 90)
        ),
        1, 0
    )

    # fallback only for extreme cases
    df["Use_Fallback"] = np.where(
        (
            (df["Intermittent_Flag"] == 1) &
            (df["Best_Model_Recent4B_WMAPE"] > 90)
        ) |
        (
            df["Unreliable_Flag"] == 1
        ) |
        (
            (df["Highly_Volatile_Flag"] == 1) &
            (df["Best_Model_Recent4B_WMAPE"] > 110)
        ),
        1, 0
    )

    df["Final_Model"] = np.where(
        df["Use_Fallback"] == 1,
        "FALLBACK",
        df["Best_Model"]
    )

    df["Fallback_Type"] = np.select(
        [
            (df["Intermittent_Flag"] == 1) & (df["Use_Fallback"] == 1),
            (df["Highly_Volatile_Flag"] == 1) & (df["Use_Fallback"] == 1),
            (df["Unreliable_Flag"] == 1) & (df["Use_Fallback"] == 1)
        ],
        [
            "ROLLING3",
            "MAX_LAG1_ROLL3",
            "ROLLING3"
        ],
        default="NONE"
    )

    return df


In [ ]:
# ------------------------------------------------------------
# 1) MODEL A EVALUATION OUTPUTS
# trained before Holdout12 start
# evaluated on Holdout12 + Recent4
# ------------------------------------------------------------
_, xgb_long_model_a_eval_df, xgb_long_model_a_eval_metrics, _ = train_eval_validation_xgb_long(
    full_data=Data,
    feature_cols=long_best_feats,
    best_params=long_best_params,
    valid_window=LONG_MODEL_A_WINDOW
)
xgb_long_model_a_eval_std = standardize_long_model_output(
    df=xgb_long_model_a_eval_df,
    actual_col=ACTUAL_TARGET_COL,
    pred_col="Pred",
    item_col="ItemCode_Original",
    year_col="Year",
    month_col="Month_Number",
    model_name="XGBOOST",
    segment="LONG"
)

_, catboost_long_model_a_eval_df, catboost_long_model_a_eval_metrics, _ = train_eval_validation_catboost_long(
    full_data=Data,
    feature_cols=catboost_long_best_feats,
    best_params=catboost_long_best_params,
    valid_window=LONG_MODEL_A_WINDOW
)
catboost_long_model_a_eval_std = standardize_long_model_output(
    df=catboost_long_model_a_eval_df,
    actual_col=ACTUAL_TARGET_COL,
    pred_col="Pred",
    item_col="ItemCode_Original",
    year_col="Year",
    month_col="Month_Number",
    model_name="CATBOOST",
    segment="LONG"
)

_, lgbm_long_model_a_eval_df, lgbm_long_model_a_eval_metrics, _ = train_eval_validation_lgbm_long(
    full_data=Data,
    feature_cols=lgbm_long_best_feats,
    best_params=lgbm_long_best_params,
    valid_window=LONG_MODEL_A_WINDOW
)
lgbm_long_model_a_eval_std = standardize_long_model_output(
    df=lgbm_long_model_a_eval_df,
    actual_col=ACTUAL_TARGET_COL,
    pred_col="Pred",
    item_col="ItemCode_Original",
    year_col="Year",
    month_col="Month_Number",
    model_name="LIGHTGBM",
    segment="LONG"
)

gru_seed_everything(GRU_SEED)
gru_long_model_a_artifacts, gru_long_model_a_eval_df_raw, gru_long_model_a_eval_std, gru_long_model_a_eval_metrics, gru_long_model_a_scalers = run_long_gru_window_evaluation(
    full_data=Data,
    valid_window=LONG_MODEL_A_WINDOW,
    run_label="LONG_MODEL_A"
)

print("\n===== LONG GRU MODEL A EVALUATION METRICS =====")
print(gru_long_model_a_eval_metrics)

In [ ]:
# ------------------------------------------------------------
# 2) MODEL B EVALUATION OUTPUTS
# trained before Recent4 start
# evaluated on Recent4 only
# ------------------------------------------------------------
_, xgb_long_model_b_recent_df, xgb_long_model_b_recent_metrics, _ = train_eval_validation_xgb_long(
    full_data=Data,
    feature_cols=long_best_feats,
    best_params=long_best_params,
    valid_window=LONG_RECENT4_WINDOW
)
xgb_long_model_b_recent_std = standardize_long_model_output(
    df=xgb_long_model_b_recent_df,
    actual_col=ACTUAL_TARGET_COL,
    pred_col="Pred",
    item_col="ItemCode_Original",
    year_col="Year",
    month_col="Month_Number",
    model_name="XGBOOST",
    segment="LONG"
)

_, catboost_long_model_b_recent_df, catboost_long_model_b_recent_metrics, _ = train_eval_validation_catboost_long(
    full_data=Data,
    feature_cols=catboost_long_best_feats,
    best_params=catboost_long_best_params,
    valid_window=LONG_RECENT4_WINDOW
)
catboost_long_model_b_recent_std = standardize_long_model_output(
    df=catboost_long_model_b_recent_df,
    actual_col=ACTUAL_TARGET_COL,
    pred_col="Pred",
    item_col="ItemCode_Original",
    year_col="Year",
    month_col="Month_Number",
    model_name="CATBOOST",
    segment="LONG"
)

_, lgbm_long_model_b_recent_df, lgbm_long_model_b_recent_metrics, _ = train_eval_validation_lgbm_long(
    full_data=Data,
    feature_cols=lgbm_long_best_feats,
    best_params=lgbm_long_best_params,
    valid_window=LONG_RECENT4_WINDOW
)
lgbm_long_model_b_recent_std = standardize_long_model_output(
    df=lgbm_long_model_b_recent_df,
    actual_col=ACTUAL_TARGET_COL,
    pred_col="Pred",
    item_col="ItemCode_Original",
    year_col="Year",
    month_col="Month_Number",
    model_name="LIGHTGBM",
    segment="LONG"
)

gru_seed_everything(GRU_SEED)
gru_long_model_b_artifacts, gru_long_model_b_recent_df_raw, gru_long_model_b_recent_std, gru_long_model_b_recent_metrics, gru_long_model_b_scalers = run_long_gru_window_evaluation(
    full_data=Data,
    valid_window=LONG_RECENT4_WINDOW,
    run_label="LONG_MODEL_B_RECENT4"
)

print("\n===== LONG GRU MODEL B RECENT4 EVALUATION METRICS =====")
print(gru_long_model_b_recent_metrics)

In [ ]:
# ------------------------------------------------------------
# 3) COMBINE MODEL A OUTPUTS
# ------------------------------------------------------------
long_model_a_eval_df = pd.concat(
    [
        xgb_long_model_a_eval_std.copy(),
        catboost_long_model_a_eval_std.copy(),
        lgbm_long_model_a_eval_std.copy(),
        gru_long_model_a_eval_std.copy()
    ],
    ignore_index=True
)

for c in ["ItemCode", "ItemCode_Original", "Model_Name"]:
    long_model_a_eval_df[c] = long_model_a_eval_df[c].astype(str)

for c in ["Actual", "Pred", "Abs_Error", "Year", "Month_Number"]:
    long_model_a_eval_df[c] = pd.to_numeric(long_model_a_eval_df[c], errors="coerce")

long_model_a_eval_df = long_model_a_eval_df.dropna(
    subset=["ItemCode", "Model_Name", "Actual", "Pred", "Abs_Error", "Year", "Month_Number"]
).copy()

long_model_a_eval_df["Period_Index"] = (
    long_model_a_eval_df["Year"].astype(int) * 12 +
    long_model_a_eval_df["Month_Number"].astype(int)
)

holdout12_periods = set(range(
    int(LONG_HOLDOUT12_WINDOW["valid_start_idx"]),
    int(LONG_HOLDOUT12_WINDOW["valid_end_idx"]) + 1
))

recent4_periods = set(range(
    int(LONG_RECENT4_WINDOW["valid_start_idx"]),
    int(LONG_RECENT4_WINDOW["valid_end_idx"]) + 1
))

long_model_a_holdout12_df = long_model_a_eval_df[
    long_model_a_eval_df["Period_Index"].isin(holdout12_periods)
].copy()

long_model_a_recent4_df = long_model_a_eval_df[
    long_model_a_eval_df["Period_Index"].isin(recent4_periods)
].copy()

print("\nMODEL A row counts")
print("Holdout12 rows:", len(long_model_a_holdout12_df))
print("Recent4 rows:", len(long_model_a_recent4_df))

# ✅ ADD HERE
print("ABC_Class in long_model_a_holdout12_df:", "ABC_Class" in long_model_a_holdout12_df.columns)
print("ABC_Class in long_model_a_recent4_df:", "ABC_Class" in long_model_a_recent4_df.columns)

# ------------------------------------------------------------
# 4) COMBINE MODEL B OUTPUTS
# ------------------------------------------------------------
long_model_b_recent_df = pd.concat(
    [
        xgb_long_model_b_recent_std.copy(),
        catboost_long_model_b_recent_std.copy(),
        lgbm_long_model_b_recent_std.copy(),
        gru_long_model_b_recent_std.copy()
    ],
    ignore_index=True
)

for c in ["ItemCode", "ItemCode_Original", "Model_Name"]:
    long_model_b_recent_df[c] = long_model_b_recent_df[c].astype(str)

for c in ["Actual", "Pred", "Abs_Error", "Year", "Month_Number"]:
    long_model_b_recent_df[c] = pd.to_numeric(long_model_b_recent_df[c], errors="coerce")

long_model_b_recent_df = long_model_b_recent_df.dropna(
    subset=["ItemCode", "Model_Name", "Actual", "Pred", "Abs_Error", "Year", "Month_Number"]
).copy()

print("Model B Recent4 rows:", len(long_model_b_recent_df))
# ✅ ADD HERE
print("ABC_Class in long_model_b_recent_df:", "ABC_Class" in long_model_b_recent_df.columns)

In [ ]:
# ============================================================
# LONG MODEL PERFORMANCE REPORTS
# ============================================================
for model_name in ["XGBOOST", "CATBOOST", "LIGHTGBM", "GRU"]:
    df_h12 = long_model_a_holdout12_df[long_model_a_holdout12_df["Model_Name"] == model_name].copy()
    if not df_h12.empty:
        print_model_eval_report(
            df_h12,
            title=f"{model_name} LONG → HOLDOU12"
        )

for model_name in ["XGBOOST", "CATBOOST", "LIGHTGBM", "GRU"]:
    df_r4a = long_model_a_recent4_df[long_model_a_recent4_df["Model_Name"] == model_name].copy()
    if not df_r4a.empty:
        print_model_eval_report(
            df_r4a,
            title=f"{model_name} LONG → RECENT4A"
        )

for model_name in ["XGBOOST", "CATBOOST", "LIGHTGBM", "GRU"]:
    df_r4b = long_model_b_recent_df[long_model_b_recent_df["Model_Name"] == model_name].copy()
    if not df_r4b.empty:
        print_model_eval_report(
            df_r4b,
            title=f"{model_name} LONG → RECENT4B"
        )

long_holdout12_model_table = build_model_summary_table(long_model_a_holdout12_df)
long_recent4a_model_table = build_model_summary_table(long_model_a_recent4_df)
long_recent4b_model_table = build_model_summary_table(long_model_b_recent_df)

print("\n===== LONG HOLDOU12 MODEL TABLE =====")
print(long_holdout12_model_table)

print("\n===== LONG RECENT4A MODEL TABLE =====")
print(long_recent4a_model_table)

print("\n===== LONG RECENT4B MODEL TABLE =====")
print(long_recent4b_model_table)

In [ ]:
# ============================================================
# ADD BEHAVIOR_TYPE TO LONG EVALUATION ROWS
# ============================================================

behavior_lookup_df = (
    Data_long.copy()
    .sort_values(["ItemCode", "Year", "Month_Number"])
    .groupby("ItemCode")
    .tail(1)[["ItemCode", "Behavior_Type"]]
)

behavior_lookup_df["ItemCode"] = behavior_lookup_df["ItemCode"].astype(str)

for df_name in [
    "long_model_a_holdout12_df",
    "long_model_a_recent4_df",
    "long_model_b_recent_df"
]:
    temp_df = globals()[df_name].copy()
    temp_df["ItemCode"] = temp_df["ItemCode"].astype(str)

    if "Behavior_Type" in temp_df.columns:
        temp_df = temp_df.drop(columns=["Behavior_Type"], errors="ignore")

    temp_df = temp_df.merge(behavior_lookup_df, on="ItemCode", how="left")
    temp_df["Behavior_Type"] = temp_df["Behavior_Type"].fillna("UNKNOWN")

    globals()[df_name] = temp_df

# ============================================================
# BEHAVIOR-LEVEL MODEL PERFORMANCE
# ============================================================

def build_behavior_model_table(df, label):
    out = (
        df.groupby(["Behavior_Type", "Model_Name"], as_index=False)
        .agg(
            Actual_Sum=("Actual", "sum"),
            Pred_Sum=("Pred", "sum"),
            Abs_Error_Sum=("Abs_Error", "sum"),
            MAE=("Abs_Error", "mean"),
            SKU_Count=("ItemCode", "nunique")
        )
    )

    out[f"{label}_WMAPE"] = np.where(
        out["Actual_Sum"] > 0,
        out["Abs_Error_Sum"] / out["Actual_Sum"] * 100,
        np.nan
    )

    out[f"{label}_Bias"] = np.where(
        out["Actual_Sum"] > 0,
        (out["Pred_Sum"] - out["Actual_Sum"]) / out["Actual_Sum"] * 100,
        np.nan
    )

    return out[[
        "Behavior_Type", "Model_Name", "SKU_Count",
        f"{label}_WMAPE", f"{label}_Bias"
    ]]


behavior_h12 = build_behavior_model_table(long_model_a_holdout12_df, "H12")
behavior_r4a = build_behavior_model_table(long_model_a_recent4_df, "R4A")
behavior_r4b = build_behavior_model_table(long_model_b_recent_df, "R4B")

behavior_model_summary = (
    behavior_h12
    .merge(behavior_r4a, on=["Behavior_Type", "Model_Name"], how="outer")
    .merge(behavior_r4b, on=["Behavior_Type", "Model_Name"], how="outer")
)

for c in ["H12_WMAPE", "R4A_WMAPE", "R4B_WMAPE"]:
    behavior_model_summary[c] = behavior_model_summary[c].fillna(behavior_model_summary[c].median())

for c in ["H12_Bias", "R4A_Bias", "R4B_Bias"]:
    behavior_model_summary[c] = behavior_model_summary[c].fillna(0)

behavior_model_summary["Behavior_Champion_Score"] = (
    0.30 * behavior_model_summary["H12_WMAPE"] +
    0.30 * behavior_model_summary["R4A_WMAPE"] +
    0.40 * behavior_model_summary["R4B_WMAPE"]
)

behavior_model_summary["Behavior_Model_Unreliable"] = np.where(
    (
        (behavior_model_summary["R4B_WMAPE"] > 100) |
        (np.abs(behavior_model_summary["R4B_Bias"]) > 45)
    ),
    1, 0
)

behavior_champion_df = (
    behavior_model_summary
    .sort_values([
        "Behavior_Type",
        "Behavior_Model_Unreliable",
        "Behavior_Champion_Score",
        "R4B_WMAPE"
    ])
    .groupby("Behavior_Type")
    .head(1)
    .rename(columns={
        "Model_Name": "Behavior_Best_Model",
        "Behavior_Champion_Score": "Behavior_Best_Model_Score",
        "R4B_WMAPE": "Behavior_Best_Model_Recent4B_WMAPE"
    })
    [["Behavior_Type", "Behavior_Best_Model", "Behavior_Best_Model_Score", "Behavior_Best_Model_Recent4B_WMAPE"]]
)

print("\n===== BEHAVIOR CHAMPION TABLE =====")
print(behavior_champion_df)

In [ ]:
# ------------------------------------------------------------
# 5) SKU-MODEL SUMMARY: HOLDOU12 FROM MODEL A
# ------------------------------------------------------------
long_sku_model_holdout12_summary = (
    long_model_a_holdout12_df
    .groupby(["ItemCode", "Model_Name"], as_index=False)
    .agg(
        Holdout12_Months=("Month_Number", "count"),
        Holdout12_Actual_Sum=("Actual", "sum"),
        Holdout12_Pred_Sum=("Pred", "sum"),
        Holdout12_MAE=("Abs_Error", "mean"),
        Holdout12_Total_Abs_Error=("Abs_Error", "sum")
    )
)

long_sku_model_holdout12_summary["Holdout12_WMAPE"] = np.where(
    long_sku_model_holdout12_summary["Holdout12_Actual_Sum"] > 0,
    long_sku_model_holdout12_summary["Holdout12_Total_Abs_Error"] /
    long_sku_model_holdout12_summary["Holdout12_Actual_Sum"] * 100,
    np.nan
)

long_sku_model_holdout12_summary["Holdout12_Bias"] = np.where(
    long_sku_model_holdout12_summary["Holdout12_Actual_Sum"] > 0,
    (long_sku_model_holdout12_summary["Holdout12_Pred_Sum"] -
     long_sku_model_holdout12_summary["Holdout12_Actual_Sum"]) /
    long_sku_model_holdout12_summary["Holdout12_Actual_Sum"] * 100,
    np.nan
)


# ------------------------------------------------------------
# 6) SKU-MODEL SUMMARY: RECENT4 FROM MODEL A
# ------------------------------------------------------------
long_sku_model_recent4_a_summary = (
    long_model_a_recent4_df
    .groupby(["ItemCode", "Model_Name"], as_index=False)
    .agg(
        Recent4A_Months=("Month_Number", "count"),
        Recent4A_Actual_Sum=("Actual", "sum"),
        Recent4A_Pred_Sum=("Pred", "sum"),
        Recent4A_MAE=("Abs_Error", "mean"),
        Recent4A_Total_Abs_Error=("Abs_Error", "sum")
    )
)

long_sku_model_recent4_a_summary["Recent4A_WMAPE"] = np.where(
    long_sku_model_recent4_a_summary["Recent4A_Actual_Sum"] > 0,
    long_sku_model_recent4_a_summary["Recent4A_Total_Abs_Error"] /
    long_sku_model_recent4_a_summary["Recent4A_Actual_Sum"] * 100,
    np.nan
)

long_sku_model_recent4_a_summary["Recent4A_Bias"] = np.where(
    long_sku_model_recent4_a_summary["Recent4A_Actual_Sum"] > 0,
    (long_sku_model_recent4_a_summary["Recent4A_Pred_Sum"] -
     long_sku_model_recent4_a_summary["Recent4A_Actual_Sum"]) /
    long_sku_model_recent4_a_summary["Recent4A_Actual_Sum"] * 100,
    np.nan
)


# ------------------------------------------------------------
# 7) SKU-MODEL SUMMARY: RECENT4 FROM MODEL B
# ------------------------------------------------------------
long_sku_model_recent4_b_summary = (
    long_model_b_recent_df
    .groupby(["ItemCode", "Model_Name"], as_index=False)
    .agg(
        Recent4B_Months=("Month_Number", "count"),
        Recent4B_Actual_Sum=("Actual", "sum"),
        Recent4B_Pred_Sum=("Pred", "sum"),
        Recent4B_MAE=("Abs_Error", "mean"),
        Recent4B_Total_Abs_Error=("Abs_Error", "sum")
    )
)

long_sku_model_recent4_b_summary["Recent4B_WMAPE"] = np.where(
    long_sku_model_recent4_b_summary["Recent4B_Actual_Sum"] > 0,
    long_sku_model_recent4_b_summary["Recent4B_Total_Abs_Error"] /
    long_sku_model_recent4_b_summary["Recent4B_Actual_Sum"] * 100,
    np.nan
)

long_sku_model_recent4_b_summary["Recent4B_Bias"] = np.where(
    long_sku_model_recent4_b_summary["Recent4B_Actual_Sum"] > 0,
    (long_sku_model_recent4_b_summary["Recent4B_Pred_Sum"] -
     long_sku_model_recent4_b_summary["Recent4B_Actual_Sum"]) /
    long_sku_model_recent4_b_summary["Recent4B_Actual_Sum"] * 100,
    np.nan
)

In [ ]:
# ------------------------------------------------------------
# 8) MERGE ALL THREE VIEWS
# ------------------------------------------------------------
long_sku_model_summary = (
    long_sku_model_holdout12_summary
    .merge(long_sku_model_recent4_a_summary, on=["ItemCode", "Model_Name"], how="left")
    .merge(long_sku_model_recent4_b_summary, on=["ItemCode", "Model_Name"], how="left")
)

fill_zero_cols = [
    "Recent4A_Months", "Recent4A_Actual_Sum", "Recent4A_Pred_Sum", "Recent4A_Total_Abs_Error",
    "Recent4B_Months", "Recent4B_Actual_Sum", "Recent4B_Pred_Sum", "Recent4B_Total_Abs_Error"
]
for c in fill_zero_cols:
    long_sku_model_summary[c] = long_sku_model_summary[c].fillna(0)

long_sku_model_summary["Recent4A_MAE"] = long_sku_model_summary["Recent4A_MAE"].fillna(
    long_sku_model_summary["Holdout12_MAE"]
)
long_sku_model_summary["Recent4A_WMAPE"] = long_sku_model_summary["Recent4A_WMAPE"].fillna(
    long_sku_model_summary["Holdout12_WMAPE"]
)
long_sku_model_summary["Recent4A_Bias"] = long_sku_model_summary["Recent4A_Bias"].fillna(
    long_sku_model_summary["Holdout12_Bias"]
)

long_sku_model_summary["Recent4B_MAE"] = long_sku_model_summary["Recent4B_MAE"].fillna(
    long_sku_model_summary["Recent4A_MAE"]
)
long_sku_model_summary["Recent4B_WMAPE"] = long_sku_model_summary["Recent4B_WMAPE"].fillna(
    long_sku_model_summary["Recent4A_WMAPE"]
)
long_sku_model_summary["Recent4B_Bias"] = long_sku_model_summary["Recent4B_Bias"].fillna(
    long_sku_model_summary["Recent4A_Bias"]
)

long_sku_model_summary["Champion_Score"] = (
    0.30 * long_sku_model_summary["Holdout12_WMAPE"] +
    0.25 * long_sku_model_summary["Recent4A_WMAPE"] +
    0.25 * long_sku_model_summary["Recent4B_WMAPE"] +
    0.20 * np.abs(long_sku_model_summary["Recent4B_Bias"])
)

long_sku_model_summary.loc[
    long_sku_model_summary["Recent4B_Bias"] > 20,
    "Champion_Score"
] += 5.0

In [ ]:
# ============================================================
# ADD LATEST SKU BEHAVIOR TO SKU-MODEL SUMMARY
# ============================================================

latest_behavior_df = (
    Data_long.copy()
    .sort_values(["ItemCode", "Year", "Month_Number"])
    .groupby("ItemCode")
    .tail(1)[["ItemCode", "Behavior_Type"]]
)

latest_behavior_df["ItemCode"] = latest_behavior_df["ItemCode"].astype(str)
long_sku_model_summary["ItemCode"] = long_sku_model_summary["ItemCode"].astype(str)

long_sku_model_summary = long_sku_model_summary.drop(columns=["Behavior_Type"], errors="ignore")

long_sku_model_summary = long_sku_model_summary.merge(
    latest_behavior_df,
    on="ItemCode",
    how="left"
)

long_sku_model_summary["Behavior_Type"] = long_sku_model_summary["Behavior_Type"].fillna("UNKNOWN")

long_sku_model_summary = long_sku_model_summary.merge(
    behavior_champion_df,
    on="Behavior_Type",
    how="left"
)

long_sku_model_summary["Behavior_Best_Model"] = (
    long_sku_model_summary["Behavior_Best_Model"].fillna("UNKNOWN")
)

In [ ]:
# ============================================================
# RELIABILITY-FIRST + BEHAVIOR-AWARE MODEL PICKING
# ============================================================

long_sku_model_summary["Champion_Score_Adjusted"] = long_sku_model_summary["Champion_Score"].copy()

# General unreliability
long_sku_model_summary["Model_Unreliable"] = np.where(
    (
        (long_sku_model_summary["Holdout12_WMAPE"] > 70) &
        (long_sku_model_summary["Recent4B_WMAPE"] > 80)
    ) |
    (
        (long_sku_model_summary["Recent4A_WMAPE"] > 100) |
        (long_sku_model_summary["Recent4B_WMAPE"] > 100)
    ) |
    (
        np.abs(long_sku_model_summary["Recent4B_Bias"]) > 35
    ),
    1, 0
)

# GRU penalty only when unstable
gru_mask = long_sku_model_summary["Model_Name"] == "GRU"

long_sku_model_summary.loc[
    gru_mask & (long_sku_model_summary["Recent4B_WMAPE"] > 35),
    "Champion_Score_Adjusted"
] += 3.0

long_sku_model_summary.loc[
    gru_mask & (long_sku_model_summary["Recent4A_WMAPE"] > 35),
    "Champion_Score_Adjusted"
] += 2.0

long_sku_model_summary.loc[
    gru_mask & (np.abs(long_sku_model_summary["Recent4B_Bias"]) > 20),
    "Champion_Score_Adjusted"
] += 2.0

long_sku_model_summary.loc[
    gru_mask & (np.abs(long_sku_model_summary["Recent4A_Bias"]) > 20),
    "Champion_Score_Adjusted"
] += 1.0

long_sku_model_summary.loc[
    gru_mask & (long_sku_model_summary["Model_Unreliable"] == 1),
    "Champion_Score_Adjusted"
] += 5.0

# Behavior champion reward
long_sku_model_summary.loc[
    long_sku_model_summary["Model_Name"] == long_sku_model_summary["Behavior_Best_Model"],
    "Champion_Score_Adjusted"
] -= 1.5

# Penalize non-behavior champion if recent performance is weak
long_sku_model_summary.loc[
    (
        long_sku_model_summary["Model_Name"] != long_sku_model_summary["Behavior_Best_Model"]
    ) &
    (
        long_sku_model_summary["Recent4B_WMAPE"] > 45
    ),
    "Champion_Score_Adjusted"
] += 2.0

# Behavior-specific penalties
long_sku_model_summary.loc[
    (
        long_sku_model_summary["Behavior_Type"].isin([
            "PROMO_FREQUENT_WEAK",
            "PROMO_RECURRING_SPECIAL_SPIKE"
        ])
    ) &
    (
        np.abs(long_sku_model_summary["Recent4B_Bias"]) > 30
    ),
    "Champion_Score_Adjusted"
] += 2.0

long_sku_model_summary.loc[
    (
        long_sku_model_summary["Behavior_Type"].isin([
            "RECENT_SPIKE",
            "RECENT_DROP_REBOUND_RISK",
            "SUPPRESSED_DEMAND"
        ])
    ) &
    (
        long_sku_model_summary["Recent4B_WMAPE"] > 60
    ),
    "Champion_Score_Adjusted"
] += 2.0

print("\nLONG SKU MODEL SUMMARY")
print(long_sku_model_summary.head())

In [ ]:
# ------------------------------------------------------------
# 9) CHAMPION MODEL PER SKU
# ------------------------------------------------------------
model_priority_map = {
    "CATBOOST": 1,
    "LIGHTGBM": 2,
    "XGBOOST": 3,
    "GRU": 4
}

long_sku_model_summary["Model_Priority"] = (
    long_sku_model_summary["Model_Name"]
    .map(model_priority_map)
    .fillna(999)
)

print("\n===== LONG COVERAGE BY SKU =====")

all_long_skus = set(
    add_history_length_from_subset(Data.copy(), Data.copy())
    .query("History_Segment == 'LONG'")["ItemCode"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .unique()
)

covered_long_skus = set(
    long_sku_model_summary["ItemCode"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .unique()
)

missing_long_skus = sorted(list(all_long_skus - covered_long_skus))

print("Actual LONG SKUs in data:", len(all_long_skus))
print("Covered LONG SKUs in comparison:", len(covered_long_skus))
print("Missing LONG SKUs from comparison:", len(missing_long_skus))
print("Sample missing LONG SKUs:", missing_long_skus[:30])


# ============================================================
# BEHAVIOR-AWARE CHAMPION MODEL PER SKU
# ============================================================

BEHAVIOR_CLOSE_SCORE_TOLERANCE = 5.0

champion_rows = []

for item_code, g in long_sku_model_summary.groupby("ItemCode", sort=False):
    g = g.copy()

    reliable_pool = g[g["Model_Unreliable"] == 0].copy()
    candidate_pool = reliable_pool if not reliable_pool.empty else g

    # normal best row by SKU
    sku_best_row = (
        candidate_pool
        .sort_values(
            by=[
                "Champion_Score_Adjusted",
                "Recent4B_WMAPE",
                "Recent4A_WMAPE",
                "Holdout12_WMAPE",
                "Holdout12_MAE",
                "Model_Priority"
            ],
            ascending=True
        )
        .iloc[0]
    )

    behavior_type = str(sku_best_row.get("Behavior_Type", "UNKNOWN"))
    behavior_best_model = str(sku_best_row.get("Behavior_Best_Model", "UNKNOWN"))

    # candidate row matching behavior champion
    behavior_candidate = candidate_pool[
        candidate_pool["Model_Name"].astype(str) == behavior_best_model
    ].copy()

    final_row = sku_best_row.copy()
    behavior_routing_applied = 0

    if not behavior_candidate.empty:
        behavior_row = (
            behavior_candidate
            .sort_values(
                by=[
                    "Champion_Score_Adjusted",
                    "Recent4B_WMAPE",
                    "Recent4A_WMAPE",
                    "Holdout12_WMAPE",
                    "Holdout12_MAE",
                    "Model_Priority"
                ],
                ascending=True
            )
            .iloc[0]
        )

        # Use behavior champion only if it is close enough to SKU-specific best
        score_gap = (
            float(behavior_row["Champion_Score_Adjusted"]) -
            float(sku_best_row["Champion_Score_Adjusted"])
        )

        if score_gap <= BEHAVIOR_CLOSE_SCORE_TOLERANCE:
            final_row = behavior_row.copy()
            behavior_routing_applied = 1

    final_row["Behavior_Routing_Applied"] = behavior_routing_applied
    final_row["Behavior_Type"] = behavior_type
    final_row["Behavior_Best_Model"] = behavior_best_model

    champion_rows.append(final_row)

champion_long_map_df = pd.DataFrame(champion_rows).reset_index(drop=True)

champion_long_map_df = champion_long_map_df.rename(columns={
    "Model_Name": "Best_Model",
    "Champion_Score": "Best_Model_Score",
    "Champion_Score_Adjusted": "Best_Model_Score_Adjusted",
    "Holdout12_WMAPE": "Best_Model_Holdout12_WMAPE",
    "Recent4A_WMAPE": "Best_Model_Recent4A_WMAPE",
    "Recent4B_WMAPE": "Best_Model_Recent4B_WMAPE",
    "Holdout12_MAE": "Best_Model_Holdout12_MAE",
    "Recent4A_MAE": "Best_Model_Recent4A_MAE",
    "Recent4B_MAE": "Best_Model_Recent4B_MAE",
    "Holdout12_Bias": "Best_Model_Holdout12_Bias",
    "Recent4A_Bias": "Best_Model_Recent4A_Bias",
    "Recent4B_Bias": "Best_Model_Recent4B_Bias",
    "Holdout12_Months": "Evaluation_Months_Holdout12",
    "Recent4A_Months": "Evaluation_Months_Recent4A",
    "Recent4B_Months": "Evaluation_Months_Recent4B"
})

champion_long_map_df["Segment"] = "LONG"

champion_long_map_df = champion_long_map_df[
    [
        "ItemCode",
        "Segment",
        "Behavior_Type",
        "Behavior_Best_Model",
        "Behavior_Routing_Applied",
        "Best_Model",
        "Best_Model_Score",
        "Best_Model_Holdout12_WMAPE",
        "Best_Model_Recent4A_WMAPE",
        "Best_Model_Recent4B_WMAPE",
        "Best_Model_Holdout12_MAE",
        "Best_Model_Recent4A_MAE",
        "Best_Model_Recent4B_MAE",
        "Best_Model_Holdout12_Bias",
        "Best_Model_Recent4A_Bias",
        "Best_Model_Recent4B_Bias",
        "Evaluation_Months_Holdout12",
        "Evaluation_Months_Recent4A",
        "Evaluation_Months_Recent4B",
        "Holdout12_Actual_Sum",
        "Holdout12_Pred_Sum",
        "Recent4A_Actual_Sum",
        "Recent4A_Pred_Sum",
        "Recent4B_Actual_Sum",
        "Recent4B_Pred_Sum"
    ]
].copy()

print("\nLONG CHAMPION MAP")
print(champion_long_map_df.head())


# ------------------------------------------------------------
# 10) ADD SKU BEHAVIOR + RELIABILITY + FINAL ROUTING
# ------------------------------------------------------------
sku_behavior_profile = build_sku_behavior_profile(Data_long)

sku_behavior_profile["ItemCode"] = sku_behavior_profile["ItemCode"].astype(str)
champion_long_map_df["ItemCode"] = champion_long_map_df["ItemCode"].astype(str)

champion_long_map_df = force_itemcode_str(champion_long_map_df)
sku_behavior_profile = force_itemcode_str(sku_behavior_profile)

champion_long_map_df = champion_long_map_df.merge(
    sku_behavior_profile[
        ["ItemCode", "Intermittent_Flag", "Highly_Volatile_Flag", "SKU_Zero_Rate", "SKU_CV"]
    ],
    on="ItemCode",
    how="left"
)

for c in ["Intermittent_Flag", "Highly_Volatile_Flag", "SKU_Zero_Rate", "SKU_CV"]:
    champion_long_map_df[c] = champion_long_map_df[c].fillna(0)

champion_long_map_df = assign_final_routing(champion_long_map_df)

# -------------------
print("\n===== LONG ROUTING DIAGNOSTICS =====")

print("\nUse_Fallback breakdown:")
print(champion_long_map_df["Use_Fallback"].value_counts(dropna=False))

print("\nFallback by reason:")
print(pd.crosstab(
    champion_long_map_df["Intermittent_Flag"],
    champion_long_map_df["Use_Fallback"],
    rownames=["Intermittent_Flag"],
    colnames=["Use_Fallback"]
))

print(pd.crosstab(
    champion_long_map_df["Unreliable_Flag"],
    champion_long_map_df["Use_Fallback"],
    rownames=["Unreliable_Flag"],
    colnames=["Use_Fallback"]
))

print(pd.crosstab(
    champion_long_map_df["Highly_Volatile_Flag"],
    champion_long_map_df["Fallback_Type"],
    rownames=["Highly_Volatile_Flag"],
    colnames=["Fallback_Type"]
))

print("\nBest_Model vs Final_Model:")
print(pd.crosstab(
    champion_long_map_df["Best_Model"],
    champion_long_map_df["Final_Model"]
))

print("\nFallback_Type counts:")
print(champion_long_map_df["Fallback_Type"].value_counts(dropna=False))

print("\nRows forced to fallback:")
forced_fb = champion_long_map_df[champion_long_map_df["Final_Model"] == "FALLBACK"].copy()
print(forced_fb[[
    "ItemCode", "Best_Model", "Best_Model_Score",
    "Best_Model_Holdout12_WMAPE", "Best_Model_Recent4A_WMAPE", "Best_Model_Recent4B_WMAPE",
    "Intermittent_Flag", "Highly_Volatile_Flag", "Unreliable_Flag", "Fallback_Type"
]].head(30))

champion_long_map_df = champion_long_map_df[
    [
        "ItemCode",
        "Segment",
        "Behavior_Type",
        "Behavior_Best_Model",
        "Behavior_Routing_Applied",
        "Best_Model",
        "Final_Model",
        "Fallback_Type",
        "Use_Fallback",
        "Unreliable_Flag",
        "Intermittent_Flag",
        "Highly_Volatile_Flag",
        "Best_Model_Score",
        "Best_Model_Holdout12_WMAPE",
        "Best_Model_Recent4A_WMAPE",
        "Best_Model_Recent4B_WMAPE",
        "Best_Model_Holdout12_MAE",
        "Best_Model_Recent4A_MAE",
        "Best_Model_Recent4B_MAE",
        "Best_Model_Holdout12_Bias",
        "Best_Model_Recent4A_Bias",
        "Best_Model_Recent4B_Bias",
        "Evaluation_Months_Holdout12",
        "Evaluation_Months_Recent4A",
        "Evaluation_Months_Recent4B",
        "Holdout12_Actual_Sum",
        "Holdout12_Pred_Sum",
        "Recent4A_Actual_Sum",
        "Recent4A_Pred_Sum",
        "Recent4B_Actual_Sum",
        "Recent4B_Pred_Sum",
        "SKU_Zero_Rate",
        "SKU_CV"
    ]
].copy()

print("\nLONG CHAMPION MAP WITH ROUTING")
print(champion_long_map_df.head())

# -------------------

# ------------------------------------------------------------
# 11) MODEL WIN COUNTS
# ------------------------------------------------------------
long_model_win_counts = (
    champion_long_map_df["Final_Model"]
    .value_counts(dropna=False)
    .reset_index()
)
long_model_win_counts.columns = ["Final_Model", "SKU_Count"]

print("\nLONG MODEL WIN COUNTS")
print(long_model_win_counts)


# ------------------------------------------------------------
# 12) MERGE CHAMPION BACK TO MODEL A ROW LEVEL
# ------------------------------------------------------------
long_model_a_eval_df = force_itemcode_str(long_model_a_eval_df)
champion_long_map_df = force_itemcode_str(champion_long_map_df)

long_eval_with_champion_df = long_model_a_eval_df.merge(
    champion_long_map_df[
        [
            "ItemCode",
            "Best_Model",
            "Best_Model_Score",
            "Intermittent_Flag",
            "Highly_Volatile_Flag",
            "Unreliable_Flag",
            "Use_Fallback",
            "Final_Model",
            "Fallback_Type"
        ]
    ],
    on="ItemCode",
    how="left"
)

long_eval_with_champion_df["Is_Champion_Model"] = (
    long_eval_with_champion_df["Model_Name"] == long_eval_with_champion_df["Best_Model"]
).astype(int)


# ------------------------------------------------------------
# 13) OVERALL MODEL SUMMARY
# ------------------------------------------------------------
overall_holdout12_summary = (
    long_model_a_holdout12_df
    .groupby("Model_Name", as_index=False)
    .agg(
        Holdout12_Actual_Sum=("Actual", "sum"),
        Holdout12_Pred_Sum=("Pred", "sum"),
        Holdout12_Total_Abs_Error=("Abs_Error", "sum"),
        Holdout12_MAE=("Abs_Error", "mean")
    )
)

overall_holdout12_summary["Holdout12_WMAPE"] = np.where(
    overall_holdout12_summary["Holdout12_Actual_Sum"] > 0,
    overall_holdout12_summary["Holdout12_Total_Abs_Error"] /
    overall_holdout12_summary["Holdout12_Actual_Sum"] * 100,
    np.nan
)

overall_recent4a_summary = (
    long_model_a_recent4_df
    .groupby("Model_Name", as_index=False)
    .agg(
        Recent4A_Actual_Sum=("Actual", "sum"),
        Recent4A_Pred_Sum=("Pred", "sum"),
        Recent4A_Total_Abs_Error=("Abs_Error", "sum"),
        Recent4A_MAE=("Abs_Error", "mean")
    )
)

overall_recent4a_summary["Recent4A_WMAPE"] = np.where(
    overall_recent4a_summary["Recent4A_Actual_Sum"] > 0,
    overall_recent4a_summary["Recent4A_Total_Abs_Error"] /
    overall_recent4a_summary["Recent4A_Actual_Sum"] * 100,
    np.nan
)

overall_recent4b_summary = (
    long_model_b_recent_df
    .groupby("Model_Name", as_index=False)
    .agg(
        Recent4B_Actual_Sum=("Actual", "sum"),
        Recent4B_Pred_Sum=("Pred", "sum"),
        Recent4B_Total_Abs_Error=("Abs_Error", "sum"),
        Recent4B_MAE=("Abs_Error", "mean")
    )
)

overall_recent4b_summary["Recent4B_WMAPE"] = np.where(
    overall_recent4b_summary["Recent4B_Actual_Sum"] > 0,
    overall_recent4b_summary["Recent4B_Total_Abs_Error"] /
    overall_recent4b_summary["Recent4B_Actual_Sum"] * 100,
    np.nan
)

overall_model_summary = (
    overall_holdout12_summary
    .merge(overall_recent4a_summary, on="Model_Name", how="left")
    .merge(overall_recent4b_summary, on="Model_Name", how="left")
)

overall_model_summary["Champion_Score"] = (
    0.40 * overall_model_summary["Holdout12_WMAPE"] +
    0.30 * overall_model_summary["Recent4A_WMAPE"] +
    0.30 * overall_model_summary["Recent4B_WMAPE"]
)

print("\nOVERALL MODEL SUMMARY")
print(overall_model_summary)


# ------------------------------------------------------------
# 14) SAVE RESULTS
# ------------------------------------------------------------
with pd.ExcelWriter("long_model_comparison_and_champion_map.xlsx", engine="openpyxl") as writer:
    long_model_a_holdout12_df.to_excel(writer, sheet_name="ModelA_Holdout12_RowLevel", index=False)
    long_model_a_recent4_df.to_excel(writer, sheet_name="ModelA_Recent4_RowLevel", index=False)
    long_model_b_recent_df.to_excel(writer, sheet_name="ModelB_Recent4_RowLevel", index=False)
    gru_long_model_a_eval_df_raw.to_excel(writer, sheet_name="GRU_ModelA_RowLevel", index=False)
    gru_long_model_b_recent_df_raw.to_excel(writer, sheet_name="GRU_ModelB_Recent4_RowLevel", index=False)
    long_sku_model_summary.to_excel(writer, sheet_name="SKU_Model_Summary", index=False)
    champion_long_map_df.to_excel(writer, sheet_name="Champion_Map", index=False)
    long_model_win_counts.to_excel(writer, sheet_name="Model_Win_Counts", index=False)
    overall_model_summary.to_excel(writer, sheet_name="Overall_Model_Summary", index=False)
    long_eval_with_champion_df.to_excel(writer, sheet_name="Eval_With_Routing", index=False)
    long_holdout12_model_table.to_excel(writer, sheet_name="Long_Holdout12_ModelPerf", index=False)
    long_recent4a_model_table.to_excel(writer, sheet_name="Long_Recent4A_ModelPerf", index=False)
    long_recent4b_model_table.to_excel(writer, sheet_name="Long_Recent4B_ModelPerf", index=False)

print("\nSaved: long_model_comparison_and_champion_map.xlsx")

joblib.dump(champion_long_map_df, "champion_long_map_df.pkl")
print("Saved: champion_long_map_df.pkl")

joblib.dump(long_eval_with_champion_df, "long_eval_with_champion_df.pkl")
print("Saved: long_eval_with_champion_df.pkl")

In [ ]:
print("\n===== BEHAVIOR ROUTING DIAGNOSTICS =====")

print("\nBehavior Type Counts:")
print(champion_long_map_df["Behavior_Type"].value_counts(dropna=False))

print("\nBehavior Best Model:")
print(pd.crosstab(
    champion_long_map_df["Behavior_Type"],
    champion_long_map_df["Behavior_Best_Model"]
))

print("\nActual Final Model by Behavior:")
print(pd.crosstab(
    champion_long_map_df["Behavior_Type"],
    champion_long_map_df["Final_Model"]
))

print("\nBehavior routing applied:")
print(champion_long_map_df["Behavior_Routing_Applied"].value_counts(dropna=False))

#### DEPLOYMENT

##### XGB

In [ ]:
def train_single_deployment_model_xgb_long(full_data, feature_cols, best_params):
    print("\n========== XGBOOST LONG DEPLOYMENT MODEL → TRAIN ON ALL COMPLETE DATA ==========")

    deploy_df, prep_artifacts = prepare_long_deploy_frame(full_data)

    assert_features_exist(deploy_df, feature_cols, where="XGB_LONG_DEPLOY_TRAIN")

    model = xgb.XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=42,
        tree_method="hist",
        n_jobs=-1,
        **best_params
    )

    model.fit(
        sanitize(deploy_df[feature_cols]),
        deploy_df[MODEL_TARGET_COL],
        sample_weight=build_train_weights(deploy_df),
        verbose=False
    )

    artifacts = {
        "model": model,
        "segment": "LONG",
        "feature_cols": feature_cols,
        "best_params": best_params,
        "itemcode_categories": prep_artifacts["itemcode_categories"],
        "abc_map": prep_artifacts["abc_map"],
        "clip_caps": prep_artifacts["clip_caps"],
        "promo_profile_df": prep_artifacts["promo_profile_df"],
        "sku_profile_df": prep_artifacts["sku_profile_df"],
        "target_mode": "residual",
        "baseline_col": BASELINE_COL,
        "actual_target_col": ACTUAL_TARGET_COL,
        "model_target_col": MODEL_TARGET_COL,
        "model_name": "XGBOOST"
    }

    return artifacts, deploy_df


In [ ]:
# =========================
# DEPLOYMENT MODELS
# =========================

xgb_long_deploy_artifacts, xgb_long_deploy_train_df = train_single_deployment_model_xgb_long(
    full_data=Data,
    feature_cols=long_best_feats,
    best_params=long_best_params
)

In [ ]:
# =========================
# SAVE ARTIFACTS
# =========================

joblib.dump(xgb_long_deploy_artifacts, "xgb_long_deploy_artifacts_residual.pkl")

##### CAT

In [ ]:
# DEPLOYMENT MODEL
def train_single_deployment_model_catboost_long(full_data, feature_cols, best_params):
    print("\n========== CATBOOST LONG DEPLOYMENT MODEL → TRAIN ON ALL COMPLETE DATA ==========")

    deploy_df, prep_artifacts = prepare_long_deploy_frame(full_data)

    assert_features_exist(deploy_df, feature_cols, where="CATBOOST_LONG_DEPLOY_TRAIN")

    model = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42,
        verbose=0,
        **best_params
    )

    model.fit(
        sanitize(deploy_df[feature_cols]),
        deploy_df[MODEL_TARGET_COL],
        sample_weight=build_train_weights(deploy_df),
        verbose=False
    )

    artifacts = {
        "model": model,
        "segment": "LONG",
        "feature_cols": feature_cols,
        "best_params": best_params,
        "itemcode_categories": prep_artifacts["itemcode_categories"],
        "abc_map": prep_artifacts["abc_map"],
        "clip_caps": prep_artifacts["clip_caps"],
        "promo_profile_df": prep_artifacts["promo_profile_df"],
        "sku_profile_df": prep_artifacts["sku_profile_df"],
        "target_mode": "residual",
        "baseline_col": BASELINE_COL,
        "actual_target_col": ACTUAL_TARGET_COL,
        "model_target_col": MODEL_TARGET_COL,
        "model_name": "CATBOOST"
    }

    return artifacts, deploy_df

In [ ]:
catboost_long_deploy_artifacts, catboost_long_deploy_train_df = train_single_deployment_model_catboost_long(
    full_data=Data,
    feature_cols=catboost_long_best_feats,
    best_params=catboost_long_best_params
)

joblib.dump(catboost_long_deploy_artifacts, "catboost_long_deploy_artifacts_residual.pkl")

##### LGBM

In [ ]:
# DEPLOYMENT MODEL
def train_single_deployment_model_lgbm_long(full_data, feature_cols, best_params):
    print("\n========== LIGHTGBM LONG DEPLOYMENT MODEL → TRAIN ON ALL COMPLETE DATA ==========")

    deploy_df, prep_artifacts = prepare_long_deploy_frame(full_data)

    assert_features_exist(deploy_df, feature_cols, where="LGBM_LONG_DEPLOY_TRAIN")

    model = LGBMRegressor(
        objective="regression",
        metric="rmse",
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
        **best_params
    )

    model.fit(
        sanitize(deploy_df[feature_cols]),
        deploy_df[MODEL_TARGET_COL],
        sample_weight=build_train_weights(deploy_df)
    )

    artifacts = {
        "model": model,
        "segment": "LONG",
        "feature_cols": feature_cols,
        "best_params": best_params,
        "itemcode_categories": prep_artifacts["itemcode_categories"],
        "abc_map": prep_artifacts["abc_map"],
        "clip_caps": prep_artifacts["clip_caps"],
        "promo_profile_df": prep_artifacts["promo_profile_df"],
        "sku_profile_df": prep_artifacts["sku_profile_df"],
        "target_mode": "residual",
        "baseline_col": BASELINE_COL,
        "actual_target_col": ACTUAL_TARGET_COL,
        "model_target_col": MODEL_TARGET_COL,
        "model_name": "LIGHTGBM"
    }

    return artifacts, deploy_df

In [ ]:
lgbm_long_deploy_artifacts, lgbm_long_deploy_train_df = train_single_deployment_model_lgbm_long(
    full_data=Data,
    feature_cols=lgbm_long_best_feats,
    best_params=lgbm_long_best_params
)

joblib.dump(lgbm_long_deploy_artifacts, "lgbm_long_deploy_artifacts_residual.pkl")

##### GRU

In [ ]:
# 16) LONG GRU DEPLOYMENT TRAINING
def train_long_gru_deployment_model(full_data):
    print("\n========== LONG GRU DEPLOYMENT TRAINING ==========")

    deploy_df, deploy_meta = prepare_long_gru_deploy_data(full_data)

    if deploy_df.empty:
        raise ValueError("No LONG rows available for GRU deployment.")

    deploy_df = deploy_df.dropna(subset=[GRU_LONG_TARGET_COL, GRU_LONG_RESIDUAL_COL]).copy()

    item_to_idx = make_item_mapping_from_train(deploy_df)
    scalers = fit_long_gru_scalers(deploy_df)

    key_col = "ItemCode_Original" if "ItemCode_Original" in deploy_df.columns else "ItemCode"
    deploy_df[key_col] = deploy_df[key_col].astype(str)
    deploy_df = deploy_df[deploy_df[key_col].isin(item_to_idx.keys())].copy()

    X_seq, X_static, X_item, y, w, meta = build_long_gru_sequences(
        deploy_df,
        item_to_idx,
        scalers,
        seq_len=GRU_SEQ_LEN
    )

    if len(y) == 0:
        raise ValueError("No LONG GRU deployment sequences created.")

    dataset = LongGRUSequenceDataset(X_seq, X_static, X_item, y, w)
    loader = DataLoader(dataset, batch_size=GRU_BATCH_SIZE, shuffle=True)

    print("GRU_DEVICE:", GRU_DEVICE)
    print("Deployment sequences:", len(y))
    print("Deployment batches:", len(loader))

    model = LongGRUResidualForecaster(
        num_items=len(item_to_idx),
        seq_input_dim=len(GRU_LONG_SEQ_FEATURES),
        static_input_dim=len(GRU_LONG_STATIC_FEATURES),
        embed_dim=GRU_EMBED_DIM,
        hidden_size=GRU_HIDDEN_SIZE,
        num_layers=GRU_NUM_LAYERS,
        dropout=GRU_DROPOUT,
    ).to(GRU_DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=GRU_LR, weight_decay=GRU_WEIGHT_DECAY)
    criterion = WeightedAsymmetricMAELoss(under_penalty=GRU_UNDER_PENALTY)

    best_loss = float("inf")
    best_state = None

    for epoch in range(1, GRU_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        batch_count = 0

        print(f"Starting deploy epoch {epoch}/{GRU_EPOCHS}...")

        for x_seq, x_static, x_item, y_batch, w_batch in loader:
            batch_count += 1

            x_seq = x_seq.to(GRU_DEVICE)
            x_static = x_static.to(GRU_DEVICE)
            x_item = x_item.to(GRU_DEVICE)
            y_batch = y_batch.to(GRU_DEVICE)
            w_batch = w_batch.to(GRU_DEVICE)

            optimizer.zero_grad()
            preds = model(x_seq, x_static, x_item)
            loss = criterion(preds, y_batch, w_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item() * len(y_batch)

            if batch_count % 20 == 0:
                print(f"  deploy epoch {epoch} batch {batch_count}/{len(loader)} loss={loss.item():.5f}")

        epoch_loss = total_loss / len(loader.dataset)
        print(f"Epoch {epoch:02d} | Train Loss: {epoch_loss:.5f}")

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    artifacts = {
        "model": model,
        "model_type": "GRU",
        "model_name": "GRU",
        "segment": "LONG",
        "seq_features": GRU_LONG_SEQ_FEATURES,
        "static_features": GRU_LONG_STATIC_FEATURES,
        "seq_len": GRU_SEQ_LEN,
        "embed_dim": GRU_EMBED_DIM,
        "hidden_size": GRU_HIDDEN_SIZE,
        "num_layers": GRU_NUM_LAYERS,
        "dropout": GRU_DROPOUT,
        "item_to_idx": item_to_idx,
        "abc_map": deploy_meta["abc_map"],
        "promo_profile_df": deploy_meta["promo_profile_df"],
        "sku_profile_df": deploy_meta["sku_profile_df"],
        "clip_caps": deploy_meta["clip_caps"],
        "itemcode_categories": deploy_meta["itemcode_categories"]
    }

    return artifacts, scalers, deploy_df

# 17) SAVE LONG GRU DEPLOY ARTIFACTS
def save_long_gru_deploy_artifacts(artifacts, scalers):
    os.makedirs(GRU_DEPLOY_DIR, exist_ok=True)

    torch.save(
        {
            "model_state_dict": artifacts["model"].state_dict(),
            "model_type": artifacts["model_type"],
            "segment": artifacts["segment"],
            "seq_features": artifacts["seq_features"],
            "static_features": artifacts["static_features"],
            "seq_len": artifacts["seq_len"],
            "embed_dim": artifacts["embed_dim"],
            "hidden_size": artifacts["hidden_size"],
            "num_layers": artifacts["num_layers"],
            "dropout": artifacts["dropout"],
            "item_to_idx": artifacts["item_to_idx"],
            "abc_map": artifacts["abc_map"],
            "clip_caps": artifacts["clip_caps"]
        },
        os.path.join(GRU_DEPLOY_DIR, "gru_long_deploy_model.pt")
    )

    joblib.dump(scalers.seq_scaler, os.path.join(GRU_DEPLOY_DIR, "gru_long_seq_scaler.pkl"))
    joblib.dump(scalers.static_scaler, os.path.join(GRU_DEPLOY_DIR, "gru_long_static_scaler.pkl"))
    joblib.dump(artifacts["promo_profile_df"], os.path.join(GRU_DEPLOY_DIR, "gru_long_promo_profile_df.pkl"))
    joblib.dump(artifacts["sku_profile_df"], os.path.join(GRU_DEPLOY_DIR, "gru_long_sku_profile_df.pkl"))
    joblib.dump(artifacts["item_to_idx"], os.path.join(GRU_DEPLOY_DIR, "gru_long_item_to_idx.pkl"))
    joblib.dump(artifacts["itemcode_categories"], os.path.join(GRU_DEPLOY_DIR, "gru_long_itemcode_categories.pkl"))

    deploy_meta = {
        "model_type": artifacts["model_type"],
        "model_name": artifacts["model_name"],
        "segment": artifacts["segment"],
        "seq_features": artifacts["seq_features"],
        "static_features": artifacts["static_features"],
        "seq_len": artifacts["seq_len"],
        "embed_dim": artifacts["embed_dim"],
        "hidden_size": artifacts["hidden_size"],
        "num_layers": artifacts["num_layers"],
        "dropout": artifacts["dropout"],
    }

    with open(os.path.join(GRU_DEPLOY_DIR, "gru_long_deploy_meta.json"), "w") as f:
        json.dump(deploy_meta, f, indent=2)

    print("\nSaved LONG GRU deploy artifacts:")
    print(f" - {GRU_DEPLOY_DIR}/gru_long_deploy_model.pt")
    print(f" - {GRU_DEPLOY_DIR}/gru_long_seq_scaler.pkl")
    print(f" - {GRU_DEPLOY_DIR}/gru_long_static_scaler.pkl")
    print(f" - {GRU_DEPLOY_DIR}/gru_long_promo_profile_df.pkl")
    print(f" - {GRU_DEPLOY_DIR}/gru_long_deploy_meta.json")
    print(f" - {GRU_DEPLOY_DIR}/gru_long_item_to_idx.pkl")
    print(f" - {GRU_DEPLOY_DIR}/gru_long_itemcode_categories.pkl")


In [ ]:
# ============================================================
# 22) RUN LONG GRU DEPLOY
# ============================================================
gru_seed_everything(GRU_SEED)

gru_long_deploy_artifacts, gru_long_deploy_scalers, gru_long_deploy_df = train_long_gru_deployment_model(
    full_data=Data
)

save_long_gru_deploy_artifacts(
    artifacts=gru_long_deploy_artifacts,
    scalers=gru_long_deploy_scalers
)

print("\nLONG GRU deployment rows:", len(gru_long_deploy_df))

##### Registry

In [ ]:
print("\n===== WINDOW SANITY CHECK =====")
print("LONG_TUNE_WINDOWS:", LONG_TUNE_WINDOWS)
print("LONG_VALID_WINDOW:", LONG_VALID_WINDOW)
print("LONG_HOLDOUT12_WINDOW:", LONG_HOLDOUT12_WINDOW)
print("LONG_RECENT4_WINDOW:", LONG_RECENT4_WINDOW)
print("LONG_MODEL_A_WINDOW:", LONG_MODEL_A_WINDOW)

assert LONG_HOLDOUT12_WINDOW["valid_end_idx"] < LONG_RECENT4_WINDOW["valid_start_idx"], "Holdout12 overlaps Recent4"
assert LONG_MODEL_A_WINDOW["valid_start_idx"] == LONG_HOLDOUT12_WINDOW["valid_start_idx"]
assert LONG_MODEL_A_WINDOW["valid_end_idx"] == LONG_RECENT4_WINDOW["valid_end_idx"]

print("Window logic OK")

print("\n===== COVERAGE CHECK =====")
print(long_sku_model_summary.groupby("Model_Name").agg(
    SKU_Count=("ItemCode", "nunique"),
    Avg_H12_Months=("Holdout12_Months", "mean"),
    Avg_R4A_Months=("Recent4A_Months", "mean"),
    Avg_R4B_Months=("Recent4B_Months", "mean")
))

In [ ]:
long_deployment_artifact_registry = pd.DataFrame([
    {"Model_Name": "XGBOOST",  "Deploy_File": "xgb_long_deploy_artifacts_residual.pkl"},
    {"Model_Name": "CATBOOST", "Deploy_File": "catboost_long_deploy_artifacts_residual.pkl"},
    {"Model_Name": "LIGHTGBM", "Deploy_File": "lgbm_long_deploy_artifacts_residual.pkl"},
    {"Model_Name": "GRU",      "Deploy_File": f"{GRU_DEPLOY_DIR}/gru_long_deploy_meta.json"}
])

print("\nLONG DEPLOYMENT ARTIFACT REGISTRY")
print(long_deployment_artifact_registry)

joblib.dump(long_deployment_artifact_registry, "long_deployment_artifact_registry.pkl")
print("Saved: long_deployment_artifact_registry.pkl")

### Medium History Segment

#### Models - XGB, CAT, RF

In [ ]:
print("\n================ MEDIUM SUBGROUP V2 PIPELINE ================\n")

MEDIUM_SUBGROUP_FEATURE_COLS = [
    f for f in MEDIUM_FEATURE_COLS
    if f not in []
] + [
    "Medium_SKU_Type_Encoded",
    "Medium_Bonus_Frequency",
    "Medium_Bonus_Demand_Share",
    "Medium_CV",
    "Medium_ZeroRate",
    "Medium_Supply_Rate",
    "Medium_Trend_Slope",
]

# remove duplicates safely
MEDIUM_SUBGROUP_FEATURE_COLS = list(dict.fromkeys(MEDIUM_SUBGROUP_FEATURE_COLS))

# ============================================================
# 1) MEDIUM WINDOWS
# ============================================================
def get_medium_recent4_window(time_windows, recent_months=4):
    valid_end_idx = int(time_windows["latest_idx"] - 1)
    valid_start_idx = int(valid_end_idx - recent_months + 1)
    return {
        "valid_start_idx": valid_start_idx,
        "valid_end_idx": valid_end_idx,
        "valid_months": int(recent_months),
        "window_name": "RECENT4"
    }

def get_medium_prev4_window(time_windows, prev_months=4, recent_months=4):
    recent_window = get_medium_recent4_window(time_windows, recent_months=recent_months)
    valid_end_idx = int(recent_window["valid_start_idx"] - 1)
    valid_start_idx = int(valid_end_idx - prev_months + 1)

    if valid_start_idx > valid_end_idx:
        raise ValueError("Invalid MEDIUM PREV4 window.")

    return {
        "valid_start_idx": valid_start_idx,
        "valid_end_idx": valid_end_idx,
        "valid_months": int(prev_months),
        "window_name": "PREV4"
    }

def get_medium_tune_windows(df, time_windows, n_folds=2, fold_size_months=4, recent_months=4):
    temp = add_period_index(df)
    periods = (
        temp[["Year", "Month_Number", "Period_Index"]]
        .drop_duplicates()
        .sort_values("Period_Index")
        .reset_index(drop=True)
    )

    recent_window = get_medium_recent4_window(time_windows, recent_months=recent_months)
    usable = periods[periods["Period_Index"] < recent_window["valid_start_idx"]].copy()

    if len(usable) < fold_size_months * (n_folds + 1):
        raise ValueError("Not enough history to build MEDIUM tune windows.")

    windows = []
    end_idx = int(usable["Period_Index"].max())

    for _ in range(n_folds):
        valid_end_idx = end_idx
        valid_start_idx = valid_end_idx - fold_size_months + 1
        windows.append({
            "valid_start_idx": int(valid_start_idx),
            "valid_end_idx": int(valid_end_idx),
            "valid_months": int(fold_size_months)
        })
        end_idx = valid_start_idx - 1

    return list(reversed(windows))

In [ ]:

MEDIUM_TUNE_WINDOWS = get_medium_tune_windows(Data, TIME_WINDOWS, n_folds=2, fold_size_months=4, recent_months=4)
MEDIUM_PREV4_WINDOW = get_medium_prev4_window(TIME_WINDOWS, prev_months=4, recent_months=4)
MEDIUM_RECENT4_WINDOW = get_medium_recent4_window(TIME_WINDOWS, recent_months=4)

print("MEDIUM_TUNE_WINDOWS:", MEDIUM_TUNE_WINDOWS)
print("MEDIUM_PREV4_WINDOW:", MEDIUM_PREV4_WINDOW)
print("MEDIUM_RECENT4_WINDOW:", MEDIUM_RECENT4_WINDOW)

# Optional alias if you still want the word VALID in code
MEDIUM_VALID_WINDOW = MEDIUM_PREV4_WINDOW

In [ ]:
# ============================================================
# 2) MEDIUM PROFILE BUILDING
# ============================================================
def build_medium_sku_profile(train_df):
    train_df = force_itemcode_str(train_df)
    train_df = train_df.copy().sort_values(["ItemCode", "Year", "Month_Number"])
    out = []

    for item, g in train_df.groupby("ItemCode"):
        g = g.copy()

        mean_demand = float(g["Clean_Demand"].mean()) if len(g) > 0 else 0.0
        std_demand = float(g["Clean_Demand"].std()) if len(g) > 1 else 0.0
        cv = 0.0 if mean_demand <= 0 else std_demand / (mean_demand + 1)

        zero_rate = float((g["Clean_Demand"] == 0).mean()) if len(g) > 0 else 0.0
        bonus_freq = float(g["Bonus_Flag"].mean()) if "Bonus_Flag" in g.columns else 0.0
        supply_rate = float(g["Supply_Constraint_Flag"].mean()) if "Supply_Constraint_Flag" in g.columns else 0.0

        total_demand = float(g["Clean_Demand"].sum())
        bonus_demand = float(g.loc[g["Bonus_Flag"] == 1, "Clean_Demand"].sum()) if total_demand > 0 else 0.0
        bonus_share = 0.0 if total_demand <= 0 else bonus_demand / total_demand

        if len(g) >= 2:
            x = np.arange(len(g))
            y = g["Clean_Demand"].values
            trend_slope = float(np.polyfit(x, y, 1)[0])
        else:
            trend_slope = 0.0

        if zero_rate > 0.5:
            sku_type = "INTERMITTENT"
        elif bonus_freq > 0.3 or bonus_share > 0.4:
            sku_type = "PROMO_HEAVY"
        elif supply_rate > 0.3:
            sku_type = "SUPPLY_AFFECTED"
        elif abs(trend_slope) > mean_demand * 0.2:
            sku_type = "TRENDING"
        else:
            sku_type = "STABLE"

        if bonus_freq > 0.25 or bonus_share > 0.35:
            subgroup = "PROMO_HEAVY"
        else:
            subgroup = "STABLE"

        out.append({
            "ItemCode": item,
            "Medium_SKU_Type": sku_type,
            "Medium_Subgroup": subgroup,
            "Medium_Bonus_Frequency": bonus_freq,
            "Medium_Bonus_Demand_Share": bonus_share,
            "Medium_CV": cv,
            "Medium_ZeroRate": zero_rate,
            "Medium_Supply_Rate": supply_rate,
            "Medium_Trend_Slope": trend_slope
        })

    out_df = pd.DataFrame(out)
    out_df = force_itemcode_str(out_df)
    return out_df

def merge_medium_sku_profile(df, profile_df):
    df = force_itemcode_str(df)
    profile_df = force_itemcode_str(profile_df)
    df = df.copy()

    keep_cols = [
        "ItemCode",
        "Medium_SKU_Type",
        "Medium_Subgroup",
        "Medium_Bonus_Frequency",
        "Medium_Bonus_Demand_Share",
        "Medium_CV",
        "Medium_ZeroRate",
        "Medium_Supply_Rate",
        "Medium_Trend_Slope"
    ]

    df = df.drop(columns=[c for c in keep_cols if c != "ItemCode"], errors="ignore")
    df = df.merge(profile_df[keep_cols], on="ItemCode", how="left")
    df = force_itemcode_str(df)

    df["Medium_SKU_Type"] = df["Medium_SKU_Type"].fillna("STABLE")
    df["Medium_Subgroup"] = df["Medium_Subgroup"].fillna("STABLE")

    for c in [
        "Medium_Bonus_Frequency",
        "Medium_Bonus_Demand_Share",
        "Medium_CV",
        "Medium_ZeroRate",
        "Medium_Supply_Rate",
        "Medium_Trend_Slope"
    ]:
        df[c] = df[c].fillna(0)

    type_map = {
        "STABLE": 0,
        "PROMO_HEAVY": 1,
        "TRENDING": 2,
        "SUPPLY_AFFECTED": 3,
        "INTERMITTENT": 4
    }
    df["Medium_SKU_Type_Encoded"] = df["Medium_SKU_Type"].map(type_map).fillna(0).astype(int)

    return df

def enforce_unique_subgroup(profile_df):
    profile_df = profile_df.copy()
    
    dup = profile_df.groupby("ItemCode")["Medium_Subgroup"].nunique()
    bad_skus = dup[dup > 1].index.tolist()
    
    if len(bad_skus) > 0:
        print("⚠️ FIXING DUPLICATE SUBGROUP SKUs:", len(bad_skus))
        profile_df = (
            profile_df
            .sort_values(["ItemCode", "Medium_Bonus_Demand_Share"], ascending=False)
            .drop_duplicates("ItemCode")
        )
    
    return profile_df

def get_current_medium_skus(df):
    df = add_history_length_from_subset(df.copy(), df.copy())
    df = df[df["History_Segment"] == "MEDIUM"]
    
    return set(
        df["ItemCode"]
        .astype(str)
        .str.replace(".0", "", regex=False)
        .unique()
    )

def split_low_history_medium_skus(df, min_months=6):
    df = force_itemcode_str(df).copy()

    hist_counts = (
        df[["ItemCode", "Year", "Month_Number"]]
        .drop_duplicates()
        .groupby("ItemCode", as_index=False)
        .size()
        .rename(columns={"size": "History_Months"})
    )

    hist_counts["ItemCode"] = hist_counts["ItemCode"].astype(str)

    good_skus = set(
        hist_counts.loc[hist_counts["History_Months"] >= min_months, "ItemCode"]
    )
    low_skus = set(
        hist_counts.loc[hist_counts["History_Months"] < min_months, "ItemCode"]
    )

    df_good = df[df["ItemCode"].astype(str).isin(good_skus)].copy()
    df_low = df[df["ItemCode"].astype(str).isin(low_skus)].copy()

    return df_good, df_low, good_skus, low_skus


# ============================================================
# 3) FOLD-SAFE MEDIUM PREP
# ============================================================
def prepare_medium_subgroup_frame_foldsafe(train_df, valid_df):
    train_df = force_itemcode_str(train_df)
    valid_df = force_itemcode_str(valid_df)

    train_df = train_df.copy().sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)
    valid_df = valid_df.copy().sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    # recurring bonus features from TRAIN only
    train_df, valid_df, _ = apply_recurring_bonus_features_foldsafe(train_df, valid_df)
    train_df = force_itemcode_str(train_df)
    valid_df = force_itemcode_str(valid_df)

    # train-derived mappings
    train_df, valid_df, abc_map = apply_fold_adjustments(train_df, valid_df)

    promo_profile_df = build_promo_profile(train_df)
    train_df = merge_promo_profile(train_df, promo_profile_df)
    valid_df = merge_promo_profile(valid_df, promo_profile_df)

    medium_profile_df = build_medium_sku_profile(train_df)
    medium_profile_df = enforce_unique_subgroup(medium_profile_df)
    
    train_df = merge_medium_sku_profile(train_df, medium_profile_df)
    valid_df = merge_medium_sku_profile(valid_df, medium_profile_df)

    train_df, valid_df, sku_profile_df = apply_sku_history_profile(train_df, valid_df)

    train_out, valid_out = rebuild_time_features_foldsafe(train_df, valid_df)
    train_out = force_itemcode_str(train_out)
    valid_out = force_itemcode_str(valid_out)

    return train_out, valid_out, abc_map, promo_profile_df, medium_profile_df, sku_profile_df


# ============================================================
# 4) SUBGROUP FILTER
# ============================================================
def filter_medium_subgroup(df, subgroup_name):
    df = force_itemcode_str(df)
    subgroup_name = str(subgroup_name).upper()

    if subgroup_name == "PROMO_HEAVY":
        return df[df["Medium_Subgroup"] == "PROMO_HEAVY"].copy()

    if subgroup_name == "STABLE":
        return df[df["Medium_Subgroup"] == "STABLE"].copy()

    raise ValueError(f"Unknown subgroup_name: {subgroup_name}")

# ============================================================
# 5) MODEL BUILDERS / PARAMS
# ============================================================
def build_medium_model(model_name, params):
    model_name = model_name.upper()

    if model_name == "XGBOOST":
        return xgb.XGBRegressor(
            objective="reg:squarederror",
            eval_metric="rmse",
            random_state=42,
            tree_method="hist",
            n_jobs=-1,
            **params
        )

    elif model_name == "RANDOM_FOREST":
        return RandomForestRegressor(
            random_state=42,
            n_jobs=-1,
            **params
        )

    elif model_name == "CATBOOST":
        return CatBoostRegressor(
            loss_function="RMSE",
            eval_metric="RMSE",
            random_seed=42,
            verbose=0,
            **params
        )

    else:
        raise ValueError(f"Unsupported model_name: {model_name}")

def get_medium_model_params(trial, model_name, subgroup_name):
    model_name = model_name.upper()
    subgroup_name = subgroup_name.upper()

    if model_name == "XGBOOST":
        if subgroup_name == "PROMO_HEAVY":
            return {
                "n_estimators": trial.suggest_int("n_estimators", 600, 1400),
                "learning_rate": trial.suggest_float("learning_rate", 0.012, 0.035),
                "max_depth": trial.suggest_int("max_depth", 5, 7),
                "max_leaves": 64,
                "grow_policy": "lossguide",
                "min_child_weight": trial.suggest_int("min_child_weight", 3, 8),
                "subsample": trial.suggest_float("subsample", 0.70, 0.88),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.70, 0.88),
                "gamma": trial.suggest_float("gamma", 0.10, 0.45),
                "reg_lambda": trial.suggest_float("reg_lambda", 12, 22),
                "reg_alpha": trial.suggest_float("reg_alpha", 1.0, 6.0),
            }
        else:
            return {
                "n_estimators": trial.suggest_int("n_estimators", 400, 1000),
                "learning_rate": trial.suggest_float("learning_rate", 0.012, 0.030),
                "max_depth": trial.suggest_int("max_depth", 3, 5),
                "max_leaves": 64,
                "grow_policy": "lossguide",
                "min_child_weight": trial.suggest_int("min_child_weight", 5, 12),
                "subsample": trial.suggest_float("subsample", 0.72, 0.90),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 0.85),
                "gamma": trial.suggest_float("gamma", 0.15, 0.55),
                "reg_lambda": trial.suggest_float("reg_lambda", 12, 24),
                "reg_alpha": trial.suggest_float("reg_alpha", 1.0, 6.0),
            }

    elif model_name == "CATBOOST":
        if subgroup_name == "PROMO_HEAVY":
            return {
                "iterations": trial.suggest_int("iterations", 500, 1300),
                "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.045),
                "depth": trial.suggest_int("depth", 5, 8),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 2.0, 15.0),
                "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 35),
                "subsample": trial.suggest_float("subsample", 0.70, 0.95),
                "rsm": trial.suggest_float("rsm", 0.70, 1.0),
                "random_strength": trial.suggest_float("random_strength", 0.0, 3.0),
                "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 3.0),
            }
        else:
            return {
                "iterations": trial.suggest_int("iterations", 300, 900),
                "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.040),
                "depth": trial.suggest_int("depth", 4, 6),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 2.0, 12.0),
                "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 40),
                "subsample": trial.suggest_float("subsample", 0.75, 0.95),
                "rsm": trial.suggest_float("rsm", 0.70, 1.0),
                "random_strength": trial.suggest_float("random_strength", 0.0, 2.5),
                "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 2.5),
            }

    elif model_name == "RANDOM_FOREST":
        if subgroup_name != "STABLE":
            raise ValueError("RANDOM_FOREST should be used only for STABLE subgroup.")

        return {
            "n_estimators": trial.suggest_int("n_estimators", 300, 900),
            "max_depth": trial.suggest_int("max_depth", 4, 14),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 12),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 8),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.6, 0.8]),
            "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
        }

    else:
        raise ValueError(f"Unsupported model_name: {model_name}")

def maybe_scale_features(X_train, X_valid=None, model_name="XGBOOST"):
    if model_name.upper() != "ELASTICNET":
        return X_train, X_valid, None

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    if X_valid is None:
        return X_train_scaled, None, scaler

    X_valid_scaled = scaler.transform(X_valid)
    return X_train_scaled, X_valid_scaled, scaler


In [ ]:
# ============================================================
# 6) DEBUG HELPERS
# ============================================================
def log_medium_step(df, label):
    print(f"[MEDIUM DEBUG] {label} | rows={len(df)} | skus={df['ItemCode'].astype(str).nunique()}")

def summarize_missing_skus(full_data, compared_df, label="MEDIUM"):
    all_skus = set(
        add_history_length_from_subset(full_data.copy(), full_data.copy())
        .query("History_Segment == 'MEDIUM'")["ItemCode"]
        .astype(str).str.replace(".0", "", regex=False).unique()
    )
    covered_skus = set(
        compared_df["ItemCode"]
        .astype(str).str.replace(".0", "", regex=False).unique()
    )
    missing = sorted(list(all_skus - covered_skus))

    print(f"\n[{label} COVERAGE]")
    print("Actual MEDIUM SKUs:", len(all_skus))
    print("Covered MEDIUM SKUs:", len(covered_skus))
    print("Missing MEDIUM SKUs:", len(missing))
    print("Sample missing:", missing[:20])

    return missing

def debug_medium_survival(full_data, subgroup_name, valid_window):
    full_data = add_period_index(full_data)

    train_df = full_data[full_data["Period_Index"] < valid_window["valid_start_idx"]].copy()
    valid_df = full_data[full_data["Period_Index"].between(valid_window["valid_start_idx"], valid_window["valid_end_idx"])].copy()

    train_df = add_history_length_from_subset(train_df, train_df)
    valid_df = add_history_length_from_subset(train_df, valid_df)

    train_df = train_df[train_df["History_Segment"] == "MEDIUM"].copy()
    valid_df = valid_df[valid_df["History_Segment"] == "MEDIUM"].copy()

    print("raw valid medium skus:", valid_df["ItemCode"].astype(str).nunique())

    train_df, valid_df, *_ = prepare_medium_subgroup_frame_foldsafe(train_df, valid_df)
    print("after foldsafe valid skus:", valid_df["ItemCode"].astype(str).nunique())

    valid_df = filter_medium_subgroup(valid_df, subgroup_name)
    print("after subgroup filter valid skus:", valid_df["ItemCode"].astype(str).nunique())

    valid_df = recompute_target(valid_df)
    valid_df = add_residual_target(valid_df)
    valid_df = valid_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()
    print("after target build valid skus:", valid_df["ItemCode"].astype(str).nunique())

    return valid_df

print("\n===== DEBUG MEDIUM SURVIVAL: PROMO_HEAVY / PREV4 =====")
_ = debug_medium_survival(Data, "PROMO_HEAVY", MEDIUM_PREV4_WINDOW)

print("\n===== DEBUG MEDIUM SURVIVAL: STABLE / PREV4 =====")
_ = debug_medium_survival(Data, "STABLE", MEDIUM_PREV4_WINDOW)

print("\n===== DEBUG MEDIUM SURVIVAL: PROMO_HEAVY / RECENT4 =====")
_ = debug_medium_survival(Data, "PROMO_HEAVY", MEDIUM_RECENT4_WINDOW)

print("\n===== DEBUG MEDIUM SURVIVAL: STABLE / RECENT4 =====")
_ = debug_medium_survival(Data, "STABLE", MEDIUM_RECENT4_WINDOW)



In [ ]:
# ============================================================
# 7) TUNING / TRAIN-EVAL
# ============================================================
def tune_residual_medium_subgroup(
    full_data,
    feature_cols,
    subgroup_name,
    model_name,
    n_trials=30,
    study_name=None
):
    full_data = full_data.copy()
    model_name = model_name.upper()
    subgroup_name = subgroup_name.upper()

    if study_name is None:
        study_name = f"residual_{model_name.lower()}_medium_{subgroup_name.lower()}"

    def objective(trial):
        params = get_medium_model_params(trial, model_name, subgroup_name)
        scores = []

        temp = add_period_index(full_data)

        for tune_window in MEDIUM_TUNE_WINDOWS:
            train_df = temp[temp["Period_Index"] < tune_window["valid_start_idx"]].copy()
            valid_df = temp[temp["Period_Index"].between(tune_window["valid_start_idx"], tune_window["valid_end_idx"])].copy()

            if train_df.empty or valid_df.empty:
                continue

            train_df = add_history_length_from_subset(train_df, train_df)
            valid_df = add_history_length_from_subset(train_df, valid_df)

            train_df = train_df[train_df["History_Segment"] == "MEDIUM"].copy()
            valid_df = valid_df[valid_df["History_Segment"] == "MEDIUM"].copy()

            train_df, _, good_skus, low_skus = split_low_history_medium_skus(train_df, min_months=6)
            valid_df = valid_df[valid_df["ItemCode"].astype(str).isin(good_skus)].copy()

            if train_df.empty or valid_df.empty:
                continue

            train_df, valid_df, _, _, _, _ = prepare_medium_subgroup_frame_foldsafe(train_df, valid_df)

            train_df = filter_medium_subgroup(train_df, subgroup_name)
            valid_df = filter_medium_subgroup(valid_df, subgroup_name)

            if train_df.empty or valid_df.empty:
                continue

            caps = compute_clip_caps(train_df, cols=["Inventory_Pressure", "Stock_Cover_Months"], q=0.99)
            train_df = apply_clip_caps(train_df, caps)
            valid_df = apply_clip_caps(valid_df, caps)

            train_df = recompute_target(train_df)
            valid_df = recompute_target(valid_df)

            train_df = add_residual_target(train_df)
            valid_df = add_residual_target(valid_df)

            train_df = train_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()
            valid_df = valid_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()

            if train_df.empty or valid_df.empty:
                continue

            train_df["ItemCode_Original"] = train_df["ItemCode"]
            valid_df["ItemCode_Original"] = valid_df["ItemCode"]

            train_df, valid_df, _ = encode_itemcode(train_df, valid_df)

            assert_features_exist(train_df, feature_cols, where=f"{model_name}_{subgroup_name}_TUNE_TRAIN")
            assert_features_exist(valid_df, feature_cols, where=f"{model_name}_{subgroup_name}_TUNE_VALID")

            model = build_medium_model(model_name, params)

            Xtr = sanitize(train_df[feature_cols])
            ytr = train_df[MODEL_TARGET_COL]
            Xva = sanitize(valid_df[feature_cols])

            Xtr, Xva, _ = maybe_scale_features(Xtr, Xva, model_name=model_name)

            w_train = recency_weights(train_df, yearly_boost=0.25).astype(float)
            w_train *= np.where(train_df["ABC_Class"] == 0, 2.5,
                       np.where(train_df["ABC_Class"] == 1, 1.2, 1.0))

            try:
                model.fit(Xtr, ytr, sample_weight=w_train)
            except TypeError:
                model.fit(Xtr, ytr)

            pred_residual = model.predict(Xva)
            pred_final = np.clip(valid_df[BASELINE_COL].values + pred_residual, 0, None)

            scores.append(wmape(valid_df[ACTUAL_TARGET_COL].values, pred_final))

        return 999999.0 if len(scores) == 0 else np.mean(scores)

    study = optuna.create_study(direction="minimize", study_name=study_name)
    study.optimize(objective, n_trials=n_trials)

    return study.best_params, study

def train_eval_validation_residual_medium_subgroup(
    full_data,
    feature_cols,
    best_params,
    subgroup_name,
    model_name,
    valid_window,
    yearly_boost=0.25
):
    model_name = model_name.upper()
    subgroup_name = subgroup_name.upper()

    full_data = add_period_index(full_data)

    train_df = full_data[full_data["Period_Index"] < valid_window["valid_start_idx"]].copy()
    valid_df = full_data[full_data["Period_Index"].between(valid_window["valid_start_idx"], valid_window["valid_end_idx"])].copy()

    if train_df.empty or valid_df.empty:
        raise ValueError(
            f"Need both train data before {valid_window['valid_start_idx']} "
            f"and validation data from {valid_window['valid_start_idx']} to {valid_window['valid_end_idx']}."
        )

    train_df = add_history_length_from_subset(train_df, train_df)
    valid_df = add_history_length_from_subset(train_df, valid_df)

    train_df = train_df[train_df["History_Segment"] == "MEDIUM"].copy()
    valid_df = valid_df[valid_df["History_Segment"] == "MEDIUM"].copy()

    train_df, _, good_skus, low_skus = split_low_history_medium_skus(train_df, min_months=6)
    valid_df = valid_df[valid_df["ItemCode"].astype(str).isin(good_skus)].copy()

    if train_df.empty or valid_df.empty:
        raise ValueError("No MEDIUM rows available.")

    train_df, valid_df, abc_map, promo_profile_df, medium_profile_df, sku_profile_df = prepare_medium_subgroup_frame_foldsafe(
        train_df, valid_df
    )

    train_df = filter_medium_subgroup(train_df, subgroup_name)
    valid_df = filter_medium_subgroup(valid_df, subgroup_name)

    if train_df.empty or valid_df.empty:
        raise ValueError(f"No usable rows for subgroup {subgroup_name}.")

    caps = compute_clip_caps(train_df, cols=["Inventory_Pressure", "Stock_Cover_Months"], q=0.99)
    train_df = apply_clip_caps(train_df, caps)
    valid_df = apply_clip_caps(valid_df, caps)

    train_df = recompute_target(train_df)
    valid_df = recompute_target(valid_df)

    train_df = add_residual_target(train_df)
    valid_df = add_residual_target(valid_df)

    train_df = train_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()
    valid_df = valid_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()

    if train_df.empty or valid_df.empty:
        raise ValueError(f"No usable rows after target creation for subgroup {subgroup_name}.")

    train_df["ItemCode_Original"] = train_df["ItemCode"]
    valid_df["ItemCode_Original"] = valid_df["ItemCode"]

    train_df, valid_df, itemcode_categories = encode_itemcode(train_df, valid_df)

    assert_features_exist(train_df, feature_cols, where=f"{model_name}_{subgroup_name}_VALID_TRAIN")
    assert_features_exist(valid_df, feature_cols, where=f"{model_name}_{subgroup_name}_VALID_EVAL")

    model = build_medium_model(model_name, best_params)

    Xtr = sanitize(train_df[feature_cols])
    ytr = train_df[MODEL_TARGET_COL]
    Xva = sanitize(valid_df[feature_cols])

    Xtr, Xva, scaler = maybe_scale_features(Xtr, Xva, model_name=model_name)

    w_train = recency_weights(train_df, yearly_boost=yearly_boost).astype(float)
    w_train *= np.where(train_df["ABC_Class"] == 0, 2.5,
               np.where(train_df["ABC_Class"] == 1, 1.2, 1.0))

    try:
        model.fit(Xtr, ytr, sample_weight=w_train)
    except TypeError:
        model.fit(Xtr, ytr)

    valid_df["Pred_Residual"] = model.predict(Xva)
    valid_df["Pred"] = np.clip(valid_df[BASELINE_COL] + valid_df["Pred_Residual"], 0, None)

    metrics = evaluate_all_metrics(valid_df[ACTUAL_TARGET_COL].values, valid_df["Pred"].values)

    aux = {
        "abc_map": abc_map,
        "promo_profile_df": promo_profile_df,
        "medium_profile_df": medium_profile_df,
        "itemcode_categories": itemcode_categories,
        "clip_caps": caps,
        "subgroup_name": subgroup_name,
        "model_name": model_name,
        "sku_profile_df": sku_profile_df,
        "feature_scaler": scaler
    }

    return model, valid_df, metrics, aux

# ============================================================
# 8) FEATURE PRUNING
# ============================================================
def iterative_feature_prune_residual_medium_subgroup(
    full_data,
    start_features,
    best_params,
    subgroup_name,
    model_name,
    drop_k=1,
    min_features=15,
    max_rounds=10,
    tolerance=0.10,
    n_repeats=5
):
    history = []
    features = start_features.copy()

    model, eval_df, m, aux = train_eval_validation_residual_medium_subgroup(
        full_data=full_data,
        feature_cols=features,
        best_params=best_params,
        subgroup_name=subgroup_name,
        model_name=model_name,
        valid_window=MEDIUM_PREV4_WINDOW
    )

    best_wmape = m["WMAPE"]
    last_accepted_state = (features.copy(), model, eval_df.copy(), m.copy(), aux.copy())

    for r in range(1, max_rounds + 1):
        if len(features) <= min_features:
            break

        imp = permutation_rank(
            model=model,
            eval_df=eval_df,
            feature_cols=features,
            target_col=MODEL_TARGET_COL,
            n_repeats=n_repeats
        )

        protected = {
            "ItemCode",
            "ABC_Class",
            "Lag1",
            "Lag2",
            "Lag3",
            "Rolling3M_Mean",
            "Rolling3M_Std",
            "Month_Sin",
            "Month_Cos",
            "Bonus_Flag",
            "Expected_Bonus_NextMonth",
            "Supply_Constraint_Flag",
            "Medium_SKU_Type_Encoded"
        }

        drop_candidates = [
            f for f in imp.sort_values("perm_importance").feature.tolist()
            if f not in protected
        ]

        to_drop = drop_candidates[:drop_k]

        if not to_drop:
            break

        new_features = [f for f in features if f not in to_drop]

        new_model, new_eval_df, new_m, new_aux = train_eval_validation_residual_medium_subgroup(
            full_data=full_data,
            feature_cols=new_features,
            best_params=best_params,
            subgroup_name=subgroup_name,
            model_name=model_name,
            valid_window=MEDIUM_PREV4_WINDOW
        )

        history.append({
            "round": r,
            "subgroup": subgroup_name,
            "model_name": model_name,
            "valid_start_idx": MEDIUM_PREV4_WINDOW["valid_start_idx"],
            "valid_end_idx": MEDIUM_PREV4_WINDOW["valid_end_idx"],
            "dropped": to_drop,
            "n_features": len(new_features),
            **new_m
        })

        if new_m["WMAPE"] <= best_wmape + tolerance:
            features = new_features
            model, eval_df = new_model, new_eval_df
            best_wmape = min(best_wmape, new_m["WMAPE"])
            last_accepted_state = (features.copy(), model, eval_df.copy(), new_m.copy(), new_aux.copy())
        else:
            break

    results_df = pd.DataFrame(history)

    return (
        last_accepted_state[0],
        last_accepted_state[1],
        last_accepted_state[2],
        last_accepted_state[3],
        last_accepted_state[4],
        results_df
    )

# ============================================================
# 9) DEPLOYMENT TRAINING
# ============================================================
def train_single_deployment_model_medium_subgroup_residual(
    full_data,
    feature_cols,
    best_params,
    subgroup_name,
    model_name
):
    model_name = model_name.upper()
    subgroup_name = subgroup_name.upper()

    print(f"\n========== {model_name} DEPLOYMENT MODEL → TRAIN ON ALL COMPLETE DATA ({subgroup_name}) ==========")

    deploy_df = full_data.copy().sort_values(["ItemCode", "Year", "Month_Number"])
    deploy_df = force_itemcode_str(deploy_df)

    deploy_df = add_history_length_from_subset(deploy_df, deploy_df)
    deploy_df = deploy_df[deploy_df["History_Segment"] == "MEDIUM"].copy()

    deploy_df, low_df, good_skus, low_skus = split_low_history_medium_skus(deploy_df, min_months=6)

    print(f"[MEDIUM DEPLOY] Skipping low-history SKUs: {len(low_skus)}")

    if deploy_df.empty:
        raise ValueError("No MEDIUM rows available for deployment training.")

    bonus_pattern_df = detect_recurring_bonus_skus(deploy_df)[[
        "ItemCode",
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ]].copy()

    deploy_df = deploy_df.drop(columns=[
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ], errors="ignore")

    bonus_pattern_df = force_itemcode_str(bonus_pattern_df)
    deploy_df = deploy_df.merge(bonus_pattern_df, on="ItemCode", how="left")
    deploy_df = force_itemcode_str(deploy_df)

    for c in ["Recurring_Bonus_SKU", "Bonus_Cycle_Length", "Avg_Bonus_Gap", "Bonus_Frequency_All"]:
        deploy_df[c] = deploy_df[c].fillna(0)
    deploy_df["Avg_Bonus_Uplift"] = deploy_df["Avg_Bonus_Uplift"].fillna(1.0)

    deploy_df, _ = apply_sku_cap(deploy_df.copy(), deploy_df.copy())

    sku_total = deploy_df.groupby("ItemCode")["Clean_Demand"].sum().sort_values(ascending=False)
    total_sum = sku_total.sum()

    if total_sum > 0:
        cum_pct = sku_total.cumsum() / total_sum
        abc_series = pd.cut(cum_pct, bins=[0, 0.7, 0.9, 1.0], labels=[0, 1, 2])
        abc_map = abc_series.to_dict()
        deploy_df["ABC_Class"] = deploy_df["ItemCode"].map(abc_map).fillna(2)
    else:
        abc_map = {}
        deploy_df["ABC_Class"] = 2

    promo_profile_df = build_promo_profile(deploy_df)
    deploy_df = merge_promo_profile(deploy_df, promo_profile_df)

    medium_profile_df = build_medium_sku_profile(deploy_df)
    medium_profile_df = enforce_unique_subgroup(medium_profile_df)
    
    deploy_df = merge_medium_sku_profile(deploy_df, medium_profile_df)

    sku_profile_df = (
        deploy_df.groupby("ItemCode")
        .agg(
            SKU_Mean_Demand=("Clean_Demand", "mean"),
            SKU_Std_Demand=("Clean_Demand", "std"),
            SKU_ZeroRate=("Clean_Demand", lambda x: (x == 0).mean())
        )
        .reset_index()
    )

    sku_profile_df["SKU_Std_Demand"] = sku_profile_df["SKU_Std_Demand"].fillna(0)
    sku_profile_df["SKU_CV"] = np.where(
        sku_profile_df["SKU_Mean_Demand"] <= 0,
        0,
        sku_profile_df["SKU_Std_Demand"] / (sku_profile_df["SKU_Mean_Demand"] + 1)
    )

    sku_profile_df = sku_profile_df[["ItemCode", "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]]
    sku_profile_df = force_itemcode_str(sku_profile_df)

    deploy_df = deploy_df.drop(columns=["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"], errors="ignore")
    deploy_df = deploy_df.merge(sku_profile_df, on="ItemCode", how="left")
    deploy_df = force_itemcode_str(deploy_df)

    for c in ["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]:
        deploy_df[c] = deploy_df[c].fillna(0)

    deploy_df = filter_medium_subgroup(deploy_df, subgroup_name)
    if deploy_df.empty:
        raise ValueError(f"No rows available for deployment subgroup {subgroup_name}.")

    deploy_df = add_bonus_cycle_features(deploy_df)
    deploy_df = rebuild_time_features(deploy_df)

    caps = compute_clip_caps(deploy_df, cols=["Inventory_Pressure", "Stock_Cover_Months"], q=0.99)
    deploy_df = apply_clip_caps(deploy_df, caps)

    deploy_df = recompute_target(deploy_df)
    deploy_df = add_residual_target(deploy_df)
    deploy_df = deploy_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()

    if deploy_df.empty:
        raise ValueError(f"No usable deployment rows after target creation for {subgroup_name}.")

    deploy_df["ItemCode_Original"] = deploy_df["ItemCode"]
    deploy_df, _, itemcode_categories = encode_itemcode(deploy_df, deploy_df)

    assert_features_exist(deploy_df, feature_cols, where=f"{model_name}_{subgroup_name}_DEPLOY_TRAIN")

    model = build_medium_model(model_name, best_params)

    Xtr = sanitize(deploy_df[feature_cols])
    ytr = deploy_df[MODEL_TARGET_COL]

    Xtr, _, scaler = maybe_scale_features(Xtr, None, model_name=model_name)

    w_train = recency_weights(deploy_df, yearly_boost=0.25).astype(float)
    w_train *= np.where(deploy_df["ABC_Class"] == 0, 2.5,
               np.where(deploy_df["ABC_Class"] == 1, 1.2, 1.0))

    try:
        model.fit(Xtr, ytr, sample_weight=w_train)
    except TypeError:
        model.fit(Xtr, ytr)

    artifacts = {
        "model": model,
        "feature_cols": feature_cols,
        "best_params": best_params,
        "itemcode_categories": itemcode_categories,
        "abc_map": abc_map,
        "clip_caps": caps,
        "promo_profile_df": promo_profile_df,
        "medium_profile_df": medium_profile_df,
        "sku_profile_df": sku_profile_df,
        "target_mode": "residual",
        "baseline_col": BASELINE_COL,
        "actual_target_col": ACTUAL_TARGET_COL,
        "model_target_col": MODEL_TARGET_COL,
        "segment": "MEDIUM",
        "subgroup_name": subgroup_name,
        "model_name": model_name.upper(),
        "feature_scaler": scaler
    }

    return artifacts, deploy_df

# ============================================================
# 10) RUN MEDIUM MODEL PIPELINE
# ============================================================
def train_single_window_validation_medium_subgroup_residual(
    full_data,
    feature_cols,
    best_params,
    subgroup_name,
    model_name,
    valid_window
):
    print(
        f"\n========== {model_name} WINDOW VALIDATION "
        f"→ VALID ON PERIODS {valid_window['valid_start_idx']} to {valid_window['valid_end_idx']} "
        f"({subgroup_name}) =========="
    )

    model, valid_df, metrics, aux = train_eval_validation_residual_medium_subgroup(
        full_data=full_data,
        feature_cols=feature_cols,
        best_params=best_params,
        subgroup_name=subgroup_name,
        model_name=model_name,
        valid_window=valid_window
    )

    return valid_df, metrics

def run_medium_model_pipeline(
    full_data,
    feature_cols,
    subgroup_name,
    model_name,
    n_trials=30,
    drop_k=1,
    min_features=15,
    max_rounds=10,
    tolerance=0.10,
    n_repeats=5
):
    best_params, study = tune_residual_medium_subgroup(
        full_data=full_data,
        feature_cols=feature_cols,
        subgroup_name=subgroup_name,
        model_name=model_name,
        n_trials=n_trials,
        study_name=f"residual_{model_name.lower()}_medium_{subgroup_name.lower()}"
    )
    print(f"{model_name} {subgroup_name} best params:", best_params)

    best_feats, _, _, best_metrics, _, prune_log = iterative_feature_prune_residual_medium_subgroup(
        full_data=full_data,
        start_features=feature_cols,
        best_params=best_params,
        subgroup_name=subgroup_name,
        model_name=model_name,
        drop_k=drop_k,
        min_features=min_features,
        max_rounds=max_rounds,
        tolerance=tolerance,
        n_repeats=n_repeats
    )

    print(f"\n===== {model_name} {subgroup_name} BEST METRICS AFTER FEATURE PRUNING =====")
    print(best_metrics)

    print(f"\n===== {model_name} {subgroup_name} BEST FEATURES =====")
    print(best_feats)

    print(f"\n===== {model_name} {subgroup_name} FEATURE PRUNE LOG =====")
    print(prune_log)

    prev4_valid_df, prev4_valid_metrics = train_single_window_validation_medium_subgroup_residual(
        full_data=full_data,
        feature_cols=best_feats,
        best_params=best_params,
        subgroup_name=subgroup_name,
        model_name=model_name,
        valid_window=MEDIUM_PREV4_WINDOW
    )

    print(f"\n===== {model_name} {subgroup_name} PREV4 VALIDATION METRICS =====")
    print(prev4_valid_metrics)

    recent4_valid_df, recent4_valid_metrics = train_single_window_validation_medium_subgroup_residual(
        full_data=full_data,
        feature_cols=best_feats,
        best_params=best_params,
        subgroup_name=subgroup_name,
        model_name=model_name,
        valid_window=MEDIUM_RECENT4_WINDOW
    )

    print(f"\n===== {model_name} {subgroup_name} RECENT4 VALIDATION METRICS =====")
    print(recent4_valid_metrics)

    deploy_artifacts, deploy_train_df = train_single_deployment_model_medium_subgroup_residual(
        full_data=full_data,
        feature_cols=best_feats,
        best_params=best_params,
        subgroup_name=subgroup_name,
        model_name=model_name
    )

    return {
        "best_params": best_params,
        "best_feats": best_feats,
        "study": study,
        "prune_log": prune_log,
        "best_metrics": best_metrics,
        "deploy_artifacts": deploy_artifacts,
        "deploy_train_df": deploy_train_df,
        "prev4_valid": prev4_valid_df,
        "prev4_valid_metrics": prev4_valid_metrics,
        "recent4_valid": recent4_valid_df,
        "recent4_valid_metrics": recent4_valid_metrics,
    }


In [ ]:
# ============================================================
# 11) RUN MEDIUM MODELS
# ============================================================
medium_runs = {}

# PROMO_HEAVY
medium_runs["PROMO_HEAVY_XGBOOST"] = run_medium_model_pipeline(
    full_data=Data,
    feature_cols=MEDIUM_SUBGROUP_FEATURE_COLS,
    subgroup_name="PROMO_HEAVY",
    model_name="XGBOOST",
    n_trials=30
)

medium_runs["PROMO_HEAVY_CATBOOST"] = run_medium_model_pipeline(
    full_data=Data,
    feature_cols=MEDIUM_SUBGROUP_FEATURE_COLS,
    subgroup_name="PROMO_HEAVY",
    model_name="CATBOOST",
    n_trials=30
)

# STABLE
medium_runs["STABLE_XGBOOST"] = run_medium_model_pipeline(
    full_data=Data,
    feature_cols=MEDIUM_SUBGROUP_FEATURE_COLS,
    subgroup_name="STABLE",
    model_name="XGBOOST",
    n_trials=30
)

medium_runs["STABLE_CATBOOST"] = run_medium_model_pipeline(
    full_data=Data,
    feature_cols=MEDIUM_SUBGROUP_FEATURE_COLS,
    subgroup_name="STABLE",
    model_name="CATBOOST",
    n_trials=30
)

medium_runs["STABLE_RANDOM_FOREST"] = run_medium_model_pipeline(
    full_data=Data,
    feature_cols=MEDIUM_SUBGROUP_FEATURE_COLS,
    subgroup_name="STABLE",
    model_name="RANDOM_FOREST",
    n_trials=20
)

for k in medium_runs:
    if "prev4_valid" in medium_runs[k]:
        medium_runs[k]["prev4_valid"] = force_itemcode_str(medium_runs[k]["prev4_valid"])
    if "recent4_valid" in medium_runs[k]:
        medium_runs[k]["recent4_valid"] = force_itemcode_str(medium_runs[k]["recent4_valid"])

for run_name, run_obj in medium_runs.items():
    deploy_name = f"{run_name.lower()}_deploy_artifacts_residual.pkl"
    joblib.dump(run_obj["deploy_artifacts"], deploy_name)
    print("Saved:", deploy_name)


#### Comparison

In [ ]:
# ============================================================
# 12) STANDARDIZE OUTPUT
# ============================================================
def standardize_medium_model_output(
    df,
    actual_col,
    pred_col,
    item_col="ItemCode",
    year_col="Year",
    month_col="Month_Number",
    model_name="UNKNOWN",
    segment="MEDIUM",
    subgroup_name=""
):
    out = df.copy()
    out = force_itemcode_str(out)

    if item_col not in out.columns:
        raise KeyError(f"Missing item column: {item_col}")
    if actual_col not in out.columns:
        raise KeyError(f"Missing actual column: {actual_col}")
    if pred_col not in out.columns:
        raise KeyError(f"Missing pred column: {pred_col}")

    out["ItemCode_Original"] = out[item_col].astype(str)
    out["ItemCode"] = out["ItemCode_Original"]
    out["Actual"] = pd.to_numeric(out[actual_col], errors="coerce")
    out["Pred"] = pd.to_numeric(out[pred_col], errors="coerce").clip(lower=0)

    if year_col in out.columns:
        out["Year"] = pd.to_numeric(out[year_col], errors="coerce")
    else:
        out["Year"] = np.nan

    if month_col in out.columns:
        out["Month_Number"] = pd.to_numeric(out[month_col], errors="coerce")
    else:
        out["Month_Number"] = np.nan

    out["Error"] = out["Actual"] - out["Pred"]
    out["Abs_Error"] = np.abs(out["Error"])
    out["Segment"] = segment
    out["Model_Name"] = str(model_name).upper()
    out["Medium_Subgroup"] = str(subgroup_name).upper()

    keep_cols = [
        "ItemCode",
        "ItemCode_Original",
        "Year",
        "Month_Number",
        "Actual",
        "Pred",
        "Error",
        "Abs_Error",
        "Segment",
        "Model_Name",
        "Medium_Subgroup"
    ]

    extra_cols = [c for c in out.columns if c not in keep_cols]
    return out[keep_cols + extra_cols].copy()


# ============================================================
# 13) BUILD PREV4 / RECENT4 ROW-LEVEL EVAL DATA
# ============================================================
medium_prev4_eval_df = pd.concat(
    [
        standardize_medium_model_output(
            medium_runs["PROMO_HEAVY_XGBOOST"]["prev4_valid"],
            actual_col=ACTUAL_TARGET_COL,
            pred_col="Pred",
            item_col="ItemCode_Original",
            year_col="Year",
            month_col="Month_Number",
            model_name="XGBOOST",
            segment="MEDIUM",
            subgroup_name="PROMO_HEAVY"
        ),
        standardize_medium_model_output(
            medium_runs["PROMO_HEAVY_CATBOOST"]["prev4_valid"],
            actual_col=ACTUAL_TARGET_COL,
            pred_col="Pred",
            item_col="ItemCode_Original",
            year_col="Year",
            month_col="Month_Number",
            model_name="CATBOOST",
            segment="MEDIUM",
            subgroup_name="PROMO_HEAVY"
        ),
        standardize_medium_model_output(
            medium_runs["STABLE_XGBOOST"]["prev4_valid"],
            actual_col=ACTUAL_TARGET_COL,
            pred_col="Pred",
            item_col="ItemCode_Original",
            year_col="Year",
            month_col="Month_Number",
            model_name="XGBOOST",
            segment="MEDIUM",
            subgroup_name="STABLE"
        ),
        standardize_medium_model_output(
            medium_runs["STABLE_CATBOOST"]["prev4_valid"],
            actual_col=ACTUAL_TARGET_COL,
            pred_col="Pred",
            item_col="ItemCode_Original",
            year_col="Year",
            month_col="Month_Number",
            model_name="CATBOOST",
            segment="MEDIUM",
            subgroup_name="STABLE"
        ),
        standardize_medium_model_output(
            medium_runs["STABLE_RANDOM_FOREST"]["prev4_valid"],
            actual_col=ACTUAL_TARGET_COL,
            pred_col="Pred",
            item_col="ItemCode_Original",
            year_col="Year",
            month_col="Month_Number",
            model_name="RANDOM_FOREST",
            segment="MEDIUM",
            subgroup_name="STABLE"
        ),
    ],
    ignore_index=True
)

medium_recent4_eval_df = pd.concat(
    [
        standardize_medium_model_output(
            medium_runs["PROMO_HEAVY_XGBOOST"]["recent4_valid"],
            actual_col=ACTUAL_TARGET_COL,
            pred_col="Pred",
            item_col="ItemCode_Original",
            year_col="Year",
            month_col="Month_Number",
            model_name="XGBOOST",
            segment="MEDIUM",
            subgroup_name="PROMO_HEAVY"
        ),
        standardize_medium_model_output(
            medium_runs["PROMO_HEAVY_CATBOOST"]["recent4_valid"],
            actual_col=ACTUAL_TARGET_COL,
            pred_col="Pred",
            item_col="ItemCode_Original",
            year_col="Year",
            month_col="Month_Number",
            model_name="CATBOOST",
            segment="MEDIUM",
            subgroup_name="PROMO_HEAVY"
        ),
        standardize_medium_model_output(
            medium_runs["STABLE_XGBOOST"]["recent4_valid"],
            actual_col=ACTUAL_TARGET_COL,
            pred_col="Pred",
            item_col="ItemCode_Original",
            year_col="Year",
            month_col="Month_Number",
            model_name="XGBOOST",
            segment="MEDIUM",
            subgroup_name="STABLE"
        ),
        standardize_medium_model_output(
            medium_runs["STABLE_CATBOOST"]["recent4_valid"],
            actual_col=ACTUAL_TARGET_COL,
            pred_col="Pred",
            item_col="ItemCode_Original",
            year_col="Year",
            month_col="Month_Number",
            model_name="CATBOOST",
            segment="MEDIUM",
            subgroup_name="STABLE"
        ),
        standardize_medium_model_output(
            medium_runs["STABLE_RANDOM_FOREST"]["recent4_valid"],
            actual_col=ACTUAL_TARGET_COL,
            pred_col="Pred",
            item_col="ItemCode_Original",
            year_col="Year",
            month_col="Month_Number",
            model_name="RANDOM_FOREST",
            segment="MEDIUM",
            subgroup_name="STABLE"
        ),
    ],
    ignore_index=True
)


# ============================================================
# 14) CLEAN TYPES
# ============================================================
for df_ in [medium_prev4_eval_df, medium_recent4_eval_df]:
    for c in ["ItemCode", "ItemCode_Original", "Model_Name", "Medium_Subgroup"]:
        df_[c] = df_[c].astype(str)

    for c in ["Actual", "Pred", "Abs_Error", "Year", "Month_Number"]:
        df_[c] = pd.to_numeric(df_[c], errors="coerce")

    df_.dropna(
        subset=["ItemCode", "Medium_Subgroup", "Model_Name", "Actual", "Pred", "Abs_Error", "Year", "Month_Number"],
        inplace=True
    )


In [ ]:
# ============================================================
# 15) REPORTS
# ============================================================
for model_name in ["XGBOOST", "CATBOOST", "RANDOM_FOREST"]:
    df_prev4 = medium_prev4_eval_df[medium_prev4_eval_df["Model_Name"] == model_name].copy()
    if not df_prev4.empty:
        print_model_eval_report(
            df_prev4,
            title=f"{model_name} MEDIUM → PREV4",
            group_cols=["Medium_Subgroup"]
        )

for model_name in ["XGBOOST", "CATBOOST", "RANDOM_FOREST"]:
    df_recent4 = medium_recent4_eval_df[medium_recent4_eval_df["Model_Name"] == model_name].copy()
    if not df_recent4.empty:
        print_model_eval_report(
            df_recent4,
            title=f"{model_name} MEDIUM → RECENT4",
            group_cols=["Medium_Subgroup"]
        )

medium_prev4_model_table = build_model_summary_table(
    medium_prev4_eval_df,
    extra_group_cols=["Medium_Subgroup"]
)

medium_recent4_model_table = build_model_summary_table(
    medium_recent4_eval_df,
    extra_group_cols=["Medium_Subgroup"]
)

print("\n===== MEDIUM PREV4 MODEL TABLE =====")
print(medium_prev4_model_table)

print("\n===== MEDIUM RECENT4 MODEL TABLE =====")
print(medium_recent4_model_table)

# ============================================================
# 16) SKU-MODEL SUMMARIES
# ============================================================
medium_sku_model_prev4_summary = (
    medium_prev4_eval_df
    .groupby(["ItemCode", "Medium_Subgroup", "Model_Name"], as_index=False)
    .agg(
        Prev4_Months=("Month_Number", "count"),
        Prev4_Actual_Sum=("Actual", "sum"),
        Prev4_Pred_Sum=("Pred", "sum"),
        Prev4_MAE=("Abs_Error", "mean"),
        Prev4_Total_Abs_Error=("Abs_Error", "sum")
    )
)

medium_sku_model_prev4_summary["Prev4_WMAPE"] = np.where(
    medium_sku_model_prev4_summary["Prev4_Actual_Sum"] > 0,
    medium_sku_model_prev4_summary["Prev4_Total_Abs_Error"] /
    medium_sku_model_prev4_summary["Prev4_Actual_Sum"] * 100,
    np.nan
)

medium_sku_model_prev4_summary["Prev4_Bias"] = np.where(
    medium_sku_model_prev4_summary["Prev4_Actual_Sum"] > 0,
    (medium_sku_model_prev4_summary["Prev4_Pred_Sum"] -
     medium_sku_model_prev4_summary["Prev4_Actual_Sum"]) /
    medium_sku_model_prev4_summary["Prev4_Actual_Sum"] * 100,
    np.nan
)

medium_sku_model_recent4_summary = (
    medium_recent4_eval_df
    .groupby(["ItemCode", "Medium_Subgroup", "Model_Name"], as_index=False)
    .agg(
        Recent4_Months=("Month_Number", "count"),
        Recent4_Actual_Sum=("Actual", "sum"),
        Recent4_Pred_Sum=("Pred", "sum"),
        Recent4_MAE=("Abs_Error", "mean"),
        Recent4_Total_Abs_Error=("Abs_Error", "sum")
    )
)

medium_sku_model_recent4_summary["Recent4_WMAPE"] = np.where(
    medium_sku_model_recent4_summary["Recent4_Actual_Sum"] > 0,
    medium_sku_model_recent4_summary["Recent4_Total_Abs_Error"] /
    medium_sku_model_recent4_summary["Recent4_Actual_Sum"] * 100,
    np.nan
)

medium_sku_model_recent4_summary["Recent4_Bias"] = np.where(
    medium_sku_model_recent4_summary["Recent4_Actual_Sum"] > 0,
    (medium_sku_model_recent4_summary["Recent4_Pred_Sum"] -
     medium_sku_model_recent4_summary["Recent4_Actual_Sum"]) /
    medium_sku_model_recent4_summary["Recent4_Actual_Sum"] * 100,
    np.nan
)

for df_ in [medium_sku_model_prev4_summary, medium_sku_model_recent4_summary]:
    df_["ItemCode"] = df_["ItemCode"].astype(str)
    df_["Model_Name"] = df_["Model_Name"].astype(str).str.upper()


In [ ]:
# ============================================================
# 17) MERGE PREV4 + RECENT4
# ============================================================
current_medium_skus = get_current_medium_skus(Data)

medium_union_base = pd.concat([
    medium_sku_model_prev4_summary[["ItemCode", "Medium_Subgroup", "Model_Name"]],
    medium_sku_model_recent4_summary[["ItemCode", "Medium_Subgroup", "Model_Name"]],
], ignore_index=True).drop_duplicates()

medium_union_base = medium_union_base[
    medium_union_base["ItemCode"].isin(current_medium_skus)
].copy()

medium_sku_model_summary = (
    medium_union_base
    .merge(
        medium_sku_model_prev4_summary,
        on=["ItemCode", "Medium_Subgroup", "Model_Name"],
        how="left"
    )
    .merge(
        medium_sku_model_recent4_summary,
        on=["ItemCode", "Medium_Subgroup", "Model_Name"],
        how="left"
    )
)

fill_zero_cols = [
    "Prev4_Months", "Prev4_Actual_Sum", "Prev4_Pred_Sum", "Prev4_Total_Abs_Error",
    "Recent4_Months", "Recent4_Actual_Sum", "Recent4_Pred_Sum", "Recent4_Total_Abs_Error"
]
for c in fill_zero_cols:
    if c in medium_sku_model_summary.columns:
        medium_sku_model_summary[c] = medium_sku_model_summary[c].fillna(0)


In [ ]:
# ============================================================
# 18) CHAMPION SCORE
# ============================================================
def compute_medium_champion_score(row):
    if row.get("Prev4_Months", 0) < 2 and row.get("Recent4_Months", 0) < 2:
        return np.nan

    prev_wmape = row.get("Prev4_WMAPE", np.nan)
    recent_wmape = row.get("Recent4_WMAPE", np.nan)
    recent_bias = row.get("Recent4_Bias", 0)

    parts = []
    weights = []

    if pd.notna(prev_wmape) and row.get("Prev4_Months", 0) > 0:
        parts.append(prev_wmape)
        weights.append(0.35)

    if pd.notna(recent_wmape) and row.get("Recent4_Months", 0) > 0:
        parts.append(recent_wmape)
        weights.append(0.35)

    if pd.notna(recent_bias):
        parts.append(abs(recent_bias))
        weights.append(0.30)

    if len(parts) == 0:
        return np.nan

    weights = np.array(weights, dtype=float)
    weights = weights / weights.sum()

    score = float(np.sum(np.array(parts) * weights))

    if pd.notna(recent_bias) and recent_bias > 20:
        score += 5.0

    return score

medium_sku_model_summary["Champion_Score"] = medium_sku_model_summary.apply(
    compute_medium_champion_score,
    axis=1
)

print("\nMEDIUM SKU MODEL SUMMARY")
print(medium_sku_model_summary.head())

# ============================================================
# 19) CHAMPION MODEL PER SKU + SUBGROUP
# ============================================================
model_priority_map = {
    "XGBOOST": 1,
    "CATBOOST": 2,
    "RANDOM_FOREST": 3
}

medium_sku_model_summary["Model_Priority"] = (
    medium_sku_model_summary["Model_Name"]
    .map(model_priority_map)
    .fillna(999)
)

champion_medium_by_subgroup_df = (
    medium_sku_model_summary
    .sort_values(
        by=[
            "ItemCode",
            "Medium_Subgroup",
            "Champion_Score",
            "Recent4_WMAPE",
            "Prev4_WMAPE",
            "Prev4_MAE",
            "Model_Priority"
        ],
        ascending=[True, True, True, True, True, True, True]
    )
    .drop_duplicates(subset=["ItemCode", "Medium_Subgroup"], keep="first")
    .reset_index(drop=True)
)

champion_medium_by_subgroup_df = champion_medium_by_subgroup_df.rename(columns={
    "Model_Name": "Best_Model",
    "Champion_Score": "Best_Model_Score",
    "Prev4_WMAPE": "Best_Model_Prev4_WMAPE",
    "Recent4_WMAPE": "Best_Model_Recent4_WMAPE",
    "Prev4_MAE": "Best_Model_Prev4_MAE",
    "Recent4_MAE": "Best_Model_Recent4_MAE",
    "Prev4_Bias": "Best_Model_Prev4_Bias",
    "Recent4_Bias": "Best_Model_Recent4_Bias",
    "Prev4_Months": "Evaluation_Months_Prev4",
    "Recent4_Months": "Evaluation_Months_Recent4"
})

champion_medium_by_subgroup_df["Segment"] = "MEDIUM"

print("\nMEDIUM CHAMPION MAP BEFORE ROUTING")
print(champion_medium_by_subgroup_df.head())

# ============================================================
# 20) ROUTING RULES
# ============================================================
def assign_medium_final_routing(df):
    df = force_itemcode_str(df)
    df = df.copy()

    for c in [
        "Best_Model_Score",
        "Best_Model_Prev4_WMAPE",
        "Best_Model_Recent4_WMAPE"
    ]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df["Use_Fallback"] = np.where(
        (
            (df["Best_Model_Prev4_WMAPE"] > 85) &
            (df["Best_Model_Recent4_WMAPE"] > 85)
        ) |
        (
            df["Best_Model_Score"] > 75
        ),
        1,
        0
    )

    df["Final_Model"] = np.where(
        df["Use_Fallback"] == 1,
        "FALLBACK",
        df["Best_Model"]
    )

    df["Fallback_Type"] = np.where(
        df["Use_Fallback"] == 1,
        "ROLLING3",
        "NONE"
    )

    return df

champion_medium_by_subgroup_df["Unreliable_Flag"] = np.where(
    (
        (champion_medium_by_subgroup_df["Best_Model_Prev4_WMAPE"] > 80) &
        (champion_medium_by_subgroup_df["Best_Model_Recent4_WMAPE"] > 85)
    ) |
    (
        champion_medium_by_subgroup_df["Best_Model_Score"] > 75
    ),
    1, 0
)

champion_medium_by_subgroup_df = assign_medium_final_routing(champion_medium_by_subgroup_df)

print("\n===== MEDIUM ROUTING DIAGNOSTICS =====")
print(champion_medium_by_subgroup_df["Use_Fallback"].value_counts(dropna=False))
print(pd.crosstab(champion_medium_by_subgroup_df["Best_Model"], champion_medium_by_subgroup_df["Final_Model"]))
print(champion_medium_by_subgroup_df[[
    "ItemCode", "Medium_Subgroup", "Best_Model", "Final_Model",
    "Best_Model_Score", "Best_Model_Prev4_WMAPE",
    "Best_Model_Recent4_WMAPE", "Unreliable_Flag"
]].head(30))

In [ ]:
# ============================================================
# 21) COVERAGE CHECK
# ============================================================
print("\n===== MEDIUM COVERAGE BY SKU =====")
print("Unique SKUs in PREV4 summary:", medium_sku_model_prev4_summary["ItemCode"].nunique())
print("Unique SKUs in RECENT4 summary:", medium_sku_model_recent4_summary["ItemCode"].nunique())
print("Unique SKUs in merged medium_sku_model_summary:", medium_sku_model_summary["ItemCode"].nunique())

all_medium_skus = set(
    add_history_length_from_subset(Data.copy(), Data.copy())
    .query("History_Segment == 'MEDIUM'")["ItemCode"]
    .astype(str)
    .str.strip()
    .str.replace(".0", "", regex=False)
    .unique()
)

covered_medium_skus = set(
    medium_sku_model_summary["ItemCode"]
    .astype(str)
    .str.strip()
    .str.replace(".0", "", regex=False)
    .unique()
)

missing_medium_skus = sorted(all_medium_skus - covered_medium_skus)
print("\n[LOW HISTORY MEDIUM SKUs → RULE BASE]")
print("Count:", len(missing_medium_skus))

extra_medium_skus = sorted(covered_medium_skus - all_medium_skus)

print("Actual MEDIUM SKUs in data:", len(all_medium_skus))
print("Covered MEDIUM SKUs in comparison:", len(covered_medium_skus))
print("Missing MEDIUM SKUs from comparison:", len(missing_medium_skus))
print("Sample missing MEDIUM SKUs:", missing_medium_skus[:20])
print("Extra count:", len(extra_medium_skus))
print("Extra sample:", extra_medium_skus[:20])


# ============================================================
# 22) FINAL CHAMPION MAP (ONE ROW PER SKU)
# ============================================================
champion_medium_map_df = champion_medium_by_subgroup_df.copy()
champion_medium_map_df["ItemCode"] = champion_medium_map_df["ItemCode"].astype(str)
champion_medium_map_df["Medium_Subgroup"] = champion_medium_map_df["Medium_Subgroup"].astype(str).str.upper()

print("\n===== MEDIUM SUBGROUP CONSISTENCY CHECK =====")
sku_subgroup_counts = champion_medium_by_subgroup_df.groupby("ItemCode")["Medium_Subgroup"].nunique()
print("SKUs appearing in multiple subgroups:", (sku_subgroup_counts > 1).sum())
print(sku_subgroup_counts[sku_subgroup_counts > 1].head(20))

champion_medium_map_df = champion_medium_map_df[
    [
        "ItemCode",
        "Medium_Subgroup",
        "Segment",
        "Best_Model",
        "Final_Model",
        "Fallback_Type",
        "Use_Fallback",
        "Unreliable_Flag",
        "Best_Model_Score",
        "Best_Model_Prev4_WMAPE",
        "Best_Model_Recent4_WMAPE",
        "Best_Model_Prev4_MAE",
        "Best_Model_Recent4_MAE",
        "Best_Model_Prev4_Bias",
        "Best_Model_Recent4_Bias",
        "Evaluation_Months_Prev4",
        "Evaluation_Months_Recent4",
        "Prev4_Actual_Sum",
        "Prev4_Pred_Sum",
        "Recent4_Actual_Sum",
        "Recent4_Pred_Sum"
    ]
].copy()

champion_medium_map_df = (
    champion_medium_map_df
    .sort_values(["ItemCode", "Best_Model_Score"])
    .drop_duplicates(subset=["ItemCode"], keep="first")
    .reset_index(drop=True)
)

fallback_rows = pd.DataFrame({
    "ItemCode": missing_medium_skus,
    "Medium_Subgroup": "STABLE",
    "Segment": "MEDIUM",
    "Behavior_Type": "UNKNOWN",
    "Best_Model": "FALLBACK",
    "Final_Model": "FALLBACK",
    "Fallback_Type": "ROLLING3",
    "Use_Fallback": 1,
    "Unreliable_Flag": 1,
    "Best_Model_Score": np.nan,
    "Best_Model_Prev4_WMAPE": np.nan,
    "Best_Model_Recent4_WMAPE": np.nan,
    "Best_Model_Prev4_MAE": np.nan,
    "Best_Model_Recent4_MAE": np.nan,
    "Best_Model_Prev4_Bias": np.nan,
    "Best_Model_Recent4_Bias": np.nan,
    "Evaluation_Months_Prev4": 0,
    "Evaluation_Months_Recent4": 0,
    "Prev4_Actual_Sum": 0.0,
    "Prev4_Pred_Sum": 0.0,
    "Recent4_Actual_Sum": 0.0,
    "Recent4_Pred_Sum": 0.0,
})

# ============================================================
# ADD BEHAVIOR_TYPE TO MEDIUM CHAMPION MAP
# ============================================================

Data_medium = add_history_length_from_subset(Data.copy(), Data.copy())

Data_medium = Data_medium[Data_medium["History_Segment"] == "MEDIUM"].copy()

if "Behavior_Type" not in Data_medium.columns:

    Data_medium = rebuild_time_features(Data_medium)

    Data_medium["Behavior_Type"] = Data_medium.apply(classify_sku_behavior, axis=1)

medium_behavior_lookup_df = (
    Data_medium.copy()
    .sort_values(["ItemCode", "Year", "Month_Number"])
    .groupby("ItemCode")
    .tail(1)[["ItemCode", "Behavior_Type"]]
)

medium_behavior_lookup_df["ItemCode"] = (
    medium_behavior_lookup_df["ItemCode"]
    .astype(str)
    .str.replace(".0", "", regex=False)
)

champion_medium_map_df["ItemCode"] = (
    champion_medium_map_df["ItemCode"]
    .astype(str)
    .str.replace(".0", "", regex=False)
)

champion_medium_map_df = champion_medium_map_df.drop(
    columns=["Behavior_Type"],
    errors="ignore"
)

champion_medium_map_df = champion_medium_map_df.merge(
    medium_behavior_lookup_df,
    on="ItemCode",
    how="left"
)

champion_medium_map_df["Behavior_Type"] = (
    champion_medium_map_df["Behavior_Type"]
    .fillna("UNKNOWN")
)


champion_medium_map_df = pd.concat(
    [champion_medium_map_df, fallback_rows],
    ignore_index=True
)

print("\nFINAL MEDIUM CHAMPION MAP (ONE ROW PER SKU)")
print(champion_medium_map_df.head())
print("Unique medium SKUs in final champion map:", champion_medium_map_df["ItemCode"].nunique())
print("Duplicate ItemCodes in champion_medium_map_df:", champion_medium_map_df["ItemCode"].duplicated().sum())
print("Null Medium_Subgroup rows:", champion_medium_map_df["Medium_Subgroup"].isna().sum())
print("Null Best_Model rows:", champion_medium_map_df["Best_Model"].isna().sum())


# ============================================================
# 23) WIN COUNTS
# ============================================================
medium_model_win_counts = (
    champion_medium_map_df
    .groupby(["Medium_Subgroup", "Final_Model"], as_index=False)
    .size()
    .rename(columns={"size": "SKU_Count"})
)

print("\nMEDIUM MODEL WIN COUNTS")
print(medium_model_win_counts)

medium_best_model_win_counts = (
    champion_medium_map_df
    .groupby(["Medium_Subgroup", "Best_Model"], as_index=False)
    .size()
    .rename(columns={"size": "SKU_Count"})
)

print("\nMEDIUM RAW BEST MODEL WIN COUNTS")
print(medium_best_model_win_counts)


# ============================================================
# 24) MERGE CHAMPION BACK TO PREV4 ROW LEVEL
# ============================================================
medium_prev4_eval_df = force_itemcode_str(medium_prev4_eval_df)
champion_medium_map_df = force_itemcode_str(champion_medium_map_df)

medium_model_compare_with_champion = medium_prev4_eval_df.merge(
    champion_medium_map_df[
        [
            "ItemCode",
            "Medium_Subgroup",
            "Best_Model",
            "Final_Model",
            "Fallback_Type",
            "Use_Fallback",
            "Unreliable_Flag",
            "Best_Model_Score"
        ]
    ],
    on=["ItemCode", "Medium_Subgroup"],
    how="left"
)

medium_model_compare_with_champion["Is_Champion_Model"] = (
    medium_model_compare_with_champion["Model_Name"] == medium_model_compare_with_champion["Best_Model"]
).astype(int)


# ============================================================
# 25) OVERALL MODEL SUMMARY
# ============================================================
overall_medium_prev4_summary = (
    medium_prev4_eval_df
    .groupby(["Medium_Subgroup", "Model_Name"], as_index=False)
    .agg(
        Prev4_Actual_Sum=("Actual", "sum"),
        Prev4_Pred_Sum=("Pred", "sum"),
        Prev4_Total_Abs_Error=("Abs_Error", "sum"),
        Prev4_MAE=("Abs_Error", "mean")
    )
)

overall_medium_prev4_summary["Prev4_WMAPE"] = np.where(
    overall_medium_prev4_summary["Prev4_Actual_Sum"] > 0,
    overall_medium_prev4_summary["Prev4_Total_Abs_Error"] /
    overall_medium_prev4_summary["Prev4_Actual_Sum"] * 100,
    np.nan
)

overall_medium_prev4_summary["Prev4_Bias"] = np.where(
    overall_medium_prev4_summary["Prev4_Actual_Sum"] > 0,
    (overall_medium_prev4_summary["Prev4_Pred_Sum"] -
     overall_medium_prev4_summary["Prev4_Actual_Sum"]) /
    overall_medium_prev4_summary["Prev4_Actual_Sum"] * 100,
    np.nan
)

overall_medium_recent4_summary = (
    medium_recent4_eval_df
    .groupby(["Medium_Subgroup", "Model_Name"], as_index=False)
    .agg(
        Recent4_Actual_Sum=("Actual", "sum"),
        Recent4_Pred_Sum=("Pred", "sum"),
        Recent4_Total_Abs_Error=("Abs_Error", "sum"),
        Recent4_MAE=("Abs_Error", "mean")
    )
)

overall_medium_recent4_summary["Recent4_WMAPE"] = np.where(
    overall_medium_recent4_summary["Recent4_Actual_Sum"] > 0,
    overall_medium_recent4_summary["Recent4_Total_Abs_Error"] /
    overall_medium_recent4_summary["Recent4_Actual_Sum"] * 100,
    np.nan
)

overall_medium_recent4_summary["Recent4_Bias"] = np.where(
    overall_medium_recent4_summary["Recent4_Actual_Sum"] > 0,
    (overall_medium_recent4_summary["Recent4_Pred_Sum"] -
     overall_medium_recent4_summary["Recent4_Actual_Sum"]) /
    overall_medium_recent4_summary["Recent4_Actual_Sum"] * 100,
    np.nan
)

overall_medium_model_summary = (
    overall_medium_prev4_summary
    .merge(overall_medium_recent4_summary, on=["Medium_Subgroup", "Model_Name"], how="outer")
)

overall_medium_model_summary["Champion_Score"] = (
    0.40 * overall_medium_model_summary["Prev4_WMAPE"] +
    0.60 * overall_medium_model_summary["Recent4_WMAPE"]
)

print("\nOVERALL MEDIUM MODEL SUMMARY")
print(overall_medium_model_summary)

print("\n===== MEDIUM COVERAGE CHECK =====")
print(
    medium_sku_model_summary.groupby("Model_Name").agg(
        SKU_Count=("ItemCode", "nunique"),
        Avg_Prev4_Months=("Prev4_Months", "mean"),
        Avg_Recent4_Months=("Recent4_Months", "mean")
    )
)

# ============================================================
# 26) SAVE CHAMPION MAP
# ============================================================
joblib.dump(champion_medium_map_df, "champion_medium_map_df.pkl")
print("Saved: champion_medium_map_df.pkl")

print(champion_medium_map_df.shape)
print(champion_medium_map_df["Medium_Subgroup"].value_counts(dropna=False))
print(champion_medium_map_df["Best_Model"].value_counts(dropna=False))
print(champion_medium_map_df["Final_Model"].value_counts(dropna=False))

print(
    medium_sku_model_summary.groupby("Model_Name").agg(
        SKU_Count=("ItemCode", "nunique"),
        Avg_Prev4_Months=("Prev4_Months", "mean"),
        Avg_Recent4_Months=("Recent4_Months", "mean")
    )
)

In [ ]:
print("\n===== MEDIUM BEHAVIOR DIAGNOSTICS =====")

print("\nBehavior Type Counts:")
print(champion_medium_map_df["Behavior_Type"].value_counts(dropna=False))

print("\nBehavior Type by Subgroup:")
print(pd.crosstab(
    champion_medium_map_df["Behavior_Type"],
    champion_medium_map_df["Medium_Subgroup"]
))

print("\nFinal Model by Behavior Type:")
print(pd.crosstab(
    champion_medium_map_df["Behavior_Type"],
    champion_medium_map_df["Final_Model"]
))

print("\nFallback by Behavior Type:")
print(pd.crosstab(
    champion_medium_map_df["Behavior_Type"],
    champion_medium_map_df["Use_Fallback"]
))

#### Deployment

In [ ]:
# ============================================================
# 27) DEPLOYMENT REGISTRY
# ============================================================
medium_deployment_artifact_registry = pd.DataFrame([
    {"Medium_Subgroup": "PROMO_HEAVY", "Model_Name": "XGBOOST",       "Run_Key": "PROMO_HEAVY_XGBOOST",  "Deploy_Artifact_Key": "deploy_artifacts"},
    {"Medium_Subgroup": "PROMO_HEAVY", "Model_Name": "CATBOOST",      "Run_Key": "PROMO_HEAVY_CATBOOST", "Deploy_Artifact_Key": "deploy_artifacts"},
    {"Medium_Subgroup": "STABLE",      "Model_Name": "XGBOOST",       "Run_Key": "STABLE_XGBOOST",       "Deploy_Artifact_Key": "deploy_artifacts"},
    {"Medium_Subgroup": "STABLE",      "Model_Name": "CATBOOST",      "Run_Key": "STABLE_CATBOOST",      "Deploy_Artifact_Key": "deploy_artifacts"},
    {"Medium_Subgroup": "STABLE",      "Model_Name": "RANDOM_FOREST", "Run_Key": "STABLE_RANDOM_FOREST", "Deploy_Artifact_Key": "deploy_artifacts"},
])

medium_deployment_artifact_registry["Medium_Subgroup"] = (
    medium_deployment_artifact_registry["Medium_Subgroup"].astype(str).str.upper()
)
medium_deployment_artifact_registry["Model_Name"] = (
    medium_deployment_artifact_registry["Model_Name"].astype(str).str.upper()
)

print("\nMEDIUM DEPLOYMENT ARTIFACT REGISTRY")
print(medium_deployment_artifact_registry)

medium_deploy_model_registry = {
    ("PROMO_HEAVY", "XGBOOST"): medium_runs["PROMO_HEAVY_XGBOOST"]["deploy_artifacts"],
    ("PROMO_HEAVY", "CATBOOST"): medium_runs["PROMO_HEAVY_CATBOOST"]["deploy_artifacts"],
    ("STABLE", "XGBOOST"): medium_runs["STABLE_XGBOOST"]["deploy_artifacts"],
    ("STABLE", "CATBOOST"): medium_runs["STABLE_CATBOOST"]["deploy_artifacts"],
    ("STABLE", "RANDOM_FOREST"): medium_runs["STABLE_RANDOM_FOREST"]["deploy_artifacts"],
}


In [ ]:
# ============================================================
# 28) WINDOW SANITY CHECK
# ============================================================
print("\n===== MEDIUM WINDOW SANITY CHECK =====")
print("MEDIUM_TUNE_WINDOWS:", MEDIUM_TUNE_WINDOWS)
print("MEDIUM_PREV4_WINDOW:", MEDIUM_PREV4_WINDOW)
print("MEDIUM_RECENT4_WINDOW:", MEDIUM_RECENT4_WINDOW)

assert MEDIUM_PREV4_WINDOW["valid_end_idx"] < MEDIUM_RECENT4_WINDOW["valid_start_idx"], "MEDIUM PREV4 overlaps RECENT4"
print("MEDIUM window logic OK")

# ============================================================
# 29) SAVE EXCEL
# ============================================================
with pd.ExcelWriter("medium_model_comparison_and_champion_map.xlsx", engine="openpyxl") as writer:
    medium_prev4_eval_df.to_excel(writer, sheet_name="Prev4_RowLevel", index=False)
    medium_recent4_eval_df.to_excel(writer, sheet_name="Recent4_RowLevel", index=False)
    medium_sku_model_summary.to_excel(writer, sheet_name="SKU_Model_Summary", index=False)
    champion_medium_by_subgroup_df.to_excel(writer, sheet_name="Champion_By_Subgroup", index=False)
    champion_medium_map_df.to_excel(writer, sheet_name="Champion_Map_Final", index=False)
    medium_model_win_counts.to_excel(writer, sheet_name="Final_Model_Win_Counts", index=False)
    medium_best_model_win_counts.to_excel(writer, sheet_name="Best_Model_Win_Counts", index=False)
    overall_medium_model_summary.to_excel(writer, sheet_name="Overall_Model_Summary", index=False)
    medium_model_compare_with_champion.to_excel(writer, sheet_name="Eval_With_Routing", index=False)
    medium_deployment_artifact_registry.to_excel(writer, sheet_name="Deployment_Artifact_Registry", index=False)
    medium_prev4_model_table.to_excel(writer, sheet_name="Medium_Prev4_ModelPerf", index=False)
    medium_recent4_model_table.to_excel(writer, sheet_name="Medium_Recent4_ModelPerf", index=False)

print("\nSaved: medium_model_comparison_and_champion_map.xlsx")
print("Unique ItemCodes:", champion_medium_map_df["ItemCode"].nunique())
print("Total rows:", len(champion_medium_map_df))
print("Duplicate ItemCodes:", champion_medium_map_df["ItemCode"].duplicated().sum())

### Short History Segmnet

#### Model for SHORT HISTORU SKUs

In [ ]:
### Short History Segment
print("\n================ SHORT SEGMENT PIPELINE ================\n")

# ============================================================
# 0) SHORT FEATURE LIST
# ============================================================
SHORT_FEATURE_COLS = [
    "Lag1",
    "Lag2",
    "Rolling3M_Mean",
    "SKU_Mean_Demand",
    "Last_Bonus_Demand",
    "Avg_Bonus_Uplift",
    "Bonus_Flag",
    "Expected_Bonus_NextMonth",

    "Supply_Constraint_Flag",
    "Available_Primary_Inventory_Qty",
    "Distributor_Inventory_Qty",

    "Recent_Drop_Flag",
    "Demand_Rebound_Risk",
    "Primary_Stockout_Risk",
    "Distributor_Buffer_Available",
    "Suppressed_Demand_Flag",
    "Near_Bonus_Cycle",
    "Promo_Frequent_Weak_Flag",
    "Recurring_Special_Bonus_Risk",

    "Short_History_Length",
    "Short_Mean_Demand",
    "Short_ZeroRate",
    "Short_Bonus_Frequency",
    "Short_Bonus_Demand_Share",
    "Short_Supply_Rate",
]

# ============================================================
# 1) SHORT PROFILE BUILDING
# ============================================================
def build_short_sku_profile(train_df):
    train_df = force_itemcode_str(train_df)
    train_df = train_df.copy().sort_values(["ItemCode", "Year", "Month_Number"])
    out = []

    for item, g in train_df.groupby("ItemCode"):
        g = g.copy()

        hist_len = g[["Year", "Month_Number"]].drop_duplicates().shape[0]
        mean_demand = float(g["Clean_Demand"].mean()) if hist_len > 0 else 0.0
        zero_rate = float((g["Clean_Demand"] == 0).mean()) if hist_len > 0 else 0.0
        bonus_freq = float(g["Bonus_Flag"].mean()) if "Bonus_Flag" in g.columns else 0.0
        supply_rate = float(g["Supply_Constraint_Flag"].mean()) if "Supply_Constraint_Flag" in g.columns else 0.0

        total_demand = float(g["Clean_Demand"].sum())
        bonus_demand = float(g.loc[g["Bonus_Flag"] == 1, "Clean_Demand"].sum()) if total_demand > 0 else 0.0
        bonus_share = 0.0 if total_demand <= 0 else bonus_demand / total_demand

        if bonus_freq >= 0.25 or bonus_share >= 0.40:
            short_type = "SHORT_PROMO"
        else:
            short_type = "SHORT_NORMAL"

        out.append({
            "ItemCode": str(item),
            "Short_SKU_Type": short_type,
            "Short_History_Length": hist_len,
            "Short_Mean_Demand": mean_demand,
            "Short_ZeroRate": zero_rate,
            "Short_Bonus_Frequency": bonus_freq,
            "Short_Bonus_Demand_Share": bonus_share,
            "Short_Supply_Rate": supply_rate
        })

    out_df = pd.DataFrame(out)
    out_df = force_itemcode_str(out_df)
    return out_df


def merge_short_sku_profile(df, profile_df):
    df = force_itemcode_str(df)
    profile_df = force_itemcode_str(profile_df)
    df = df.copy()

    keep_cols = [
        "ItemCode",
        "Short_SKU_Type",
        "Short_History_Length",
        "Short_Mean_Demand",
        "Short_ZeroRate",
        "Short_Bonus_Frequency",
        "Short_Bonus_Demand_Share",
        "Short_Supply_Rate"
    ]

    df = df.drop(columns=[c for c in keep_cols if c != "ItemCode"], errors="ignore")
    df = df.merge(profile_df[keep_cols], on="ItemCode", how="left")
    df = force_itemcode_str(df)

    df["Short_SKU_Type"] = df["Short_SKU_Type"].fillna("SHORT_NORMAL")

    for c in [
        "Short_History_Length",
        "Short_Mean_Demand",
        "Short_ZeroRate",
        "Short_Bonus_Frequency",
        "Short_Bonus_Demand_Share",
        "Short_Supply_Rate"
    ]:
        df[c] = df[c].fillna(0)

    return df


def filter_short_subgroup(df, subgroup_name):
    df = force_itemcode_str(df)
    df = df.copy()

    subgroup_name = str(subgroup_name).upper()

    if subgroup_name == "SHORT_PROMO":
        return df[df["Short_SKU_Type"] == "SHORT_PROMO"].copy()

    if subgroup_name == "SHORT_NORMAL":
        return df[df["Short_SKU_Type"] == "SHORT_NORMAL"].copy()

    raise ValueError(f"Unknown short subgroup: {subgroup_name}")


# ============================================================
# 2) FOLD-SAFE SHORT PREP
# ============================================================
def prepare_short_frame_foldsafe(train_df, valid_df):
    train_df = force_itemcode_str(train_df)
    valid_df = force_itemcode_str(valid_df)

    train_df = train_df.copy().sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)
    valid_df = valid_df.copy().sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    # recurring bonus features from TRAIN only
    train_df, valid_df, _ = apply_recurring_bonus_features_foldsafe(train_df, valid_df)
    train_df = force_itemcode_str(train_df)
    valid_df = force_itemcode_str(valid_df)

    # train-derived mappings
    train_df, valid_df, abc_map = apply_fold_adjustments(train_df, valid_df)

    promo_profile_df = build_promo_profile(train_df)
    train_df = merge_promo_profile(train_df, promo_profile_df)
    valid_df = merge_promo_profile(valid_df, promo_profile_df)
    train_df = force_itemcode_str(train_df)
    valid_df = force_itemcode_str(valid_df)

    short_profile_df = build_short_sku_profile(train_df)
    train_df = merge_short_sku_profile(train_df, short_profile_df)
    valid_df = merge_short_sku_profile(valid_df, short_profile_df)
    train_df = force_itemcode_str(train_df)
    valid_df = force_itemcode_str(valid_df)

    train_df, valid_df, sku_profile_df = apply_sku_history_profile(train_df, valid_df)
    train_df = force_itemcode_str(train_df)
    valid_df = force_itemcode_str(valid_df)

    train_out, valid_out = rebuild_time_features_foldsafe(train_df, valid_df)
    train_out = force_itemcode_str(train_out)
    valid_out = force_itemcode_str(valid_out)

    return train_out, valid_out, abc_map, promo_profile_df, short_profile_df, sku_profile_df


# ============================================================
# 3) SHORT RULE MODEL
# ============================================================
def short_rule_predict(row):
    lag1 = float(row.get("Lag1", 0) or 0)
    lag2 = float(row.get("Lag2", 0) or 0)
    rolling3 = float(row.get("Rolling3M_Mean", 0) or 0)
    sku_mean = float(row.get("SKU_Mean_Demand", 0) or 0)

    last_bonus_demand = float(row.get("Last_Bonus_Demand", 0) or 0)
    avg_bonus_uplift = float(row.get("Avg_Bonus_Uplift", 1.0) or 1.0)

    expected_bonus = int(row.get("Expected_Bonus_NextMonth", 0) or 0)
    supply_flag = int(row.get("Supply_Constraint_Flag", 0) or 0)

    primary_stock = float(row.get("Available_Primary_Inventory_Qty", 0) or 0)
    distributor_stock = float(row.get("Distributor_Inventory_Qty", 0) or 0)

    short_type = row.get("Short_SKU_Type", "SHORT_NORMAL")
    hist_len = int(row.get("Short_History_Length", 0) or 0)

    anchors = [x for x in [lag1, lag2, rolling3, sku_mean] if x > 0]

    if len(anchors) == 0:
        pred = 0.0
    elif hist_len <= 2:
        pred = float(np.mean(anchors))
    else:
        pred = max(
            0.50 * float(np.mean(anchors)) + 0.50 * float(np.max(anchors)),
            0.75 * sku_mean if sku_mean > 0 else 0.0
        )

    if short_type == "SHORT_PROMO" and expected_bonus == 1 and last_bonus_demand > 0:
        pred = 0.50 * pred + 0.50 * last_bonus_demand
        pred = pred * min(max(avg_bonus_uplift, 1.0), 1.6)

    if supply_flag == 1:
        safe_cap = max(lag1, rolling3, sku_mean, 0)
        pred = min(pred, 1.10 * safe_cap)

    if (primary_stock + distributor_stock) <= 0:
        pred *= 0.90

    return max(pred, 0.0)


# ============================================================
# 4) SHORT HOLDOUT EVALUATION
# ============================================================
def evaluate_short_rule_model(full_data, subgroup_name=None):
    full_data = add_period_index(full_data)

    train_df = full_data[full_data["Period_Index"] < TIME_WINDOWS["test_start_idx"]].copy()
    test_df = full_data[
        full_data["Period_Index"].between(TIME_WINDOWS["test_start_idx"], TIME_WINDOWS["latest_idx"])
    ].copy()

    if train_df.empty or test_df.empty:
        raise ValueError("Need both pre-holdout train data and holdout test data.")

    train_df = add_history_length_from_subset(train_df, train_df)
    test_df = add_history_length_from_subset(train_df, test_df)

    train_df = train_df[train_df["History_Segment"] == "SHORT"].copy()
    test_df = test_df[test_df["History_Segment"] == "SHORT"].copy()

    if train_df.empty or test_df.empty:
        raise ValueError("No SHORT rows available.")

    train_df, test_df, abc_map, promo_profile_df, short_profile_df, sku_profile_df = prepare_short_frame_foldsafe(
        train_df, test_df
    )

    if subgroup_name is not None:
        subgroup_name = str(subgroup_name).upper()
        train_df = filter_short_subgroup(train_df, subgroup_name)
        test_df = filter_short_subgroup(test_df, subgroup_name)

    if train_df.empty or test_df.empty:
        raise ValueError(f"No usable SHORT rows for subgroup: {subgroup_name}")

    caps = compute_clip_caps(train_df, cols=["Inventory_Pressure", "Stock_Cover_Months"], q=0.99)
    train_df = apply_clip_caps(train_df, caps)
    test_df = apply_clip_caps(test_df, caps)

    train_df = recompute_target(train_df)
    test_df = recompute_target(test_df)

    test_df = test_df.dropna(subset=[ACTUAL_TARGET_COL]).copy()
    test_df = force_itemcode_str(test_df)

    if test_df.empty:
        raise ValueError("No usable SHORT rows after target creation.")

    test_df["ItemCode_Original"] = test_df["ItemCode"].astype(str)
    test_df["Pred"] = test_df.apply(short_rule_predict, axis=1)

    metrics = evaluate_all_metrics(test_df[ACTUAL_TARGET_COL].values, test_df["Pred"].values)

    artifacts = {
        "model": None,
        "feature_cols": SHORT_FEATURE_COLS,
        "best_params": None,
        "itemcode_categories": None,
        "abc_map": abc_map,
        "clip_caps": caps,
        "promo_profile_df": promo_profile_df,
        "short_profile_df": short_profile_df,
        "sku_profile_df": sku_profile_df,
        "target_mode": "rule_based",
        "baseline_col": None,
        "actual_target_col": ACTUAL_TARGET_COL,
        "model_target_col": None,
        "segment": "SHORT",
        "subgroup_name": subgroup_name if subgroup_name is not None else "ALL_SHORT",
        "rule_name": "short_rule_v2"
    }

    return artifacts, test_df, metrics


# ============================================================
# 5) SHORT DEPLOYMENT PREP
# ============================================================
def prepare_short_rule_deployment(full_data, subgroup_name=None):
    deploy_df = force_itemcode_str(full_data)
    deploy_df = deploy_df.copy().sort_values(["ItemCode", "Year", "Month_Number"])

    deploy_df = add_history_length_from_subset(deploy_df, deploy_df)
    deploy_df = deploy_df[deploy_df["History_Segment"] == "SHORT"].copy()

    if deploy_df.empty:
        artifacts = {
            "model": None,
            "feature_cols": SHORT_FEATURE_COLS,
            "best_params": None,
            "itemcode_categories": None,
            "abc_map": {},
            "clip_caps": {},
            "promo_profile_df": pd.DataFrame(),
            "short_profile_df": pd.DataFrame(),
            "sku_profile_df": pd.DataFrame(),
            "target_mode": "rule_based",
            "baseline_col": None,
            "actual_target_col": ACTUAL_TARGET_COL,
            "model_target_col": None,
            "segment": "SHORT",
            "subgroup_name": subgroup_name if subgroup_name is not None else "ALL_SHORT",
            "rule_name": "short_rule_v2"
        }
        return artifacts, pd.DataFrame()

    bonus_pattern_df = detect_recurring_bonus_skus(deploy_df)[[
        "ItemCode",
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ]].copy()

    deploy_df = deploy_df.drop(columns=[
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ], errors="ignore")

    bonus_pattern_df = force_itemcode_str(bonus_pattern_df)
    deploy_df = force_itemcode_str(deploy_df)
    deploy_df = deploy_df.merge(bonus_pattern_df, on="ItemCode", how="left")

    for c in ["Recurring_Bonus_SKU", "Bonus_Cycle_Length", "Avg_Bonus_Gap", "Bonus_Frequency_All"]:
        deploy_df[c] = deploy_df[c].fillna(0)
    deploy_df["Avg_Bonus_Uplift"] = deploy_df["Avg_Bonus_Uplift"].fillna(1.0)

    deploy_df, _ = apply_sku_cap(deploy_df.copy(), deploy_df.copy())

    sku_total = deploy_df.groupby("ItemCode")["Clean_Demand"].sum().sort_values(ascending=False)
    total_sum = sku_total.sum()

    if total_sum > 0:
        cum_pct = sku_total.cumsum() / total_sum
        abc_series = pd.cut(cum_pct, bins=[0, 0.7, 0.9, 1.0], labels=[0, 1, 2])
        abc_map = abc_series.to_dict()
        deploy_df["ABC_Class"] = deploy_df["ItemCode"].map(abc_map).fillna(2)
    else:
        abc_map = {}
        deploy_df["ABC_Class"] = 2

    promo_profile_df = build_promo_profile(deploy_df)
    deploy_df = merge_promo_profile(deploy_df, promo_profile_df)

    short_profile_df = build_short_sku_profile(deploy_df)
    deploy_df = merge_short_sku_profile(deploy_df, short_profile_df)

    sku_profile_df = (
        deploy_df.groupby("ItemCode")
        .agg(
            SKU_Mean_Demand=("Clean_Demand", "mean"),
            SKU_Std_Demand=("Clean_Demand", "std"),
            SKU_ZeroRate=("Clean_Demand", lambda x: (x == 0).mean())
        )
        .reset_index()
    )

    sku_profile_df["SKU_Std_Demand"] = sku_profile_df["SKU_Std_Demand"].fillna(0)
    sku_profile_df["SKU_CV"] = np.where(
        sku_profile_df["SKU_Mean_Demand"] <= 0,
        0,
        sku_profile_df["SKU_Std_Demand"] / (sku_profile_df["SKU_Mean_Demand"] + 1)
    )

    sku_profile_df = force_itemcode_str(sku_profile_df)
    sku_profile_df = sku_profile_df[["ItemCode", "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]]

    deploy_df = force_itemcode_str(deploy_df)
    deploy_df = deploy_df.drop(columns=["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"], errors="ignore")
    deploy_df = deploy_df.merge(sku_profile_df, on="ItemCode", how="left")

    for c in ["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]:
        deploy_df[c] = deploy_df[c].fillna(0)

    if subgroup_name is not None:
        deploy_df = filter_short_subgroup(deploy_df, subgroup_name)

    if deploy_df.empty:
        artifacts = {
            "model": None,
            "feature_cols": SHORT_FEATURE_COLS,
            "best_params": None,
            "itemcode_categories": None,
            "abc_map": abc_map,
            "clip_caps": {},
            "promo_profile_df": promo_profile_df,
            "short_profile_df": short_profile_df,
            "sku_profile_df": sku_profile_df,
            "target_mode": "rule_based",
            "baseline_col": None,
            "actual_target_col": ACTUAL_TARGET_COL,
            "model_target_col": None,
            "segment": "SHORT",
            "subgroup_name": subgroup_name if subgroup_name is not None else "ALL_SHORT",
            "rule_name": "short_rule_v2"
        }
        return artifacts, pd.DataFrame()

    deploy_df = add_bonus_cycle_features(deploy_df)
    deploy_df = rebuild_time_features(deploy_df)

    caps = compute_clip_caps(deploy_df, cols=["Inventory_Pressure", "Stock_Cover_Months"], q=0.99)
    deploy_df = apply_clip_caps(deploy_df, caps)

    artifacts = {
        "model": None,
        "feature_cols": SHORT_FEATURE_COLS,
        "best_params": None,
        "itemcode_categories": None,
        "abc_map": abc_map,
        "clip_caps": caps,
        "promo_profile_df": promo_profile_df,
        "short_profile_df": short_profile_df,
        "sku_profile_df": sku_profile_df,
        "target_mode": "rule_based",
        "baseline_col": None,
        "actual_target_col": ACTUAL_TARGET_COL,
        "model_target_col": None,
        "segment": "SHORT",
        "subgroup_name": subgroup_name if subgroup_name is not None else "ALL_SHORT",
        "rule_name": "short_rule_v2"
    }

    return artifacts, deploy_df


# ============================================================
# 6) RUN SHORT HOLDOUT EVALUATION
# ============================================================
short_eval_artifacts, short_holdout_test_df, short_eval_metrics = evaluate_short_rule_model(
    full_data=Data,
    subgroup_name=None
)
print("\n===== SHORT ALL RULE EVALUATION METRICS =====")
print(short_eval_metrics)

short_promo_eval_artifacts, short_promo_holdout_test_df, short_promo_eval_metrics = evaluate_short_rule_model(
    full_data=Data,
    subgroup_name="SHORT_PROMO"
)
print("\n===== SHORT_PROMO RULE EVALUATION METRICS =====")
print(short_promo_eval_metrics)

short_normal_eval_artifacts, short_normal_holdout_test_df, short_normal_eval_metrics = evaluate_short_rule_model(
    full_data=Data,
    subgroup_name="SHORT_NORMAL"
)
print("\n===== SHORT_NORMAL RULE EVALUATION METRICS =====")
print(short_normal_eval_metrics)

def standardize_short_rule_output(df, subgroup_label="ALL_SHORT"):
    out = df.copy()
    out = force_itemcode_str(out)

    out["ItemCode_Original"] = out["ItemCode"].astype(str)
    out["ItemCode"] = out["ItemCode_Original"]

    out["Actual"] = pd.to_numeric(out[ACTUAL_TARGET_COL], errors="coerce")
    out["Pred"] = pd.to_numeric(out["Pred"], errors="coerce").clip(lower=0)
    out["Error"] = out["Actual"] - out["Pred"]
    out["Abs_Error"] = np.abs(out["Error"])

    out["Segment"] = "SHORT"
    out["Model_Name"] = "RULE_BASED"
    out["Short_Subgroup"] = subgroup_label

    return out

short_holdout_test_std = standardize_short_rule_output(
    short_holdout_test_df,
    subgroup_label="ALL_SHORT"
)

short_promo_holdout_test_std = standardize_short_rule_output(
    short_promo_holdout_test_df,
    subgroup_label="SHORT_PROMO"
)

short_normal_holdout_test_std = standardize_short_rule_output(
    short_normal_holdout_test_df,
    subgroup_label="SHORT_NORMAL"
)

# ============================================================
# 7) SHORT MODEL PERFORMANCE REPORTS
# ============================================================
print_model_eval_report(
    short_holdout_test_std,
    title="RULE_BASED SHORT → HOLDOUT",
    group_cols=["Short_SKU_Type"] if "Short_SKU_Type" in short_holdout_test_std.columns else None
)

print_model_eval_report(
    short_promo_holdout_test_std,
    title="RULE_BASED SHORT_PROMO → HOLDOUT",
    group_cols=["Short_SKU_Type"] if "Short_SKU_Type" in short_promo_holdout_test_std.columns else None
)

print_model_eval_report(
    short_normal_holdout_test_std,
    title="RULE_BASED SHORT_NORMAL → HOLDOUT",
    group_cols=["Short_SKU_Type"] if "Short_SKU_Type" in short_normal_holdout_test_std.columns else None
)

short_holdout_model_table = build_model_summary_table(
    short_holdout_test_std,
    extra_group_cols=["Short_SKU_Type"] if "Short_SKU_Type" in short_holdout_test_std.columns else None
)

print("\n===== SHORT HOLDOUT MODEL TABLE =====")
print(short_holdout_model_table)


# ============================================================
# 8) RUN SHORT DEPLOYMENT PREP
# ============================================================
short_deploy_artifacts, short_deploy_train_df = prepare_short_rule_deployment(
    full_data=Data,
    subgroup_name=None
)

short_promo_deploy_artifacts, short_promo_deploy_df = prepare_short_rule_deployment(
    full_data=Data,
    subgroup_name="SHORT_PROMO"
)

short_normal_deploy_artifacts, short_normal_deploy_df = prepare_short_rule_deployment(
    full_data=Data,
    subgroup_name="SHORT_NORMAL"
)

print("\nSHORT deployment rows:", len(short_deploy_train_df))
print("SHORT_PROMO deployment rows:", len(short_promo_deploy_df))
print("SHORT_NORMAL deployment rows:", len(short_normal_deploy_df))


# ============================================================
# 9) SHORT OPTIONAL ROW-LEVEL TAGS
# ============================================================
short_holdout_test_df = force_itemcode_str(short_holdout_test_df)
short_promo_holdout_test_df = force_itemcode_str(short_promo_holdout_test_df)
short_normal_holdout_test_df = force_itemcode_str(short_normal_holdout_test_df)

if "Short_SKU_Type" in short_holdout_test_df.columns:
    short_holdout_test_df["Short_Subgroup"] = short_holdout_test_df["Short_SKU_Type"].astype(str).str.upper()
else:
    short_holdout_test_df["Short_Subgroup"] = "UNKNOWN"

short_promo_holdout_test_df["Short_Subgroup"] = "SHORT_PROMO"
short_normal_holdout_test_df["Short_Subgroup"] = "SHORT_NORMAL"

print("\nSHORT HOLDOUT TEST ROWS (ALL):", len(short_holdout_test_df))
print("SHORT HOLDOUT TEST ROWS (PROMO):", len(short_promo_holdout_test_df))
print("SHORT HOLDOUT TEST ROWS (NORMAL):", len(short_normal_holdout_test_df))


# ============================================================
# 10) SAVE SHORT ARTIFACTS
# ============================================================
joblib.dump(short_eval_artifacts, "short_rule_eval_artifacts.pkl")
joblib.dump(short_deploy_artifacts, "short_rule_deploy_artifacts.pkl")

joblib.dump(short_promo_eval_artifacts, "short_promo_rule_eval_artifacts.pkl")
joblib.dump(short_promo_deploy_artifacts, "short_promo_rule_deploy_artifacts.pkl")

joblib.dump(short_normal_eval_artifacts, "short_normal_rule_eval_artifacts.pkl")
joblib.dump(short_normal_deploy_artifacts, "short_normal_rule_deploy_artifacts.pkl")

print("\nSaved:")
print(" - short_rule_eval_artifacts.pkl")
print(" - short_rule_deploy_artifacts.pkl")
print(" - short_promo_rule_eval_artifacts.pkl")
print(" - short_promo_rule_deploy_artifacts.pkl")
print(" - short_normal_rule_eval_artifacts.pkl")
print(" - short_normal_rule_deploy_artifacts.pkl")


# ============================================================
# 11) SAVE SHORT REPORT
# ============================================================
with pd.ExcelWriter("short_model_comparison_and_rule_outputs.xlsx", engine="openpyxl") as writer:
    short_holdout_test_std.to_excel(writer, sheet_name="Short_Holdout_All", index=False)
    short_promo_holdout_test_std.to_excel(writer, sheet_name="Short_Holdout_Promo", index=False)
    short_normal_holdout_test_std.to_excel(writer, sheet_name="Short_Holdout_Normal", index=False)
    short_holdout_model_table.to_excel(writer, sheet_name="Short_Holdout_ModelPerf", index=False)

print("\nSaved: short_model_comparison_and_rule_outputs.xlsx")

# Inference

In [ ]:
''' ============================================================
# CLEAN FINAL INFERENCE BLOCK
# ============================================================
# PURPOSE
# - Use latest snapshot data already available in Cleaned_Base_Data
# - Re-segment each SKU by history length
# - Route each SKU to LONG / MEDIUM / SHORT
# - Within LONG / MEDIUM, use champion map to choose best model
# - If champion map says fallback, use fallback
# - If LONG champion is GRU, use GRU-specific inference
# - Predict NEXT MONTH only
#
# EXPECTED IN MEMORY
# - Cleaned_Base_Data
# - PHARMA_SKUS
# - champion_long_map_df
# - champion_medium_map_df
# - medium_deploy_artifact_registry
# - medium_runs
# - short_deploy_artifacts
# - short_promo_deploy_artifacts
# - short_normal_deploy_artifacts
# - long_deploy_artifacts
# - catboost_long_deploy_artifacts
# - lgbm_long_deploy_artifacts
# - gru_long_deploy_artifacts
# - gru_long_deploy_scalers
#
# SHARED HELPERS ALREADY DEFINED EARLIER IN NOTEBOOK
# - add_history_length_from_subset
# - detect_recurring_bonus_skus
# - add_bonus_cycle_features
# - merge_promo_profile
# - build_promo_profile
# - merge_medium_sku_profile                  (for medium only)
# - build_short_sku_profile
# - merge_short_sku_profile
# - rebuild_time_features
# - recompute_target
# - add_residual_target
# - apply_clip_caps
# - assert_features_exist
# - sanitize
# - gru_signed_log_inverse
# - LongGRUResidualForecaster
# ============================================================'''

In [ ]:
# ============================================================
# LONG INFERENCE + ROUTING (CLEAN REWRITE)
# ============================================================

# ------------------------------------------------------------
# 0) SMALL SHARED HELPERS LONG+MEDIUM+SHORT
# ------------------------------------------------------------
def force_itemcode_str(df):
    df = df.copy()
    if "ItemCode" in df.columns:
        df["ItemCode"] = df["ItemCode"].astype(str)
    if "ItemCode_Original" in df.columns:
        df["ItemCode_Original"] = df["ItemCode_Original"].astype(str)
    return df


def sanitize_model_input(df):
    x = df.copy()
    x = x.replace([np.inf, -np.inf], np.nan)
    for c in x.columns:
        x[c] = pd.to_numeric(x[c], errors="coerce")
    return x.fillna(0)


def ensure_inference_features(df, feature_cols):
    df = df.copy()
    for c in feature_cols:
        if c not in df.columns:
            df[c] = 0
    return df


def choose_segment_by_history(raw_data, sku_code):
    sku_code = str(sku_code)
    work = raw_data.copy()
    work["ItemCode"] = work["ItemCode"].astype(str)

    sku_df = work[work["ItemCode"] == sku_code].copy()
    if sku_df.empty:
        return "SHORT"

    if "History_Segment" in sku_df.columns:
        sku_df = sku_df.sort_values(["Year", "Month_Number"])
        return str(sku_df["History_Segment"].iloc[-1])

    hist_len = sku_df[["Year", "Month_Number"]].drop_duplicates().shape[0]
    if hist_len >= 18:
        return "LONG"
    elif hist_len >= 6:
        return "MEDIUM"
    return "SHORT"


def get_next_period_from_history(df, sku_code):
    sku_code = str(sku_code)

    work = df.copy()
    work["ItemCode_Original"] = work["ItemCode"].astype(str)

    sku_hist = work[work["ItemCode_Original"] == sku_code].copy()
    if sku_hist.empty:
        return None, None, None

    last_row = sku_hist.sort_values(["Year", "Month_Number"]).iloc[-1:].copy()

    next_month = int(last_row["Month_Number"].iloc[0]) + 1
    next_year = int(last_row["Year"].iloc[0])

    if next_month > 12:
        next_month = 1
        next_year += 1

    return last_row, next_year, next_month


def infer_expected_bonus_flag(sku_code, prepared_df):
    sku_code = str(sku_code)

    work = prepared_df.copy()
    if "ItemCode_Original" not in work.columns:
        work["ItemCode_Original"] = work["ItemCode"].astype(str)

    g = work[work["ItemCode_Original"] == sku_code].copy()
    g = g.sort_values(["Year", "Month_Number"])

    if g.empty:
        return 0

    recent_bonus_rate = g["Bonus_Flag"].tail(12).mean() if "Bonus_Flag" in g.columns else 0
    recurring = g["Recurring_Bonus_SKU"].iloc[-1] if "Recurring_Bonus_SKU" in g.columns else 0
    cycle_len = g["Bonus_Cycle_Length"].iloc[-1] if "Bonus_Cycle_Length" in g.columns else 0
    months_since = g["Months_Since_Last_Bonus"].iloc[-1] if "Months_Since_Last_Bonus" in g.columns else 999

    if recurring == 1 and cycle_len > 0 and abs((months_since + 1) - cycle_len) <= 1:
        return 1

    if recurring == 1 and recent_bonus_rate >= 0.25:
        return 1

    return 0


def apply_residual_strength(row, pred_residual, segment="LONG"):
    baseline = float(row.get(BASELINE_COL, 0) or 0)
    rolling3 = float(row.get("Rolling3M_Mean", 0) or 0)
    lag1 = float(row.get("Lag1", 0) or 0)
    sku_mean = float(row.get("SKU_Mean_Demand", 0) or 0)

    anchor = max(baseline, rolling3, lag1, sku_mean, 1.0)

    if segment == "LONG":
        pos_strength = 1.25
        neg_strength = 1.05
    elif segment == "MEDIUM":
        pos_strength = 1.15
        neg_strength = 1.00
    else:
        pos_strength = 1.00
        neg_strength = 1.00

    residual = float(pred_residual)

    if residual >= 0:
        adjusted = residual * pos_strength
    else:
        adjusted = residual * neg_strength

    lower = -0.45 * anchor
    upper = 1.20 * anchor

    return float(np.clip(adjusted, lower, upper))


def classify_sku_behavior(row):
    cv = float(row.get("SKU_CV", 0) or 0)
    zero_rate = float(row.get("SKU_ZeroRate", 0) or 0)
    promo_profile = str(row.get("Promo_Profile", "NORMAL"))
    bonus_freq = float(row.get("Bonus_Frequency_All", 0) or 0)

    lag1 = float(row.get("Lag1", 0) or 0)
    lag2 = float(row.get("Lag2", 0) or 0)
    rolling3 = float(row.get("Rolling3M_Mean", 0) or 0)
    rolling6 = float(row.get("Rolling6M_Mean", rolling3) or rolling3)
    sku_mean = float(row.get("SKU_Mean_Demand", 0) or 0)

    anchor = max(rolling3, rolling6, sku_mean, 1)

    # Priority order matters
    if zero_rate >= 0.40:
        return "INTERMITTENT"

    if lag1 > 2.5 * anchor or lag2 > 2.5 * anchor:
        return "RECENT_SPIKE"

    if lag1 < 0.35 * anchor and rolling3 > 100:
        return "RECENT_DROP"

    if sku_mean < 100 and cv >= 1.0:
        return "LOW_VOLUME_VOLATILE"

    if promo_profile in ["PROMO_INFLUENCED", "PURE_PROMO"] or bonus_freq >= 0.25:
        return "PROMO_DRIVEN"

    if cv >= 1.0:
        return "VOLATILE"

    return "STABLE"


def get_latest_behavior_type(raw_data, sku_code):
    sku_code = str(sku_code)

    if "Behavior_Type" not in raw_data.columns:
        return "UNKNOWN"

    g = raw_data[raw_data["ItemCode"].astype(str) == sku_code].copy()

    if g.empty:
        return "UNKNOWN"

    g = g.sort_values(["Year", "Month_Number"])
    return str(g["Behavior_Type"].iloc[-1])

In [ ]:
# =========================
# SAFE BEHAVIOR TYPE SETUP FOR INFERENCE
# =========================
Cleaned_Base_Data = force_itemcode_str(Cleaned_Base_Data)

if "History_Segment" not in Cleaned_Base_Data.columns:
    Cleaned_Base_Data = add_history_length_from_subset(
        Cleaned_Base_Data.copy(),
        Cleaned_Base_Data.copy()
    )

if "Behavior_Type" not in Cleaned_Base_Data.columns:
    print("⚠️ Behavior_Type missing in Cleaned_Base_Data. Adding inference-only Behavior_Type.")
    temp_behavior_df = rebuild_time_features(Cleaned_Base_Data.copy())
    temp_behavior_df["Behavior_Type"] = temp_behavior_df.apply(classify_sku_behavior, axis=1)

    latest_behavior_df = (
        temp_behavior_df
        .sort_values(["ItemCode", "Year", "Month_Number"])
        .groupby("ItemCode")
        .tail(1)[["ItemCode", "Behavior_Type"]]
    )

    Cleaned_Base_Data = Cleaned_Base_Data.drop(columns=["Behavior_Type"], errors="ignore")
    Cleaned_Base_Data = Cleaned_Base_Data.merge(
        latest_behavior_df,
        on="ItemCode",
        how="left"
    )

if "Behavior_Type" not in Cleaned_Base_Data.columns:
    Cleaned_Base_Data["Behavior_Type"] = "UNKNOWN"

Cleaned_Base_Data["Behavior_Type"] = Cleaned_Base_Data["Behavior_Type"].fillna("UNKNOWN")

In [ ]:
# ------------------------------------------------------------
# BEHAVIOR TYPE LOOKUP
# ------------------------------------------------------------
def build_behavior_type_map(raw_data):
    work = raw_data.copy()
    work["ItemCode"] = work["ItemCode"].astype(str)

    latest_behavior = (
        work.sort_values(["ItemCode", "Year", "Month_Number"])
        .groupby("ItemCode")["Behavior_Type"]
        .last()
        .to_dict()
    )

    return {str(k): str(v) for k, v in latest_behavior.items()}

behavior_type_map = build_behavior_type_map(Cleaned_Base_Data)

def get_behavior_type(sku_code):
    return behavior_type_map.get(str(sku_code), "UNKNOWN")

### LONG SEGMENT

In [ ]:
# ------------------------------------------------------------
# 1) LONG CHAMPION LOOKUP
# ------------------------------------------------------------
def get_long_routing_row(sku_code):
    sku_code = str(sku_code)
    df = champion_long_map_df.copy()
    df["ItemCode"] = df["ItemCode"].astype(str)

    hit = df[df["ItemCode"] == sku_code].copy()
    if hit.empty:
        return None
    return hit.iloc[0].to_dict()


def get_long_deploy_artifact(model_name):
    model_name = str(model_name).upper()

    if model_name == "XGBOOST":
        return globals().get("xgb_long_deploy_artifacts")
    if model_name == "CATBOOST":
        return globals().get("catboost_long_deploy_artifacts")
    if model_name == "LIGHTGBM":
        return globals().get("lgbm_long_deploy_artifacts")

    return None


# ------------------------------------------------------------
# 2) TREE MODEL POST-PROCESSING
# ------------------------------------------------------------
def apply_promo_aware_adjustment(row, raw_final_pred):
    raw_final_pred = max(float(raw_final_pred), 0.0)

    promo_profile = row.get("Promo_Profile", "NORMAL")
    next_bonus = int(row.get("Bonus_Flag", 0))

    rolling_mean = float(row.get("Rolling3M_Mean", 0) or 0)
    lag1 = float(row.get("Lag1", 0) or 0)
    last_bonus_demand = float(row.get("Last_Bonus_Demand", 0) or 0)

    bonus_avg = float(row.get("Bonus_Avg_Demand", 0) or 0)
    non_bonus_avg = float(row.get("NonBonus_Avg_Demand", 0) or 0)
    uplift_ratio = float(row.get("Bonus_Uplift_Ratio_Profile", 1.0) or 1.0)

    normal_anchor = max(rolling_mean, lag1, non_bonus_avg, 0)
    bonus_anchor = max(bonus_avg, last_bonus_demand, rolling_mean, lag1, 0)

    if promo_profile == "NORMAL":
        if normal_anchor > 0:
            raw_final_pred = min(raw_final_pred, normal_anchor * 2.5)
        return raw_final_pred

    if promo_profile == "PROMO_INFLUENCED":
        if next_bonus == 1:
            bounded_uplift = min(max(uplift_ratio, 1.0), 1.60)
            uplift_pred = raw_final_pred * bounded_uplift

            if bonus_anchor > 0:
                adjusted = 0.55 * uplift_pred + 0.45 * bonus_anchor
            else:
                adjusted = uplift_pred

            floor_val = max(raw_final_pred, normal_anchor * 1.05)
            cap_val = max(floor_val, bonus_anchor * 1.35 if bonus_anchor > 0 else uplift_pred)

            return min(max(adjusted, floor_val), cap_val)
        else:
            if non_bonus_avg > 0:
                return max(0.70 * raw_final_pred + 0.30 * non_bonus_avg, 0)
            return raw_final_pred

    if promo_profile == "PURE_PROMO":
        if next_bonus == 1:
            if bonus_avg > 0 and last_bonus_demand > 0:
                adjusted = 0.60 * bonus_avg + 0.40 * last_bonus_demand
            elif bonus_avg > 0:
                adjusted = bonus_avg
            else:
                adjusted = raw_final_pred

            if bonus_anchor > 0:
                adjusted = min(adjusted, bonus_anchor * 1.20)

            return max(adjusted, raw_final_pred * 0.90)
        else:
            if non_bonus_avg > 0:
                return min(raw_final_pred, max(non_bonus_avg, normal_anchor))
            return min(raw_final_pred, max(rolling_mean * 0.60, lag1 * 0.60, 0))

    return raw_final_pred


def apply_final_forecast_guardrails(row, pred):
    pred = max(float(pred), 0.0)

    rolling_mean = float(row.get("Rolling3M_Mean", 0) or 0)
    lag1 = float(row.get("Lag1", 0) or 0)
    sku_mean = float(row.get("SKU_Mean_Demand", 0) or 0)
    non_bonus_avg = float(row.get("NonBonus_Avg_Demand", 0) or 0)
    promo_profile = row.get("Promo_Profile", "NORMAL")
    next_bonus = int(row.get("Bonus_Flag", 0) or 0)

    base_anchor = max(rolling_mean, lag1, sku_mean, non_bonus_avg, 0)

    if base_anchor < 100:
        pred = min(pred, max(base_anchor * 1.8, 30))
    elif base_anchor < 500:
        pred = min(pred, base_anchor * 2.2)
    elif promo_profile == "NORMAL" or next_bonus == 0:
        pred = min(pred, base_anchor * 2.5)
    else:
        pred = min(pred, base_anchor * 3.0)

    return max(pred, 0.0)


def apply_production_calibration(row, pred):
    pred = max(float(pred), 0.0)

    segment = row.get("Segment_For_Calibration", "")
    abc_class = str(row.get("ABC_Class", "2"))

    rolling_mean = float(row.get("Rolling3M_Mean", 0) or 0)
    lag1 = float(row.get("Lag1", 0) or 0)
    sku_mean = float(row.get("SKU_Mean_Demand", 0) or 0)
    promo_profile = row.get("Promo_Profile", "NORMAL")
    expected_bonus = int(row.get("Bonus_Flag", 0) or 0)

    anchor = max(rolling_mean, lag1, sku_mean, 0.0)

    if segment == "LONG":
        factor = 1.03
    elif segment == "MEDIUM":
        factor = 1.05
    elif segment == "SHORT":
        factor = 1.08
    else:
        factor = 1.00

    if abc_class == "0":
        factor *= 1.02
    elif abc_class == "1":
        factor *= 1.01
    else:
        factor *= 0.99

    if promo_profile in ["PROMO_INFLUENCED", "PURE_PROMO"] and expected_bonus == 1:
        factor *= 1.04

    calibrated = pred * factor

    if anchor > 0:
        if segment == "LONG":
            calibrated = min(calibrated, anchor * 3.0)
        elif segment == "MEDIUM":
            calibrated = min(calibrated, anchor * 2.8)
        else:
            calibrated = min(calibrated, anchor * 2.5)

    return max(calibrated, 0.0)


def fallback_forecast_from_row(row, fallback_type="ROLLING3"):
    lag1 = float(row.get("Lag1", 0) or 0)
    rolling3 = float(row.get("Rolling3M_Mean", 0) or 0)
    sku_mean = float(row.get("SKU_Mean_Demand", 0) or 0)

    if fallback_type == "MAX_LAG1_ROLL3":
        pred = max(lag1, rolling3, 0)
    elif fallback_type == "LAG1":
        pred = max(lag1, 0)
    elif fallback_type == "MEAN":
        pred = max(sku_mean, 0)
    else:
        pred = max(rolling3, 0)

    return pred


# ------------------------------------------------------------
# 3) CLEAN TREE PREP USING DEPLOYMENT ARTIFACTS
# ------------------------------------------------------------
def prepare_long_tree_inference_frame(raw_data, artifacts):
    df = raw_data.copy().sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)
    df = force_itemcode_str(df)

    df = add_history_length_from_subset(df, df)
    df = force_itemcode_str(df)
    df = df[df["History_Segment"] == "LONG"].copy()
    df = force_itemcode_str(df)

    if df.empty:
        return df

    bonus_pattern_df = detect_recurring_bonus_skus(df)[[
        "ItemCode",
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ]].copy()
    bonus_pattern_df = force_itemcode_str(bonus_pattern_df)

    df = df.drop(columns=[
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ], errors="ignore")

    df = force_itemcode_str(df)
    df = df.merge(bonus_pattern_df, on="ItemCode", how="left")

    for c in ["Recurring_Bonus_SKU", "Bonus_Cycle_Length", "Avg_Bonus_Gap", "Bonus_Frequency_All"]:
        df[c] = df[c].fillna(0)
    df["Avg_Bonus_Uplift"] = df["Avg_Bonus_Uplift"].fillna(1.0)

    df, _ = apply_sku_cap(df.copy(), df.copy())
    df = force_itemcode_str(df)

    # artifact-based ABC map
    abc_map = artifacts.get("abc_map", {})
    df["ABC_Class"] = df["ItemCode"].map(abc_map).fillna(2)

    # artifact-based promo profile
    promo_profile_df = artifacts.get("promo_profile_df", pd.DataFrame())
    if isinstance(promo_profile_df, pd.DataFrame) and not promo_profile_df.empty:
        promo_profile_df = force_itemcode_str(promo_profile_df)
        df = force_itemcode_str(df)
        df = merge_promo_profile(df, promo_profile_df)
        df = force_itemcode_str(df)
    else:
        tmp_promo = build_promo_profile(df)
        tmp_promo = force_itemcode_str(tmp_promo)
        df = force_itemcode_str(df)
        df = merge_promo_profile(df, tmp_promo)
        df = force_itemcode_str(df)

    # artifact-based SKU profile
    sku_profile_df = artifacts.get("sku_profile_df", pd.DataFrame())
    if isinstance(sku_profile_df, pd.DataFrame) and not sku_profile_df.empty:
        sku_profile_df = force_itemcode_str(sku_profile_df)
        keep_cols = [c for c in ["ItemCode", "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"] if c in sku_profile_df.columns]

        df = df.drop(columns=["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"], errors="ignore")
        df = force_itemcode_str(df)
        df = df.merge(sku_profile_df[keep_cols], on="ItemCode", how="left")

        for c in ["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]:
            if c in df.columns:
                df[c] = df[c].fillna(0)
    else:
        tmp_sku = (
            df.groupby("ItemCode")
            .agg(
                SKU_Mean_Demand=("Clean_Demand", "mean"),
                SKU_Std_Demand=("Clean_Demand", "std"),
                SKU_ZeroRate=("Clean_Demand", lambda x: (x == 0).mean())
            )
            .reset_index()
        )
        tmp_sku = force_itemcode_str(tmp_sku)
        tmp_sku["SKU_Std_Demand"] = tmp_sku["SKU_Std_Demand"].fillna(0)
        tmp_sku["SKU_CV"] = np.where(
            tmp_sku["SKU_Mean_Demand"] <= 0,
            0,
            tmp_sku["SKU_Std_Demand"] / (tmp_sku["SKU_Mean_Demand"] + 1)
        )
        tmp_sku = tmp_sku[["ItemCode", "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]]

        df = df.drop(columns=["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"], errors="ignore")
        df = force_itemcode_str(df)
        df = df.merge(tmp_sku, on="ItemCode", how="left")

        for c in ["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]:
            df[c] = df[c].fillna(0)

    df = add_bonus_cycle_features(df)
    df = force_itemcode_str(df)
    df = rebuild_time_features(df)
    df = force_itemcode_str(df)

    if "clip_caps" in artifacts:
        df = apply_clip_caps(df, artifacts["clip_caps"])
        df = force_itemcode_str(df)

    df = recompute_target(df)
    df = force_itemcode_str(df)
    df = add_residual_target(df)
    df = force_itemcode_str(df)

    itemcode_categories = artifacts.get("itemcode_categories", pd.Index([]))
    cat_to_code = {str(k): i for i, k in enumerate(itemcode_categories)}
    unk_code = len(cat_to_code)

    df["ItemCode_Original"] = df["ItemCode"].astype(str)
    df["ItemCode_Encoded"] = df["ItemCode"].astype(str).map(cat_to_code).fillna(unk_code).astype(int)

    return df


# ------------------------------------------------------------
# 4) LONG TREE FORECAST
# ------------------------------------------------------------
def forecast_long_tree_sku(sku_code, raw_data, artifacts, used_model_name):
    sku_code = str(sku_code)
    model = artifacts["model"]
    feature_cols = artifacts["feature_cols"]

    prepared_df = prepare_long_tree_inference_frame(raw_data, artifacts)
    prepared_df = force_itemcode_str(prepared_df)

    if prepared_df.empty:
        return None

    sku_hist = prepared_df[prepared_df["ItemCode_Original"] == sku_code].copy().sort_values(["Year", "Month_Number"])
    if sku_hist.empty:
        return None

    last_row, next_year, next_month = get_next_period_from_history(prepared_df, sku_code)
    if last_row is None:
        return None

    next_bonus = infer_expected_bonus_flag(sku_code, prepared_df)

    new_row = last_row.copy()
    new_row = force_itemcode_str(new_row)
    new_row["Year"] = next_year
    new_row["Month_Number"] = next_month
    new_row["Bonus_Flag"] = int(next_bonus)

    # for future row, avoid leaking actual future sales
    for c in ["Secondary_Sales_Qty", "Primary_Sales_Qty", "Free_Qty", "Observed_Demand", "Effective_Demand", "Clean_Demand"]:
        if c in new_row.columns:
            new_row[c] = 0

    work_df = prepared_df.drop(columns=["ItemCode_Encoded"], errors="ignore").copy()
    work_df = force_itemcode_str(work_df)
    work_df = pd.concat([work_df, new_row], ignore_index=True)
    work_df = force_itemcode_str(work_df)
    work_df = work_df.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    # rebuild recurring bonus flags on full history + new row
    bonus_pattern_df_f = detect_recurring_bonus_skus(work_df)[[
        "ItemCode",
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ]].copy()
    bonus_pattern_df_f = force_itemcode_str(bonus_pattern_df_f)

    work_df = work_df.drop(columns=[
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ], errors="ignore")
    work_df = force_itemcode_str(work_df)
    work_df = work_df.merge(bonus_pattern_df_f, on="ItemCode", how="left")

    for c in ["Recurring_Bonus_SKU", "Bonus_Cycle_Length", "Avg_Bonus_Gap", "Bonus_Frequency_All"]:
        work_df[c] = work_df[c].fillna(0)
    work_df["Avg_Bonus_Uplift"] = work_df["Avg_Bonus_Uplift"].fillna(1.0)

    # keep artifact-based ABC
    abc_map = artifacts.get("abc_map", {})
    work_df["ABC_Class"] = work_df["ItemCode"].map(abc_map).fillna(2)

    # keep artifact-based promo profile
    promo_profile_df = artifacts.get("promo_profile_df", pd.DataFrame())
    if isinstance(promo_profile_df, pd.DataFrame) and not promo_profile_df.empty:
        promo_profile_df = force_itemcode_str(promo_profile_df)
        work_df = force_itemcode_str(work_df)
        work_df = merge_promo_profile(work_df, promo_profile_df)
        work_df = force_itemcode_str(work_df)
    else:
        tmp_promo = build_promo_profile(work_df)
        tmp_promo = force_itemcode_str(tmp_promo)
        work_df = force_itemcode_str(work_df)
        work_df = merge_promo_profile(work_df, tmp_promo)
        work_df = force_itemcode_str(work_df)

    # keep artifact-based sku profile
    sku_profile_df = artifacts.get("sku_profile_df", pd.DataFrame())
    if isinstance(sku_profile_df, pd.DataFrame) and not sku_profile_df.empty:
        sku_profile_df = force_itemcode_str(sku_profile_df)
        keep_cols = [c for c in ["ItemCode", "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"] if c in sku_profile_df.columns]

        work_df = work_df.drop(columns=["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"], errors="ignore")
        work_df = force_itemcode_str(work_df)
        work_df = work_df.merge(sku_profile_df[keep_cols], on="ItemCode", how="left")

        for c in ["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]:
            if c in work_df.columns:
                work_df[c] = work_df[c].fillna(0)
    else:
        tmp_sku = (
            work_df.groupby("ItemCode")
            .agg(
                SKU_Mean_Demand=("Clean_Demand", "mean"),
                SKU_Std_Demand=("Clean_Demand", "std"),
                SKU_ZeroRate=("Clean_Demand", lambda x: (x == 0).mean())
            )
            .reset_index()
        )
        tmp_sku = force_itemcode_str(tmp_sku)
        tmp_sku["SKU_Std_Demand"] = tmp_sku["SKU_Std_Demand"].fillna(0)
        tmp_sku["SKU_CV"] = np.where(
            tmp_sku["SKU_Mean_Demand"] <= 0,
            0,
            tmp_sku["SKU_Std_Demand"] / (tmp_sku["SKU_Mean_Demand"] + 1)
        )
        tmp_sku = tmp_sku[["ItemCode", "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]]

        work_df = work_df.drop(columns=["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"], errors="ignore")
        work_df = force_itemcode_str(work_df)
        work_df = work_df.merge(tmp_sku, on="ItemCode", how="left")

        for c in ["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]:
            work_df[c] = work_df[c].fillna(0)

    work_df = add_bonus_cycle_features(work_df)
    work_df = force_itemcode_str(work_df)
    work_df = rebuild_time_features(work_df)
    work_df = force_itemcode_str(work_df)

    if "clip_caps" in artifacts:
        work_df = apply_clip_caps(work_df, artifacts["clip_caps"])
        work_df = force_itemcode_str(work_df)

    work_df = recompute_target(work_df)
    work_df = force_itemcode_str(work_df)
    work_df = add_residual_target(work_df)
    work_df = force_itemcode_str(work_df)

    itemcode_categories = artifacts.get("itemcode_categories", pd.Index([]))
    cat_to_code = {str(k): i for i, k in enumerate(itemcode_categories)}
    unk_code = len(cat_to_code)

    work_df["ItemCode_Original"] = work_df["ItemCode"].astype(str)
    work_df["ItemCode_Encoded"] = work_df["ItemCode"].astype(str).map(cat_to_code).fillna(unk_code).astype(int)

    work_df_model = work_df.copy()
    work_df_model["ItemCode"] = work_df_model["ItemCode_Encoded"]

    next_row = work_df_model[
        (work_df_model["ItemCode_Original"] == sku_code) &
        (work_df_model["Year"] == next_year) &
        (work_df_model["Month_Number"] == next_month)
    ].copy()

    if next_row.empty:
        return None

    next_row["Segment_For_Calibration"] = "LONG"
    next_row = ensure_inference_features(next_row, feature_cols)

    assert_features_exist(next_row, feature_cols, where="LONG_TREE_INFERENCE_NEXT_ROW")

    X_next = sanitize_model_input(next_row[feature_cols])
    pred_residual = float(np.array(model.predict(X_next)).reshape(-1)[0])

    baseline = float(next_row[BASELINE_COL].iloc[0])
    
    pred_residual_raw = float(pred_residual)
    pred_residual = apply_residual_strength(
        next_row.iloc[0].to_dict(),
        pred_residual_raw,
        segment=next_row["Segment_For_Calibration"].iloc[0]
    )

    raw_forecast = max(baseline + pred_residual, 0.0)

    row_dict = next_row.iloc[0].to_dict()

    forecast = apply_promo_aware_adjustment(row_dict, raw_forecast)
    forecast = apply_final_forecast_guardrails(row_dict, forecast)
    forecast = apply_production_calibration(row_dict, forecast)
    
    # prevent over-aggressive low forecast
    lag1 = float(row_dict.get("Lag1", 0) or 0)
    rolling_mean = float(row_dict.get("Rolling3M_Mean", 0) or 0)
    
    if baseline > 100 and forecast < 0.25 * baseline:
        forecast = max(0.60 * baseline, 0.50 * lag1, 0.50 * rolling_mean)

    return {
        "ItemCode": int(float(sku_code)),
        "Forecast_Year": int(next_year),
        "Forecast_Month": int(next_month),
        "Forecast_Prediction": float(forecast),
        "Segment": "LONG",
        "Subgroup": "",
        "Champion_Model": str(used_model_name).upper(),
        "Used_Model": str(used_model_name).upper(),
        "Fallback_Used": 0,
        "Expected_Bonus": int(next_bonus),
        "Residual_Baseline": float(baseline),
        "Predicted_Residual": float(pred_residual),
        "Status": "Success",
        "Predicted_Residual": float(pred_residual),
        "Predicted_Residual_Raw": float(pred_residual_raw),
    }


# ------------------------------------------------------------
# 5) LONG FALLBACK FORECAST
# ------------------------------------------------------------
def forecast_long_fallback_sku(sku_code, raw_data, fallback_type="ROLLING3"):
    sku_code = str(sku_code)

    routing = get_long_routing_row(sku_code)
    if routing is None:
        return None

    best_model = str(routing.get("Best_Model", "XGBOOST")).upper()

    # use some tree artifact only to build the frame
    if best_model in ["XGBOOST", "CATBOOST", "LIGHTGBM"]:
        artifacts = get_long_deploy_artifact(best_model)
    else:
        artifacts = get_long_deploy_artifact("XGBOOST")

    if artifacts is None:
        return None

    prepared_df = prepare_long_tree_inference_frame(raw_data, artifacts)
    prepared_df = force_itemcode_str(prepared_df)
    if prepared_df.empty:
        return None

    sku_hist = prepared_df[prepared_df["ItemCode_Original"] == sku_code].copy().sort_values(["Year", "Month_Number"])
    if sku_hist.empty:
        return None

    last_row, next_year, next_month = get_next_period_from_history(prepared_df, sku_code)
    if last_row is None:
        return None

    next_bonus = infer_expected_bonus_flag(sku_code, prepared_df)

    new_row = last_row.copy()
    new_row = force_itemcode_str(new_row)
    new_row["Year"] = next_year
    new_row["Month_Number"] = next_month
    new_row["Bonus_Flag"] = int(next_bonus)

    for c in ["Secondary_Sales_Qty", "Primary_Sales_Qty", "Free_Qty", "Observed_Demand", "Effective_Demand", "Clean_Demand"]:
        if c in new_row.columns:
            new_row[c] = 0

    work_df = prepared_df.drop(columns=["ItemCode_Encoded"], errors="ignore").copy()
    work_df = force_itemcode_str(work_df)
    work_df = pd.concat([work_df, new_row], ignore_index=True)
    work_df = force_itemcode_str(work_df)
    work_df = work_df.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    bonus_pattern_df_f = detect_recurring_bonus_skus(work_df)[[
        "ItemCode",
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ]].copy()
    bonus_pattern_df_f = force_itemcode_str(bonus_pattern_df_f)

    work_df = work_df.drop(columns=[
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ], errors="ignore")
    work_df = force_itemcode_str(work_df)
    work_df = work_df.merge(bonus_pattern_df_f, on="ItemCode", how="left")

    for c in ["Recurring_Bonus_SKU", "Bonus_Cycle_Length", "Avg_Bonus_Gap", "Bonus_Frequency_All"]:
        work_df[c] = work_df[c].fillna(0)
    work_df["Avg_Bonus_Uplift"] = work_df["Avg_Bonus_Uplift"].fillna(1.0)

    work_df["ABC_Class"] = work_df["ItemCode"].map(artifacts.get("abc_map", {})).fillna(2)

    promo_profile_df = artifacts.get("promo_profile_df", pd.DataFrame())
    if isinstance(promo_profile_df, pd.DataFrame) and not promo_profile_df.empty:
        promo_profile_df = force_itemcode_str(promo_profile_df)
        work_df = force_itemcode_str(work_df)
        work_df = merge_promo_profile(work_df, promo_profile_df)
        work_df = force_itemcode_str(work_df)

    sku_profile_df = artifacts.get("sku_profile_df", pd.DataFrame())
    if isinstance(sku_profile_df, pd.DataFrame) and not sku_profile_df.empty:
        sku_profile_df = force_itemcode_str(sku_profile_df)
        keep_cols = [c for c in ["ItemCode", "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"] if c in sku_profile_df.columns]
        work_df = work_df.drop(columns=["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"], errors="ignore")
        work_df = force_itemcode_str(work_df)
        work_df = work_df.merge(sku_profile_df[keep_cols], on="ItemCode", how="left")
        for c in ["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]:
            if c in work_df.columns:
                work_df[c] = work_df[c].fillna(0)

    work_df = add_bonus_cycle_features(work_df)
    work_df = force_itemcode_str(work_df)
    work_df = rebuild_time_features(work_df)
    work_df = force_itemcode_str(work_df)

    if "clip_caps" in artifacts:
        work_df = apply_clip_caps(work_df, artifacts["clip_caps"])
        work_df = force_itemcode_str(work_df)

    work_df = recompute_target(work_df)
    work_df = force_itemcode_str(work_df)
    work_df = add_residual_target(work_df)
    work_df = force_itemcode_str(work_df)

    next_row = work_df[
        (work_df["ItemCode"].astype(str) == sku_code) &
        (work_df["Year"] == next_year) &
        (work_df["Month_Number"] == next_month)
    ].copy()

    if next_row.empty:
        return None

    next_row["Segment_For_Calibration"] = "LONG"

    row_dict = next_row.iloc[0].to_dict()

    pred = fallback_forecast_from_row(row_dict, fallback_type=fallback_type)

    # Conservative fallback correction
    lag1 = float(row_dict.get("Lag1", 0) or 0)
    roll3 = float(row_dict.get("Rolling3M_Mean", 0) or 0)
    roll6 = float(row_dict.get("Rolling6M_Mean", roll3) or roll3)
    
    anchors = [x for x in [lag1, roll3, roll6] if x > 0]
    
    if len(anchors) > 0:
        robust_pred = float(np.median(anchors))
        pred = min(pred, robust_pred * 1.10)

    expected_bonus = int(row_dict.get("Expected_Bonus_NextMonth", 0) or 0)
    bonus_flag = int(row_dict.get("Bonus_Flag", 0) or 0)

    if expected_bonus != 1 and bonus_flag != 1:
        pred *= 0.90

    pred = apply_production_calibration(row_dict, pred)

    return {
        "ItemCode": int(float(sku_code)),
        "Forecast_Year": int(next_year),
        "Forecast_Month": int(next_month),
        "Forecast_Prediction": float(pred),
        "Segment": "LONG",
        "Subgroup": "",
        "Champion_Model": str(best_model).upper(),
        "Used_Model": f"FALLBACK_{fallback_type}",
        "Fallback_Used": 1,
        "Expected_Bonus": int(next_bonus),
        "Residual_Baseline": np.nan,
        "Predicted_Residual": np.nan,
        "Status": "Success"
    }


# ------------------------------------------------------------
# 6) GRU LOAD HELPERS
# ------------------------------------------------------------
def load_long_gru_deploy_bundle_if_needed():
    global gru_long_deploy_artifacts
    global gru_long_deploy_scalers

    if "gru_long_deploy_artifacts" in globals() and "gru_long_deploy_scalers" in globals():
        if gru_long_deploy_artifacts is not None and gru_long_deploy_scalers is not None:
            return gru_long_deploy_artifacts, gru_long_deploy_scalers

    artifact = torch.load(
        os.path.join(GRU_DEPLOY_DIR, "gru_long_deploy_model.pt"),
        map_location=GRU_DEVICE
    )

    seq_scaler = joblib.load(os.path.join(GRU_DEPLOY_DIR, "gru_long_seq_scaler.pkl"))
    static_scaler = joblib.load(os.path.join(GRU_DEPLOY_DIR, "gru_long_static_scaler.pkl"))
    promo_profile_df = joblib.load(os.path.join(GRU_DEPLOY_DIR, "gru_long_promo_profile_df.pkl"))
    sku_profile_df = joblib.load(os.path.join(GRU_DEPLOY_DIR, "gru_long_sku_profile_df.pkl"))
    item_to_idx = joblib.load(os.path.join(GRU_DEPLOY_DIR, "gru_long_item_to_idx.pkl")) \
        if os.path.exists(os.path.join(GRU_DEPLOY_DIR, "gru_long_item_to_idx.pkl")) \
        else artifact["item_to_idx"]
    
    if GRU_DEBUG:
        sample_keys = list(item_to_idx.keys())[:10]
        print("\n[GRU LOAD DEBUG]")
        print("Sample item_to_idx keys:", sample_keys)
        print("Contains 600308?", "600308" in item_to_idx)
        print("Contains 600311?", "600311" in item_to_idx)
        print("Contains 600315?", "600315" in item_to_idx)
        print("Contains 600319?", "600319" in item_to_idx)

    model = LongGRUResidualForecaster(
        num_items=len(item_to_idx),
        seq_input_dim=len(artifact["seq_features"]),
        static_input_dim=len(artifact["static_features"]),
        embed_dim=artifact["embed_dim"],
        hidden_size=artifact["hidden_size"],
        num_layers=artifact["num_layers"],
        dropout=artifact["dropout"],
    ).to(GRU_DEVICE)

    model.load_state_dict(artifact["model_state_dict"])
    model.eval()

    loaded_artifacts = {
        "model": model,
        "model_type": artifact["model_type"],
        "model_name": "GRU",
        "segment": artifact["segment"],
        "seq_features": artifact["seq_features"],
        "static_features": artifact["static_features"],
        "seq_len": artifact["seq_len"],
        "embed_dim": artifact["embed_dim"],
        "hidden_size": artifact["hidden_size"],
        "num_layers": artifact["num_layers"],
        "dropout": artifact["dropout"],
        "item_to_idx": item_to_idx,
        "abc_map": artifact["abc_map"],
        "clip_caps": artifact["clip_caps"],
        "promo_profile_df": promo_profile_df,
        "sku_profile_df": sku_profile_df
    }

    loaded_scalers = LongGRUScalerBundle(
        seq_scaler=seq_scaler,
        static_scaler=static_scaler
    )

    gru_long_deploy_artifacts = loaded_artifacts
    gru_long_deploy_scalers = loaded_scalers

    return loaded_artifacts, loaded_scalers

#  Clear rejection rule for GRU forecast.
def should_reject_gru_prediction(
    pred_res_log,
    pred_residual_raw,
    baseline,
    lag1,
    rolling3,
    sku_mean,
    forecast_value,
    next_bonus,
    promo_profile
):
    # hard log instability
    if pred_res_log <= -6.95 or pred_res_log >= 6.95:
        return False, "CLIPPED_BUT_ACCEPTED"

    # residual too negative vs baseline
    if pred_residual_raw < -1.5 * max(baseline, 1.0):
        return True, "EXTREME_NEGATIVE_RESIDUAL"

    # collapse after non-promo spike
    if (
        next_bonus == 0 and
        promo_profile == "NORMAL" and
        lag1 > 2.5 * max(rolling3, 1.0) and
        forecast_value < 0.25 * max(rolling3, 1.0)
    ):
        return True, "NON_PROMO_SPIKE_COLLAPSE"

    # forecast much too low relative to recent demand
    if forecast_value < 0.20 * max(min(lag1, baseline), 1.0):
        return True, "FORECAST_TOO_LOW"

    # forecast much too high can also be guarded later if needed
    return False, "OK"

# Spike-aware smoothing rule only when spike is not expected
def apply_spike_guardrail(raw_forecast, lag1, rolling3, sku_mean, next_bonus, promo_profile):
    raw_forecast = max(float(raw_forecast), 0.0)

    # suspected one-off spike, not expected promo
    if (
        next_bonus == 0 and
        promo_profile == "NORMAL" and
        lag1 > 2.5 * max(rolling3, 1.0)
    ):
        # reduce collapse / overreaction after one-off spike
        guarded = max(raw_forecast, 0.85 * rolling3)
        return guarded, True

    return raw_forecast, False

# ------------------------------------------------------------
# 7) PREPARE LONG GRU INFERENCE FRAME
# ------------------------------------------------------------
def prepare_long_gru_inference_frame(raw_data, loaded_artifacts):
    df = raw_data.copy().sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)
    df = force_itemcode_str(df)

    df = add_history_length_from_subset(df, df)
    df = force_itemcode_str(df)
    df = df[df["History_Segment"] == "LONG"].copy()
    df = force_itemcode_str(df)

    if df.empty:
        return df

    bonus_pattern_df = detect_recurring_bonus_skus(df)[[
        "ItemCode",
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ]].copy()
    bonus_pattern_df = force_itemcode_str(bonus_pattern_df)

    df = df.drop(columns=[
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ], errors="ignore")

    df = force_itemcode_str(df)
    df = df.merge(bonus_pattern_df, on="ItemCode", how="left")

    for c in ["Recurring_Bonus_SKU", "Bonus_Cycle_Length", "Avg_Bonus_Gap", "Bonus_Frequency_All"]:
        df[c] = df[c].fillna(0)
    df["Avg_Bonus_Uplift"] = df["Avg_Bonus_Uplift"].fillna(1.0)

    df, _ = apply_sku_cap(df.copy(), df.copy())
    df = force_itemcode_str(df)

    df["ABC_Class"] = df["ItemCode"].map(loaded_artifacts["abc_map"]).fillna(2)

    promo_profile_df = force_itemcode_str(loaded_artifacts["promo_profile_df"])
    df = force_itemcode_str(df)
    df = merge_promo_profile(df, promo_profile_df)
    df = force_itemcode_str(df)

    if "sku_profile_df" in loaded_artifacts and isinstance(loaded_artifacts["sku_profile_df"], pd.DataFrame):
        sku_profile_df = loaded_artifacts["sku_profile_df"].copy()
        sku_profile_df = force_itemcode_str(sku_profile_df)
        keep_cols = [c for c in ["ItemCode", "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"] if c in sku_profile_df.columns]

        if keep_cols:
            df = df.drop(columns=["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"], errors="ignore")
            df = force_itemcode_str(df)
            df = df.merge(sku_profile_df[keep_cols], on="ItemCode", how="left")

            for c in ["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]:
                if c in df.columns:
                    df[c] = df[c].fillna(0)

    df = add_bonus_cycle_features(df)
    df = force_itemcode_str(df)
    df = rebuild_time_features(df)
    df = force_itemcode_str(df)

    df = apply_clip_caps(df, loaded_artifacts["clip_caps"])
    df = force_itemcode_str(df)

    df = recompute_target(df)
    df = force_itemcode_str(df)
    df = add_residual_target(df)
    df = force_itemcode_str(df)

    df["ItemCode_Original"] = df["ItemCode"].astype(str)
    df["Residual_Target_Log"] = gru_signed_log_transform(df[MODEL_TARGET_COL].fillna(0))

    return df

# ------------------------------------------------------------
# 8) LONG GRU FORECAST
# ------------------------------------------------------------
def forecast_long_gru_sku(sku_code, raw_data):
    sku_code = str(sku_code)

    loaded_artifacts, loaded_scalers = load_long_gru_deploy_bundle_if_needed()
    model = loaded_artifacts["model"]
    item_to_idx = loaded_artifacts["item_to_idx"]
    seq_len = loaded_artifacts["seq_len"]

    prepared_df = prepare_long_gru_inference_frame(raw_data, loaded_artifacts)
    prepared_df = force_itemcode_str(prepared_df)

    if prepared_df.empty:
        if gru_should_debug(sku_code):
            print(f"[GRU DEBUG] SKU={sku_code} | prepared_df empty")
        return None

    sku_hist = prepared_df[prepared_df["ItemCode_Original"] == sku_code].copy()
    sku_hist = sku_hist.sort_values(["Year", "Month_Number"]).reset_index(drop=True)

    if sku_hist.empty:
        if gru_should_debug(sku_code):
            print(f"[GRU DEBUG] SKU={sku_code} | sku_hist empty")
        return None

    if sku_code not in item_to_idx:
        if gru_should_debug(sku_code):
            print(f"[GRU DEBUG] SKU={sku_code} | not in item_to_idx")
        return None

    last_row, next_year, next_month = get_next_period_from_history(prepared_df, sku_code)
    if last_row is None:
        if gru_should_debug(sku_code):
            print(f"[GRU DEBUG] SKU={sku_code} | last_row is None")
        return None

    next_bonus = infer_expected_bonus_flag(sku_code, prepared_df)

    new_row = last_row.copy()
    new_row = force_itemcode_str(new_row)
    new_row["Year"] = next_year
    new_row["Month_Number"] = next_month
    new_row["Bonus_Flag"] = int(next_bonus)

    for c in [
        "Secondary_Sales_Qty",
        "Primary_Sales_Qty",
        "Free_Qty",
        "Observed_Demand",
        "Effective_Demand",
        "Clean_Demand"
    ]:
        if c in new_row.columns:
            new_row[c] = 0

    future_df = pd.concat([prepared_df.copy(), new_row], ignore_index=True)
    future_df = force_itemcode_str(future_df)
    future_df = future_df.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    bonus_pattern_df_f = detect_recurring_bonus_skus(future_df)[[
        "ItemCode",
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ]].copy()
    bonus_pattern_df_f = force_itemcode_str(bonus_pattern_df_f)

    future_df = future_df.drop(columns=[
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ], errors="ignore")
    future_df = force_itemcode_str(future_df)
    future_df = future_df.merge(bonus_pattern_df_f, on="ItemCode", how="left")

    for c in ["Recurring_Bonus_SKU", "Bonus_Cycle_Length", "Avg_Bonus_Gap", "Bonus_Frequency_All"]:
        future_df[c] = future_df[c].fillna(0)
    future_df["Avg_Bonus_Uplift"] = future_df["Avg_Bonus_Uplift"].fillna(1.0)

    future_df["ABC_Class"] = future_df["ItemCode"].map(loaded_artifacts["abc_map"]).fillna(2)

    promo_profile_df = force_itemcode_str(loaded_artifacts["promo_profile_df"])
    future_df = force_itemcode_str(future_df)
    future_df = merge_promo_profile(future_df, promo_profile_df)
    future_df = force_itemcode_str(future_df)

    if "sku_profile_df" in loaded_artifacts and isinstance(loaded_artifacts["sku_profile_df"], pd.DataFrame):
        sku_profile_df = loaded_artifacts["sku_profile_df"].copy()
        sku_profile_df = force_itemcode_str(sku_profile_df)
        keep_cols = [c for c in ["ItemCode", "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"] if c in sku_profile_df.columns]

        if keep_cols:
            future_df = future_df.drop(columns=["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"], errors="ignore")
            future_df = force_itemcode_str(future_df)
            future_df = future_df.merge(sku_profile_df[keep_cols], on="ItemCode", how="left")

            for c in ["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]:
                if c in future_df.columns:
                    future_df[c] = future_df[c].fillna(0)

    future_df = add_bonus_cycle_features(future_df)
    future_df = force_itemcode_str(future_df)
    future_df = rebuild_time_features(future_df)
    future_df = force_itemcode_str(future_df)
    future_df = apply_clip_caps(future_df, loaded_artifacts["clip_caps"])
    future_df = force_itemcode_str(future_df)
    future_df = recompute_target(future_df)
    future_df = force_itemcode_str(future_df)
    future_df = add_residual_target(future_df)
    future_df = force_itemcode_str(future_df)

    future_df["Residual_Target_Log"] = gru_signed_log_transform(future_df[MODEL_TARGET_COL].fillna(0))
    future_df["Residual_Target_Log"] = future_df["Residual_Target_Log"].clip(-7.0, 7.0)
    future_df["ItemCode_Original"] = future_df["ItemCode"].astype(str)

    g = future_df[future_df["ItemCode_Original"] == sku_code].copy()
    g = g.sort_values(["Year", "Month_Number"]).reset_index(drop=True)

    target_idx = g[(g["Year"] == next_year) & (g["Month_Number"] == next_month)].index
    if len(target_idx) == 0:
        if gru_should_debug(sku_code):
            print(f"[GRU DEBUG] SKU={sku_code} | target_idx missing")
        return None

    idx = target_idx[0]
    if idx < seq_len - 1:
        if gru_should_debug(sku_code):
            print(f"[GRU DEBUG] SKU={sku_code} | not enough seq length | idx={idx} | seq_len={seq_len}")
        return None

    seq_slice = g.iloc[idx - seq_len + 1: idx + 1].copy()
    if len(seq_slice) != seq_len:
        if gru_should_debug(sku_code):
            print(f"[GRU DEBUG] SKU={sku_code} | seq_slice len mismatch | len={len(seq_slice)}")
        return None

    if gru_should_debug(sku_code):
        cols = [
            "Year", "Month_Number", "Clean_Demand", "Lag1", "Rolling3M_Mean",
            "Bonus_Flag", "Expected_Bonus_NextMonth", "Supply_Constraint_Flag",
            BASELINE_COL, MODEL_TARGET_COL
        ]
        cols = [c for c in cols if c in seq_slice.columns]
        print(f"\n[GRU TRACE] SKU={sku_code} | seq tail")
        print(seq_slice[cols].tail(GRU_DEBUG_MAX_SEQ_ROWS))

    seq_slice = ensure_inference_features(seq_slice, loaded_artifacts["seq_features"])
    seq_slice = ensure_inference_features(seq_slice, loaded_artifacts["static_features"])

    seq_vals = loaded_scalers.seq_scaler.transform(
        seq_slice[loaded_artifacts["seq_features"]].fillna(0).values
    )
    static_vals = loaded_scalers.static_scaler.transform(
        seq_slice.iloc[-1][loaded_artifacts["static_features"]].fillna(0).values.reshape(1, -1)
    )[0]

    x_seq = torch.tensor(seq_vals[np.newaxis, :, :], dtype=torch.float32).to(GRU_DEVICE)
    x_static = torch.tensor(static_vals[np.newaxis, :], dtype=torch.float32).to(GRU_DEVICE)
    x_item = torch.tensor([item_to_idx[sku_code]], dtype=torch.long).to(GRU_DEVICE)

    with torch.no_grad():
        pred_res_log = float(model(x_seq, x_static, x_item).cpu().numpy()[0])

    # ---- stabilize GRU output in log space ----
    pred_res_log = float(np.clip(pred_res_log, -6.0, 6.0))
    pred_residual = float(gru_signed_log_inverse(pred_res_log))

    baseline = float(seq_slice.iloc[-1][BASELINE_COL])

    target_row = seq_slice.iloc[-1].to_dict()
    target_row["Segment_For_Calibration"] = "LONG"

    rolling_mean = float(target_row.get("Rolling3M_Mean", 0) or 0)
    rolling3 = rolling_mean
    lag1 = float(target_row.get("Lag1", 0) or 0)
    sku_mean = float(target_row.get("SKU_Mean_Demand", 0) or 0)
    promo_profile = str(target_row.get("Promo_Profile", "NORMAL"))

    anchor = max(rolling3, lag1, sku_mean, 0.0)

    if baseline > 0:
        lower_cap = -0.85 * baseline
        upper_cap = max(1.5 * baseline, anchor * 2.0)
    else:
        lower_cap = -max(anchor, 100.0)
        upper_cap = max(anchor * 2.0, 100.0)

    pred_residual_raw = float(pred_residual)
    pred_residual = float(np.clip(pred_residual, lower_cap, upper_cap))

    raw_forecast = baseline + pred_residual
    raw_forecast = max(raw_forecast, 0.0)

    raw_forecast, spike_guard_used = apply_spike_guardrail(
        raw_forecast=raw_forecast,
        lag1=lag1,
        rolling3=rolling3,
        sku_mean=sku_mean,
        next_bonus=next_bonus,
        promo_profile=promo_profile
    )

    forecast = apply_promo_aware_adjustment(target_row, raw_forecast)
    forecast = apply_final_forecast_guardrails(target_row, forecast)
    forecast = apply_production_calibration(target_row, forecast)

    reject_gru, reject_reason = should_reject_gru_prediction(
        pred_res_log=pred_res_log,
        pred_residual_raw=pred_residual_raw,
        baseline=baseline,
        lag1=lag1,
        rolling3=rolling3,
        sku_mean=sku_mean,
        forecast_value=forecast,
        next_bonus=next_bonus,
        promo_profile=promo_profile
    )

    if reject_gru:
        print(
            f"[GRU FALLBACK TRIGGER] SKU={sku_code} reason={reject_reason} "
            f"forecast={forecast} anchor={lag1} baseline={baseline} raw_residual={pred_residual_raw}"
        )
        return None

    if gru_should_debug(sku_code):
        print(f"\n[GRU TRACE] SKU={sku_code}")
        print("baseline:", baseline)
        print("pred_res_log:", pred_res_log)
        print("pred_residual_raw:", pred_residual_raw)
        print("pred_residual_clipped:", pred_residual)
        print("lower_cap:", lower_cap)
        print("upper_cap:", upper_cap)
        print("anchor:", anchor)
        print("raw_forecast:", raw_forecast)
        print("final_forecast:", forecast)
        print("next_bonus:", next_bonus)
        print("promo_profile:", promo_profile)
        print("lag1:", lag1)
        print("rolling3:", rolling3)
        print("sku_mean:", sku_mean)
        print("spike_guard_used:", spike_guard_used)

    if baseline > 0 and pred_residual_raw < -0.8 * baseline:
        print(
            f"[GRU WARNING] SKU={sku_code} "
            f"baseline={baseline:.2f} "
            f"raw_residual={pred_residual_raw:.2f} "
            f"clipped_residual={pred_residual:.2f}"
        )

    if (
        not np.isfinite(forecast)
        or forecast < 0
        or (
            anchor > 0
            and forecast < 0.15 * anchor
            and baseline > 0
            and pred_residual_raw < -0.8 * baseline
        )
    ):
        print(
            f"[GRU FALLBACK TRIGGER] SKU={sku_code} "
            f"forecast={forecast} anchor={anchor:.2f} "
            f"baseline={baseline:.2f} raw_residual={pred_residual_raw:.2f}"
        )
        return None

    return {
        "ItemCode": int(float(sku_code)),
        "Forecast_Year": int(next_year),
        "Forecast_Month": int(next_month),
        "Forecast_Prediction": float(forecast),
        "Segment": "LONG",
        "Subgroup": "",
        "Champion_Model": "GRU",
        "Used_Model": "GRU",
        "Fallback_Used": 0,
        "Expected_Bonus": int(next_bonus),
        "Residual_Baseline": float(baseline),
        "Predicted_Residual": float(pred_residual),
        "Status": "Success"
    }

# ------------------------------------------------------------
# 9) FINAL LONG ROUTER
# ------------------------------------------------------------
def forecast_long_sku(sku_code, raw_data):
    sku_code = str(sku_code)
    routing = get_long_routing_row(sku_code)

    if routing is None:
        return {
            "ItemCode": int(float(sku_code)),
            "Forecast_Year": np.nan,
            "Forecast_Month": np.nan,
            "Forecast_Prediction": np.nan,
            "Segment": "LONG",
            "Subgroup": "",
            "Champion_Model": "UNKNOWN",
            "Used_Model": "UNKNOWN",
            "Fallback_Used": 0,
            "Expected_Bonus": np.nan,
            "Residual_Baseline": np.nan,
            "Predicted_Residual": np.nan,
            "Status": "FAILED: LONG routing row missing"
        }

    final_model = str(routing.get("Final_Model", routing.get("Best_Model", "XGBOOST"))).upper()
    best_model = str(routing.get("Best_Model", "XGBOOST")).upper()
    fallback_type = str(routing.get("Fallback_Type", "ROLLING3"))

    # 1) forced fallback route
    if final_model == "FALLBACK":
        result = forecast_long_fallback_sku(
            sku_code=sku_code,
            raw_data=raw_data,
            fallback_type=fallback_type
        )
        if result is not None:
            result["Champion_Model"] = best_model
            return result

        return {
            "ItemCode": int(float(sku_code)),
            "Forecast_Year": np.nan,
            "Forecast_Month": np.nan,
            "Forecast_Prediction": np.nan,
            "Segment": "LONG",
            "Subgroup": "",
            "Champion_Model": best_model,
            "Used_Model": f"FALLBACK_{fallback_type}",
            "Fallback_Used": 1,
            "Expected_Bonus": np.nan,
            "Residual_Baseline": np.nan,
            "Predicted_Residual": np.nan,
            "Status": f"FAILED: LONG fallback returned None ({fallback_type})"
        }

    # 2) GRU champion route
    if best_model == "GRU":
        try:
            result = forecast_long_gru_sku(sku_code, raw_data)
            if result is not None:
                result["Champion_Model"] = "GRU"
                return result
        except Exception as e:
            gru_error = str(e)
        else:
            gru_error = "GRU returned None"

        print(f"[GRU DEBUG] SKU={sku_code} | error={gru_error}")

        fb = forecast_long_fallback_sku(
            sku_code=sku_code,
            raw_data=raw_data,
            fallback_type=fallback_type
        )
        if fb is not None:
            fb["Champion_Model"] = "GRU"
            fb["Used_Model"] = f"FALLBACK_{fallback_type}_AFTER_GRU_FAIL"
            fb["Status"] = f"Success (GRU failed -> fallback used: {gru_error})"
            return fb

        return {
            "ItemCode": int(float(sku_code)),
            "Forecast_Year": np.nan,
            "Forecast_Month": np.nan,
            "Forecast_Prediction": np.nan,
            "Segment": "LONG",
            "Subgroup": "",
            "Champion_Model": "GRU",
            "Used_Model": "GRU",
            "Fallback_Used": 0,
            "Expected_Bonus": np.nan,
            "Residual_Baseline": np.nan,
            "Predicted_Residual": np.nan,
            "Status": f"FAILED: GRU failed and fallback failed | {gru_error}"
        }

    # 3) tree champion route
    artifacts = get_long_deploy_artifact(best_model)
    if artifacts is None:
        fb = forecast_long_fallback_sku(
            sku_code=sku_code,
            raw_data=raw_data,
            fallback_type=fallback_type
        )
        if fb is not None:
            fb["Champion_Model"] = best_model
            fb["Used_Model"] = f"FALLBACK_{fallback_type}_NO_ARTIFACT"
            fb["Status"] = f"Success (missing {best_model} artifact -> fallback used)"
            return fb

        return {
            "ItemCode": int(float(sku_code)),
            "Forecast_Year": np.nan,
            "Forecast_Month": np.nan,
            "Forecast_Prediction": np.nan,
            "Segment": "LONG",
            "Subgroup": "",
            "Champion_Model": best_model,
            "Used_Model": best_model,
            "Fallback_Used": 0,
            "Expected_Bonus": np.nan,
            "Residual_Baseline": np.nan,
            "Predicted_Residual": np.nan,
            "Status": f"FAILED: {best_model} artifact missing"
        }

    try:
        result = forecast_long_tree_sku(
            sku_code=sku_code,
            raw_data=raw_data,
            artifacts=artifacts,
            used_model_name=best_model
        )
        if result is not None:
            result["Champion_Model"] = best_model
            return result
        tree_error = "tree forecast returned None"
    except Exception as e:
        tree_error = str(e)

    fb = forecast_long_fallback_sku(
        sku_code=sku_code,
        raw_data=raw_data,
        fallback_type=fallback_type
    )
    if fb is not None:
        fb["Champion_Model"] = best_model
        fb["Used_Model"] = f"FALLBACK_{fallback_type}_AFTER_{best_model}_FAIL"
        fb["Status"] = f"Success ({best_model} failed -> fallback used: {tree_error})"
        return fb

    return {
        "ItemCode": int(float(sku_code)),
        "Forecast_Year": np.nan,
        "Forecast_Month": np.nan,
        "Forecast_Prediction": np.nan,
        "Segment": "LONG",
        "Subgroup": "",
        "Champion_Model": best_model,
        "Used_Model": best_model,
        "Fallback_Used": 0,
        "Expected_Bonus": np.nan,
        "Residual_Baseline": np.nan,
        "Predicted_Residual": np.nan,
        "Status": f"FAILED: {best_model} failed and fallback failed | {tree_error}"
    }


### MEDIUM SEGMENT

In [ ]:
# ------------------------------------------------------------
# A) MEDIUM REGISTRY
# ------------------------------------------------------------
medium_deploy_artifact_registry = pd.DataFrame([
    {"Medium_Subgroup": "PROMO_HEAVY", "Model_Name": "XGBOOST",       "Run_Key": "PROMO_HEAVY_XGBOOST",   "Deploy_Artifact_Key": "deploy_artifacts"},
    {"Medium_Subgroup": "PROMO_HEAVY", "Model_Name": "CATBOOST",      "Run_Key": "PROMO_HEAVY_CATBOOST",  "Deploy_Artifact_Key": "deploy_artifacts"},
    {"Medium_Subgroup": "STABLE",      "Model_Name": "XGBOOST",       "Run_Key": "STABLE_XGBOOST",        "Deploy_Artifact_Key": "deploy_artifacts"},
    {"Medium_Subgroup": "STABLE",      "Model_Name": "CATBOOST",      "Run_Key": "STABLE_CATBOOST",       "Deploy_Artifact_Key": "deploy_artifacts"},
    {"Medium_Subgroup": "STABLE",      "Model_Name": "RANDOM_FOREST", "Run_Key": "STABLE_RANDOM_FOREST",  "Deploy_Artifact_Key": "deploy_artifacts"},
])


def get_sku_history_length(raw_data, sku_code):
    sku_code = str(sku_code)
    g = raw_data[raw_data["ItemCode"].astype(str) == sku_code].copy()
    if g.empty:
        return 0
    return g[["Year", "Month_Number"]].drop_duplicates().shape[0]

# ------------------------------------------------------------
# C) CHAMPION LOOKUPS
# ------------------------------------------------------------
def get_medium_routing_row(sku_code):
    sku_code = str(sku_code)
    df = champion_medium_map_df.copy()
    df["ItemCode"] = df["ItemCode"].astype(str)

    hit = df[df["ItemCode"] == sku_code].copy()
    if hit.empty:
        return None
    return hit.iloc[0].to_dict()

# ------------------------------------------------------------
# D) DEPLOYMENT ARTIFACT LOOKUPS
# ------------------------------------------------------------
def get_medium_deploy_artifact(subgroup_name, model_name):
    subgroup_name = str(subgroup_name).upper()
    model_name = str(model_name).upper()

    hit = medium_deploy_artifact_registry[
        (medium_deploy_artifact_registry["Medium_Subgroup"].astype(str).str.upper() == subgroup_name) &
        (medium_deploy_artifact_registry["Model_Name"].astype(str).str.upper() == model_name)
    ].copy()

    if hit.empty:
        return None

    run_key = hit["Run_Key"].iloc[0]
    artifact_key = hit["Deploy_Artifact_Key"].iloc[0]

    if run_key not in medium_runs:
        return None
    if artifact_key not in medium_runs[run_key]:
        return None

    return medium_runs[run_key][artifact_key]

# ------------------------------------------------------------
# F) PREPARE RESIDUAL FORECAST FRAME
# used for LONG/MEDIUM tree models
# ------------------------------------------------------------
def prepare_residual_forecast_frame(artifacts, raw_data):
    df = raw_data.copy().sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)
    df = force_itemcode_str(df)
    df["ItemCode_Original"] = df["ItemCode"].astype(str)

    df = add_history_length_from_subset(df, df)
    df = force_itemcode_str(df)

    segment = str(artifacts.get("segment", "")).upper()
    if segment in ["LONG", "MEDIUM"]:
        df = df[df["History_Segment"] == segment].copy()
        df = force_itemcode_str(df)

    if df.empty:
        return df

    # recurring bonus profile from current history
    bonus_pattern_df = detect_recurring_bonus_skus(df)[[
        "ItemCode",
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ]].copy()
    bonus_pattern_df = force_itemcode_str(bonus_pattern_df)

    df = df.drop(columns=[
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ], errors="ignore")

    df = force_itemcode_str(df)
    df = df.merge(bonus_pattern_df, on="ItemCode", how="left")

    for c in ["Recurring_Bonus_SKU", "Bonus_Cycle_Length", "Avg_Bonus_Gap", "Bonus_Frequency_All"]:
        df[c] = df[c].fillna(0)
    df["Avg_Bonus_Uplift"] = df["Avg_Bonus_Uplift"].fillna(1.0)

    # artifact-based ABC
    abc_map = artifacts.get("abc_map", {})
    df["ABC_Class"] = df["ItemCode"].map(abc_map).fillna(2)

    # artifact-based promo profile
    promo_profile_df = artifacts.get("promo_profile_df", pd.DataFrame())
    if isinstance(promo_profile_df, pd.DataFrame) and not promo_profile_df.empty:
        promo_profile_df = force_itemcode_str(promo_profile_df)
        df = force_itemcode_str(df)
        df = merge_promo_profile(df, promo_profile_df)
        df = force_itemcode_str(df)
    else:
        tmp_promo = build_promo_profile(df)
        tmp_promo = force_itemcode_str(tmp_promo)
        df = force_itemcode_str(df)
        df = merge_promo_profile(df, tmp_promo)
        df = force_itemcode_str(df)

    # artifact-based SKU profile
    sku_profile_df = artifacts.get("sku_profile_df", pd.DataFrame())
    if isinstance(sku_profile_df, pd.DataFrame) and not sku_profile_df.empty:
        sku_profile_df = force_itemcode_str(sku_profile_df)
        keep_cols = [c for c in ["ItemCode", "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"] if c in sku_profile_df.columns]

        df = df.drop(columns=["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"], errors="ignore")
        df = force_itemcode_str(df)
        df = df.merge(sku_profile_df[keep_cols], on="ItemCode", how="left")

        for c in ["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]:
            if c in df.columns:
                df[c] = df[c].fillna(0)
    else:
        tmp_sku = (
            df.groupby("ItemCode")
            .agg(
                SKU_Mean_Demand=("Clean_Demand", "mean"),
                SKU_Std_Demand=("Clean_Demand", "std"),
                SKU_ZeroRate=("Clean_Demand", lambda x: (x == 0).mean())
            )
            .reset_index()
        )
        tmp_sku = force_itemcode_str(tmp_sku)
        tmp_sku["SKU_Std_Demand"] = tmp_sku["SKU_Std_Demand"].fillna(0)
        tmp_sku["SKU_CV"] = np.where(
            tmp_sku["SKU_Mean_Demand"] <= 0,
            0,
            tmp_sku["SKU_Std_Demand"] / (tmp_sku["SKU_Mean_Demand"] + 1)
        )
        tmp_sku = tmp_sku[["ItemCode", "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]]

        df = df.drop(columns=["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"], errors="ignore")
        df = force_itemcode_str(df)
        df = df.merge(tmp_sku, on="ItemCode", how="left")

        for c in ["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]:
            df[c] = df[c].fillna(0)

    # artifact-based medium subgroup profile
    if segment == "MEDIUM":
        medium_profile_df = artifacts.get("medium_profile_df", pd.DataFrame())
        if isinstance(medium_profile_df, pd.DataFrame) and not medium_profile_df.empty:
            medium_profile_df = force_itemcode_str(medium_profile_df)
            df = force_itemcode_str(df)
            df = merge_medium_sku_profile(df, medium_profile_df)
            df = force_itemcode_str(df)

    df = add_bonus_cycle_features(df)
    df = force_itemcode_str(df)
    df = rebuild_time_features(df)
    df = force_itemcode_str(df)

    if "clip_caps" in artifacts and artifacts["clip_caps"] is not None:
        df = apply_clip_caps(df, artifacts["clip_caps"])
        df = force_itemcode_str(df)

    df = recompute_target(df)
    df = force_itemcode_str(df)
    df = add_residual_target(df)
    df = force_itemcode_str(df)

    itemcode_categories = artifacts.get("itemcode_categories", pd.Index([]))
    cat_to_code = {str(k): i for i, k in enumerate(itemcode_categories)}
    unk_code = len(cat_to_code)

    df["ItemCode_Encoded"] = df["ItemCode"].astype(str).map(cat_to_code).fillna(unk_code).astype(int)

    return df

# ------------------------------------------------------------
# I) GENERIC TREE MODEL PREDICTOR
# ------------------------------------------------------------
def predict_tree_model_sku(
    sku_code,
    raw_data,
    artifacts,
    used_model_name,
    subgroup_name=""
):
    sku_code = str(sku_code)
    model = artifacts["model"]
    feature_cols = artifacts["feature_cols"]

    prepared_df = prepare_residual_forecast_frame(artifacts, raw_data)
    prepared_df = force_itemcode_str(prepared_df)
    if prepared_df.empty:
        return None

    sku_hist = prepared_df[prepared_df["ItemCode_Original"] == sku_code].copy().sort_values(["Year", "Month_Number"])
    if sku_hist.empty:
        return None

    last_row, next_year, next_month = get_next_period_from_history(prepared_df, sku_code)
    if last_row is None:
        return None

    next_bonus = infer_expected_bonus_flag(sku_code, prepared_df)

    new_row = last_row.copy()
    new_row = force_itemcode_str(new_row)
    new_row["Year"] = next_year
    new_row["Month_Number"] = next_month
    new_row["Bonus_Flag"] = int(next_bonus)

    # prevent leakage
    for c in ["Secondary_Sales_Qty", "Primary_Sales_Qty", "Free_Qty", "Observed_Demand", "Effective_Demand", "Clean_Demand"]:
        if c in new_row.columns:
            new_row[c] = 0

    work_df = prepared_df.drop(columns=["ItemCode_Encoded"], errors="ignore").copy()
    work_df = force_itemcode_str(work_df)
    work_df = pd.concat([work_df, new_row], ignore_index=True)
    work_df = force_itemcode_str(work_df)
    work_df = work_df.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    # refresh recurring bonus fields on extended history
    bonus_pattern_df_f = detect_recurring_bonus_skus(work_df)[[
        "ItemCode",
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ]].copy()
    bonus_pattern_df_f = force_itemcode_str(bonus_pattern_df_f)

    work_df = work_df.drop(columns=[
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ], errors="ignore")
    work_df = force_itemcode_str(work_df)
    work_df = work_df.merge(bonus_pattern_df_f, on="ItemCode", how="left")

    for c in ["Recurring_Bonus_SKU", "Bonus_Cycle_Length", "Avg_Bonus_Gap", "Bonus_Frequency_All"]:
        work_df[c] = work_df[c].fillna(0)
    work_df["Avg_Bonus_Uplift"] = work_df["Avg_Bonus_Uplift"].fillna(1.0)

    # artifact-based ABC
    abc_map = artifacts.get("abc_map", {})
    work_df["ABC_Class"] = work_df["ItemCode"].map(abc_map).fillna(2)

    # artifact-based promo profile
    promo_profile_df = artifacts.get("promo_profile_df", pd.DataFrame())
    if isinstance(promo_profile_df, pd.DataFrame) and not promo_profile_df.empty:
        promo_profile_df = force_itemcode_str(promo_profile_df)
        work_df = force_itemcode_str(work_df)
        work_df = merge_promo_profile(work_df, promo_profile_df)
        work_df = force_itemcode_str(work_df)
    else:
        tmp_promo = build_promo_profile(work_df)
        tmp_promo = force_itemcode_str(tmp_promo)
        work_df = force_itemcode_str(work_df)
        work_df = merge_promo_profile(work_df, tmp_promo)
        work_df = force_itemcode_str(work_df)

    # artifact-based sku profile
    sku_profile_df = artifacts.get("sku_profile_df", pd.DataFrame())
    if isinstance(sku_profile_df, pd.DataFrame) and not sku_profile_df.empty:
        sku_profile_df = force_itemcode_str(sku_profile_df)
        keep_cols = [c for c in ["ItemCode", "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"] if c in sku_profile_df.columns]

        work_df = work_df.drop(columns=["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"], errors="ignore")
        work_df = force_itemcode_str(work_df)
        work_df = work_df.merge(sku_profile_df[keep_cols], on="ItemCode", how="left")

        for c in ["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]:
            if c in work_df.columns:
                work_df[c] = work_df[c].fillna(0)
    else:
        tmp_sku = (
            work_df.groupby("ItemCode")
            .agg(
                SKU_Mean_Demand=("Clean_Demand", "mean"),
                SKU_Std_Demand=("Clean_Demand", "std"),
                SKU_ZeroRate=("Clean_Demand", lambda x: (x == 0).mean())
            )
            .reset_index()
        )
        tmp_sku = force_itemcode_str(tmp_sku)
        tmp_sku["SKU_Std_Demand"] = tmp_sku["SKU_Std_Demand"].fillna(0)
        tmp_sku["SKU_CV"] = np.where(
            tmp_sku["SKU_Mean_Demand"] <= 0,
            0,
            tmp_sku["SKU_Std_Demand"] / (tmp_sku["SKU_Mean_Demand"] + 1)
        )
        tmp_sku = tmp_sku[["ItemCode", "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]]

        work_df = work_df.drop(columns=["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"], errors="ignore")
        work_df = force_itemcode_str(work_df)
        work_df = work_df.merge(tmp_sku, on="ItemCode", how="left")

        for c in ["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]:
            work_df[c] = work_df[c].fillna(0)

    # artifact-based medium subgroup profile
    if str(artifacts.get("segment", "")).upper() == "MEDIUM":
        medium_profile_df = artifacts.get("medium_profile_df", pd.DataFrame())
        if isinstance(medium_profile_df, pd.DataFrame) and not medium_profile_df.empty:
            medium_profile_df = force_itemcode_str(medium_profile_df)
            work_df = force_itemcode_str(work_df)
            work_df = merge_medium_sku_profile(work_df, medium_profile_df)
            work_df = force_itemcode_str(work_df)

    work_df = add_bonus_cycle_features(work_df)
    work_df = force_itemcode_str(work_df)
    work_df = rebuild_time_features(work_df)
    work_df = force_itemcode_str(work_df)

    if "clip_caps" in artifacts and artifacts["clip_caps"] is not None:
        work_df = apply_clip_caps(work_df, artifacts["clip_caps"])
        work_df = force_itemcode_str(work_df)

    work_df = recompute_target(work_df)
    work_df = force_itemcode_str(work_df)
    work_df = add_residual_target(work_df)
    work_df = force_itemcode_str(work_df)

    itemcode_categories = artifacts.get("itemcode_categories", pd.Index([]))
    cat_to_code = {str(k): i for i, k in enumerate(itemcode_categories)}
    unk_code = len(cat_to_code)

    work_df["ItemCode_Original"] = work_df["ItemCode"].astype(str)
    work_df["ItemCode_Encoded"] = work_df["ItemCode"].astype(str).map(cat_to_code).fillna(unk_code).astype(int)

    work_df_model = work_df.copy()
    work_df_model["ItemCode"] = work_df_model["ItemCode_Encoded"]

    next_row = work_df_model[
        (work_df_model["ItemCode_Original"] == sku_code) &
        (work_df_model["Year"] == next_year) &
        (work_df_model["Month_Number"] == next_month)
    ].copy()

    if next_row.empty:
        return None

    next_row["Segment_For_Calibration"] = artifacts.get("segment", "")
    next_row = ensure_inference_features(next_row, feature_cols)
    
    assert_features_exist(next_row, feature_cols, where="FINAL_INFERENCE_NEXT_ROW")

    X_next = sanitize_model_input(next_row[feature_cols])

    scaler = artifacts.get("feature_scaler", None)
    if scaler is not None:
        X_next = scaler.transform(X_next)

    pred_residual = float(np.array(model.predict(X_next)).reshape(-1)[0])

    baseline = float(next_row[BASELINE_COL].iloc[0])
    
    pred_residual_raw = float(pred_residual)
    pred_residual = apply_residual_strength(
        next_row.iloc[0].to_dict(),
        pred_residual_raw,
        segment=next_row["Segment_For_Calibration"].iloc[0]
    )

    raw_forecast = max(baseline + pred_residual, 0.0)

    row_dict = next_row.iloc[0].to_dict()
    forecast = apply_promo_aware_adjustment(row_dict, raw_forecast)
    forecast = apply_final_forecast_guardrails(row_dict, forecast)
    forecast = apply_production_calibration(row_dict, forecast)

    # prevent over-aggressive low forecast
    lag1 = float(row_dict.get("Lag1", 0) or 0)
    rolling_mean = float(row_dict.get("Rolling3M_Mean", 0) or 0)
    if baseline > 100 and forecast < 0.25 * baseline:
        forecast = max(0.60 * baseline, 0.50 * lag1, 0.50 * rolling_mean)

    subgroup_val = subgroup_name
    if str(artifacts.get("segment", "")).upper() == "MEDIUM" and subgroup_val == "":
        subgroup_val = next_row["Medium_Subgroup"].iloc[0] if "Medium_Subgroup" in next_row.columns else ""

    return {
        "ItemCode": int(float(sku_code)),
        "Forecast_Year": next_year,
        "Forecast_Month": next_month,
        "Forecast_Prediction": forecast,
        "Segment": artifacts.get("segment", "UNKNOWN"),
        "Subgroup": subgroup_val,
        "Champion_Model": str(used_model_name).upper(),
        "Used_Model": str(used_model_name).upper(),
        "Fallback_Used": 0,
        "Expected_Bonus": int(next_bonus),
        "Residual_Baseline": baseline,
        "Predicted_Residual": pred_residual,
        "Status": "Success",
        "Predicted_Residual": float(pred_residual),
        "Predicted_Residual_Raw": float(pred_residual_raw),
    }

# ------------------------------------------------------------
# J) FALLBACK FORECASTER
# ------------------------------------------------------------
def forecast_with_fallback(sku_code, raw_data, segment, fallback_type="ROLLING3", subgroup_name=""):
    sku_code = str(sku_code)
    fallback_type = str(fallback_type).upper()

    if segment == "LONG":
        routing = get_long_routing_row(sku_code)
        if routing is None:
            return None

        best_model = str(routing.get("Best_Model", "XGBOOST")).upper()

        if best_model in ["XGBOOST", "CATBOOST", "LIGHTGBM"]:
            artifacts = get_long_deploy_artifact(best_model)
        else:
            artifacts = get_long_deploy_artifact("XGBOOST")

    elif segment == "MEDIUM":
        if subgroup_name == "":
            routing = get_medium_routing_row(sku_code)
            if routing is None:
                return None
            subgroup_name = str(routing.get("Medium_Subgroup", "STABLE")).upper()

        subgroup_hits = medium_deploy_artifact_registry[
            medium_deploy_artifact_registry["Medium_Subgroup"].astype(str).str.upper() == subgroup_name
        ].copy()

        if subgroup_hits.empty:
            return None

        artifacts = None
        for candidate_model in subgroup_hits["Model_Name"].astype(str).str.upper().tolist():
            artifacts = get_medium_deploy_artifact(subgroup_name, candidate_model)
            if artifacts is not None:
                break
    else:
        return None

    if artifacts is None:
        return None

    prepared_df = prepare_residual_forecast_frame(artifacts, raw_data)
    prepared_df = force_itemcode_str(prepared_df)
    if prepared_df.empty:
        return None

    sku_hist = prepared_df[prepared_df["ItemCode_Original"] == sku_code].copy().sort_values(["Year", "Month_Number"])
    if sku_hist.empty:
        return None

    last_row, next_year, next_month = get_next_period_from_history(prepared_df, sku_code)
    if last_row is None:
        return None

    next_bonus = infer_expected_bonus_flag(sku_code, prepared_df)

    new_row = last_row.copy()
    new_row = force_itemcode_str(new_row)
    new_row["Year"] = next_year
    new_row["Month_Number"] = next_month
    new_row["Bonus_Flag"] = int(next_bonus)

    for c in ["Secondary_Sales_Qty", "Primary_Sales_Qty", "Free_Qty", "Observed_Demand", "Effective_Demand", "Clean_Demand"]:
        if c in new_row.columns:
            new_row[c] = 0

    work_df = prepared_df.drop(columns=["ItemCode_Encoded"], errors="ignore").copy()
    work_df = force_itemcode_str(work_df)
    work_df = pd.concat([work_df, new_row], ignore_index=True)
    work_df = force_itemcode_str(work_df)
    work_df = work_df.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    bonus_pattern_df_f = detect_recurring_bonus_skus(work_df)[[
        "ItemCode",
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ]].copy()
    bonus_pattern_df_f = force_itemcode_str(bonus_pattern_df_f)

    work_df = work_df.drop(columns=[
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ], errors="ignore")
    work_df = force_itemcode_str(work_df)
    work_df = work_df.merge(bonus_pattern_df_f, on="ItemCode", how="left")

    for c in ["Recurring_Bonus_SKU", "Bonus_Cycle_Length", "Avg_Bonus_Gap", "Bonus_Frequency_All"]:
        work_df[c] = work_df[c].fillna(0)
    work_df["Avg_Bonus_Uplift"] = work_df["Avg_Bonus_Uplift"].fillna(1.0)

    abc_map = artifacts.get("abc_map", {})
    work_df["ABC_Class"] = work_df["ItemCode"].map(abc_map).fillna(2)

    promo_profile_df = artifacts.get("promo_profile_df", pd.DataFrame())
    if isinstance(promo_profile_df, pd.DataFrame) and not promo_profile_df.empty:
        promo_profile_df = force_itemcode_str(promo_profile_df)
        work_df = force_itemcode_str(work_df)
        work_df = merge_promo_profile(work_df, promo_profile_df)
        work_df = force_itemcode_str(work_df)

    sku_profile_df = artifacts.get("sku_profile_df", pd.DataFrame())
    if isinstance(sku_profile_df, pd.DataFrame) and not sku_profile_df.empty:
        sku_profile_df = force_itemcode_str(sku_profile_df)
        keep_cols = [c for c in ["ItemCode", "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"] if c in sku_profile_df.columns]

        work_df = work_df.drop(columns=["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"], errors="ignore")
        work_df = force_itemcode_str(work_df)
        work_df = work_df.merge(sku_profile_df[keep_cols], on="ItemCode", how="left")

        for c in ["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]:
            if c in work_df.columns:
                work_df[c] = work_df[c].fillna(0)

    if str(artifacts.get("segment", "")).upper() == "MEDIUM":
        medium_profile_df = artifacts.get("medium_profile_df", pd.DataFrame())
        if isinstance(medium_profile_df, pd.DataFrame) and not medium_profile_df.empty:
            medium_profile_df = force_itemcode_str(medium_profile_df)
            work_df = force_itemcode_str(work_df)
            work_df = merge_medium_sku_profile(work_df, medium_profile_df)
            work_df = force_itemcode_str(work_df)

    work_df = add_bonus_cycle_features(work_df)
    work_df = force_itemcode_str(work_df)
    work_df = rebuild_time_features(work_df)
    work_df = force_itemcode_str(work_df)

    if "clip_caps" in artifacts and artifacts["clip_caps"] is not None:
        work_df = apply_clip_caps(work_df, artifacts["clip_caps"])
        work_df = force_itemcode_str(work_df)

    work_df = recompute_target(work_df)
    work_df = force_itemcode_str(work_df)
    work_df = add_residual_target(work_df)
    work_df = force_itemcode_str(work_df)

    work_df["ItemCode_Original"] = work_df["ItemCode"].astype(str)

    next_row = work_df[
        (work_df["ItemCode_Original"] == sku_code) &
        (work_df["Year"] == next_year) &
        (work_df["Month_Number"] == next_month)
    ].copy()

    if next_row.empty:
        return None

    next_row["Segment_For_Calibration"] = segment
    row_dict = next_row.iloc[0].to_dict()

    if segment == "MEDIUM" and fallback_type in ["MEDIUM_RULE", "RULE_BASED"]:
        pred = medium_rule_fallback_predict(row_dict)
    else:
        pred = fallback_forecast_from_row(row_dict, fallback_type=fallback_type)

    lag1 = float(row_dict.get("Lag1", 0) or 0)
    roll3 = float(row_dict.get("Rolling3M_Mean", 0) or 0)
    roll6 = float(row_dict.get("Rolling6M_Mean", roll3) or roll3)

    anchors = [x for x in [lag1, roll3, roll6] if x > 0]

    if len(anchors) > 0:
        robust_pred = float(np.median(anchors))
        pred = min(pred, robust_pred * 1.10)

    expected_bonus = int(row_dict.get("Expected_Bonus_NextMonth", 0) or 0)
    bonus_flag = int(row_dict.get("Bonus_Flag", 0) or 0)

    if expected_bonus != 1 and bonus_flag != 1:
        pred *= 0.90

    pred = apply_production_calibration(row_dict, pred)

    subgroup_val = subgroup_name if segment == "MEDIUM" else ""

    return {
        "ItemCode": int(float(sku_code)),
        "Forecast_Year": next_year,
        "Forecast_Month": next_month,
        "Forecast_Prediction": pred,
        "Segment": segment,
        "Subgroup": subgroup_val,
        "Champion_Model": "FALLBACK",
        "Used_Model": f"FALLBACK_{fallback_type}",
        "Fallback_Used": 1,
        "Expected_Bonus": int(next_bonus),
        "Residual_Baseline": np.nan,
        "Predicted_Residual": np.nan,
        "Status": "Success"
    }

def medium_rule_fallback_predict(row):
    lag1 = float(row.get("Lag1", 0) or 0)
    lag2 = float(row.get("Lag2", 0) or 0)
    lag3 = float(row.get("Lag3", 0) or 0)
    rolling3 = float(row.get("Rolling3M_Mean", 0) or 0)
    sku_mean = float(row.get("SKU_Mean_Demand", 0) or 0)

    last_bonus_demand = float(row.get("Last_Bonus_Demand", 0) or 0)
    avg_bonus_uplift = float(row.get("Avg_Bonus_Uplift", 1.0) or 1.0)

    expected_bonus = int(row.get("Expected_Bonus_NextMonth", row.get("Bonus_Flag", 0)) or 0)
    supply_flag = int(row.get("Supply_Constraint_Flag", 0) or 0)

    primary_stock = float(row.get("Available_Primary_Inventory_Qty", 0) or 0)
    distributor_stock = float(row.get("Distributor_Inventory_Qty", 0) or 0)

    subgroup = str(row.get("Medium_Subgroup", "STABLE")).upper()

    anchors = [x for x in [lag1, lag2, lag3, rolling3, sku_mean] if x > 0]

    if len(anchors) == 0:
        pred = 0.0
    else:
        pred = float(np.median(anchors))

    if subgroup == "PROMO_HEAVY":
        if expected_bonus == 1 and last_bonus_demand > 0:
            pred = 0.65 * pred + 0.35 * last_bonus_demand
        elif expected_bonus == 1:
            pred = pred * min(max(avg_bonus_uplift, 1.0), 1.6)

    if supply_flag == 1:
        pred = min(pred, max(lag1, rolling3, 0))

    if (primary_stock + distributor_stock) <= 0:
        pred *= 0.90

    return max(pred, 0.0)


# ------------------------------------------------------------
# M) MEDIUM ROUTER
# ------------------------------------------------------------
def forecast_medium_sku(sku_code, raw_data):
    sku_code = str(sku_code)
    routing = get_medium_routing_row(sku_code)

    if routing is None:
        return {
            "ItemCode": int(float(sku_code)),
            "Forecast_Year": np.nan,
            "Forecast_Month": np.nan,
            "Forecast_Prediction": np.nan,
            "Segment": "MEDIUM",
            "Subgroup": "",
            "Champion_Model": "UNKNOWN",
            "Used_Model": "UNKNOWN",
            "Fallback_Used": 0,
            "Expected_Bonus": np.nan,
            "Residual_Baseline": np.nan,
            "Predicted_Residual": np.nan,
            "Status": "FAILED: MEDIUM routing row missing"
        }

    subgroup = str(routing.get("Medium_Subgroup", "STABLE")).upper()
    final_model = str(routing.get("Final_Model", routing.get("Best_Model", "XGBOOST"))).upper()
    best_model = str(routing.get("Best_Model", "XGBOOST")).upper()
    fallback_type = str(routing.get("Fallback_Type", "ROLLING3")).upper()

    if final_model == "FALLBACK":
        result = forecast_with_fallback(
            sku_code=sku_code,
            raw_data=raw_data,
            segment="MEDIUM",
            fallback_type=fallback_type,
            subgroup_name=subgroup
        )
        if result is not None:
            result["Champion_Model"] = best_model
            result["Subgroup"] = subgroup
            return result

        return {
            "ItemCode": int(float(sku_code)),
            "Forecast_Year": np.nan,
            "Forecast_Month": np.nan,
            "Forecast_Prediction": np.nan,
            "Segment": "MEDIUM",
            "Subgroup": subgroup,
            "Champion_Model": best_model,
            "Used_Model": f"FALLBACK_{fallback_type}",
            "Fallback_Used": 1,
            "Expected_Bonus": np.nan,
            "Residual_Baseline": np.nan,
            "Predicted_Residual": np.nan,
            "Status": f"FAILED: MEDIUM fallback returned None ({fallback_type})"
        }

    artifacts = get_medium_deploy_artifact(subgroup, best_model)
    if artifacts is None:
        result = forecast_with_fallback(
            sku_code=sku_code,
            raw_data=raw_data,
            segment="MEDIUM",
            fallback_type=fallback_type,
            subgroup_name=subgroup
        )
        if result is not None:
            result["Champion_Model"] = best_model
            result["Subgroup"] = subgroup
            result["Used_Model"] = f"FALLBACK_{fallback_type}_NO_ARTIFACT"
            result["Status"] = f"Success (missing {best_model} artifact -> fallback used)"
            return result

        return {
            "ItemCode": int(float(sku_code)),
            "Forecast_Year": np.nan,
            "Forecast_Month": np.nan,
            "Forecast_Prediction": np.nan,
            "Segment": "MEDIUM",
            "Subgroup": subgroup,
            "Champion_Model": best_model,
            "Used_Model": best_model,
            "Fallback_Used": 0,
            "Expected_Bonus": np.nan,
            "Residual_Baseline": np.nan,
            "Predicted_Residual": np.nan,
            "Status": f"FAILED: MEDIUM artifact missing ({subgroup}/{best_model})"
        }

    try:
        result = predict_tree_model_sku(
            sku_code=sku_code,
            raw_data=raw_data,
            artifacts=artifacts,
            used_model_name=best_model,
            subgroup_name=subgroup
        )
        if result is not None:
            result["Champion_Model"] = best_model
            result["Subgroup"] = subgroup
            return result
        model_error = "predict_tree_model_sku returned None"
    except Exception as e:
        model_error = str(e)

    fb = forecast_with_fallback(
        sku_code=sku_code,
        raw_data=raw_data,
        segment="MEDIUM",
        fallback_type=fallback_type,
        subgroup_name=subgroup
    )
    if fb is not None:
        fb["Champion_Model"] = best_model
        fb["Subgroup"] = subgroup
        fb["Used_Model"] = f"FALLBACK_{fallback_type}_AFTER_{best_model}_FAIL"
        fb["Status"] = f"Success ({best_model} failed -> fallback used: {model_error})"
        return fb

    return {
        "ItemCode": int(float(sku_code)),
        "Forecast_Year": np.nan,
        "Forecast_Month": np.nan,
        "Forecast_Prediction": np.nan,
        "Segment": "MEDIUM",
        "Subgroup": subgroup,
        "Champion_Model": best_model,
        "Used_Model": best_model,
        "Fallback_Used": 0,
        "Expected_Bonus": np.nan,
        "Residual_Baseline": np.nan,
        "Predicted_Residual": np.nan,
        "Status": f"FAILED: MEDIUM {best_model} failed and fallback failed | {model_error}"
    }

### SHORT

In [ ]:
### SHORT SEGMENT
# ============================================================
# SHORT RULE-BASED INFERENCE
# ============================================================

# ------------------------------------------------------------
# K) SHORT SUBGROUP FILTER
# ------------------------------------------------------------
def filter_short_subgroup(df, subgroup_name):
    df = force_itemcode_str(df)
    df = df.copy()

    subgroup_name = str(subgroup_name).upper()

    if subgroup_name == "SHORT_PROMO":
        return df[df["Short_SKU_Type"] == "SHORT_PROMO"].copy()

    if subgroup_name == "SHORT_NORMAL":
        return df[df["Short_SKU_Type"] == "SHORT_NORMAL"].copy()

    raise ValueError(f"Unknown short subgroup: {subgroup_name}")


# ------------------------------------------------------------
# L) SHORT RULE HELPERS
# ------------------------------------------------------------
def build_short_sku_profile(df):
    df = force_itemcode_str(df)
    df = df.copy().sort_values(["ItemCode", "Year", "Month_Number"])
    out = []

    for item, g in df.groupby("ItemCode"):
        hist_len = g[["Year", "Month_Number"]].drop_duplicates().shape[0]
        mean_demand = float(g["Clean_Demand"].mean()) if hist_len > 0 else 0.0
        zero_rate = float((g["Clean_Demand"] == 0).mean()) if hist_len > 0 else 0.0
        bonus_freq = float(g["Bonus_Flag"].mean()) if "Bonus_Flag" in g.columns else 0.0
        supply_rate = float(g["Supply_Constraint_Flag"].mean()) if "Supply_Constraint_Flag" in g.columns else 0.0

        total_demand = float(g["Clean_Demand"].sum())
        bonus_demand = float(g.loc[g["Bonus_Flag"] == 1, "Clean_Demand"].sum()) if total_demand > 0 else 0.0
        bonus_share = 0.0 if total_demand <= 0 else bonus_demand / total_demand

        if bonus_freq >= 0.25 or bonus_share >= 0.40:
            short_type = "SHORT_PROMO"
        else:
            short_type = "SHORT_NORMAL"

        out.append({
            "ItemCode": str(item),
            "Short_SKU_Type": short_type,
            "Short_History_Length": hist_len,
            "Short_Mean_Demand": mean_demand,
            "Short_ZeroRate": zero_rate,
            "Short_Bonus_Frequency": bonus_freq,
            "Short_Bonus_Demand_Share": bonus_share,
            "Short_Supply_Rate": supply_rate
        })

    out_df = pd.DataFrame(out)
    out_df = force_itemcode_str(out_df)
    return out_df


def merge_short_sku_profile(df, profile_df):
    df = force_itemcode_str(df)
    profile_df = force_itemcode_str(profile_df)

    keep_cols = [
        "ItemCode",
        "Short_SKU_Type",
        "Short_History_Length",
        "Short_Mean_Demand",
        "Short_ZeroRate",
        "Short_Bonus_Frequency",
        "Short_Bonus_Demand_Share",
        "Short_Supply_Rate"
    ]

    df = df.drop(columns=[c for c in keep_cols if c != "ItemCode"], errors="ignore")
    df = df.merge(profile_df[keep_cols], on="ItemCode", how="left")

    df["Short_SKU_Type"] = df["Short_SKU_Type"].fillna("SHORT_NORMAL")
    for c in [
        "Short_History_Length",
        "Short_Mean_Demand",
        "Short_ZeroRate",
        "Short_Bonus_Frequency",
        "Short_Bonus_Demand_Share",
        "Short_Supply_Rate"
    ]:
        df[c] = df[c].fillna(0)

    return df


def short_rule_predict(row):
    lag1 = float(row.get("Lag1", 0) or 0)
    lag2 = float(row.get("Lag2", 0) or 0)
    rolling3 = float(row.get("Rolling3M_Mean", 0) or 0)
    sku_mean = float(row.get("SKU_Mean_Demand", 0) or 0)

    last_bonus_demand = float(row.get("Last_Bonus_Demand", 0) or 0)
    avg_bonus_uplift = float(row.get("Avg_Bonus_Uplift", 1.0) or 1.0)

    bonus_flag = int(row.get("Bonus_Flag", 0) or 0)
    expected_bonus = int(row.get("Expected_Bonus_NextMonth", 0) or 0)
    supply_flag = int(row.get("Supply_Constraint_Flag", 0) or 0)

    primary_stock = float(row.get("Available_Primary_Inventory_Qty", 0) or 0)
    distributor_stock = float(row.get("Distributor_Inventory_Qty", 0) or 0)

    short_type = row.get("Short_SKU_Type", "SHORT_NORMAL")
    hist_len = int(row.get("Short_History_Length", 0) or 0)

    anchors = [x for x in [lag1, lag2, rolling3, sku_mean] if x > 0]

    if len(anchors) == 0:
        pred = 0.0
    elif hist_len <= 2:
        pred = float(np.mean(anchors))
    else:
        pred = float(np.median(anchors))

    if short_type == "SHORT_PROMO":
        if expected_bonus == 1 and last_bonus_demand > 0:
            pred = 0.60 * pred + 0.40 * last_bonus_demand
        elif bonus_flag == 1:
            pred = pred * min(max(avg_bonus_uplift, 1.0), 1.8)

    if supply_flag == 1:
        pred = min(pred, max(lag1, rolling3, 0))

    if (primary_stock + distributor_stock) <= 0:
        pred *= 0.85

    return max(pred, 0.0)


# ------------------------------------------------------------
# M) SHORT DEPLOY ARTIFACT LOOKUP
# ------------------------------------------------------------
def get_short_deploy_artifact(subgroup_name=None):
    subgroup_name = None if subgroup_name is None else str(subgroup_name).upper()

    if subgroup_name == "SHORT_PROMO":
        return globals().get("short_promo_deploy_artifacts", globals().get("short_deploy_artifacts"))

    if subgroup_name == "SHORT_NORMAL":
        return globals().get("short_normal_deploy_artifacts", globals().get("short_deploy_artifacts"))

    return globals().get("short_deploy_artifacts")


# ------------------------------------------------------------
# N) PREPARE SHORT INFERENCE FRAME FROM DEPLOY ARTIFACTS
# ------------------------------------------------------------
def prepare_short_inference_frame(raw_data, artifacts, subgroup_name=None):
    df = raw_data.copy().sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)
    df = force_itemcode_str(df)

    df = add_history_length_from_subset(df, df)
    df = force_itemcode_str(df)
    df = df[df["History_Segment"] == "SHORT"].copy()
    df = force_itemcode_str(df)

    if df.empty:
        return df

    bonus_pattern_df = detect_recurring_bonus_skus(df)[[
        "ItemCode",
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ]].copy()
    bonus_pattern_df = force_itemcode_str(bonus_pattern_df)

    df = df.drop(columns=[
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift"
    ], errors="ignore")
    df = force_itemcode_str(df)
    df = df.merge(bonus_pattern_df, on="ItemCode", how="left")

    for c in ["Recurring_Bonus_SKU", "Bonus_Cycle_Length", "Avg_Bonus_Gap", "Bonus_Frequency_All"]:
        df[c] = df[c].fillna(0)
    df["Avg_Bonus_Uplift"] = df["Avg_Bonus_Uplift"].fillna(1.0)

    df, _ = apply_sku_cap(df.copy(), df.copy())
    df = force_itemcode_str(df)

    abc_map = artifacts.get("abc_map", {})
    df["ABC_Class"] = df["ItemCode"].map(abc_map).fillna(2)

    promo_profile_df = artifacts.get("promo_profile_df", pd.DataFrame())
    if isinstance(promo_profile_df, pd.DataFrame) and not promo_profile_df.empty:
        promo_profile_df = force_itemcode_str(promo_profile_df)
        df = force_itemcode_str(df)
        df = merge_promo_profile(df, promo_profile_df)
        df = force_itemcode_str(df)
    else:
        tmp_promo = build_promo_profile(df)
        tmp_promo = force_itemcode_str(tmp_promo)
        df = force_itemcode_str(df)
        df = merge_promo_profile(df, tmp_promo)
        df = force_itemcode_str(df)

    short_profile_df = artifacts.get("short_profile_df", pd.DataFrame())
    if isinstance(short_profile_df, pd.DataFrame) and not short_profile_df.empty:
        short_profile_df = force_itemcode_str(short_profile_df)
        df = force_itemcode_str(df)
        df = merge_short_sku_profile(df, short_profile_df)
        df = force_itemcode_str(df)
    else:
        tmp_short_profile = build_short_sku_profile(df)
        tmp_short_profile = force_itemcode_str(tmp_short_profile)
        df = force_itemcode_str(df)
        df = merge_short_sku_profile(df, tmp_short_profile)
        df = force_itemcode_str(df)

    sku_profile_df = artifacts.get("sku_profile_df", pd.DataFrame())
    if isinstance(sku_profile_df, pd.DataFrame) and not sku_profile_df.empty:
        sku_profile_df = force_itemcode_str(sku_profile_df)
        keep_cols = [c for c in ["ItemCode", "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"] if c in sku_profile_df.columns]

        df = df.drop(columns=["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"], errors="ignore")
        df = force_itemcode_str(df)
        df = df.merge(sku_profile_df[keep_cols], on="ItemCode", how="left")

        for c in ["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]:
            if c in df.columns:
                df[c] = df[c].fillna(0)
    else:
        tmp_sku = (
            df.groupby("ItemCode")
            .agg(
                SKU_Mean_Demand=("Clean_Demand", "mean"),
                SKU_Std_Demand=("Clean_Demand", "std"),
                SKU_ZeroRate=("Clean_Demand", lambda x: (x == 0).mean())
            )
            .reset_index()
        )
        tmp_sku = force_itemcode_str(tmp_sku)
        tmp_sku["SKU_Std_Demand"] = tmp_sku["SKU_Std_Demand"].fillna(0)
        tmp_sku["SKU_CV"] = np.where(
            tmp_sku["SKU_Mean_Demand"] <= 0,
            0,
            tmp_sku["SKU_Std_Demand"] / (tmp_sku["SKU_Mean_Demand"] + 1)
        )
        tmp_sku = tmp_sku[["ItemCode", "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]]

        df = df.drop(columns=["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"], errors="ignore")
        df = force_itemcode_str(df)
        df = df.merge(tmp_sku, on="ItemCode", how="left")

        for c in ["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]:
            df[c] = df[c].fillna(0)

    if subgroup_name is not None:
        df = filter_short_subgroup(df, subgroup_name)

    if df.empty:
        return df

    df = add_bonus_cycle_features(df)
    df = force_itemcode_str(df)
    df = rebuild_time_features(df)
    df = force_itemcode_str(df)

    if "clip_caps" in artifacts and artifacts["clip_caps"] is not None:
        df = apply_clip_caps(df, artifacts["clip_caps"])
        df = force_itemcode_str(df)

    return df


# ------------------------------------------------------------
# O) SHORT FORECAST
# ------------------------------------------------------------
def forecast_short_sku(sku_code, raw_data):
    sku_code = str(sku_code)

    try:
        base_artifacts = get_short_deploy_artifact(None)
        if base_artifacts is None:
            return {
                "ItemCode": int(float(sku_code)),
                "Forecast_Year": np.nan,
                "Forecast_Month": np.nan,
                "Forecast_Prediction": np.nan,
                "Segment": "SHORT",
                "Subgroup": "",
                "Champion_Model": "RULE_BASED",
                "Used_Model": "RULE_BASED",
                "Fallback_Used": 0,
                "Expected_Bonus": np.nan,
                "Residual_Baseline": np.nan,
                "Predicted_Residual": np.nan,
                "Status": "FAILED: short deploy artifacts missing"
            }

        prepared_df_all = prepare_short_inference_frame(raw_data, base_artifacts, subgroup_name=None)
        prepared_df_all = force_itemcode_str(prepared_df_all)

        if prepared_df_all.empty:
            return {
                "ItemCode": int(float(sku_code)),
                "Forecast_Year": np.nan,
                "Forecast_Month": np.nan,
                "Forecast_Prediction": np.nan,
                "Segment": "SHORT",
                "Subgroup": "",
                "Champion_Model": "RULE_BASED",
                "Used_Model": "RULE_BASED",
                "Fallback_Used": 0,
                "Expected_Bonus": np.nan,
                "Residual_Baseline": np.nan,
                "Predicted_Residual": np.nan,
                "Status": "FAILED: prepared short frame empty"
            }

        sku_hist_all = prepared_df_all[prepared_df_all["ItemCode"].astype(str) == sku_code].copy()
        sku_hist_all = sku_hist_all.sort_values(["Year", "Month_Number"])

        if sku_hist_all.empty:
            return {
                "ItemCode": int(float(sku_code)),
                "Forecast_Year": np.nan,
                "Forecast_Month": np.nan,
                "Forecast_Prediction": np.nan,
                "Segment": "SHORT",
                "Subgroup": "",
                "Champion_Model": "RULE_BASED",
                "Used_Model": "RULE_BASED",
                "Fallback_Used": 0,
                "Expected_Bonus": np.nan,
                "Residual_Baseline": np.nan,
                "Predicted_Residual": np.nan,
                "Status": "FAILED: short sku not found in prepared frame"
            }

        short_type = sku_hist_all["Short_SKU_Type"].iloc[-1] if "Short_SKU_Type" in sku_hist_all.columns else "SHORT_NORMAL"

        subgroup_artifacts = get_short_deploy_artifact(short_type)
        if subgroup_artifacts is None:
            subgroup_artifacts = base_artifacts

        prepared_df = prepare_short_inference_frame(raw_data, subgroup_artifacts, subgroup_name=short_type)
        prepared_df = force_itemcode_str(prepared_df)

        if prepared_df.empty:
            return {
                "ItemCode": int(float(sku_code)),
                "Forecast_Year": np.nan,
                "Forecast_Month": np.nan,
                "Forecast_Prediction": np.nan,
                "Segment": "SHORT",
                "Subgroup": short_type,
                "Champion_Model": "RULE_BASED",
                "Used_Model": "RULE_BASED",
                "Fallback_Used": 0,
                "Expected_Bonus": np.nan,
                "Residual_Baseline": np.nan,
                "Predicted_Residual": np.nan,
                "Status": "FAILED: subgroup short frame empty"
            }

        sku_hist = prepared_df[prepared_df["ItemCode"].astype(str) == sku_code].copy()
        sku_hist = sku_hist.sort_values(["Year", "Month_Number"])

        if sku_hist.empty:
            return {
                "ItemCode": int(float(sku_code)),
                "Forecast_Year": np.nan,
                "Forecast_Month": np.nan,
                "Forecast_Prediction": np.nan,
                "Segment": "SHORT",
                "Subgroup": short_type,
                "Champion_Model": "RULE_BASED",
                "Used_Model": "RULE_BASED",
                "Fallback_Used": 0,
                "Expected_Bonus": np.nan,
                "Residual_Baseline": np.nan,
                "Predicted_Residual": np.nan,
                "Status": "FAILED: short sku missing after subgroup filter"
            }

        last_row = sku_hist.iloc[-1:].copy()

        next_month = int(last_row["Month_Number"].iloc[0]) + 1
        next_year = int(last_row["Year"].iloc[0])
        if next_month > 12:
            next_month = 1
            next_year += 1

        next_bonus = infer_expected_bonus_flag(sku_code, prepared_df)

        new_row = last_row.copy()
        new_row["Year"] = next_year
        new_row["Month_Number"] = next_month
        new_row["Bonus_Flag"] = int(next_bonus)

        for c in ["Secondary_Sales_Qty", "Primary_Sales_Qty", "Free_Qty", "Observed_Demand", "Effective_Demand", "Clean_Demand"]:
            if c in new_row.columns:
                new_row[c] = 0

        future_df = pd.concat([prepared_df.copy(), new_row], ignore_index=True)
        future_df = force_itemcode_str(future_df)
        future_df = future_df.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

        future_df = add_bonus_cycle_features(future_df)
        future_df = force_itemcode_str(future_df)
        future_df = rebuild_time_features(future_df)
        future_df = force_itemcode_str(future_df)

        if "clip_caps" in subgroup_artifacts and subgroup_artifacts["clip_caps"] is not None:
            future_df = apply_clip_caps(future_df, subgroup_artifacts["clip_caps"])
            future_df = force_itemcode_str(future_df)

        future_df = recompute_target(future_df)
        future_df = force_itemcode_str(future_df)

        next_row = future_df[
            (future_df["ItemCode"].astype(str) == sku_code) &
            (future_df["Year"] == next_year) &
            (future_df["Month_Number"] == next_month)
        ].copy()

        if next_row.empty:
            return {
                "ItemCode": int(float(sku_code)),
                "Forecast_Year": np.nan,
                "Forecast_Month": np.nan,
                "Forecast_Prediction": np.nan,
                "Segment": "SHORT",
                "Subgroup": short_type,
                "Champion_Model": "RULE_BASED",
                "Used_Model": "RULE_BASED",
                "Fallback_Used": 0,
                "Expected_Bonus": int(next_bonus),
                "Residual_Baseline": np.nan,
                "Predicted_Residual": np.nan,
                "Status": "FAILED: next short row not created"
            }

        next_row["Segment_For_Calibration"] = "SHORT"

        row_dict = next_row.iloc[0].to_dict()
        pred = short_rule_predict(row_dict)
        pred = apply_production_calibration(row_dict, pred)

        short_type = next_row["Short_SKU_Type"].iloc[0] if "Short_SKU_Type" in next_row.columns else "SHORT_NORMAL"

        return {
            "ItemCode": int(float(sku_code)),
            "Forecast_Year": next_year,
            "Forecast_Month": next_month,
            "Forecast_Prediction": float(pred),
            "Segment": "SHORT",
            "Subgroup": short_type,
            "Champion_Model": "RULE_BASED",
            "Used_Model": "RULE_BASED",
            "Fallback_Used": 0,
            "Expected_Bonus": int(next_bonus),
            "Residual_Baseline": np.nan,
            "Predicted_Residual": np.nan,
            "Status": "Success"
        }

    except Exception as e:
        return {
            "ItemCode": int(float(sku_code)),
            "Forecast_Year": np.nan,
            "Forecast_Month": np.nan,
            "Forecast_Prediction": np.nan,
            "Segment": "SHORT",
            "Subgroup": "",
            "Champion_Model": "RULE_BASED",
            "Used_Model": "RULE_BASED",
            "Fallback_Used": 0,
            "Expected_Bonus": np.nan,
            "Residual_Baseline": np.nan,
            "Predicted_Residual": np.nan,
            "Status": f"FAILED: SHORT exception | {str(e)}"
        }

### UNIFIED

In [ ]:
# ------------------------------------------------------------
# N) UNIFIED ROUTER
# ------------------------------------------------------------
def forecast_one_sku(sku_code, raw_data):
    sku_code = str(sku_code)
    segment = choose_segment_by_history(raw_data, sku_code)
    behavior_type = get_latest_behavior_type(raw_data, sku_code)

    if segment == "LONG":
        result = forecast_long_sku(sku_code, raw_data)
    elif segment == "MEDIUM":
        result = forecast_medium_sku(sku_code, raw_data)
    else:
        result = forecast_short_sku(sku_code, raw_data)

    if result is not None:
        result["Behavior_Type"] = behavior_type

    return result


In [ ]:
# ------------------------------------------------------------
# O) BULK FORECAST
# ------------------------------------------------------------
def bulk_segmented_forecast(raw_data, sku_list):
    results = []
    failed = []

    for sku in sku_list:
        sku = str(sku)
        result = forecast_one_sku(sku, raw_data)

        if result is None:
            failed.append({
                "ItemCode": int(float(sku)),
                "Segment": choose_segment_by_history(raw_data, sku),
                "Behavior_Type": get_latest_behavior_type(raw_data, sku),   
                "Used_Model": "UNKNOWN",
                "Status": "FAILED: forecast_one_sku returned None"
            })
            continue

        status = str(result.get("Status", ""))
        used_model = str(result.get("Used_Model", "UNKNOWN"))
        segment = str(result.get("Segment", choose_segment_by_history(raw_data, sku)))

        if status.upper().startswith("FAILED"):
            failed.append({
                "ItemCode": int(float(sku)),
                "Segment": segment,
                "Behavior_Type": get_latest_behavior_type(raw_data, sku),
                "Used_Model": used_model,
                "Status": status
            })
        else:
            results.append(result)

    results_df = pd.DataFrame(results)
    failed_df = pd.DataFrame(failed)

    if not results_df.empty:
        results_df = results_df.sort_values(
            ["Forecast_Year", "Forecast_Month", "Segment", "ItemCode"]
        ).reset_index(drop=True)

    if not failed_df.empty:
        failed_df = failed_df.sort_values(["ItemCode"]).reset_index(drop=True)

    return results_df, failed_df


def build_recent_calibration_table(long_recent_df, medium_recent_df=None):
    parts = []

    long_calib = long_recent_df.copy()
    long_calib["Segment"] = "LONG"
    long_calib["Subgroup"] = ""
    parts.append(long_calib)

    if medium_recent_df is not None:
        medium_calib = medium_recent_df.copy()
        medium_calib["Segment"] = "MEDIUM"
        parts.append(medium_calib)

    df = pd.concat(parts, ignore_index=True)

    calib = (
        df.groupby(["ItemCode", "Segment", "Model_Name"], as_index=False)
        .agg(
            Actual_Sum=("Actual", "sum"),
            Pred_Sum=("Pred", "sum")
        )
    )

    calib["Calibration_Factor"] = np.where(
        calib["Pred_Sum"] > 0,
        calib["Actual_Sum"] / calib["Pred_Sum"],
        1.0
    )

    calib["Calibration_Factor"] = calib["Calibration_Factor"].clip(0.65, 1.15)

    return calib[["ItemCode", "Segment", "Model_Name", "Calibration_Factor"]]


# ------------------------------------------------------------
# T) PATCH LONG ROUTER TO INCLUDE GRU
# Replace your existing forecast_long_sku with this version
# ------------------------------------------------------------
def build_actual_segment_sku_sets(raw_data, sku_list):
    work = raw_data.copy()
    work["ItemCode"] = work["ItemCode"].astype(str)

    requested = set(map(str, sku_list))
    work = work[work["ItemCode"].isin(requested)].copy()

    in_data = set(work["ItemCode"].unique())

    long_skus = set(work.loc[work["History_Segment"] == "LONG", "ItemCode"].unique())
    medium_skus = set(work.loc[work["History_Segment"] == "MEDIUM", "ItemCode"].unique())
    short_skus = set(work.loc[work["History_Segment"] == "SHORT", "ItemCode"].unique())

    return {
        "requested": requested,
        "in_data": in_data,
        "long": long_skus,
        "medium": medium_skus,
        "short": short_skus
    }

def patch_missing_long_champion_rows(raw_data, champion_long_map_df, sku_list):
    out = champion_long_map_df.copy()
    out["ItemCode"] = out["ItemCode"].astype(str)

    seg_sets = build_actual_segment_sku_sets(raw_data, sku_list)
    actual_long = seg_sets["long"]

    mapped_long = set(out["ItemCode"].unique())
    missing_long = sorted(actual_long - mapped_long)

    if not missing_long:
        return out, missing_long

    filler = pd.DataFrame({
        "ItemCode": missing_long,
        "Segment": "LONG",
        "Best_Model": "XGBOOST",
        "Final_Model": "FALLBACK",
        "Fallback_Type": "ROLLING3",
        "Use_Fallback": 1,
        "Unreliable_Flag": 1,
        "Intermittent_Flag": 0,
        "Highly_Volatile_Flag": 0,
        "Best_Model_Score": np.nan,
        "Best_Model_Holdout12_WMAPE": np.nan,
        "Best_Model_Recent4A_WMAPE": np.nan,
        "Best_Model_Recent4B_WMAPE": np.nan,
        "Best_Model_Holdout12_MAE": np.nan,
        "Best_Model_Recent4A_MAE": np.nan,
        "Best_Model_Recent4B_MAE": np.nan,
        "Best_Model_Holdout12_Bias": np.nan,
        "Best_Model_Recent4A_Bias": np.nan,
        "Best_Model_Recent4B_Bias": np.nan,
        "Evaluation_Months_Holdout12": 0,
        "Evaluation_Months_Recent4A": 0,
        "Evaluation_Months_Recent4B": 0,
        "Holdout12_Actual_Sum": 0,
        "Holdout12_Pred_Sum": 0,
        "Recent4A_Actual_Sum": 0,
        "Recent4A_Pred_Sum": 0,
        "Recent4B_Actual_Sum": 0,
        "Recent4B_Pred_Sum": 0,
        "SKU_Zero_Rate": np.nan,
        "SKU_CV": np.nan,
    })

    out = pd.concat([out, filler], ignore_index=True)
    out = out.drop_duplicates(subset=["ItemCode"], keep="first").reset_index(drop=True)

    return out, missing_long


def post_prediction_pattern_validator(pred_df, history_df):
    pred_df = pred_df.copy()
    history_df = history_df.copy()

    pred_df["ItemCode"] = pred_df["ItemCode"].astype(str)
    history_df["ItemCode"] = history_df["ItemCode"].astype(str)

    adjusted_rows = []

    for _, row in pred_df.iterrows():
        item = row["ItemCode"]
        pred = float(row["Forecast_Prediction"])

        h = history_df[history_df["ItemCode"] == item].copy()
        h = h.sort_values(["Year", "Month_Number"])

        if h.empty or len(h) < 6:
            row["Forecast_Prediction_Before_Pattern_Validation"] = pred
            row["Pattern_Validation_Status"] = "NO_HISTORY"
            adjusted_rows.append(row)
            continue

        recent = h.tail(12)

        roll3 = recent["Clean_Demand"].tail(3).mean()
        roll6 = recent["Clean_Demand"].tail(6).mean()
        median12 = recent["Clean_Demand"].median()
        max12 = recent["Clean_Demand"].max()

        bonus_months = recent[recent["Bonus_Flag"] == 1]
        normal_months = recent[recent["Bonus_Flag"] == 0]

        normal_base = normal_months["Clean_Demand"].median() if len(normal_months) > 0 else median12
        bonus_base = bonus_months["Clean_Demand"].median() if len(bonus_months) > 0 else normal_base

        expected_bonus = int(row.get("Expected_Bonus", row.get("Expected_Bonus_NextMonth", 0)) or 0)

        row["Forecast_Prediction_Before_Pattern_Validation"] = pred

        # Case 1: no bonus expected, but prediction is bonus-like spike
        if expected_bonus != 1 and pred > max(roll3, roll6, normal_base) * 1.40:
            new_pred = max(roll3, normal_base) * 1.10
            row["Forecast_Prediction"] = new_pred
            row["Pattern_Validation_Status"] = "NO_BONUS_SPIKE_REDUCED"

        # Case 2: bonus expected, but prediction is beyond historical bonus behavior
        elif expected_bonus == 1 and len(bonus_months) > 0 and pred > bonus_base * 1.35:
            new_pred = bonus_base * 1.15
            row["Forecast_Prediction"] = new_pred
            row["Pattern_Validation_Status"] = "BONUS_SPIKE_CAPPED"

        # Case 3: prediction is far above any recent historical demand
        elif pred > max12 * 1.50:
            new_pred = max12 * 1.10
            row["Forecast_Prediction"] = new_pred
            row["Pattern_Validation_Status"] = "ABOVE_HISTORY_CAPPED"

        # Case 4: prediction is believable
        else:
            row["Forecast_Prediction"] = pred
            row["Pattern_Validation_Status"] = "PASS"

        adjusted_rows.append(row)

    return pd.DataFrame(adjusted_rows)

# ============================================================
# PATCH CHAMPION MAP COVERAGE BEFORE INFERENCE
# ============================================================
champion_long_map_df["ItemCode"] = champion_long_map_df["ItemCode"].astype(str)
champion_medium_map_df["ItemCode"] = champion_medium_map_df["ItemCode"].astype(str)
Cleaned_Base_Data["ItemCode"] = Cleaned_Base_Data["ItemCode"].astype(str)

champion_long_map_df, patched_missing_long = patch_missing_long_champion_rows(
    raw_data=Cleaned_Base_Data,
    champion_long_map_df=champion_long_map_df,
    sku_list=PHARMA_SKUS
)


print("\n--- Champion Map Coverage Patch ---")
print("Patched LONG missing SKUs:", len(patched_missing_long))
if len(patched_missing_long) > 0:
    print("Sample patched LONG SKUs:", patched_missing_long[:10])

print("MEDIUM champion map is used directly from retraining output.")
print("MEDIUM champion rows:", len(champion_medium_map_df))

In [ ]:
required_medium_cols = [
    "ItemCode", "Medium_Subgroup", "Best_Model", "Final_Model",
    "Fallback_Type", "Use_Fallback"
]

missing_cols = [c for c in required_medium_cols if c not in champion_medium_map_df.columns]
if missing_cols:
    raise ValueError(f"❌ champion_medium_map_df missing columns: {missing_cols}")

required_long_cols = [
    "ItemCode", "Best_Model", "Final_Model",
    "Fallback_Type", "Use_Fallback"
]

missing_cols = [c for c in required_long_cols if c not in champion_long_map_df.columns]
if missing_cols:
    raise ValueError(f"❌ champion_long_map_df missing columns: {missing_cols}")

print("✅ Champion map column validation passed")

In [ ]:
print("\n========== PRE-RUN VALIDATION ==========\n")

Cleaned_Base_Data["ItemCode"] = Cleaned_Base_Data["ItemCode"].astype(str)
champion_long_map_df["ItemCode"] = champion_long_map_df["ItemCode"].astype(str)
champion_medium_map_df["ItemCode"] = champion_medium_map_df["ItemCode"].astype(str)

forecast_skus = sorted(set(map(str, PHARMA_SKUS)))
data_skus = set(Cleaned_Base_Data["ItemCode"].unique())

print("Data rows:", len(Cleaned_Base_Data))
print("Unique SKUs in data:", Cleaned_Base_Data["ItemCode"].nunique())
print("SKUs to forecast:", len(PHARMA_SKUS))
print("Unique forecast SKUs:", len(forecast_skus))

if len(Cleaned_Base_Data) == 0:
    raise ValueError("❌ Cleaned_Base_Data is EMPTY")

if len(PHARMA_SKUS) == 0:
    raise ValueError("❌ PHARMA_SKUS is EMPTY")

missing_from_data = sorted(list(set(forecast_skus) - data_skus))
extra_in_data = sorted(list(data_skus - set(forecast_skus)))

print("\n--- Dataset Coverage ---")
print("Missing from dataset:", len(missing_from_data))
if missing_from_data:
    print("Sample missing from dataset:", missing_from_data[:10])

print("Extra in dataset not in forecast list:", len(extra_in_data))

segment_summary = (
    Cleaned_Base_Data[
        Cleaned_Base_Data["ItemCode"].astype(str).isin(forecast_skus)
    ][["ItemCode", "History_Segment"]]
    .drop_duplicates()
    .groupby("History_Segment")
    .size()
    .reset_index(name="SKU_Count")
    .rename(columns={"History_Segment": "Segment"})
    .sort_values("Segment")
    .reset_index(drop=True)
)

print("\n--- Forecast SKU Segment Split ---")
print(segment_summary)

actual_long_skus = set(
    Cleaned_Base_Data.loc[Cleaned_Base_Data["History_Segment"] == "LONG", "ItemCode"].unique()
)
actual_medium_skus = set(
    Cleaned_Base_Data.loc[Cleaned_Base_Data["History_Segment"] == "MEDIUM", "ItemCode"].unique()
)
actual_short_skus = set(
    Cleaned_Base_Data.loc[Cleaned_Base_Data["History_Segment"] == "SHORT", "ItemCode"].unique()
)

# LONG
print("\n--- Champion Map (LONG) ---")
print("Rows:", len(champion_long_map_df))

if champion_long_map_df.empty:
    raise ValueError("❌ champion_long_map_df is EMPTY")

print("\nBest_Model distribution:")
print(champion_long_map_df["Best_Model"].value_counts(dropna=False))

print("\nFinal_Model distribution:")
print(champion_long_map_df["Final_Model"].value_counts(dropna=False))

long_map_skus = set(champion_long_map_df["ItemCode"].astype(str).unique())
long_missing = sorted(list(actual_long_skus - long_map_skus))
long_extra = sorted(list(long_map_skus - actual_long_skus))

print("\nActual LONG forecast SKUs:", len(actual_long_skus))
print("LONG SKUs present in champion map:", len(actual_long_skus & long_map_skus))
print("Actual LONG SKUs missing in champion map:", len(long_missing))
print("Extra LONG champion-map SKUs not in current forecast list:", len(long_extra))

if long_missing:
    print("Sample missing LONG SKUs:", long_missing[:10])


# MEDIUM

print("\n--- Champion Map (MEDIUM) ---")
print("Rows:", len(champion_medium_map_df))

if champion_medium_map_df.empty:
    raise ValueError("❌ champion_medium_map_df is EMPTY")

print("\nBest_Model distribution:")
print(champion_medium_map_df["Best_Model"].value_counts(dropna=False))

print("\nFinal_Model distribution:")
print(champion_medium_map_df["Final_Model"].value_counts(dropna=False))

medium_map_skus = set(champion_medium_map_df["ItemCode"].astype(str).unique())
medium_missing = sorted(list(actual_medium_skus - medium_map_skus))
medium_extra = sorted(list(medium_map_skus - actual_medium_skus))

print("\nActual MEDIUM forecast SKUs:", len(actual_medium_skus))
print("MEDIUM SKUs present in champion map:", len(actual_medium_skus & medium_map_skus))
print("Actual MEDIUM SKUs missing in champion map:", len(medium_missing))
print("Extra MEDIUM champion-map SKUs not in current forecast list:", len(medium_extra))

if medium_missing:
    print("Sample missing MEDIUM SKUs:", medium_missing[:10])


# SHORT
print("\n--- SHORT Segment Check ---")
print("SHORT forecast SKUs:", len(actual_short_skus))
print("SHORT uses rule-based routing, so no champion map is required.")

# LONG ARTIFACTS
print("\n--- Checking LONG tree artifacts ---")


def check_artifact(name, obj):
    if obj is None:
        print(f"❌ {name} NOT LOADED")
    else:
        print(f"✅ {name} OK")

check_artifact("XGB LONG", globals().get("xgb_long_deploy_artifacts"))
check_artifact("CATBOOST LONG", globals().get("catboost_long_deploy_artifacts"))
check_artifact("LGBM LONG", globals().get("lgbm_long_deploy_artifacts"))

# MEDIUM ARTIFACTS
print("\n--- Checking MEDIUM deploy artifacts ---")
for _, row in medium_deploy_artifact_registry.iterrows():
    run_key = row["Run_Key"]
    artifact_key = row["Deploy_Artifact_Key"]

    if run_key in medium_runs and artifact_key in medium_runs[run_key]:
        print(f"✅ {run_key} {artifact_key} OK")
    else:
        print(f"❌ {run_key} {artifact_key} MISSING")

# GRU
print("\n--- Checking GRU artifact ---")
try:
    _art, _scalers = load_long_gru_deploy_bundle_if_needed()
    print("✅ GRU LONG OK")
except Exception as e:
    print("❌ GRU LONG FAILED:", str(e))

# SAMPLE TEST
print("\n--- Running SAMPLE FORECAST TEST ---")

sample_skus = list(map(str, PHARMA_SKUS[:5]))
test_results = []

for sku in sample_skus:
    try:
        res = forecast_one_sku(sku, Cleaned_Base_Data)
        test_results.append(res)

        if res is None:
            print(f"SKU {sku} → FAILED | Segment: {choose_segment_by_history(Cleaned_Base_Data, sku)} | Reason: returned None")
        else:
            status = str(res.get("Status", ""))
            if status.upper().startswith("FAILED"):
                print(
                    f"SKU {sku} → FAILED | "
                    f"Segment: {res.get('Segment', 'N/A')} | "
                    f"Model: {res.get('Used_Model', 'N/A')} | "
                    f"Reason: {status}"
                )
            else:
                print(
                    f"SKU {sku} → OK | "
                    f"Segment: {res.get('Segment', 'N/A')} | "
                    f"Model: {res.get('Used_Model', 'N/A')}"
                )

    except Exception as e:
        print(f"SKU {sku} → ERROR: {str(e)}")
        test_results.append(None)

test_df = pd.DataFrame([r for r in test_results if r is not None])

print("\nSample Output:")
print(test_df.head())

failed_samples = [
    s for s, r in zip(sample_skus, test_results)
    if (r is None) or str(r.get("Status", "")).upper().startswith("FAILED")
]
if failed_samples:
    print("\n⚠️ Failed sample SKUs:", failed_samples)
else:
    print("\n✅ All sample SKUs passed")

sample_fail_count = len(failed_samples)
sample_total = len(sample_skus)

if sample_fail_count > 0:
    raise ValueError(
        f"❌ Sample validation failed for {sample_fail_count}/{sample_total} SKUs: {failed_samples}. "
        "Do NOT run full pipeline."
    )

print("\n========== VALIDATION COMPLETE — READY TO RUN ==========\n")

In [ ]:
print("\n--- GRU DEBUG SAMPLE TEST ---")
for dbg_sku in sorted(GRU_DEBUG_SKUS):
    if dbg_sku in set(Cleaned_Base_Data["ItemCode"].astype(str).unique()):
        print(f"\n######## DEBUG SKU {dbg_sku} ########")
        res = forecast_one_sku(dbg_sku, Cleaned_Base_Data)
        print("Result:", res)
    else:
        print(f"\n######## DEBUG SKU {dbg_sku} ########")
        print("SKU not in Cleaned_Base_Data")

In [ ]:
# ------------------------------------------------------------
# P) FINAL RUN
# ------------------------------------------------------------
sku_list = sorted(set(int(x) for x in PHARMA_SKUS))

segmented_predictions, segmented_failed_df = bulk_segmented_forecast(
    raw_data=Cleaned_Base_Data.copy(),
    sku_list=sku_list
)

calibration_df = build_recent_calibration_table(
    long_recent_df=long_model_b_recent_df,
    medium_recent_df=medium_recent4_eval_df
)

segmented_predictions["ItemCode"] = segmented_predictions["ItemCode"].astype(str)
calibration_df["ItemCode"] = calibration_df["ItemCode"].astype(str)

segmented_predictions["Used_Model_Clean"] = (
    segmented_predictions["Used_Model"]
    .astype(str)
    .str.replace("FALLBACK_", "", regex=False)
    .str.split("_")
    .str[0]
)

segmented_predictions = segmented_predictions.merge(
    calibration_df,
    left_on=["ItemCode", "Segment", "Used_Model_Clean"],
    right_on=["ItemCode", "Segment", "Model_Name"],
    how="left"
)

segmented_predictions["Calibration_Factor"] = segmented_predictions["Calibration_Factor"].fillna(1.0)
segmented_predictions["Forecast_Prediction_Raw"] = segmented_predictions["Forecast_Prediction"]

segmented_predictions["Forecast_Prediction"] = (
    segmented_predictions["Forecast_Prediction"] *
    segmented_predictions["Calibration_Factor"]
).clip(lower=0)

segmented_predictions = post_prediction_pattern_validator(
    pred_df=segmented_predictions,
    history_df=Cleaned_Base_Data
)

# ---------------- LONG inference routing debug ----------------
print("\n--- LONG INFERENCE ROUTING SUMMARY ---")
long_preds = segmented_predictions[segmented_predictions["Segment"] == "LONG"].copy()

print("LONG total predictions:", len(long_preds))

print("\nLONG Champion_Model distribution:")
print(long_preds["Champion_Model"].value_counts(dropna=False))

print("\nLONG Used_Model distribution:")
print(long_preds["Used_Model"].value_counts(dropna=False))

print("\nLONG Fallback_Used distribution:")
print(long_preds["Fallback_Used"].value_counts(dropna=False))

print("\nLONG fallback rows sample:")
fb_sample = long_preds[long_preds["Fallback_Used"] == 1].copy()
if fb_sample.empty:
    print("No LONG fallback rows")
else:
    print(fb_sample[[
        "ItemCode", "Champion_Model", "Used_Model", "Status"
    ]].head(30))
# -------------------------------------------------------------


if segmented_predictions.empty:
    raise ValueError("No predictions were generated.")

total_requested = len(sku_list)
total_failed = len(segmented_failed_df)
fail_rate = total_failed / total_requested if total_requested > 0 else 0

print(f"\nTotal requested SKUs: {total_requested}")
print(f"Successful forecasts: {len(segmented_predictions)}")
print(f"Failed forecasts: {total_failed}")
print(f"Failure rate: {fail_rate:.2%}")

if total_failed > 0:
    print("\nFailure reason summary:")
    print(segmented_failed_df["Status"].value_counts(dropna=False).head(20))

if fail_rate > 0.02:
    raise ValueError(
        f"❌ Too many failed forecasts: {total_failed}/{total_requested} ({fail_rate:.2%}). "
        "Fix inference issues before saving final file."
    )

if segmented_predictions.empty:
    raise ValueError("No predictions were generated.")

latest_forecast_year = segmented_predictions["Forecast_Year"].max()
latest_forecast_month = segmented_predictions.loc[
    segmented_predictions["Forecast_Year"] == latest_forecast_year,
    "Forecast_Month"
].max()

target_predictions = segmented_predictions[
    (segmented_predictions["Forecast_Year"] == latest_forecast_year) &
    (segmented_predictions["Forecast_Month"] == latest_forecast_month)
].copy()

preferred_cols = [
    "ItemCode",
    "Forecast_Year",
    "Forecast_Month",
    "Forecast_Prediction",
    "Forecast_Prediction_Before_Calibration",
    "Forecast_Prediction_Before_Pattern_Validation",
    "Calibration_Factor",
    "Pattern_Validation_Status",
    "Segment",
    "Behavior_Type",
    "Subgroup",
    "Champion_Model",
    "Used_Model",
    "Fallback_Used",
    "Expected_Bonus",
    "Residual_Baseline",
    "Predicted_Residual",
    "Status"
]
target_predictions = target_predictions[[c for c in preferred_cols if c in target_predictions.columns]]

month_name_map = {
    1: "jan", 2: "feb", 3: "mar", 4: "apr", 5: "may", 6: "jun",
    7: "jul", 8: "aug", 9: "sep", 10: "oct", 11: "nov", 12: "dec"
}

file_month_name = month_name_map[int(latest_forecast_month)]
output_file = f"pharma_forecast_{file_month_name}_{int(latest_forecast_year)}.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    target_predictions.to_excel(writer, sheet_name="Forecasts", index=False)
    segmented_predictions.to_excel(writer, sheet_name="All_Predictions", index=False)

    if not segmented_failed_df.empty:
        segmented_failed_df.to_excel(writer, sheet_name="Failed_SKUs", index=False)

print(f"Saved file: {output_file}")
print("\nForecast preview:")
print(target_predictions.head())

print("\nForecast row count:", len(target_predictions))
print("Unique forecast SKUs:", target_predictions["ItemCode"].nunique())

if not segmented_failed_df.empty:
    print("\nFailed SKUs preview:")
    print(segmented_failed_df.head(10))
    print("\nFailed SKU count:", len(segmented_failed_df))
else:
    print("\nNo failed SKUs")

In [ ]:
# ============================================================
# POST-RUN AUDIT BLOCK
# ============================================================
print("\n========== POST-RUN AUDIT ==========\n")

audit_df = segmented_predictions.copy()
audit_df["ItemCode"] = audit_df["ItemCode"].astype(str)

# 1) basic counts
print("Total forecast rows:", len(audit_df))
print("Unique forecast SKUs:", audit_df["ItemCode"].nunique())

print("\nForecast rows by segment:")
print(audit_df["Segment"].value_counts(dropna=False))

print("\nForecast rows by used model:")
print(audit_df["Used_Model"].value_counts(dropna=False).head(20))

print("\nFallback usage by segment:")
print(pd.crosstab(audit_df["Segment"], audit_df["Fallback_Used"], dropna=False))

# 2) data quality checks
null_forecast = audit_df["Forecast_Prediction"].isna().sum()
neg_forecast = (pd.to_numeric(audit_df["Forecast_Prediction"], errors="coerce") < 0).sum()
zero_forecast = (pd.to_numeric(audit_df["Forecast_Prediction"], errors="coerce").fillna(0) == 0).sum()

print("\nNull forecast count:", null_forecast)
print("Negative forecast count:", neg_forecast)
print("Zero forecast count:", zero_forecast)

# 3) failed sku summary
if not segmented_failed_df.empty:
    print("\nFailed SKU count:", len(segmented_failed_df))
    print("Failed SKU unique count:", segmented_failed_df["ItemCode"].astype(str).nunique())
    print("\nTop failure reasons:")
    print(segmented_failed_df["Status"].value_counts(dropna=False).head(20))
else:
    print("\nNo failed SKUs")

# 4) missing requested SKUs check
requested_skus = set(map(str, PHARMA_SKUS))
predicted_skus = set(audit_df["ItemCode"].astype(str).unique())
failed_skus = set(segmented_failed_df["ItemCode"].astype(str).unique()) if not segmented_failed_df.empty else set()

uncovered_skus = sorted(requested_skus - predicted_skus - failed_skus)

print("\nRequested SKU count:", len(requested_skus))
print("Predicted SKU count:", len(predicted_skus))
print("Failed SKU count:", len(failed_skus))
print("Uncovered SKU count:", len(uncovered_skus))
if uncovered_skus:
    print("Sample uncovered SKUs:", uncovered_skus[:20])

# 5) sanity distribution by segment
print("\nForecast summary by segment:")
print(
    audit_df.groupby("Segment")["Forecast_Prediction"]
    .agg(["count", "min", "median", "mean", "max"])
)

# 6) extreme forecasts
print("\nTop 20 highest forecasts:")
print(
    audit_df.sort_values("Forecast_Prediction", ascending=False)[
        ["ItemCode", "Segment", "Subgroup", "Champion_Model", "Used_Model", "Forecast_Prediction"]
    ].head(20)
)

print("\nTop 20 lowest non-zero forecasts:")
print(
    audit_df[audit_df["Forecast_Prediction"] > 0]
    .sort_values("Forecast_Prediction", ascending=True)[
        ["ItemCode", "Segment", "Subgroup", "Champion_Model", "Used_Model", "Forecast_Prediction"]
    ].head(20)
)

# 7) optional guardrail checks by anchor if available
guard_cols = ["Forecast_Prediction", "Residual_Baseline"]
if all(c in audit_df.columns for c in guard_cols):
    audit_df["Forecast_vs_Baseline_Ratio"] = np.where(
        pd.to_numeric(audit_df["Residual_Baseline"], errors="coerce") > 0,
        pd.to_numeric(audit_df["Forecast_Prediction"], errors="coerce") /
        pd.to_numeric(audit_df["Residual_Baseline"], errors="coerce"),
        np.nan
    )

    suspicious_low = audit_df[audit_df["Forecast_vs_Baseline_Ratio"] < 0.25]
    suspicious_high = audit_df[audit_df["Forecast_vs_Baseline_Ratio"] > 3.50]

    print("\nSuspiciously LOW vs baseline:", len(suspicious_low))
    if not suspicious_low.empty:
        print(
            suspicious_low[
                ["ItemCode", "Segment", "Used_Model", "Forecast_Prediction", "Residual_Baseline", "Forecast_vs_Baseline_Ratio"]
            ].head(20)
        )

    print("\nSuspiciously HIGH vs baseline:", len(suspicious_high))
    if not suspicious_high.empty:
        print(
            suspicious_high[
                ["ItemCode", "Segment", "Used_Model", "Forecast_Prediction", "Residual_Baseline", "Forecast_vs_Baseline_Ratio"]
            ].head(20)
        )

# 8) save audit sheet if you want
audit_summary_rows = []

for seg, g in audit_df.groupby("Segment"):
    audit_summary_rows.append({
        "Segment": seg,
        "Rows": len(g),
        "Unique_SKUs": g["ItemCode"].nunique(),
        "Fallback_Count": int((g["Fallback_Used"] == 1).sum()) if "Fallback_Used" in g.columns else 0,
        "Zero_Forecast_Count": int((pd.to_numeric(g["Forecast_Prediction"], errors="coerce").fillna(0) == 0).sum()),
        "Mean_Forecast": float(pd.to_numeric(g["Forecast_Prediction"], errors="coerce").mean()),
        "Median_Forecast": float(pd.to_numeric(g["Forecast_Prediction"], errors="coerce").median()),
        "Max_Forecast": float(pd.to_numeric(g["Forecast_Prediction"], errors="coerce").max())
    })

post_run_audit_summary_df = pd.DataFrame(audit_summary_rows)

print("\nPost-run audit summary:")
print(post_run_audit_summary_df)

print("\n========== POST-RUN AUDIT COMPLETE ==========\n")

# Debug

In [ ]:
low_skus = [
600310,600478,600603,600698,600828,602647,602868,602882,604003,604035,
604058,604096,604097,604145,605164,605187,605349,606354,606597,606632,
606666,601112,601115,601131,601142,601865,601869,601940,602413,603203,
603277,603318,603640,604624,604968,605657,605685,605929,606043,606066,
606068,606154,606756,606838,606897,606900,607000,607002,607017,607031,
607034,606950,607046,607050,607053,607055,607070,607095,607119,607277,
607285,607303,607304,607317,607307,607595,607606,607612,607810,607875,
607901,607905,607906,607911,607951,607959,608056,608059,608114,608131,
608134,608137,608162,608528,608534,610020,610029,610105,610702,610936,
611050,611202,611240,611241,611403,611406,611434,611565,611592,611605,
611606,611608,611611,611612,611631,611588,611689,612095,612101,612115,
612047,612049,612050,612053,612056,612057,612058,612061,612063,612065,
612067,612070,612071,612072,612073,612033,612045,612308
]

In [ ]:
accuracy_df = pd.read_excel(
    "/Users/dhanujiamanda/Documents/Projects/Agentic AI /Doc/ProjectFlow.xlsx",
    sheet_name=0,
    header=1,
    engine="openpyxl"
)

accuracy_df.columns = accuracy_df.columns.astype(str).str.strip()

print(accuracy_df.columns.tolist())
print(accuracy_df.head())

In [ ]:
diag = accuracy_df.copy()
diag["ItemCode"] = diag["ProductCode"].astype(str).str.replace(".0", "", regex=False)

feature_base = Cleaned_Base_Data.copy()
feature_base["ItemCode"] = feature_base["ItemCode"].astype(str).str.replace(".0", "", regex=False)

latest_features = (
    feature_base
    .sort_values(["ItemCode", "Year", "Month_Number"])
    .groupby("ItemCode")
    .tail(1)
)

diagnostic_cols = [
    "ItemCode",
    "Clean_Demand",
    "Lag1", "Lag2", "Lag3",
    "Rolling3M_Mean", "Rolling6M_Mean",

    "Bonus_Flag",
    "Bonus_Flag_Lag1",
    "Bonus_Frequency_All",
    "Bonus_Frequency_12M",
    "Expected_Bonus_NextMonth",
    "Recurring_Bonus_SKU",
    "Bonus_Cycle_Length",
    "Months_Since_Last_Bonus",
    "Avg_Bonus_Uplift",
    "Last_Bonus_Demand",
    "Promo_Uplift_6M",

    "Supply_Constraint_Flag",
    "Supply_Constraint_Lag1",
    "Supply_Constraint_Lag2",
    "Supply_Shock",
    "Available_Primary_Inventory_Qty",
    "Distributor_Inventory_Qty",
    "Stock_Cover_Months",

    "Behavior_Type",
    "Behavior_CV_6M",
    "Behavior_Peak_Ratio_6M",
    "Behavior_PromoRate_12M",
    "Recent_Spike_Flag",
    "Post_Spike_Drop_Risk"
]

latest_features = latest_features[[c for c in diagnostic_cols if c in latest_features.columns]]

diag = diag.merge(latest_features, on="ItemCode", how="left", suffixes=("", "_feature"))

if "Behavior_Type_feature" in diag.columns:
    diag["Behavior_Type"] = diag["Behavior_Type"].fillna(diag["Behavior_Type_feature"])
    diag = diag.drop(columns=["Behavior_Type_feature"], errors="ignore")

fill_defaults = {
    "Rolling3M_Mean": 0,
    "Rolling6M_Mean": 0,
    "Last_Bonus_Demand": 0,
    "Bonus_Frequency_All": 0,
    "Bonus_Frequency_12M": 0,
    "Expected_Bonus_NextMonth": 0,
    "Avg_Bonus_Uplift": 1,
    "Supply_Constraint_Lag1": 0,
    "Supply_Constraint_Lag2": 0,
    "Recent_Spike_Flag": 0,
    "Post_Spike_Drop_Risk": 0,
    "Behavior_Type": "UNKNOWN"
}

for col, val in fill_defaults.items():
    if col not in diag.columns:
        diag[col] = val
    else:
        diag[col] = diag[col].fillna(val)

diag["Forecast"] = pd.to_numeric(diag["Forecast"], errors="coerce")
diag["ActualSale"] = pd.to_numeric(diag["ActualSale"], errors="coerce")
diag["ABS Error"] = pd.to_numeric(diag["ABS Error"], errors="coerce")

diag["Forecast_Error"] = diag["ActualSale"] - diag["Forecast"]
diag["Error_Direction"] = np.where(
    diag["Forecast"] < diag["ActualSale"],
    "UNDER_FORECAST",
    "OVER_FORECAST"
)

diag["Actual_vs_Rolling3"] = diag["ActualSale"] / (diag["Rolling3M_Mean"] + 1)
diag["Forecast_vs_Rolling3"] = diag["Forecast"] / (diag["Rolling3M_Mean"] + 1)
diag["Actual_vs_LastBonus"] = diag["ActualSale"] / (diag["Last_Bonus_Demand"] + 1)
diag["Forecast_vs_LastBonus"] = diag["Forecast"] / (diag["Last_Bonus_Demand"] + 1)

print(diag[[
    "ItemCode", "Behavior_Type", "Model Type", "Forecast", "ActualSale",
    "Error_Direction", "Expected_Bonus_NextMonth",
    "Bonus_Frequency_All", "Bonus_Frequency_12M",
    "Avg_Bonus_Uplift", "Last_Bonus_Demand",
    "Rolling3M_Mean", "Actual_vs_Rolling3"
]].head(20))

In [ ]:
def assign_error_cause(row):
    direction = row.get("Error_Direction", "")
    behavior = str(row.get("Behavior_Type", "UNKNOWN"))

    expected_bonus = int(row.get("Expected_Bonus_NextMonth", 0) or 0)
    bonus_freq_all = float(row.get("Bonus_Frequency_All", 0) or 0)
    bonus_freq_12m = float(row.get("Bonus_Frequency_12M", 0) or 0)
    avg_uplift = float(row.get("Avg_Bonus_Uplift", 1) or 1)

    rolling3 = float(row.get("Rolling3M_Mean", 0) or 0)
    last_bonus = float(row.get("Last_Bonus_Demand", 0) or 0)
    actual = float(row.get("ActualSale", 0) or 0)
    forecast = float(row.get("Forecast", 0) or 0)

    actual_vs_roll3 = actual / (rolling3 + 1)
    forecast_vs_roll3 = forecast / (rolling3 + 1)

    supply_lag1 = int(row.get("Supply_Constraint_Lag1", 0) or 0)
    supply_lag2 = int(row.get("Supply_Constraint_Lag2", 0) or 0)

    recent_spike = int(row.get("Recent_Spike_Flag", 0) or 0)
    post_spike = int(row.get("Post_Spike_Drop_Risk", 0) or 0)

    promo_like = (
        behavior == "PROMO_DRIVEN"
        or bonus_freq_all >= 0.25
        or bonus_freq_12m >= 0.25
        or avg_uplift >= 1.4
    )

    if direction == "UNDER_FORECAST":
        if expected_bonus == 1 and forecast < 0.5 * max(last_bonus, rolling3, 1):
            return "MISSED_EXPECTED_PROMO_UPLIFT"

        if promo_like and actual_vs_roll3 > 1.5:
            return "PROMO_UPLIFT_UNDER_LEARNED"

        if supply_lag1 == 1 or supply_lag2 == 1:
            return "SUPPLY_RECOVERY_REBOUND_MISSED"

        if actual_vs_roll3 > 2.0:
            return "UNEXPECTED_DEMAND_SPIKE"

        if behavior == "RECENT_DROP":
            return "RECENT_DROP_REBOUND_MISSED"

        return "GENERAL_UNDER_FORECAST"

    if direction == "OVER_FORECAST":
        if recent_spike == 1 or post_spike == 1:
            return "OLD_SPIKE_CARRIED_FORWARD"

        if promo_like and expected_bonus == 0 and forecast_vs_roll3 > 2.0:
            return "PROMO_EXPECTED_BUT_DID_NOT_HAPPEN"

        if actual < 0.4 * max(rolling3, 1):
            return "RECENT_DEMAND_COLLAPSE_NOT_CAPTURED"

        return "GENERAL_OVER_FORECAST"

    return "UNKNOWN"

In [ ]:
diag["Likely_Error_Cause"] = diag.apply(assign_error_cause, axis=1)

summary = (
    diag.groupby(["Behavior_Type", "Model Type", "Error_Direction", "Likely_Error_Cause"])
    .agg(
        SKU_Count=("ItemCode", "count"),
        Actual_Sum=("ActualSale", "sum"),
        Forecast_Sum=("Forecast", "sum"),
        Abs_Error_Sum=("ABS Error", "sum")
    )
    .reset_index()
)

summary["WMAPE"] = np.where(
    summary["Actual_Sum"] > 0,
    summary["Abs_Error_Sum"] / summary["Actual_Sum"] * 100,
    np.nan
)

summary = summary.sort_values("Abs_Error_Sum", ascending=False)

print(summary.head(30))

In [ ]:
def build_sku_behavior_audit(df):
    work = df.copy().sort_values(["ItemCode", "Year", "Month_Number"])
    work["ItemCode"] = work["ItemCode"].astype(str)

    rows = []

    for sku, g in work.groupby("ItemCode"):
        y = g["Clean_Demand"].fillna(0).astype(float)

        mean_demand = y.mean()
        std_demand = y.std()
        cv = std_demand / (mean_demand + 1)
        zero_rate = (y == 0).mean()

        peak_ratio = y.max() / (mean_demand + 1)

        # repeating pattern signal
        autocorr_3 = y.autocorr(lag=3) if len(y) > 6 else 0
        autocorr_6 = y.autocorr(lag=6) if len(y) > 12 else 0
        autocorr_12 = y.autocorr(lag=12) if len(y) > 18 else 0

        # simple trend strength
        if len(y) >= 6:
            recent = y.tail(6).values
            trend = np.polyfit(range(len(recent)), recent, 1)[0]
        else:
            trend = 0

        if zero_rate >= 0.40:
            behavior = "SPORADIC"
        elif max(autocorr_3, autocorr_6, autocorr_12) >= 0.45 and peak_ratio >= 2:
            behavior = "CYCLIC_SPIKE"
        elif abs(trend) > mean_demand * 0.15:
            behavior = "TRENDING"
        elif cv <= 0.6:
            behavior = "STABLE"
        else:
            behavior = "VOLATILE"

        rows.append({
            "ItemCode": sku,
            "Mean_Demand": mean_demand,
            "CV": cv,
            "Zero_Rate": zero_rate,
            "Peak_Ratio": peak_ratio,
            "Autocorr_3": autocorr_3,
            "Autocorr_6": autocorr_6,
            "Autocorr_12": autocorr_12,
            "Trend_Strength": trend,
            "Behavior_Type": behavior
        })

    return pd.DataFrame(rows)

In [ ]:
sku_behavior_audit_df = build_sku_behavior_audit(Cleaned_Base_Data)

sku_behavior_audit_df["ItemCode"] = sku_behavior_audit_df["ItemCode"].astype(str)
target_predictions["ItemCode"] = target_predictions["ItemCode"].astype(str)

forecast_with_behavior = target_predictions.merge(
    sku_behavior_audit_df,
    on="ItemCode",
    how="left"
)

forecast_with_behavior.to_excel("forecast_with_behavior_audit.xlsx", index=False)

print(forecast_with_behavior["Behavior_Type"].value_counts())

In [ ]:
segment_behavior_dist = (
    forecast_with_behavior
    .groupby(["Segment", "Behavior_Type"])
    .size()
    .reset_index(name="SKU_Count")
    .sort_values(["Segment", "SKU_Count"], ascending=[True, False])
)

print(segment_behavior_dist)

In [ ]:
pivot = segment_behavior_dist.pivot(
    index="Segment",
    columns="Behavior_Type",
    values="SKU_Count"
).fillna(0)

print(pivot)

In [ ]:
import pandas as pd

# Your main dataset
df = forecast_with_behavior.copy()

# Optional: add accuracy if available
# df = df.merge(accuracy_df, on="ItemCode", how="left")

# Create Excel writer
output_path = "behavior_segment_sku_breakdown.xlsx"

with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:

    # 1️⃣ Summary
    summary = (
        df.groupby(["Segment", "Behavior_Type"])
        .size()
        .reset_index(name="SKU_Count")
    )
    summary.to_excel(writer, sheet_name="Summary", index=False)

    # 2️⃣ Pivot (nice view)
    pivot = summary.pivot(
        index="Segment",
        columns="Behavior_Type",
        values="SKU_Count"
    ).fillna(0)

    pivot.to_excel(writer, sheet_name="Pivot")

    # 3️⃣ EXPORT EACH GROUP (IMPORTANT)
    for segment in df["Segment"].unique():
        for behavior in df["Behavior_Type"].unique():

            subset = df[
                (df["Segment"] == segment) &
                (df["Behavior_Type"] == behavior)
            ]

            if len(subset) == 0:
                continue

            sheet_name = f"{segment}_{behavior}"[:31]  # Excel limit

            subset.sort_values(
                by=["Forecast_Prediction"],
                ascending=False
            ).to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Saved: {output_path}")

In [ ]:
df = Data.copy()

# Make sure types match
df["ItemCode"] = df["ItemCode"].astype(str)
low_skus = [str(x) for x in low_skus]

# Filter only your abnormal LONG SKUs
df_low = df[
    (df["History_Segment"] == "LONG") &
    (df["ItemCode"].isin(low_skus))
].copy()

df_low = df_low.sort_values(["ItemCode", "Year", "Month_Number"])

print("Rows:", len(df_low))
print("SKUs found:", df_low["ItemCode"].nunique())
print("Missing SKUs:", sorted(set(low_skus) - set(df_low["ItemCode"].unique()))[:30])

In [ ]:
import numpy as np
from scipy.stats import skew

value_col = "Clean_Demand"

def safe_autocorr(x, lag):
    x = x.dropna()
    if len(x) <= lag or x.std() == 0:
        return 0
    return x.autocorr(lag=lag)

sku_diag = df_low.groupby("ItemCode")[value_col].agg(
    Count="count",
    Mean="mean",
    Median="median",
    Std="std",
    Min="min",
    Max="max"
).reset_index()

sku_diag["CV"] = sku_diag["Std"] / (sku_diag["Mean"] + 1)
sku_diag["Peak_Ratio"] = sku_diag["Max"] / (sku_diag["Mean"] + 1)

extra = df_low.groupby("ItemCode").agg(
    Zero_Rate=(value_col, lambda x: (x == 0).mean()),
    Skewness=(value_col, lambda x: skew(x.dropna()) if len(x.dropna()) > 2 else 0),
    Autocorr_Lag3=(value_col, lambda x: safe_autocorr(x, 3)),
    Autocorr_Lag6=(value_col, lambda x: safe_autocorr(x, 6)),
    Promo_Rate=("Bonus_Flag", "mean"),
    Supply_Rate=("Supply_Constraint_Flag", "mean"),
    Avg_Uplift=("Uplift_vs_Baseline", "mean"),
    Bonus_Shock_Rate=("Bonus_Shock", "mean"),
    Supply_Shock_Rate=("Supply_Shock", "mean")
).reset_index()

sku_diag = sku_diag.merge(extra, on="ItemCode", how="left").fillna(0)

def classify_behavior(row):
    if row["Zero_Rate"] > 0.40:
        return "INTERMITTENT"
    elif row["Autocorr_Lag3"] > 0.50 or row["Autocorr_Lag6"] > 0.50:
        return "CYCLIC"
    elif row["Peak_Ratio"] > 2.50:
        return "SPIKY"
    elif row["CV"] < 0.30:
        return "STABLE"
    else:
        return "MIXED"

sku_diag["Behavior"] = sku_diag.apply(classify_behavior, axis=1)

print(sku_diag["Behavior"].value_counts())
display(sku_diag.sort_values(["Behavior", "CV"], ascending=[True, False]))

In [ ]:
# ============================================================
# CLEAN FINAL FAILURE TRACE DEBUG BLOCK
# Paste AFTER your inference functions are defined
# ============================================================

print("\n================ FAILURE TRACE DEBUG PATCH ================\n")

# ------------------------------------------------------------
# 1) STANDARD FAILURE RESULT
# ------------------------------------------------------------
def make_fail_result(
    sku_code,
    segment="",
    subgroup="",
    champion_model="UNKNOWN",
    used_model="UNKNOWN",
    status="FAILED"
):
    return {
        "ItemCode": int(float(sku_code)),
        "Forecast_Year": np.nan,
        "Forecast_Month": np.nan,
        "Forecast_Prediction": np.nan,
        "Segment": segment,
        "Subgroup": subgroup,
        "Champion_Model": champion_model,
        "Used_Model": used_model,
        "Fallback_Used": 0,
        "Expected_Bonus": np.nan,
        "Residual_Baseline": np.nan,
        "Predicted_Residual": np.nan,
        "Status": status
    }


# ------------------------------------------------------------
# 2) PATCH LONG TREE FORECAST WITH STAGE-LEVEL DEBUG
# ------------------------------------------------------------
def forecast_long_tree_sku_debug(sku_code, raw_data, artifacts, used_model_name):
    sku_code = str(sku_code)
    used_model_name = str(used_model_name).upper()

    try:
        model = artifacts["model"]
        feature_cols = artifacts["feature_cols"]
    except Exception as e:
        return make_fail_result(
            sku_code, "LONG", "", used_model_name, used_model_name,
            f"LONG_TREE_ARTIFACT_ERROR: {str(e)}"
        )

    try:
        prepared_df = prepare_long_tree_inference_frame(raw_data, artifacts)
        prepared_df = force_itemcode_str(prepared_df)
    except Exception as e:
        return make_fail_result(
            sku_code, "LONG", "", used_model_name, used_model_name,
            f"LONG_TREE_PREP_ERROR: {str(e)}"
        )

    if prepared_df.empty:
        return make_fail_result(
            sku_code, "LONG", "", used_model_name, used_model_name,
            "LONG_TREE_PREPARED_DF_EMPTY"
        )

    sku_hist = prepared_df[prepared_df["ItemCode_Original"] == sku_code].copy().sort_values(["Year", "Month_Number"])
    if sku_hist.empty:
        return make_fail_result(
            sku_code, "LONG", "", used_model_name, used_model_name,
            "LONG_TREE_SKU_HIST_EMPTY"
        )

    last_row, next_year, next_month = get_next_period_from_history(prepared_df, sku_code)
    if last_row is None:
        return make_fail_result(
            sku_code, "LONG", "", used_model_name, used_model_name,
            "LONG_TREE_NEXT_PERIOD_MISSING"
        )

    try:
        next_bonus = infer_expected_bonus_flag(sku_code, prepared_df)
    except Exception as e:
        return make_fail_result(
            sku_code, "LONG", "", used_model_name, used_model_name,
            f"LONG_TREE_BONUS_FLAG_ERROR: {str(e)}"
        )

    try:
        new_row = last_row.copy()
        new_row = force_itemcode_str(new_row)
        new_row["Year"] = next_year
        new_row["Month_Number"] = next_month
        new_row["Bonus_Flag"] = int(next_bonus)

        for c in ["Secondary_Sales_Qty", "Primary_Sales_Qty", "Free_Qty", "Observed_Demand", "Effective_Demand", "Clean_Demand"]:
            if c in new_row.columns:
                new_row[c] = 0

        work_df = prepared_df.drop(columns=["ItemCode_Encoded"], errors="ignore").copy()
        work_df = force_itemcode_str(work_df)
        work_df = pd.concat([work_df, new_row], ignore_index=True)
        work_df = force_itemcode_str(work_df)
        work_df = work_df.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)
    except Exception as e:
        return make_fail_result(
            sku_code, "LONG", "", used_model_name, used_model_name,
            f"LONG_TREE_FUTURE_ROW_BUILD_ERROR: {str(e)}"
        )

    try:
        bonus_pattern_df_f = detect_recurring_bonus_skus(work_df)[[
            "ItemCode",
            "Recurring_Bonus_SKU",
            "Bonus_Cycle_Length",
            "Avg_Bonus_Gap",
            "Bonus_Frequency_All",
            "Avg_Bonus_Uplift"
        ]].copy()
        bonus_pattern_df_f = force_itemcode_str(bonus_pattern_df_f)

        work_df = work_df.drop(columns=[
            "Recurring_Bonus_SKU",
            "Bonus_Cycle_Length",
            "Avg_Bonus_Gap",
            "Bonus_Frequency_All",
            "Avg_Bonus_Uplift"
        ], errors="ignore")
        work_df = force_itemcode_str(work_df)
        work_df = work_df.merge(bonus_pattern_df_f, on="ItemCode", how="left")

        for c in ["Recurring_Bonus_SKU", "Bonus_Cycle_Length", "Avg_Bonus_Gap", "Bonus_Frequency_All"]:
            work_df[c] = work_df[c].fillna(0)
        work_df["Avg_Bonus_Uplift"] = work_df["Avg_Bonus_Uplift"].fillna(1.0)

        abc_map = artifacts.get("abc_map", {})
        work_df["ABC_Class"] = work_df["ItemCode"].map(abc_map).fillna(2)

        promo_profile_df = artifacts.get("promo_profile_df", pd.DataFrame())
        if isinstance(promo_profile_df, pd.DataFrame) and not promo_profile_df.empty:
            promo_profile_df = force_itemcode_str(promo_profile_df)
            work_df = force_itemcode_str(work_df)
            work_df = merge_promo_profile(work_df, promo_profile_df)
            work_df = force_itemcode_str(work_df)

        sku_profile_df = artifacts.get("sku_profile_df", pd.DataFrame())
        if isinstance(sku_profile_df, pd.DataFrame) and not sku_profile_df.empty:
            sku_profile_df = force_itemcode_str(sku_profile_df)
            keep_cols = [c for c in ["ItemCode", "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"] if c in sku_profile_df.columns]

            work_df = work_df.drop(columns=["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"], errors="ignore")
            work_df = force_itemcode_str(work_df)
            work_df = work_df.merge(sku_profile_df[keep_cols], on="ItemCode", how="left")

            for c in ["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]:
                if c in work_df.columns:
                    work_df[c] = work_df[c].fillna(0)

        work_df = add_bonus_cycle_features(work_df)
        work_df = force_itemcode_str(work_df)
        work_df = rebuild_time_features(work_df)
        work_df = force_itemcode_str(work_df)

        if "clip_caps" in artifacts:
            work_df = apply_clip_caps(work_df, artifacts["clip_caps"])
            work_df = force_itemcode_str(work_df)

        work_df = recompute_target(work_df)
        work_df = force_itemcode_str(work_df)
        work_df = add_residual_target(work_df)
        work_df = force_itemcode_str(work_df)
    except Exception as e:
        return make_fail_result(
            sku_code, "LONG", "", used_model_name, used_model_name,
            f"LONG_TREE_FEATURE_BUILD_ERROR: {str(e)}"
        )

    try:
        itemcode_categories = artifacts.get("itemcode_categories", pd.Index([]))
        cat_to_code = {str(k): i for i, k in enumerate(itemcode_categories)}
        unk_code = len(cat_to_code)

        work_df["ItemCode_Original"] = work_df["ItemCode"].astype(str)
        work_df["ItemCode_Encoded"] = work_df["ItemCode"].astype(str).map(cat_to_code).fillna(unk_code).astype(int)

        work_df_model = work_df.copy()
        work_df_model["ItemCode"] = work_df_model["ItemCode_Encoded"]

        next_row = work_df_model[
            (work_df_model["ItemCode_Original"] == sku_code) &
            (work_df_model["Year"] == next_year) &
            (work_df_model["Month_Number"] == next_month)
        ].copy()

        if next_row.empty:
            return make_fail_result(
                sku_code, "LONG", "", used_model_name, used_model_name,
                "LONG_TREE_NEXT_ROW_EMPTY"
            )

        next_row["Segment_For_Calibration"] = "LONG"

        assert_features_exist(next_row, feature_cols, where="LONG_TREE_INFERENCE_NEXT_ROW")

        X_next = sanitize_model_input(next_row[feature_cols])
        pred_residual = float(np.array(model.predict(X_next)).reshape(-1)[0])

        baseline = float(next_row[BASELINE_COL].iloc[0])
        raw_forecast = max(baseline + pred_residual, 0.0)

        row_dict = next_row.iloc[0].to_dict()
        forecast = apply_promo_aware_adjustment(row_dict, raw_forecast)
        forecast = apply_final_forecast_guardrails(row_dict, forecast)
        forecast = apply_production_calibration(row_dict, forecast)
    except Exception as e:
        return make_fail_result(
            sku_code, "LONG", "", used_model_name, used_model_name,
            f"LONG_TREE_PREDICT_ERROR: {str(e)}"
        )

    return {
        "ItemCode": int(float(sku_code)),
        "Forecast_Year": int(next_year),
        "Forecast_Month": int(next_month),
        "Forecast_Prediction": float(forecast),
        "Segment": "LONG",
        "Subgroup": "",
        "Champion_Model": used_model_name,
        "Used_Model": used_model_name,
        "Fallback_Used": 0,
        "Expected_Bonus": int(next_bonus),
        "Residual_Baseline": float(baseline),
        "Predicted_Residual": float(pred_residual),
        "Status": "Success"
    }


# ------------------------------------------------------------
# 3) PATCH LONG GRU FORECAST WITH STAGE-LEVEL DEBUG
# ------------------------------------------------------------
def forecast_long_gru_sku_debug(sku_code, raw_data):
    sku_code = str(sku_code)

    try:
        loaded_artifacts, loaded_scalers = load_long_gru_deploy_bundle_if_needed()
        model = loaded_artifacts["model"]
        item_to_idx = loaded_artifacts["item_to_idx"]
        seq_len = loaded_artifacts["seq_len"]
    except Exception as e:
        return make_fail_result(sku_code, "LONG", "", "GRU", "GRU", f"LONG_GRU_LOAD_ERROR: {str(e)}")

    try:
        prepared_df = prepare_long_gru_inference_frame(raw_data, loaded_artifacts)
        prepared_df = force_itemcode_str(prepared_df)
    except Exception as e:
        return make_fail_result(sku_code, "LONG", "", "GRU", "GRU", f"LONG_GRU_PREP_ERROR: {str(e)}")

    if prepared_df.empty:
        return make_fail_result(sku_code, "LONG", "", "GRU", "GRU", "LONG_GRU_PREPARED_DF_EMPTY")

    sku_hist = prepared_df[prepared_df["ItemCode_Original"] == sku_code].copy()
    sku_hist = sku_hist.sort_values(["Year", "Month_Number"]).reset_index(drop=True)

    if sku_hist.empty:
        return make_fail_result(sku_code, "LONG", "", "GRU", "GRU", "LONG_GRU_SKU_HIST_EMPTY")

    if sku_code not in item_to_idx:
        return make_fail_result(sku_code, "LONG", "", "GRU", "GRU", "LONG_GRU_ITEM_NOT_IN_MAPPING")

    last_row, next_year, next_month = get_next_period_from_history(prepared_df, sku_code)
    if last_row is None:
        return make_fail_result(sku_code, "LONG", "", "GRU", "GRU", "LONG_GRU_NEXT_PERIOD_MISSING")

    try:
        next_bonus = infer_expected_bonus_flag(sku_code, prepared_df)

        new_row = last_row.copy()
        new_row = force_itemcode_str(new_row)
        new_row["Year"] = next_year
        new_row["Month_Number"] = next_month
        new_row["Bonus_Flag"] = int(next_bonus)

        for c in ["Secondary_Sales_Qty", "Primary_Sales_Qty", "Free_Qty", "Observed_Demand", "Effective_Demand", "Clean_Demand"]:
            if c in new_row.columns:
                new_row[c] = 0

        future_df = pd.concat([prepared_df.copy(), new_row], ignore_index=True)
        future_df = force_itemcode_str(future_df)
        future_df = future_df.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)
    except Exception as e:
        return make_fail_result(sku_code, "LONG", "", "GRU", "GRU", f"LONG_GRU_FUTURE_ROW_ERROR: {str(e)}")

    try:
        bonus_pattern_df_f = detect_recurring_bonus_skus(future_df)[[
            "ItemCode",
            "Recurring_Bonus_SKU",
            "Bonus_Cycle_Length",
            "Avg_Bonus_Gap",
            "Bonus_Frequency_All",
            "Avg_Bonus_Uplift"
        ]].copy()
        bonus_pattern_df_f = force_itemcode_str(bonus_pattern_df_f)

        future_df = future_df.drop(columns=[
            "Recurring_Bonus_SKU",
            "Bonus_Cycle_Length",
            "Avg_Bonus_Gap",
            "Bonus_Frequency_All",
            "Avg_Bonus_Uplift"
        ], errors="ignore")
        future_df = force_itemcode_str(future_df)
        future_df = future_df.merge(bonus_pattern_df_f, on="ItemCode", how="left")

        for c in ["Recurring_Bonus_SKU", "Bonus_Cycle_Length", "Avg_Bonus_Gap", "Bonus_Frequency_All"]:
            future_df[c] = future_df[c].fillna(0)
        future_df["Avg_Bonus_Uplift"] = future_df["Avg_Bonus_Uplift"].fillna(1.0)

        future_df["ABC_Class"] = future_df["ItemCode"].map(loaded_artifacts["abc_map"]).fillna(2)

        promo_profile_df = force_itemcode_str(loaded_artifacts["promo_profile_df"])
        future_df = force_itemcode_str(future_df)
        future_df = merge_promo_profile(future_df, promo_profile_df)
        future_df = force_itemcode_str(future_df)

        if "sku_profile_df" in loaded_artifacts and isinstance(loaded_artifacts["sku_profile_df"], pd.DataFrame):
            sku_profile_df = loaded_artifacts["sku_profile_df"].copy()
            sku_profile_df = force_itemcode_str(sku_profile_df)
            keep_cols = [c for c in ["ItemCode", "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"] if c in sku_profile_df.columns]

            if keep_cols:
                future_df = future_df.drop(columns=["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"], errors="ignore")
                future_df = force_itemcode_str(future_df)
                future_df = future_df.merge(sku_profile_df[keep_cols], on="ItemCode", how="left")
                for c in ["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]:
                    if c in future_df.columns:
                        future_df[c] = future_df[c].fillna(0)

        future_df = add_bonus_cycle_features(future_df)
        future_df = force_itemcode_str(future_df)
        future_df = rebuild_time_features(future_df)
        future_df = force_itemcode_str(future_df)
        future_df = apply_clip_caps(future_df, loaded_artifacts["clip_caps"])
        future_df = force_itemcode_str(future_df)
        future_df = recompute_target(future_df)
        future_df = force_itemcode_str(future_df)
        future_df = add_residual_target(future_df)
        future_df = force_itemcode_str(future_df)

        future_df["Residual_Target_Log"] = gru_signed_log_transform(future_df[MODEL_TARGET_COL].fillna(0))
        future_df["ItemCode_Original"] = future_df["ItemCode"].astype(str)
    except Exception as e:
        return make_fail_result(sku_code, "LONG", "", "GRU", "GRU", f"LONG_GRU_FEATURE_BUILD_ERROR: {str(e)}")

    g = future_df[future_df["ItemCode_Original"] == sku_code].copy()
    g = g.sort_values(["Year", "Month_Number"]).reset_index(drop=True)

    target_idx = g[(g["Year"] == next_year) & (g["Month_Number"] == next_month)].index
    if len(target_idx) == 0:
        return make_fail_result(sku_code, "LONG", "", "GRU", "GRU", "LONG_GRU_TARGET_IDX_EMPTY")

    idx = target_idx[0]
    if idx < seq_len - 1:
        return make_fail_result(sku_code, "LONG", "", "GRU", "GRU", f"LONG_GRU_SEQ_TOO_SHORT idx={idx} seq_len={seq_len}")

    seq_slice = g.iloc[idx - seq_len + 1: idx + 1].copy()
    if len(seq_slice) != seq_len:
        return make_fail_result(sku_code, "LONG", "", "GRU", "GRU", f"LONG_GRU_SEQ_SLICE_BAD len={len(seq_slice)}")

    try:
        seq_vals = loaded_scalers.seq_scaler.transform(
            seq_slice[loaded_artifacts["seq_features"]].fillna(0).values
        )
        static_vals = loaded_scalers.static_scaler.transform(
            seq_slice.iloc[-1][loaded_artifacts["static_features"]].fillna(0).values.reshape(1, -1)
        )[0]

        x_seq = torch.tensor(seq_vals[np.newaxis, :, :], dtype=torch.float32).to(GRU_DEVICE)
        x_static = torch.tensor(static_vals[np.newaxis, :], dtype=torch.float32).to(GRU_DEVICE)
        x_item = torch.tensor([item_to_idx[sku_code]], dtype=torch.long).to(GRU_DEVICE)

        with torch.no_grad():
            pred_res_log = model(x_seq, x_static, x_item).cpu().numpy()[0]

        pred_residual = float(gru_signed_log_inverse(pred_res_log))
        baseline = float(seq_slice.iloc[-1][BASELINE_COL])
        raw_forecast = max(baseline + pred_residual, 0.0)

        target_row = seq_slice.iloc[-1].to_dict()
        target_row["Segment_For_Calibration"] = "LONG"

        forecast = apply_promo_aware_adjustment(target_row, raw_forecast)
        forecast = apply_final_forecast_guardrails(target_row, forecast)
        forecast = apply_production_calibration(target_row, forecast)
    except Exception as e:
        return make_fail_result(sku_code, "LONG", "", "GRU", "GRU", f"LONG_GRU_PREDICT_ERROR: {str(e)}")

    return {
        "ItemCode": int(float(sku_code)),
        "Forecast_Year": int(next_year),
        "Forecast_Month": int(next_month),
        "Forecast_Prediction": float(forecast),
        "Segment": "LONG",
        "Subgroup": "",
        "Champion_Model": "GRU",
        "Used_Model": "GRU",
        "Fallback_Used": 0,
        "Expected_Bonus": int(next_bonus),
        "Residual_Baseline": float(baseline),
        "Predicted_Residual": float(pred_residual),
        "Status": "Success"
    }


# ------------------------------------------------------------
# 4) PATCH LONG FALLBACK WITH STAGE-LEVEL DEBUG
# ------------------------------------------------------------
def forecast_long_fallback_sku_debug(sku_code, raw_data, fallback_type="ROLLING3"):
    sku_code = str(sku_code)

    routing = get_long_routing_row(sku_code)
    if routing is None:
        return make_fail_result(sku_code, "LONG", "", "UNKNOWN", f"FALLBACK_{fallback_type}", "LONG_FALLBACK_ROUTING_MISSING")

    best_model = str(routing.get("Best_Model", "XGBOOST")).upper()

    if best_model in ["XGBOOST", "CATBOOST", "LIGHTGBM"]:
        artifacts = get_long_deploy_artifact(best_model)
    else:
        artifacts = get_long_deploy_artifact("XGBOOST")

    if artifacts is None:
        return make_fail_result(sku_code, "LONG", "", best_model, f"FALLBACK_{fallback_type}", "LONG_FALLBACK_BACKBONE_ARTIFACT_MISSING")

    try:
        prepared_df = prepare_long_tree_inference_frame(raw_data, artifacts)
        prepared_df = force_itemcode_str(prepared_df)
    except Exception as e:
        return make_fail_result(sku_code, "LONG", "", best_model, f"FALLBACK_{fallback_type}", f"LONG_FALLBACK_PREP_ERROR: {str(e)}")

    if prepared_df.empty:
        return make_fail_result(sku_code, "LONG", "", best_model, f"FALLBACK_{fallback_type}", "LONG_FALLBACK_PREPARED_DF_EMPTY")

    sku_hist = prepared_df[prepared_df["ItemCode_Original"] == sku_code].copy().sort_values(["Year", "Month_Number"])
    if sku_hist.empty:
        return make_fail_result(sku_code, "LONG", "", best_model, f"FALLBACK_{fallback_type}", "LONG_FALLBACK_SKU_HIST_EMPTY")

    last_row, next_year, next_month = get_next_period_from_history(prepared_df, sku_code)
    if last_row is None:
        return make_fail_result(sku_code, "LONG", "", best_model, f"FALLBACK_{fallback_type}", "LONG_FALLBACK_NEXT_PERIOD_MISSING")

    try:
        next_bonus = infer_expected_bonus_flag(sku_code, prepared_df)

        new_row = last_row.copy()
        new_row = force_itemcode_str(new_row)
        new_row["Year"] = next_year
        new_row["Month_Number"] = next_month
        new_row["Bonus_Flag"] = int(next_bonus)

        for c in ["Secondary_Sales_Qty", "Primary_Sales_Qty", "Free_Qty", "Observed_Demand", "Effective_Demand", "Clean_Demand"]:
            if c in new_row.columns:
                new_row[c] = 0

        work_df = prepared_df.drop(columns=["ItemCode_Encoded"], errors="ignore").copy()
        work_df = force_itemcode_str(work_df)
        work_df = pd.concat([work_df, new_row], ignore_index=True)
        work_df = force_itemcode_str(work_df)
        work_df = work_df.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

        bonus_pattern_df_f = detect_recurring_bonus_skus(work_df)[[
            "ItemCode",
            "Recurring_Bonus_SKU",
            "Bonus_Cycle_Length",
            "Avg_Bonus_Gap",
            "Bonus_Frequency_All",
            "Avg_Bonus_Uplift"
        ]].copy()
        bonus_pattern_df_f = force_itemcode_str(bonus_pattern_df_f)

        work_df = work_df.drop(columns=[
            "Recurring_Bonus_SKU",
            "Bonus_Cycle_Length",
            "Avg_Bonus_Gap",
            "Bonus_Frequency_All",
            "Avg_Bonus_Uplift"
        ], errors="ignore")
        work_df = force_itemcode_str(work_df)
        work_df = work_df.merge(bonus_pattern_df_f, on="ItemCode", how="left")

        for c in ["Recurring_Bonus_SKU", "Bonus_Cycle_Length", "Avg_Bonus_Gap", "Bonus_Frequency_All"]:
            work_df[c] = work_df[c].fillna(0)
        work_df["Avg_Bonus_Uplift"] = work_df["Avg_Bonus_Uplift"].fillna(1.0)

        work_df["ABC_Class"] = work_df["ItemCode"].map(artifacts.get("abc_map", {})).fillna(2)

        promo_profile_df = artifacts.get("promo_profile_df", pd.DataFrame())
        if isinstance(promo_profile_df, pd.DataFrame) and not promo_profile_df.empty:
            promo_profile_df = force_itemcode_str(promo_profile_df)
            work_df = force_itemcode_str(work_df)
            work_df = merge_promo_profile(work_df, promo_profile_df)
            work_df = force_itemcode_str(work_df)

        sku_profile_df = artifacts.get("sku_profile_df", pd.DataFrame())
        if isinstance(sku_profile_df, pd.DataFrame) and not sku_profile_df.empty:
            sku_profile_df = force_itemcode_str(sku_profile_df)
            keep_cols = [c for c in ["ItemCode", "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"] if c in sku_profile_df.columns]
            work_df = work_df.drop(columns=["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"], errors="ignore")
            work_df = force_itemcode_str(work_df)
            work_df = work_df.merge(sku_profile_df[keep_cols], on="ItemCode", how="left")
            for c in ["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]:
                if c in work_df.columns:
                    work_df[c] = work_df[c].fillna(0)

        work_df = add_bonus_cycle_features(work_df)
        work_df = force_itemcode_str(work_df)
        work_df = rebuild_time_features(work_df)
        work_df = force_itemcode_str(work_df)

        if "clip_caps" in artifacts:
            work_df = apply_clip_caps(work_df, artifacts["clip_caps"])
            work_df = force_itemcode_str(work_df)

        work_df = recompute_target(work_df)
        work_df = force_itemcode_str(work_df)
        work_df = add_residual_target(work_df)
        work_df = force_itemcode_str(work_df)
    except Exception as e:
        return make_fail_result(sku_code, "LONG", "", best_model, f"FALLBACK_{fallback_type}", f"LONG_FALLBACK_FEATURE_BUILD_ERROR: {str(e)}")

    next_row = work_df[
        (work_df["ItemCode"].astype(str) == sku_code) &
        (work_df["Year"] == next_year) &
        (work_df["Month_Number"] == next_month)
    ].copy()

    if next_row.empty:
        return make_fail_result(sku_code, "LONG", "", best_model, f"FALLBACK_{fallback_type}", "LONG_FALLBACK_NEXT_ROW_EMPTY")

    try:
        next_row["Segment_For_Calibration"] = "LONG"
        row_dict = next_row.iloc[0].to_dict()

        pred = fallback_forecast_from_row(row_dict, fallback_type=fallback_type)
        pred = apply_production_calibration(row_dict, pred)
    except Exception as e:
        return make_fail_result(sku_code, "LONG", "", best_model, f"FALLBACK_{fallback_type}", f"LONG_FALLBACK_PREDICT_ERROR: {str(e)}")

    return {
        "ItemCode": int(float(sku_code)),
        "Forecast_Year": int(next_year),
        "Forecast_Month": int(next_month),
        "Forecast_Prediction": float(pred),
        "Segment": "LONG",
        "Subgroup": "",
        "Champion_Model": best_model,
        "Used_Model": f"FALLBACK_{fallback_type}",
        "Fallback_Used": 1,
        "Expected_Bonus": int(next_bonus),
        "Residual_Baseline": np.nan,
        "Predicted_Residual": np.nan,
        "Status": "Success"
    }


# ------------------------------------------------------------
# 5) PATCH LONG ROUTER TO AUTO-FALLBACK AFTER FAILURE
# ------------------------------------------------------------
def forecast_long_sku_debug(sku_code, raw_data):
    sku_code = str(sku_code)
    routing = get_long_routing_row(sku_code)

    if routing is None:
        return make_fail_result(sku_code, "LONG", "", "UNKNOWN", "UNKNOWN", "LONG_ROUTING_ROW_MISSING")

    final_model = str(routing.get("Final_Model", routing.get("Best_Model", "XGBOOST"))).upper()
    best_model = str(routing.get("Best_Model", "XGBOOST")).upper()
    fallback_type = str(routing.get("Fallback_Type", "ROLLING3"))

    if final_model == "FALLBACK":
        result = forecast_long_fallback_sku_debug(
            sku_code=sku_code,
            raw_data=raw_data,
            fallback_type=fallback_type
        )
        if result is not None:
            result["Champion_Model"] = best_model
        return result

    if best_model == "GRU":
        result = forecast_long_gru_sku_debug(sku_code, raw_data)

        if result is not None and result.get("Status") == "Success":
            result["Champion_Model"] = "GRU"
            return result

        fallback_result = forecast_long_fallback_sku_debug(
            sku_code=sku_code,
            raw_data=raw_data,
            fallback_type=fallback_type
        )
        if fallback_result is not None and fallback_result.get("Status") == "Success":
            fallback_result["Champion_Model"] = "GRU"
            fallback_result["Used_Model"] = f"GRU_FAILED_{fallback_result['Used_Model']}"
            return fallback_result

        return result

    artifacts = get_long_deploy_artifact(best_model)
    if artifacts is None:
        return make_fail_result(sku_code, "LONG", "", best_model, "UNKNOWN", "LONG_DEPLOY_ARTIFACT_MISSING")

    result = forecast_long_tree_sku_debug(
        sku_code=sku_code,
        raw_data=raw_data,
        artifacts=artifacts,
        used_model_name=best_model
    )

    if result is not None and result.get("Status") == "Success":
        result["Champion_Model"] = best_model
        return result

    fallback_result = forecast_long_fallback_sku_debug(
        sku_code=sku_code,
        raw_data=raw_data,
        fallback_type=fallback_type
    )
    if fallback_result is not None and fallback_result.get("Status") == "Success":
        fallback_result["Champion_Model"] = best_model
        fallback_result["Used_Model"] = f"{best_model}_FAILED_{fallback_result['Used_Model']}"
        return fallback_result

    return result


# ------------------------------------------------------------
# 6) PATCH TOP-LEVEL ROUTER
# ------------------------------------------------------------
def forecast_one_sku_debug(sku_code, raw_data):
    sku_code = str(sku_code)
    segment = choose_segment_by_history(raw_data, sku_code)

    try:
        if segment == "LONG":
            result = forecast_long_sku_debug(sku_code, raw_data)
        elif segment == "MEDIUM":
            result = forecast_medium_sku(sku_code, raw_data)
        else:
            result = forecast_short_sku(sku_code, raw_data)

        if result is None:
            return make_fail_result(
                sku_code=sku_code,
                segment=segment,
                status=f"{segment}_RETURNED_NONE"
            )

        return result

    except Exception as e:
        return make_fail_result(
            sku_code=sku_code,
            segment=segment,
            status=f"{segment}_ERROR: {str(e)}"
        )


# ------------------------------------------------------------
# 7) DEBUG BULK FORECAST
# ------------------------------------------------------------
def bulk_segmented_forecast_debug(raw_data, sku_list):
    results = []

    for sku in sku_list:
        result = forecast_one_sku_debug(sku, raw_data)
        results.append(result)

    results_df = pd.DataFrame(results)

    if not results_df.empty:
        results_df = results_df.sort_values(
            ["Segment", "ItemCode"]
        ).reset_index(drop=True)

    success_df = results_df[results_df["Status"] == "Success"].copy()
    failed_df = results_df[results_df["Status"] != "Success"].copy()

    return results_df, success_df, failed_df


# ------------------------------------------------------------
# 8) RUN DEBUG FORECAST
# ------------------------------------------------------------
debug_all_results_df, debug_success_df, debug_failed_df = bulk_segmented_forecast_debug(
    raw_data=Cleaned_Base_Data.copy(),
    sku_list=sorted(set(int(x) for x in PHARMA_SKUS))
)

print("\n--- DEBUG STATUS SUMMARY ---")
print(debug_all_results_df["Status"].value_counts(dropna=False))

print("\n--- DEBUG FAILURES BY SEGMENT ---")
print(
    debug_failed_df.groupby(["Segment", "Status"], dropna=False)
    .size()
    .reset_index(name="Count")
    .sort_values(["Segment", "Count"], ascending=[True, False])
)

print("\n--- SAMPLE DEBUG FAILURES ---")
print(debug_failed_df.head(50))

print("\n--- SUCCESS COUNT ---")
print("Success rows:", len(debug_success_df))
print("Unique success SKUs:", debug_success_df["ItemCode"].nunique())

print("\n--- FAILURE COUNT ---")
print("Failed rows:", len(debug_failed_df))
print("Unique failed SKUs:", debug_failed_df["ItemCode"].nunique())


# ------------------------------------------------------------
# 9) OPTIONAL: SAVE DEBUG REPORT
# ------------------------------------------------------------
with pd.ExcelWriter("forecast_failure_trace_debug.xlsx", engine="openpyxl") as writer:
    debug_all_results_df.to_excel(writer, sheet_name="All_Debug_Results", index=False)
    debug_success_df.to_excel(writer, sheet_name="Success", index=False)
    debug_failed_df.to_excel(writer, sheet_name="Failures", index=False)

print("\nSaved: forecast_failure_trace_debug.xlsx")

In [ ]:
def debug_short_sku_path(full_data, sku_code):
    sku_code = str(sku_code)

    print(f"\n===== DEBUG SHORT SKU PATH: {sku_code} =====")

    df0 = force_itemcode_str(full_data.copy())
    df0["ItemCode"] = df0["ItemCode"].astype(str)

    raw_rows = df0[df0["ItemCode"] == sku_code].copy()
    print("Rows in full_data:", len(raw_rows))

    if not raw_rows.empty:
        print(raw_rows[["ItemCode", "Year", "Month_Number", "Clean_Demand"]].tail(10))

    df1 = add_history_length_from_subset(df0, df0)
    seg_rows = df1[df1["ItemCode"] == sku_code].copy()
    print("Rows after segmentation:", len(seg_rows))

    if not seg_rows.empty:
        print("History_Segment values:", seg_rows["History_Segment"].dropna().unique())
        print(seg_rows[["ItemCode", "Year", "Month_Number", "History_Length", "History_Segment"]].tail(10))

    short_rows = df1[(df1["ItemCode"] == sku_code) & (df1["History_Segment"] == "SHORT")].copy()
    print("Rows in SHORT segment:", len(short_rows))

    if not short_rows.empty:
        print(short_rows[["ItemCode", "Year", "Month_Number", "Clean_Demand"]].tail(10))



debug_short_sku_path(Data, "612103")

# Forecasts for 6 Months

In [ ]:
# ============================================================
# ALL INFERENCE SKUs - 6 MONTH HISTORICAL ROLLING FORECAST
# Forecast months:
# 2025-09, 2025-10, 2025-11, 2025-12, 2026-01, 2026-02
# Adds raw actual sale from Secondary_Sales_Qty
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1) Use all SKUs used in normal inference
# ------------------------------------------------------------
all_inference_skus = sorted(set(int(x) for x in PHARMA_SKUS))

print("All inference SKU count:", len(all_inference_skus))


# ------------------------------------------------------------
# 2) Target forecast months
# ------------------------------------------------------------
target_periods = [
    (2025, 9),
    (2025, 10),
    (2025, 11),
    (2025, 12),
    (2026, 1),
    (2026, 2),
]


# ------------------------------------------------------------
# 3) Cutoff dataset builder
# ------------------------------------------------------------
def build_cutoff_dataset(raw_df, cutoff_year, cutoff_month):
    df = raw_df.copy()
    df["ItemCode"] = df["ItemCode"].astype(str)

    df = df[
        (df["Year"] < cutoff_year) |
        (
            (df["Year"] == cutoff_year) &
            (df["Month_Number"] <= cutoff_month)
        )
    ].copy()

    return df.sort_values(
        ["ItemCode", "Year", "Month_Number"]
    ).reset_index(drop=True)


# ------------------------------------------------------------
# 4) Rolling forecast runner for all SKUs
# ------------------------------------------------------------
def run_all_sku_rolling_forecast(raw_data, sku_list, target_periods):
    all_predictions = []
    all_failed = []

    for forecast_year, forecast_month in target_periods:

        print("\n" + "=" * 70)
        print(f"Running forecast for {forecast_year}-{forecast_month:02d}")
        print("=" * 70)

        cutoff_year = forecast_year
        cutoff_month = forecast_month - 1

        if cutoff_month == 0:
            cutoff_month = 12
            cutoff_year -= 1

        hist_df = build_cutoff_dataset(
            raw_df=raw_data,
            cutoff_year=cutoff_year,
            cutoff_month=cutoff_month
        )

        print(f"Cutoff data until: {cutoff_year}-{cutoff_month:02d}")
        print("Historical rows:", len(hist_df))
        print("Historical SKUs:", hist_df["ItemCode"].nunique())

        preds_df, failed_df = bulk_segmented_forecast(
            raw_data=hist_df.copy(),
            sku_list=sku_list
        )

        if preds_df is not None and not preds_df.empty:
            preds_df = preds_df.copy()

            preds_df = preds_df[
                (preds_df["Forecast_Year"] == forecast_year) &
                (preds_df["Forecast_Month"] == forecast_month)
            ].copy()

            preds_df["Forecast_Period"] = (
                preds_df["Forecast_Year"].astype(int).astype(str)
                + "-"
                + preds_df["Forecast_Month"].astype(int).astype(str).str.zfill(2)
            )

            keep_cols = [
                "ItemCode",
                "Forecast_Period",
                "Forecast_Year",
                "Forecast_Month",
                "Forecast_Prediction",
                "Segment",
                "Behavior_Type",
                "Subgroup",
                "Champion_Model",
                "Used_Model",
                "Fallback_Used",
                "Expected_Bonus",
                "Status"
            ]

            preds_df = preds_df[[c for c in keep_cols if c in preds_df.columns]]

            all_predictions.append(preds_df)

            print("Successful predictions:", len(preds_df))
        else:
            print("Successful predictions: 0")

        if failed_df is not None and not failed_df.empty:
            failed_df = failed_df.copy()
            failed_df["Target_Forecast_Year"] = forecast_year
            failed_df["Target_Forecast_Month"] = forecast_month
            failed_df["Target_Forecast_Period"] = f"{forecast_year}-{forecast_month:02d}"
            all_failed.append(failed_df)

            print("Failed predictions:", len(failed_df))
        else:
            print("Failed predictions: 0")

    final_predictions = (
        pd.concat(all_predictions, ignore_index=True)
        if len(all_predictions) > 0
        else pd.DataFrame()
    )

    final_failed = (
        pd.concat(all_failed, ignore_index=True)
        if len(all_failed) > 0
        else pd.DataFrame()
    )

    if not final_predictions.empty:
        final_predictions["ItemCode"] = final_predictions["ItemCode"].astype(str)
        final_predictions = final_predictions.sort_values(
            ["ItemCode", "Forecast_Year", "Forecast_Month"]
        ).reset_index(drop=True)

    return final_predictions, final_failed


# ------------------------------------------------------------
# 5) Run forecast
# ------------------------------------------------------------
all_sku_historical_forecasts, all_sku_historical_failed = run_all_sku_rolling_forecast(
    raw_data=Cleaned_Base_Data.copy(),
    sku_list=all_inference_skus,
    target_periods=target_periods
)


# ------------------------------------------------------------
# 6) Add raw actual sale
# ------------------------------------------------------------
actuals_df = Cleaned_Base_Data.copy()
actuals_df["ItemCode"] = actuals_df["ItemCode"].astype(str)

actuals_df["Forecast_Period"] = (
    actuals_df["Year"].astype(int).astype(str)
    + "-"
    + actuals_df["Month_Number"].astype(int).astype(str).str.zfill(2)
)

actual_col = "Secondary_Sales_Qty"

if actual_col not in actuals_df.columns:
    raise ValueError(f"{actual_col} not found in Cleaned_Base_Data")

actuals_lookup = actuals_df[
    ["ItemCode", "Forecast_Period", actual_col]
].copy()

actuals_lookup = actuals_lookup.rename(
    columns={actual_col: "Actual_Sale"}
)

actuals_lookup = actuals_lookup.drop_duplicates(
    subset=["ItemCode", "Forecast_Period"],
    keep="last"
)

all_sku_historical_forecasts = all_sku_historical_forecasts.merge(
    actuals_lookup,
    on=["ItemCode", "Forecast_Period"],
    how="left"
)


# ------------------------------------------------------------
# 7) Accuracy calculation
# Accuracy = max(0, 1 - abs(actual - forecast) / actual) * 100
# ------------------------------------------------------------
all_sku_historical_forecasts["Absolute_Error"] = np.where(
    all_sku_historical_forecasts["Actual_Sale"].notna(),
    abs(
        all_sku_historical_forecasts["Actual_Sale"] -
        all_sku_historical_forecasts["Forecast_Prediction"]
    ),
    np.nan
)

all_sku_historical_forecasts["Accuracy_%"] = np.where(
    all_sku_historical_forecasts["Actual_Sale"].notna() &
    (all_sku_historical_forecasts["Actual_Sale"] != 0),
    (
        1 -
        all_sku_historical_forecasts["Absolute_Error"] /
        all_sku_historical_forecasts["Actual_Sale"]
    ).clip(lower=0) * 100,
    np.nan
)


# ------------------------------------------------------------
# 8) Simple output
# ------------------------------------------------------------
simple_all_sku_forecasts = all_sku_historical_forecasts[
    [
        "ItemCode",
        "Forecast_Period",
        "Forecast_Prediction",
        "Actual_Sale",
        "Absolute_Error",
        "Accuracy_%"
    ]
].copy()

simple_all_sku_forecasts = simple_all_sku_forecasts.sort_values(
    ["ItemCode", "Forecast_Period"]
).reset_index(drop=True)

print("\nPreview:")
print(simple_all_sku_forecasts.head(20))

print("\nTotal forecast rows:", len(simple_all_sku_forecasts))
print("Unique SKUs forecasted:", simple_all_sku_forecasts["ItemCode"].nunique())


# ------------------------------------------------------------
# 9) Summary by month
# ------------------------------------------------------------
monthly_summary = (
    simple_all_sku_forecasts
    .groupby("Forecast_Period")
    .agg(
        Forecast_Rows=("ItemCode", "count"),
        Unique_SKUs=("ItemCode", "nunique"),
        Actuals_Available=("Actual_Sale", lambda x: x.notna().sum()),
        Missing_Actuals=("Actual_Sale", lambda x: x.isna().sum()),
        Mean_Accuracy=("Accuracy_%", "mean"),
        Median_Accuracy=("Accuracy_%", "median"),
        Total_Actual=("Actual_Sale", "sum"),
        Total_Forecast=("Forecast_Prediction", "sum"),
        Total_Abs_Error=("Absolute_Error", "sum")
    )
    .reset_index()
)

monthly_summary["WMAPE_%"] = np.where(
    monthly_summary["Total_Actual"] > 0,
    monthly_summary["Total_Abs_Error"] / monthly_summary["Total_Actual"] * 100,
    np.nan
)

monthly_summary["Weighted_Accuracy_%"] = np.where(
    monthly_summary["WMAPE_%"].notna(),
    (100 - monthly_summary["WMAPE_%"]).clip(lower=0),
    np.nan
)

print("\nMonthly Summary:")
print(monthly_summary)


# ------------------------------------------------------------
# 10) Save Excel
# ------------------------------------------------------------
output_file = "all_sku_historical_forecasts_sep2025_to_feb2026.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    simple_all_sku_forecasts.to_excel(
        writer,
        sheet_name="Simple_Forecasts",
        index=False
    )

    all_sku_historical_forecasts.to_excel(
        writer,
        sheet_name="Detailed_Forecasts",
        index=False
    )

    monthly_summary.to_excel(
        writer,
        sheet_name="Monthly_Summary",
        index=False
    )

    if all_sku_historical_failed is not None and not all_sku_historical_failed.empty:
        all_sku_historical_failed.to_excel(
            writer,
            sheet_name="Failed_SKUs",
            index=False
        )

print(f"\nSaved file: {output_file}")

print("\nMissing actual sale count by period:")
print(
    simple_all_sku_forecasts.groupby("Forecast_Period")["Actual_Sale"]
    .apply(lambda x: x.isna().sum())
)